# Avaliação formal de desempenho do Sentinela

Este notebook avalia quantitativamente o desempenho do algoritmo do Sentinela utilizando um ground truth independente das regras estatísticas do próprio sistema.

## Objetivo

Comparar os sinais produzidos pelo Sentinela com evidências epidemiológicas externas e independentes.

A unidade de avaliação será:

`região × ano × semana epidemiológica`

## Ground truth

Cada semana poderá receber uma das seguintes classificações:

- `1` — evento epidemiológico confirmado por fonte externa;
- `0` — ausência de evento sustentada por evidência epidemiológica externa;
- `NA` — evidência insuficiente para classificação.

Semanas `NA` não serão utilizadas no cálculo das métricas de classificação.

## Princípio fundamental

O ground truth não utilizará:

- Q75 do Sentinela;
- mediana histórica do Sentinela;
- nowcast do Sentinela;
- estado `SIGNAL` ou `ALERT` do Sentinela.

Essas informações serão incorporadas somente após a construção independente dos rótulos externos.

In [314]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

In [315]:
signal_sp_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_sp_signal_evaluation.csv"
)

assert signal_sp_2026_path.exists(), (
    f"Arquivo não encontrado: {signal_sp_2026_path}"
)

signal_evaluation_sp_2026 = pd.read_csv(
    signal_sp_2026_path,
    sep=";",
)

print(
    "Arquivo carregado:",
    signal_sp_2026_path.name,
)

print(
    "Linhas:",
    len(signal_evaluation_sp_2026),
)

display(
    signal_evaluation_sp_2026.head()
)

Arquivo carregado: srag_2026_sp_signal_evaluation.csv
Linhas: 52


,epi_week,epi_year,week_start,week_end,record_count,region,calendar_status,data_status,cumulative_record_count,historical_median,q25,q75,historical_years,has_minimum_history,median_threshold,ratio_to_median,above_q75,above_factor_threshold,signal_status,signal_reason
0,1,2026,2026-01-04,2026-01-10,647.0,SP,encerrada,observado,647.0,927.0,423.5,5111.0,7,True,1390.5,0.697950,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
1,2,2026,2026-01-11,2026-01-17,541.0,SP,encerrada,observado,1188.0,817.0,338.0,5707.5,7,True,1225.5,0.662179,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
2,3,2026,2026-01-18,2026-01-24,565.0,SP,encerrada,observado,1753.0,812.0,351.0,5273.0,7,True,1218.0,0.695813,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
3,4,2026,2026-01-25,2026-01-31,491.0,SP,encerrada,observado,2244.0,791.0,394.0,4530.0,7,True,1186.5,0.620733,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
4,5,2026,2026-02-01,2026-02-07,703.0,SP,encerrada,observado,2947.0,913.0,401.0,3836.0,7,True,1369.5,0.769989,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.


In [316]:
signal_mg_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2025_mg_signal_evaluation.csv"
)

assert signal_mg_2025_path.exists(), (
    f"Arquivo não encontrado: {signal_mg_2025_path}"
)

signal_evaluation_2025 = pd.read_csv(
    signal_mg_2025_path,
    sep=";",
)

print(
    "Arquivo carregado:",
    signal_mg_2025_path.name,
)

print(
    "Linhas:",
    len(signal_evaluation_2025),
)

display(
    signal_evaluation_2025.head()
)

Arquivo carregado: srag_2025_mg_signal_evaluation.csv
Linhas: 53


,epi_week,epi_year,week_start,week_end,record_count,cumulative_record_count,calendar_status,data_status,historical_median,q25,q75,historical_years,has_minimum_history,region,median_threshold,ratio_to_median,above_q75,above_factor_threshold,signal_status,signal_reason
0,1,2025,2024-12-29,2025-01-04,397,397,encerrada,observado,586.0,111.00,2993.75,6.0,True,MG,879.00,0.677474,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
1,2,2025,2025-01-05,2025-01-11,422,819,encerrada,observado,471.5,105.00,3472.00,6.0,True,MG,707.25,0.895016,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
2,3,2025,2025-01-12,2025-01-18,413,1232,encerrada,observado,427.0,129.75,3170.75,6.0,True,MG,640.50,0.967213,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
3,4,2025,2025-01-19,2025-01-25,412,1644,encerrada,observado,363.0,105.25,2947.25,6.0,True,MG,544.50,1.134986,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
4,5,2025,2025-01-26,2025-02-01,359,2003,encerrada,observado,433.5,130.50,2613.75,6.0,True,MG,650.25,0.828143,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.


In [317]:
def build_epi_calendar(
    region: str,
    year: int,
    first_week_start: str,
    weeks: int = 52,
) -> pd.DataFrame:
    calendar = pd.DataFrame(
        {
            "epi_week": range(1, weeks + 1),
        }
    )

    calendar["region"] = region
    calendar["year"] = year

    first_start = pd.Timestamp(
        first_week_start
    )

    calendar["week_start"] = (
        first_start
        + pd.to_timedelta(
            (calendar["epi_week"] - 1) * 7,
            unit="D",
        )
    )

    calendar["week_end"] = (
        calendar["week_start"]
        + pd.Timedelta(days=6)
    )

    return calendar[
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]

In [318]:
ground_truth_base = pd.concat(
    [
        build_epi_calendar(
            region="MG",
            year=2025,
            first_week_start="2024-12-29",
            weeks=52,
        ),
        build_epi_calendar(
            region="SP",
            year=2026,
            first_week_start="2026-01-04",
            weeks=52,
        ),
    ],
    ignore_index=True,
)

display(ground_truth_base.head())

,region,year,epi_week,week_start,week_end
0,MG,2025,1,2024-12-29,2025-01-04
1,MG,2025,2,2025-01-05,2025-01-11
2,MG,2025,3,2025-01-12,2025-01-18
3,MG,2025,4,2025-01-19,2025-01-25
4,MG,2025,5,2025-01-26,2025-02-01


In [319]:
ground_truth = (
    ground_truth_base.copy()
)

ground_truth["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth.index,
    dtype="Int64",
)

ground_truth["evidence_level"] = pd.NA

ground_truth["event_type"] = pd.NA

ground_truth["evidence_summary"] = pd.NA

ground_truth["source_count"] = 0

ground_truth["source_ids"] = pd.NA

In [320]:
EVIDENCE_LEVELS = {
    "STRONG",
    "MODERATE",
    "WEAK",
    "UNLABELED",
}

In [321]:
external_events = pd.DataFrame(
    [
        {
            "event_id": "MG2025_001",
            "region": "MG",
            "event_date": "2025-05-02",
            "event_type": "STATE_EMERGENCY",
            "source_type": "government",
            "source_name": "Governo de Minas Gerais",
            "description": (
                "Situação de emergência em saúde pública "
                "diante do aumento de doenças respiratórias."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_002",
            "region": "MG",
            "event_date": "2025-05-02",
            "event_type": "LEGAL_DECREE",
            "source_type": "legislation",
            "source_name": (
                "Assembleia Legislativa de Minas Gerais"
            ),
            "description": (
                "Decreto com Numeração Especial nº 411."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_003",
            "region": "MG",
            "event_date": "2025-05-06",
            "event_type": "MUNICIPAL_EMERGENCY",
            "source_type": "government",
            "source_name": "Prefeitura de Uberlândia",
            "description": (
                "Situação de emergência devido ao aumento "
                "de casos de SRAG."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_004",
            "region": "MG",
            "event_date": "2025-05-08",
            "event_type": "EMERGENCY_OPERATIONS",
            "source_type": "government",
            "source_name": "Governo de Minas Gerais",
            "description": (
                "Implementação do Centro de Operações "
                "de Emergência em Saúde por SRAG."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_005",
            "region": "MG",
            "event_date": "2025-05-21",
            "event_type": "RESPONSE_PLAN",
            "source_type": "government",
            "source_name": "Prefeitura de Uberlândia",
            "description": (
                "Plano de ação para enfrentamento "
                "das síndromes respiratórias."
            ),
            "evidence_strength": "MODERATE",
        },
    ]
)

external_events["event_date"] = pd.to_datetime(
    external_events["event_date"]
)

display(external_events)

,event_id,region,event_date,event_type,source_type,source_name,description,evidence_strength
0,MG2025_001,MG,2025-05-02,STATE_EMERGENCY,government,Governo de Minas Gerais,Situação de emergência em saúde pública diante...,STRONG
1,MG2025_002,MG,2025-05-02,LEGAL_DECREE,legislation,Assembleia Legislativa de Minas Gerais,Decreto com Numeração Especial nº 411.,STRONG
2,MG2025_003,MG,2025-05-06,MUNICIPAL_EMERGENCY,government,Prefeitura de Uberlândia,Situação de emergência devido ao aumento de ca...,STRONG
3,MG2025_004,MG,2025-05-08,EMERGENCY_OPERATIONS,government,Governo de Minas Gerais,Implementação do Centro de Operações de Emergê...,STRONG
4,MG2025_005,MG,2025-05-21,RESPONSE_PLAN,government,Prefeitura de Uberlândia,Plano de ação para enfrentamento das síndromes...,MODERATE


In [322]:
def assign_epi_week(
    event_row: pd.Series,
    calendar: pd.DataFrame,
):
    candidates = calendar.loc[
        calendar["region"].eq(
            event_row["region"]
        )
        & (
            calendar["week_start"]
            <= event_row["event_date"]
        )
        & (
            calendar["week_end"]
            >= event_row["event_date"]
        )
    ]

    if candidates.empty:
        return pd.NA

    return int(
        candidates.iloc[0]["epi_week"]
    )

In [323]:
external_events["epi_week"] = (
    external_events.apply(
        lambda row: assign_epi_week(
            row,
            ground_truth_base,
        ),
        axis=1,
    )
)

display(
    external_events[
        [
            "event_id",
            "event_date",
            "region",
            "epi_week",
            "event_type",
        ]
    ]
)

,event_id,event_date,region,epi_week,event_type
0,MG2025_001,2025-05-02,MG,18,STATE_EMERGENCY
1,MG2025_002,2025-05-02,MG,18,LEGAL_DECREE
2,MG2025_003,2025-05-06,MG,19,MUNICIPAL_EMERGENCY
3,MG2025_004,2025-05-08,MG,19,EMERGENCY_OPERATIONS
4,MG2025_005,2025-05-21,MG,21,RESPONSE_PLAN


In [324]:
event_group_map = {
    "MG2025_001": "MG2025_EMERGENCY_STATE",
    "MG2025_002": "MG2025_EMERGENCY_STATE",
    "MG2025_003": "MG2025_UBERLANDIA_EMERGENCY",
    "MG2025_004": "MG2025_COE_ACTIVATION",
    "MG2025_005": "MG2025_UBERLANDIA_RESPONSE_PLAN",
}

external_events["event_group_id"] = (
    external_events["event_id"]
    .map(event_group_map)
)

In [325]:
assert (
    external_events["event_group_id"]
    .notna()
    .all()
)

display(
    external_events[
        [
            "event_id",
            "event_group_id",
            "epi_week",
            "event_type",
            "evidence_strength",
        ]
    ]
)

,event_id,event_group_id,epi_week,event_type,evidence_strength
0,MG2025_001,MG2025_EMERGENCY_STATE,18,STATE_EMERGENCY,STRONG
1,MG2025_002,MG2025_EMERGENCY_STATE,18,LEGAL_DECREE,STRONG
2,MG2025_003,MG2025_UBERLANDIA_EMERGENCY,19,MUNICIPAL_EMERGENCY,STRONG
3,MG2025_004,MG2025_COE_ACTIVATION,19,EMERGENCY_OPERATIONS,STRONG
4,MG2025_005,MG2025_UBERLANDIA_RESPONSE_PLAN,21,RESPONSE_PLAN,MODERATE


In [326]:
evidence_rank = {
    "WEAK": 1,
    "MODERATE": 2,
    "STRONG": 3,
}

external_events["evidence_rank"] = (
    external_events["evidence_strength"]
    .map(evidence_rank)
)

In [327]:
weekly_external_evidence = (
    external_events
    .groupby(
        [
            "region",
            "epi_week",
        ],
        as_index=False,
    )
    .agg(
        independent_events=(
            "event_group_id",
            "nunique",
        ),
        source_count=(
            "event_id",
            "nunique",
        ),
        max_evidence_rank=(
            "evidence_rank",
            "max",
        ),
        source_ids=(
            "event_id",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
        event_group_ids=(
            "event_group_id",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
        evidence_summary=(
            "description",
            lambda values: " | ".join(
                dict.fromkeys(values)
            ),
        ),
    )
)

In [328]:
rank_to_evidence = {
    1: "WEAK",
    2: "MODERATE",
    3: "STRONG",
}

weekly_external_evidence[
    "evidence_level"
] = (
    weekly_external_evidence[
        "max_evidence_rank"
    ]
    .map(rank_to_evidence)
)

In [329]:
display(
    weekly_external_evidence[
        [
            "region",
            "epi_week",
            "independent_events",
            "source_count",
            "evidence_level",
            "event_group_ids",
        ]
    ]
)

,region,epi_week,independent_events,source_count,evidence_level,event_group_ids
0,MG,18,1,2,STRONG,MG2025_EMERGENCY_STATE
1,MG,19,2,2,STRONG,"MG2025_COE_ACTIVATION,MG2025_UBERLANDIA_EMERGENCY"
2,MG,21,1,1,MODERATE,MG2025_UBERLANDIA_RESPONSE_PLAN


In [330]:
ground_truth = (
    ground_truth_base
    .merge(
        weekly_external_evidence,
        on=[
           "region",
           "epi_week", 
        ],
        how="left",
    )
)

In [331]:
ground_truth["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth.index,
    dtype="Int64",
)

In [332]:
positive_mask = (
    ground_truth["evidence_level"]
    .isin(
        [
           "STRONG",
           "MODERATE", 
        ]
    )
)

ground_truth.loc[
    positive_mask,
    "ground_truth"
] = 1

In [333]:
display(
    ground_truth.loc[
        ground_truth[
            "ground_truth"
        ].eq(1)
    ][
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
            "ground_truth",
            "evidence_level",
            "independent_events",
            "source_count",
        ]
    ]
)

,region,year,epi_week,week_start,week_end,ground_truth,evidence_level,independent_events,source_count
17,MG,2025,18,2025-04-27,2025-05-03,1,STRONG,1.0,2.0
18,MG,2025,19,2025-05-04,2025-05-10,1,STRONG,2.0,2.0
20,MG,2025,21,2025-05-18,2025-05-24,1,MODERATE,1.0,1.0


In [334]:
epidemiological_episodes = pd.DataFrame(
    [
        {
            "episode_id": "MG2025_RESP_001",
            "region": "MG",
            "year": 2025,
            "start_epi_week": 18,
            "end_epi_week": 25,
            "ground_truth": 1,
            "evidence_level": "STRONG",
            "description": (
                "Episódio estadual de aumento de SRAG com "
                "decreto de emergência, ativação do COE, "
                "expansão municipal e ampliação assistencial."
            ),
        },
    ]
)

In [335]:
for episode in epidemiological_episodes.itertuples(
    index=False
):
    mask = (
        ground_truth["region"].eq(
            episode.region
        )
        & ground_truth["year"].eq(
            episode.year
        )
        & ground_truth["epi_week"].between(
            episode.start_epi_week,
            episode.end_epi_week,
        )
    )

    ground_truth.loc[
        mask,
        "ground_truth",
    ] = episode.ground_truth

    ground_truth.loc[
        mask,
        "evidence_level",
    ] = episode.evidence_level

In [336]:
display(
    ground_truth.loc[
        ground_truth["ground_truth"].eq(1),
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
            "ground_truth",
            "evidence_level",
        ],
    ]
)

,region,year,epi_week,week_start,week_end,ground_truth,evidence_level
17,MG,2025,18,2025-04-27,2025-05-03,1,STRONG
18,MG,2025,19,2025-05-04,2025-05-10,1,STRONG
19,MG,2025,20,2025-05-11,2025-05-17,1,STRONG
20,MG,2025,21,2025-05-18,2025-05-24,1,STRONG
21,MG,2025,22,2025-05-25,2025-05-31,1,STRONG
22,MG,2025,23,2025-06-01,2025-06-07,1,STRONG
23,MG,2025,24,2025-06-08,2025-06-14,1,STRONG
24,MG,2025,25,2025-06-15,2025-06-21,1,STRONG


## 11. Ground truth independente — SP 2026

A revisão de fontes externas mostrou que São Paulo não permaneceu em situação epidemiológica estável durante todo o período analisado.

Até a SE 16, o Ministério da Saúde/InfoGripe indicava São Paulo entre as UFs que não apresentavam incidência de SRAG em nível de alerta, risco ou alto risco nas duas semanas anteriores.

Entretanto, até a SE 20, São Paulo já apresentava:

- incidência de SRAG em nível de alerta, risco ou alto risco;
- sinal de crescimento na tendência de longo prazo;
- aumento das hospitalizações por Influenza A;
- aumento de SRAG associado ao VSR.

Por esse motivo, SP/2026 não será tratado como um caso totalmente negativo.

A rotulagem inicial será:

- SE 15–16: evidência forte de ausência de evento;
- SE 17–19: período de transição, mantido como `NA`;
- SE 20 em diante: requer reconstrução da duração do episódio com boletins subsequentes.

In [337]:
sp_negative_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(15, 16)
)

ground_truth.loc[
    sp_negative_mask,
    "ground_truth",
] = 0

ground_truth.loc[
    sp_negative_mask,
    "evidence_level",
] = "STRONG"

In [338]:
sp_transition_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(17, 19)
)

ground_truth.loc[
    sp_transition_mask,
    "ground_truth",
] = pd.NA

ground_truth.loc[
    sp_transition_mask,
    "evidence_level",
] = "UNLABELED"

In [339]:
sp_positive_episode = pd.DataFrame(
    [
        {
            "episode_id": "SP2026_RESP_001",
            "region": "SP",
            "year": 2026,
            "start_epi_week": 20,
            "end_epi_week": pd.NA,
            "ground_truth": 1,
            "evidence_level": "STRONG",
            "description": (
                "Crescimento de SRAG documentado pelo InfoGripe, "
                "com aumento de hospitalizações por Influenza A e VSR."
            ),
        }
    ]
)

### Janela epidemiológica externa — SP 2026

A revisão dos informes do Ministério da Saúde/InfoGripe indica:

- SE 20: São Paulo apresentava incidência de SRAG em nível de alerta, risco ou alto risco e sinal de crescimento na tendência de longo prazo;
- SE 28: a incidência permanecia elevada, porém sem crescimento na tendência de longo prazo;
- SE 32: São Paulo já não figurava entre as UFs com incidência geral de SRAG em alerta, risco ou alto risco;
- SE 36: permanecia fora desse grupo, embora houvesse leve aumento de SRAG por Covid-19 em nível baixo de incidência.

Para reduzir viés de classificação, as semanas de transição não são rotuladas automaticamente.

Rotulagem adotada:

- SE 15–16 → negativo;
- SE 17–19 → não rotulado;
- SE 20–28 → positivo;
- SE 29–31 → não rotulado;
- SE 32–37 → negativo;
- SE 38 → não rotulado por imaturidade operacional.

In [340]:
mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(15, 16)
)

ground_truth.loc[mask, "ground_truth"] = 0
ground_truth.loc[mask, "evidence_level"] = "STRONG"


mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(20, 28)
)

ground_truth.loc[mask, "ground_truth"] = 1
ground_truth.loc[mask, "evidence_level"] = "STRONG"

mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(32, 37)
)

ground_truth.loc[mask, "ground_truth"] = 0
ground_truth.loc[mask, "evidence_level"] = "STRONG"

transition_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].isin(
        [17, 18, 19, 29, 30, 31, 38]
    )
)

ground_truth.loc[
    transition_mask,
    "ground_truth",
] = pd.NA

ground_truth.loc[
    transition_mask,
    "evidence_level",
] = "UNLABELED"

In [341]:
display(
    ground_truth.loc[
        ground_truth["region"].eq("SP")
        & ground_truth["year"].eq(2026)
        & ground_truth["epi_week"].between(14, 38),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "evidence_level",
        ],
    ]
)

,region,year,epi_week,ground_truth,evidence_level
65,SP,2026,14,<NA>,NaN
66,SP,2026,15,0,STRONG
67,SP,2026,16,0,STRONG
68,SP,2026,17,<NA>,UNLABELED
69,SP,2026,18,<NA>,UNLABELED
70,SP,2026,19,<NA>,UNLABELED
71,SP,2026,20,1,STRONG
72,SP,2026,21,1,STRONG
73,SP,2026,22,1,STRONG
74,SP,2026,23,1,STRONG


## 12. Matriz de confusão — primeira avaliação formal

O ground truth externo será comparado com a saída do Sentinela.

Semanas não rotuladas (`NA`) serão excluídas da avaliação.

A classificação utilizará:

- `ground_truth = 1` → evento epidemiológico real;
- `ground_truth = 0` → ausência de evento;
- `sentinela_positive = True` → semana sinalizada pelo Sentinela;
- `sentinela_positive = False` → semana sem sinal.

A partir dessa comparação serão calculados:

- verdadeiro positivo (`TP`);
- falso positivo (`FP`);
- falso negativo (`FN`);
- verdadeiro negativo (`TN`);
- sensibilidade;
- especificidade;
- valor preditivo positivo;
- valor preditivo negativo;
- F1-score.

In [342]:
sentinela_sp_2026 = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "signal_status",
        ]
    ]
    .copy()
)

sentinela_sp_2026["sentinela_positive"] = (
    sentinela_sp_2026[
        "signal_status"
    ].eq("SIGNAL")
)

sentinela_sp_2026["region"] = "SP"
sentinela_sp_2026["year"] = 2026

display(sentinela_sp_2026.head())

,epi_week,signal_status,sentinela_positive,region,year
0,1,NO_SIGNAL,False,SP,2026
1,2,NO_SIGNAL,False,SP,2026
2,3,NO_SIGNAL,False,SP,2026
3,4,NO_SIGNAL,False,SP,2026
4,5,NO_SIGNAL,False,SP,2026


In [343]:
sentinela_mg_2025 = (
    signal_evaluation_2025[
        [
            "epi_week",
            "signal_status",
        ]
    ]
    .copy()
)

sentinela_mg_2025["sentinela_positive"] = (
    sentinela_mg_2025[
        "signal_status"
    ].eq("SIGNAL")
)

sentinela_mg_2025["region"] = "MG"
sentinela_mg_2025["year"] = 2025

In [344]:
sentinela_predictions = pd.concat(
    [
        sentinela_mg_2025,
        sentinela_sp_2026,
    ],
    ignore_index=True,
)

In [345]:
evaluation = (
    ground_truth
    .merge(
        sentinela_predictions[
            [
                "region",
                "year",
                "epi_week",
                "sentinela_positive",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="left",
    )
)

In [346]:
evaluation_labeled = (
    evaluation.loc[
        evaluation[
            "ground_truth"
        ].notna()
    ]
    .copy()
)

evaluation_labeled[
    "ground_truth"
] = (
    evaluation_labeled[
        "ground_truth"
    ].astype(int)
)

evaluation_labeled[
    "sentinela_positive"
] = (
    evaluation_labeled[
        "sentinela_positive"
    ]
    .fillna(False)
    .astype(bool)
)

In [347]:
def classify_outcome(row):
    truth = row["ground_truth"]
    pred = row["sentinela_positive"]

    if truth == 1 and pred:
        return "TP"

    if truth == 0 and pred:
        return "FP"

    if truth == 1 and not pred:
        return "FN"

    if truth == 0 and not pred:
        return "TN"

    raise ValueError("Combinação inválida.")

In [348]:
evaluation_labeled[
    "outcome"
] = (
    evaluation_labeled.apply(
        classify_outcome,
        axis=1
    )
)

In [349]:
display(
    evaluation_labeled[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "sentinela_positive",
            "outcome",
        ]
    ]
)

,region,year,epi_week,ground_truth,sentinela_positive,outcome
17,MG,2025,18,1,True,TP
18,MG,2025,19,1,True,TP
19,MG,2025,20,1,True,TP
20,MG,2025,21,1,True,TP
21,MG,2025,22,1,True,TP
22,MG,2025,23,1,False,FN
23,MG,2025,24,1,False,FN
24,MG,2025,25,1,False,FN
66,SP,2026,15,0,False,TN
67,SP,2026,16,0,False,TN


In [350]:
confusion_counts = (
    evaluation_labeled[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(confusion_counts)

outcome
TP     5
FP     0
FN    12
TN     8
Name: count, dtype: int64

In [351]:
TP = int(confusion_counts["TP"])
FP = int(confusion_counts["FP"])
FN = int(confusion_counts["FN"])
TN = int(confusion_counts["TN"])

sensitivity = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else float("nan")
)

specificity = (
    TN / (TN + FP)
    if (TN + FP) > 0
    else float("nan")
)

ppv = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else float("nan")
)

npv = (
    TN / (TN + FN)
    if (TN + FN) > 0
    else float("nan")
)

f1 = (
    2 * TP / (2 * TP + FP + FN)
    if (2 * TP + FP + FN) > 0
    else float("nan")
)

In [352]:
metrics = pd.DataFrame(
    [
        {
            "metric": "Sensitivity",
            "value": sensitivity,
        },
        {
            "metric": "Specificity",
            "value": specificity,
        },
        {
            "metric": "PPV",
            "value": ppv,
        },
        {
            "metric": "NPV",
            "value": npv,
        },
        {
            "metric": "F1",
            "value": f1,
        },
    ]
)

display(metrics)

,metric,value
0,Sensitivity,0.294118
1,Specificity,1.000000
2,PPV,1.000000
3,NPV,0.400000
4,F1,0.454545


## 13. Otimização de parâmetros

A primeira avaliação formal mostrou que o algoritmo atual apresenta alta especificidade, porém baixa sensibilidade.

Resultados iniciais:

- Sensibilidade: 29,4%
- Especificidade: 100%
- PPV: 100%
- NPV: 40%
- F1-score: 45,5%

Esse padrão indica um classificador conservador, com poucos falsos positivos, porém elevado número de falsos negativos.

A próxima etapa consiste em avaliar sistematicamente diferentes combinações de limiares, sem utilizar o ground truth para definir manualmente uma regra específica para uma região.

Serão avaliados:

- quantis históricos;
- fatores relativos à mediana;
- operadores de combinação entre os critérios.

O objetivo é encontrar configurações que aumentem a sensibilidade sem provocar crescimento excessivo de falsos positivos.

In [353]:
quantiles = [
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

median_factors = [
    1.10,
    1.20,
    1.25,
    1.30,
    1.40,
    1.50,
]

In [354]:
historical_mg_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_weekly_2019_2026.csv"
)

historical_sp_path = (
    project_root
    / "data"
    / "processed"
    / "srag_sp_historical_weekly_2019_2026.csv"
)

historical_mg = pd.read_csv(
    historical_mg_path,
    sep=";",
)

historical_sp = pd.read_csv(
    historical_sp_path,
    sep=";",
)

In [355]:
def build_reference_for_parameters(
    historical_weekly: pd.DataFrame,
    target_year: int,
    quantile: float,
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly["epi_year"] < target_year
    ].copy()

    history = history.loc[
        history["epi_week"].between(1, 52)
    ].copy()


    return (
        history
        .groupby("epi_week")
        .agg(
            historical_median=(
                "record_count",
                "median",
            ),
            historical_quantile=(
                "record_count",
                lambda values: values.quantile(
                    quantile
                ),
            ),
        )
        .reset_index()
    )

In [356]:
def evaluate_parameters(
    weekly_observed: pd.DataFrame,
    historical_weekly: pd.DataFrame,
    ground_truth_subset: pd.DataFrame,
    target_year: int,
    quantile: float,
    median_factor: float, 
) -> dict:
    reference =build_reference_for_parameters(
        historical_weekly=historical_weekly,
        target_year=target_year,
        quantile=quantile
    )

    evaluated = (
        weeklly_observed[
            [
                "epi_week",
                "record_count", 
            ]
        ]
        .merge(
            reference,
            on="epi_week",
            how="left",
        )
    )

    evaluated["prediction"] =(
        (
            evaluated["record_count"]
            > evaluated[
                "historical_quantile"
            ]
        )
        & (
            evaluated["record_count"]
            >= (
                evaluated[
                    "historical_median"
                ]
                * median_factor
            )
        )
    )

    labeled = (
        ground_truth_subset[
            ground_truth_subset[
                "ground_truth"
            ].notna()
        ]
        .merge(
            evaluated[
                [
                    "epi_week",
                    "prediction",
                ]
            ],
            on="epi_week",
            how="inner",
        )
    )

    truth = labeled[
        "ground_truth"
    ].astype(int)

    pred = labeled[
        "prediction"
    ].astype(bool)

    TP = int(
        ((truth == 1) & pred).sum()
    )

    FP = int(
        ((truth == 0) & pred).sum()
    )

    FN = int(
        ((truth == 1) & ~pred).sum()
    )

    TN = int(
        ((truth == 0) & ~pred).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if TP + FN
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if TN + FP
        else float("nan")
    )

    f1 = (
        2 * TP
        / (2 * TP + FP + FN)
        if (2 * TP + FP + FN)
        else float("nan")
    )

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
    }



In [357]:
evaluation_cases = [
    {
        "region": "MG",
        "year": 2025,
        "weekly_observed": signal_evaluation_2025,
        "historical_weekly": historical_mg,
    },
    {
        "region": "SP",
        "year": 2026,
        "weekly_observed": signal_evaluation_sp_2026,
        "historical_weekly": historical_sp,
    },
]

In [358]:
def evaluate_combined_parameters(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    quantile: float,
    median_factor: float,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly_observed = case[
            "weekly_observed"
        ].copy()

        historical_weekly = case[
            "historical_weekly"
        ].copy()

        reference = (
            build_reference_for_parameters(
                historical_weekly=historical_weekly,
                target_year=year,
                quantile=quantile,
            )
        )

        evaluated = (
            weekly_observed[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        evaluated["sentinela_positive"] = (
            (
                evaluated["record_count"]
                > evaluated[
                    "historical_quantile"
                ]
            )
            & (
                evaluated["record_count"]
                >= (
                    evaluated[
                        "historical_median"
                    ]
                    * median_factor
                )
            )
        )

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive", 
                ]
            ]
        )
    
    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )
    
    labeled = (
        ground_truth.loc[
            ground_truth[
                "ground_truth"
            ].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = (
        labeled["ground_truth"]
        .astype(int)
    )

    prediction = (
        labeled["sentinela_positive"]
        .astype(bool)
    )

    TP = int(
        (
            (truth == 1)
            & prediction
        ).sum()
    )

    FP = int(
        (
            (truth == 0)
            & prediction
        ).sum()
    )

    FN = int(
        (
            (truth == 1)
            & ~prediction
        ).sum()
    )

    TN = int(
        (
            (truth == 0)
            & ~prediction
        ).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP
        / (
            2 * TP
            + FP
            + FN
        )
        if (
            2 * TP
            + FP
            + FN
        ) > 0
        else float("nan")
    )

    balanced_accuracy = (
        (
            sensitivity
            + specificity
        )
        / 2
    )

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy":
            balanced_accuracy,
    }

In [359]:
optimization_results = []

for quantile in quantiles:
    for median_factor in median_factors:
        result = (
            evaluate_combined_parameters(
                cases=evaluation_cases,
                ground_truth=ground_truth,
                quantile=quantile,
                median_factor=median_factor,
            )
        )

        optimization_results.append(
            result
        )

optimization_df = pd.DataFrame(
    optimization_results
)

In [360]:
optimization_ranked = (
    optimization_df
    .sort_values(
        [
            "f1",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    optimization_ranked.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [361]:
high_specificity = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ] >= 0.90
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    high_specificity.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [362]:
perfect_specificity = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ].eq(1.0)
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    perfect_specificity.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [363]:
current_configuration = (
    optimization_df.loc[
        optimization_df[
            "quantile"
        ].eq(0.75)
        & optimization_df[
            "median_factor"
        ].eq(1.50)
    ]
)

display(current_configuration)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
17,0.75,1.5,5,0,12,8,0.294118,1.0,1.0,0.4,0.454545,0.647059


In [364]:
best_configuration = (
    optimization_ranked.iloc[0]
)

print("Configuração atual:")
display(current_configuration)

print("\nMelhor configuração encontrada:")
display(
    best_configuration.to_frame().T
)

Configuração atual:


,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
17,0.75,1.5,5,0,12,8,0.294118,1.0,1.0,0.4,0.454545,0.647059



Melhor configuração encontrada:


,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.1,6.0,0.0,11.0,8.0,0.352941,1.0,1.0,0.421053,0.521739,0.676471


## 14. Avaliação da regra de combinação dos limiares

A otimização dos valores de quantil e fator da mediana produziu apenas
ganho modesto de sensibilidade.

Isso sugere que a baixa sensibilidade pode não depender apenas dos valores
dos limiares, mas também da regra lógica utilizada para combiná-los.

A regra atual exige simultaneamente:

`valor > quantil histórico AND valor >= fator × mediana`

Nesta etapa serão comparadas:

- `AND`: ambos os critérios precisam ser atendidos;
- `OR`: pelo menos um dos critérios precisa ser atendido.

A análise será realizada sobre o mesmo ground truth externo, mantendo
separação entre otimização e posterior validação.

In [365]:
def evaluate_combined_parameters(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    quantile: float,
    median_factor: float,
    combination: str,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly_observed = case[
            "weekly_observed"
        ].copy()

        historical_weekly = case[
            "historical_weekly"
        ].copy()

        reference = build_reference_for_parameters(
            historical_weekly=historical_weekly,
            target_year=year,
            quantile=quantile,
        )

        evaluated = (
            weekly_observed[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        above_quantile = (
            evaluated["record_count"]
            > evaluated["historical_quantile"]
        )

        above_median_factor = (
            evaluated["record_count"]
            >= (
                evaluated["historical_median"]
                * median_factor
            )
        )

        if combination == "AND":
            prediction = (
                above_quantile
                & above_median_factor
            )

        elif combination == "OR":
            prediction = (
                above_quantile
                | above_median_factor
            )

        else:
            raise ValueError(
                f"Combinação inválida: {combination}"
            )

        evaluated[
            "sentinela_positive"
        ] = prediction

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive",
                ]
            ]
        )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    labeled = (
        ground_truth.loc[
            ground_truth[
                "ground_truth"
            ].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = labeled[
        "ground_truth"
    ].astype(int)

    pred = labeled[
        "sentinela_positive"
    ].astype(bool)

    TP = int(
        ((truth == 1) & pred).sum()
    )
    FP = int(
        ((truth == 0) & pred).sum()
    )
    FN = int(
        ((truth == 1) & ~pred).sum()
    )
    TN = int(
        ((truth == 0) & ~pred).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP / (2 * TP + FP + FN)
        if (2 * TP + FP + FN) > 0
        else float("nan")
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "combination": combination,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy":
            balanced_accuracy,
    }

In [366]:
quantiles = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

median_factors = [
    1.00,
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
    1.40,
    1.50,
]

combinations = [
    "AND",
    "OR",
]

In [367]:
optimization_results = []

for combination in combinations:
    for quantile in quantiles:
        for median_factor in median_factors:
            optimization_results.append(
                evaluate_combined_parameters(
                    cases=evaluation_cases,
                    ground_truth=ground_truth,
                    quantile=quantile,
                    median_factor=median_factor,
                    combination=combination,
                )
            )

optimization_df = pd.DataFrame(
    optimization_results
)

In [368]:
candidate_models = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ] >= 0.90
    ]
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "sensitivity",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    candidate_models.head(20)
)

,quantile,median_factor,combination,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.60,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
1,0.60,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
2,0.65,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
3,0.65,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
4,0.70,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
5,0.70,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
6,0.75,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
7,0.75,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
8,0.80,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
9,0.80,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706


In [369]:
display(
    optimization_df.loc[
        optimization_df[
            "specificity"
        ].eq(1.0)
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
        ],
        ascending=False,
    )
    .head(20)
)

,quantile,median_factor,combination,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
63,0.60,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
64,0.60,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
72,0.65,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
73,0.65,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
81,0.70,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
82,0.70,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
90,0.75,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
91,0.75,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
99,0.80,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
100,0.80,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706


In [370]:
pd.crosstab(
    evaluation_labeled["region"],
    evaluation_labeled["outcome"],
)

outcome,FN,TN,TP
region,,,
MG,3,0,5
SP,9,8,0


In [371]:
BEST_QUANTILE = 0.60
BEST_MEDIAN_FACTOR = 1.00
BEST_COMBINATION = "OR"

In [372]:
def predict_configuration(
    cases: list[dict],
    quantile: float,
    median_factor: float,
    combination: str,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly = case[
            "weekly_observed"
        ].copy()

        historical = case[
            "historical_weekly"
        ].copy()

        reference = (
            build_reference_for_parameters(
                historical_weekly=historical,
                target_year=year,
                quantile=quantile,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        evaluated[
            "ratio_to_median"
        ] = (
            evaluated["record_count"]
            / evaluated[
                "historical_median"
            ]
        )

        evaluated[
            "above_quantile"
        ] = (
            evaluated["record_count"]
            > evaluated[
                "historical_quantile"
            ]
        )

        evaluated[
            "above_median_factor"
        ] = (
            evaluated["record_count"]
            >= (
                evaluated[
                    "historical_median"
                ]
                * median_factor
            )
        )

        if combination == "AND":
            prediction = (
                evaluated["above_quantile"]
                & evaluated[
                    "above_median_factor"
                ]
            )

        elif combination == "OR":
            prediction = (
                evaluated["above_quantile"]
                | evaluated[
                    "above_median_factor"
                ]
            )

        else:
            raise ValueError(
                f"Combinação inválida: {combination}"
            )

        evaluated[
            "sentinela_positive"
        ] = prediction

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [373]:
candidate_predictions = predict_configuration(
    cases=evaluation_cases,
    quantile=0.60,
    median_factor=1.00,
    combination="OR",
)

In [374]:
candidate_evaluation = (
    ground_truth.loc[
        ground_truth[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions,
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

candidate_evaluation[
    "ground_truth"
] = (
    candidate_evaluation[
        "ground_truth"
    ].astype(int)
)

In [375]:
candidate_evaluation[
    "outcome"
] = candidate_evaluation.apply(
    classify_outcome,
    axis=1,
)

In [376]:
candidate_fn = (
    candidate_evaluation.loc[
        candidate_evaluation[
            "outcome"
        ].eq("FN")
    ][
        [
            "region",
            "year",
            "epi_week",
            "record_count",
            "historical_median",
            "historical_quantile",
            "ratio_to_median",
            "above_quantile",
            "above_median_factor",
        ]
    ]
    .sort_values(
        [
            "region",
            "epi_week",
        ]
    )
)

display(candidate_fn)

,region,year,epi_week,record_count,historical_median,historical_quantile,ratio_to_median,above_quantile,above_median_factor
10,SP,2026,20,2345.0,3053.0,3060.2,0.768097,False,False
11,SP,2026,21,2271.0,3141.0,3604.2,0.723018,False,False
12,SP,2026,22,2477.0,2777.0,3674.0,0.891970,False,False
13,SP,2026,23,2232.0,2746.0,3624.4,0.812819,False,False
14,SP,2026,24,2217.0,2323.0,3412.0,0.954369,False,False
15,SP,2026,25,2071.0,2399.0,3568.4,0.863276,False,False
17,SP,2026,27,1704.0,1768.0,2772.4,0.963801,False,False
18,SP,2026,28,1224.0,1555.0,2410.0,0.787138,False,False


In [377]:
display(
    pd.crosstab(
        candidate_evaluation[
            "region"
        ],
        candidate_evaluation[
            "outcome"
        ],
    )
)

outcome,FN,TN,TP
region,,,
MG,0,0,8
SP,8,8,1


In [378]:
def build_reference_selected_years(
    historical_weekly: pd.DataFrame,
    years: list[int],
    quantile: float,
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly[
            "epi_year"
        ].isin(years)
        & historical_weekly[
            "epi_week"
        ].between(1, 52)
    ].copy()

    reference = (
        history
        .groupby("epi_week")
        .agg(
            historical_median=(
                "record_count",
                "median",
            ),
            historical_quantile=(
                "record_count",
                lambda values: (
                    values.quantile(
                        quantile
                    )
                ),
            ),
            historical_years=(
                "epi_year",
                "nunique",
            ),
        )
        .reset_index()
    )

    return reference

In [379]:
reference_sp_without_pandemic = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
            2025,
        ],
        quantile=0.75,
    )
)

In [380]:
comparison_sp_reference = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "record_count",
        ]
    ]
    .merge(
        build_reference_for_parameters(
            historical_weekly=historical_sp,
            target_year=2026,
            quantile=0.75,
        ).rename(
            columns={
                "historical_median":
                    "median_all",
                "historical_quantile":
                    "q75_all",
            }
        ),
        on="epi_week",
    )
    .merge(
        reference_sp_without_pandemic.rename(
            columns={
                "historical_median":
                    "median_without_pandemic",
                "historical_quantile":
                    "q75_without_pandemic",
            }
        ),
        on="epi_week",
    )
)

display(
    comparison_sp_reference.loc[
        comparison_sp_reference[
            "epi_week"
        ].between(20, 28)
    ]
)

,epi_week,record_count,median_all,q75_all,median_without_pandemic,q75_without_pandemic,historical_years
19,20,2345.0,3053.0,7284.5,1869.0,3053.0,5
20,21,2271.0,3141.0,7292.0,1872.0,3141.0,5
21,22,2477.0,2777.0,7213.0,1692.0,2777.0,5
22,23,2232.0,2746.0,7843.0,1574.0,2746.0,5
23,24,2217.0,2323.0,7378.0,1629.0,2323.0,5
24,25,2071.0,2399.0,7482.5,1637.0,2399.0,5
25,26,2052.0,1891.0,6932.5,1527.0,1891.0,5
26,27,1704.0,1768.0,6239.5,1457.0,1768.0,5
27,28,1224.0,1555.0,5192.0,1206.0,1555.0,5


## 15. Detecção por desvio robusto

A análise de SP/2026 mostrou que o episódio epidemiológico apresentou
elevação sustentada em relação à mediana histórica, porém sem ultrapassar
frequentemente os quantis superiores.

Isso sugere que uma regra baseada exclusivamente em Q75 pode ter baixa
sensibilidade para episódios de crescimento moderado e persistente.

Nesta etapa será avaliado um escore robusto baseado na mediana e no
Median Absolute Deviation (MAD).

O método reduz a influência de anos extremos e mede o afastamento da
contagem semanal em relação ao comportamento histórico típico.

In [381]:
import numpy as np

def build_robust_reference(
    historical_weekly: pd.DataFrame,
    years: list[int],
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly["epi_year"].isin(years)
        & historical_weekly["epi_week"].between(1, 52)
    ].copy()

    rows = []

    for epi_week, group in history.groupby("epi_week"):
        values = group["record_count"].astype(float)

        median = values.median()

        mad = np.median(
            np.abs(values - median)
        )

        rows.append(
            {
                "epi_week": int(epi_week),
                "historical_median": median,
                "mad": mad,
                "historical_years": (
                    group["epi_year"].nunique()
                ),
            }
        )

    return pd.DataFrame(rows)

In [382]:
robust_reference_sp = build_robust_reference(
    historical_weekly=historical_sp,
    years=[
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
)

display(
    robust_reference_sp.head()
)

,epi_week,historical_median,mad,historical_years
0,1,927.0,778.0,5
1,2,817.0,579.0,5
2,3,812.0,520.0,5
3,4,791.0,416.0,5
4,5,913.0,568.0,5


In [383]:
robust_sp_2026 = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "record_count",
        ]
    ]
    .merge(
        robust_reference_sp,
        on="epi_week",
        how="left",
    )
)

In [384]:
robust_sp_2026[
    "robust_z"
] = np.where(
    robust_sp_2026["mad"] > 0,
    (
        0.6745
        * (
            robust_sp_2026["record_count"]
            - robust_sp_2026[
                "historical_median"
            ]
        )
        / robust_sp_2026["mad"]
    ),
    np.nan,
)

In [385]:
display(
    robust_sp_2026.loc[
        robust_sp_2026[
            "epi_week"
        ].between(15, 37),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "mad",
            "robust_z",
        ],
    ]
)

,epi_week,record_count,historical_median,mad,robust_z
14,15,1599.0,1870.0,115.0,-1.589474
15,16,1759.0,1909.0,226.0,-0.447677
16,17,1736.0,1814.0,243.0,-0.216506
17,18,1780.0,1813.0,550.0,-0.040470
18,19,2186.0,1688.0,968.0,0.347005
19,20,2345.0,1869.0,1184.0,0.271167
20,21,2271.0,1872.0,1269.0,0.212077
21,22,2477.0,1692.0,1085.0,0.488002
22,23,2232.0,1574.0,1015.0,0.437262
23,24,2217.0,1629.0,694.0,0.571478


In [386]:
robust_thresholds = [
    0.5,
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
]

In [387]:
for threshold in robust_thresholds:
    robust_sp_2026[
        f"signal_z_{threshold}"
    ] = (
        robust_sp_2026[
            "robust_z"
        ]
        >= threshold
    )

In [388]:
baseline_sets = {
    "all_history": [
        2019,
        2020,
        2021,
        2022,
        2023,
        2024,
        2025,
    ],
    "without_pandemic": [
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
    "recent_post_pandemic": [
        2022,
        2023,
        2024,
        2025,
    ],
}

## 16. Detecção por excesso relativo persistente

A avaliação por MAD mostrou que o episódio de SP/2026 não apresentou
desvios semanais extremos em relação à dispersão histórica.

Entretanto, entre as SE 20 e 28, observou-se uma sequência prolongada
de semanas acima da mediana histórica.

Isso sugere que episódios epidemiológicos podem se manifestar de duas formas:

1. aumento abrupto de grande magnitude;
2. aumento moderado, porém persistente.

Nesta etapa será avaliado um indicador de excesso relativo combinado
com persistência temporal.

In [389]:
robust_sp_2026["ratio_to_median"] = (
    robust_sp_2026["record_count"]
    / robust_sp_2026["historical_median"]
)

robust_sp_2026["relative_excess"] = (
    robust_sp_2026["ratio_to_median"] - 1
)

In [390]:
display(
    robust_sp_2026.loc[
        robust_sp_2026["epi_week"].between(15, 37),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "relative_excess",
        ],
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,relative_excess
14,15,1599.0,1870.0,0.855080,-0.144920
15,16,1759.0,1909.0,0.921425,-0.078575
16,17,1736.0,1814.0,0.957001,-0.042999
17,18,1780.0,1813.0,0.981798,-0.018202
18,19,2186.0,1688.0,1.295024,0.295024
19,20,2345.0,1869.0,1.254682,0.254682
20,21,2271.0,1872.0,1.213141,0.213141
21,22,2477.0,1692.0,1.463948,0.463948
22,23,2232.0,1574.0,1.418043,0.418043
23,24,2217.0,1629.0,1.360958,0.360958


In [391]:
ratio_thresholds = [
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
]

persistence_windows = [
    1,
    2,
    3,
    4,
]

In [392]:
def apply_persistent_signal(
    df: pd.DataFrame,
    ratio_threshold: float,
    persistence: int,
) -> pd.DataFrame:
    result = (
        df.sort_values("epi_week")
        .copy()
    )

    result["above_ratio"] = (
        result["ratio_to_median"]
        >= ratio_threshold
    )

    result["persistent_signal"] = (
        result["above_ratio"]
        .rolling(
            window=persistence,
            min_periods=persistence,
        )
        .sum()
        .eq(persistence)
    )

    return result

In [393]:
test_sp = apply_persistent_signal(
    robust_sp_2026,
    ratio_threshold=1.15,
    persistence=2,
)

display(
    test_sp.loc[
        test_sp["epi_week"].between(15, 37),
        [
            "epi_week",
            "ratio_to_median",
            "above_ratio",
            "persistent_signal",
        ],
    ]
)

,epi_week,ratio_to_median,above_ratio,persistent_signal
14,15,0.855080,False,False
15,16,0.921425,False,False
16,17,0.957001,False,False
17,18,0.981798,False,False
18,19,1.295024,True,False
19,20,1.254682,True,True
20,21,1.213141,True,True
21,22,1.463948,True,True
22,23,1.418043,True,True
23,24,1.360958,True,True


In [394]:
def evaluate_persistent_ratio(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    baseline_years_by_case: dict,
    ratio_threshold: float,
    persistence: int,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        key = (region, year)

        historical = case["historical_weekly"]

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=historical,
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["above_ratio"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["sentinela_positive"] = (
            evaluated["above_ratio"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive",
                ]
            ]
        )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    labeled = (
        ground_truth.loc[
            ground_truth["ground_truth"].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = labeled["ground_truth"].astype(int)
    pred = labeled["sentinela_positive"].astype(bool)

    TP = int(((truth == 1) & pred).sum())
    FP = int(((truth == 0) & pred).sum())
    FN = int(((truth == 1) & ~pred).sum())
    TN = int(((truth == 0) & ~pred).sum())

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP / (2 * TP + FP + FN)
        if (2 * TP + FP + FN) > 0
        else float("nan")
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    return {
        "ratio_threshold": ratio_threshold,
        "persistence": persistence,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
    }

In [395]:
baseline_years_by_case = {
    ("MG", 2025): [
        2019,
        2022,
        2023,
        2024,
    ],
    ("SP", 2026): [
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
}

In [396]:
persistent_results = []

for threshold in [
    1.00,
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
]:
    for persistence in [
        1,
        2,
        3,
        4,
    ]:
        persistent_results.append(
            evaluate_persistent_ratio(
                cases=evaluation_cases,
                ground_truth=ground_truth,
                baseline_years_by_case=(
                    baseline_years_by_case
                ),
                ratio_threshold=threshold,
                persistence=persistence,
            )
        )

persistent_results_df = (
    pd.DataFrame(
        persistent_results
    )
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "sensitivity",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    persistent_results_df.head(20)
)

,ratio_threshold,persistence,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,1.00,1,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
1,1.00,2,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
2,1.00,3,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
3,1.05,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
4,1.05,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
5,1.10,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
6,1.10,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
7,1.15,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
8,1.15,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
9,1.00,4,15,0,2,8,0.882353,1.0,1.0,0.800000,0.937500,0.941176


In [397]:
display(
    persistent_results_df.loc[
        persistent_results_df[
            "specificity"
        ] >= 0.90
    ].head(20)
)

,ratio_threshold,persistence,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,1.00,1,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
1,1.00,2,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
2,1.00,3,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
3,1.05,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
4,1.05,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
5,1.10,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
6,1.10,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
7,1.15,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
8,1.15,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
9,1.00,4,15,0,2,8,0.882353,1.0,1.0,0.800000,0.937500,0.941176


In [398]:
def predict_persistent_ratio(
    cases: list[dict],
    baseline_years_by_case: dict,
    ratio_threshold: float,
    persistence: int,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]
        key = (region, year)

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=case["historical_weekly"],
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count", 
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["above_ratio"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["sentinela_positive"] = (
            evaluated["above_ratio"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)
    
    return pd.concat(
        frames,
        ignore_index=True,
    )

In [399]:
candidate_predictions = (
    predict_persistent_ratio(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [400]:
candidate_validation = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions,
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

candidate_validation["ground_truth"] = (
    candidate_validation[
        "ground_truth"
    ].astype(int)
)

candidate_validation["outcome"] = (
    candidate_validation.apply(
        classify_outcome,
        axis=1,
    )
)

In [401]:
display(
    candidate_validation.loc[
        candidate_validation[
            "outcome"
        ].isin(["FN", "FP"]),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "above_ratio",
            "sentinela_positive",
            "outcome",
        ],
    ]
)

,region,year,epi_week,ground_truth,record_count,historical_median,ratio_to_median,above_ratio,sentinela_positive,outcome
18,SP,2026,28,1,1224.0,1206.0,1.014925,False,False,FN


In [402]:
display(
    pd.crosstab(
        candidate_validation["region"],
        candidate_validation["outcome"],
    )
)

outcome,FN,TN,TP
region,,,
MG,0,0,8
SP,1,8,8


### Análise do único falso negativo

O único falso negativo da configuração candidata ocorreu em SP/2026 na SE 28.

Nessa semana:

- casos observados: 1.224;
- mediana histórica: 1.206;
- razão observado/mediana: 1,015;
- limiar candidato: 1,15.

Embora a SE 28 tenha sido incluída na janela positiva externa, a contagem já havia retornado para um nível muito próximo da mediana histórica.

Esse resultado sugere que a divergência pode representar uma diferença entre:

- persistência institucional/epidemiológica do episódio;
- normalização estatística da série semanal.

A SE 28 deve, portanto, ser tratada como uma semana de borda do episódio durante análises posteriores.

In [403]:
sp_episode_phases = {
    20: "ONSET",
    21: "ACTIVE",
    22: "ACTIVE",
    23: "ACTIVE",
    24: "ACTIVE",
    25: "ACTIVE",
    26: "ACTIVE",
    27: "ACTIVE",
    28: "RESOLUTION",
}

In [404]:
mg_episode_phases = {
    18: "ONSET",
    19: "ACTIVE",
    20: "ACTIVE",
    21: "ACTIVE",
    22: "ACTIVE",
    23: "ACTIVE",
    24: "ACTIVE",
    25: "RESOLUTION",
}

In [405]:
ground_truth["episode_phase"] = pd.NA

for week, phase in mg_episode_phases.items():
    mask = (
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
        & ground_truth["epi_week"].eq(week)
    )

    ground_truth.loc[
        mask,
        "episode_phase",
    ] = phase


for week, phase in sp_episode_phases.items():
    mask = (
        ground_truth["region"].eq("SP")
        & ground_truth["year"].eq(2026)
        & ground_truth["epi_week"].eq(week)
    )

    ground_truth.loc[
        mask,
        "episode_phase",
    ] = phase

## 17. Validação fora da amostra — SP 2025

A configuração candidata foi definida utilizando apenas:

- MG/2025;
- SP/2026.

Nesta etapa, seus parâmetros são congelados:

- baseline excluindo 2020 e 2021;
- razão observado/mediana >= 1,15;
- persistência mínima de 2 semanas.

Nenhum parâmetro será modificado durante a avaliação de SP/2025.

Fontes oficiais do Estado de São Paulo mostram crescimento importante
das hospitalizações por SRAG ao longo do outono de 2025, com pico
aproximadamente nas SE 20–21 e queda posterior.

SP/2025 será utilizado como conjunto independente para avaliar a
generalização do modelo candidato.

In [406]:
weekly_sp_2025 = (
    historical_sp.loc[
        historical_sp["epi_year"].eq(2025)
    ][
        [
            "epi_week",
            "record_count",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

display(weekly_sp_2025.head())
display(weekly_sp_2025.tail())

,epi_week,record_count
0,1,927
1,2,817
2,3,812
3,4,791
4,5,700


,epi_week,record_count
48,49,1204
49,50,999
50,51,824
51,52,687
52,53,681


In [407]:
baseline_sp_2025_years = [
    2019,
    2022,
    2023,
    2024,
]

reference_sp_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=baseline_sp_2025_years,
        quantile=0.75,
    )
)

In [408]:
validation_sp_2025 = (
    weekly_sp_2025
    .merge(
        reference_sp_2025[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left"
    )
)

validation_sp_2025["ratio_to_median"] = (
    validation_sp_2025["record_count"]
    / validation_sp_2025["historical_median"]
)

validation_sp_2025["above_ratio"] = (
    validation_sp_2025["ratio_to_median"]
    >= 1.15
)

validation_sp_2025["sentinela_positive"] = (
    validation_sp_2025["above_ratio"]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

display(
    validation_sp_2025.loc[
        validation_sp_2025[
            "epi_week"
        ].between(10, 35)
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,above_ratio,sentinela_positive
9,10,1166,1806.0,0.645626,False,False
10,11,1113,1768.5,0.629347,False,False
11,12,1263,1794.5,0.703817,False,False
12,13,1462,1746.5,0.837103,False,False
13,14,1608,1873.0,0.858516,False,False
14,15,1791,1927.5,0.929183,False,False
15,16,1992,1796.0,1.109131,False,False
16,17,2313,1711.5,1.351446,True,False
17,18,2526,1755.0,1.439316,True,True
18,19,2728,1592.5,1.713030,True,True


In [409]:
ground_truth_sp_2025 = pd.DataFrame(
    {
        "epi_week": range(1, 53),
    }
)

ground_truth_sp_2025["region"] = "SP"
ground_truth_sp_2025["year"] = 2025

ground_truth_sp_2025["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth_sp_2025.index,
    dtype="Int64",
)

ground_truth_sp_2025["evidence_level"] = pd.NA

In [410]:
positive_mask = (
    ground_truth_sp_2025[
        "epi_week"
    ].between(17, 25)
)

ground_truth_sp_2025.loc[
    positive_mask,
    "ground_truth",
] = 1

ground_truth_sp_2025.loc[
    positive_mask,
    "evidence_level",
] = "STRONG"

In [411]:
negative_mask = (
    ground_truth_sp_2025[
        "epi_week"
    ].between(30, 40)
)

ground_truth_sp_2025.loc[
    negative_mask,
    "ground_truth",
] = 0

ground_truth_sp_2025.loc[
    negative_mask,
    "evidence_level",
] = "MODERATE"

In [412]:
sp_2025_evaluation = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_sp_2025[
            [
               "epi_week",
                "record_count",
                "historical_median",
                "ratio_to_median",
                "sentinela_positive", 
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

sp_2025_evaluation[
    "ground_truth"
] = (
    sp_2025_evaluation[
        "ground_truth"
    ].astype(int)
)

sp_2025_evaluation[
    "outcome"
] = sp_2025_evaluation.apply(
    classify_outcome,
    axis=1,
)

In [413]:
display(
    sp_2025_evaluation[
        [
            "epi_week",
            "ground_truth",
            "ratio_to_median",
            "sentinela_positive",
            "outcome",
        ]
    ]
)

display(
    sp_2025_evaluation[
        "outcome"
    ].value_counts()
)

,epi_week,ground_truth,ratio_to_median,sentinela_positive,outcome
0,17,1,1.351446,False,FN
1,18,1,1.439316,True,TP
2,19,1,1.713030,True,TP
3,20,1,1.821599,True,TP
4,21,1,1.940686,True,TP
5,22,1,1.813255,True,TP
6,23,1,1.745154,True,TP
7,24,1,1.517805,True,TP
8,25,1,1.492379,True,TP
9,30,0,1.096226,False,TN


outcome
TN    11
TP     8
FN     1
Name: count, dtype: int64

In [414]:
validation_sp_2025["signal_detected"] = (
    validation_sp_2025["ratio_to_median"]
    >= 1.15
)

validation_sp_2025["alert_confirmed"] = (
    validation_sp_2025["signal_detected"]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

In [415]:
display(
    validation_sp_2025.loc[
        validation_sp_2025["epi_week"].between(15, 27),
        [
            "epi_week",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ],
    ]
)

,epi_week,ratio_to_median,signal_detected,alert_confirmed
14,15,0.929183,False,False
15,16,1.109131,False,False
16,17,1.351446,True,False
17,18,1.439316,True,True
18,19,1.713030,True,True
19,20,1.821599,True,True
20,21,1.940686,True,True
21,22,1.813255,True,True
22,23,1.745154,True,True
23,24,1.517805,True,True


## 18. Validação fora da amostra — MG 2024

A configuração candidata permanece congelada:

- baseline histórico excluindo 2020 e 2021;
- `SIGNAL` quando observado/mediana histórica >= 1,15;
- `ALERT` após duas semanas consecutivas com `SIGNAL`.

Nenhum parâmetro será ajustado durante a avaliação de MG/2024.

In [416]:
weekly_mg_2024 = (
    historical_mg.loc[
        historical_mg["epi_year"].eq(2024)
    ][
        [
            "epi_week",
            "record_count",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

display(weekly_mg_2024.head())
display(weekly_mg_2024.tail())

,epi_week,record_count
0,1,285
1,2,276
2,3,330
3,4,274
4,5,363


,epi_week,record_count
47,48,482
48,49,543
49,50,456
50,51,405
51,52,409


In [417]:
baseline_mg_2024_years = [
    2019,
    2022,
    2023,
]

In [418]:
reference_mg_2024 = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=baseline_mg_2024_years,
        quantile=0.75,
    )
)

In [419]:
validation_mg_2024 = (
    weekly_mg_2024
    .merge(
        reference_mg_2024[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

validation_mg_2024["ratio_to_median"] = (
    validation_mg_2024["record_count"]
    / validation_mg_2024["historical_median"]
)

validation_mg_2024["signal_detected"] = (
    validation_mg_2024[
        "ratio_to_median"
    ] >= 1.15
)

validation_mg_2024["alert_confirmed"] = (
    validation_mg_2024[
        "signal_detected"
    ]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

In [420]:
display(
    validation_mg_2024[
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,signal_detected,alert_confirmed
0,1,285,887.0,0.321308,False,False
1,2,276,667.0,0.413793,False,False
2,3,330,524.0,0.629771,False,False
3,4,274,452.0,0.606195,False,False
4,5,363,504.0,0.720238,False,False
5,6,363,612.0,0.593137,False,False
6,7,535,659.0,0.811836,False,False
7,8,650,657.0,0.989346,False,False
8,9,718,657.0,1.092846,False,False
9,10,603,648.0,0.930556,False,False


In [421]:
display(
    validation_mg_2024.loc[
        validation_mg_2024[
            "signal_detected"
        ],
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ],
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,signal_detected,alert_confirmed
13,14,717,565.0,1.269027,True,False
14,15,662,535.0,1.237383,True,True
15,16,735,561.0,1.310160,True,True
16,17,699,509.0,1.373281,True,True
17,18,713,605.0,1.178512,True,True
18,19,708,556.0,1.273381,True,True
19,20,722,618.0,1.168285,True,True
20,21,732,577.0,1.268631,True,True
22,23,631,533.0,1.183865,True,False
23,24,610,478.0,1.276151,True,True


## 19. Ground truth externo — MG 2024

A revisão de fontes independentes indica múltiplos períodos de aumento
de SRAG em Minas Gerais durante 2024.

Foram identificados três episódios principais:

### Episódio 1 — outono/inverno
- início externo: aproximadamente SE 14;
- crescimento documentado durante abril e maio;
- MG permaneceu com tendência de crescimento até pelo menos SE 21;
- encerramento exato não definido.

Rotulagem conservadora:
- SE 14–21 → positivo;
- SE 22–27 → transição / não rotulado.

### Episódio 2 — recrudescimento no segundo semestre
- crescimento documentado em setembro;
- manutenção de tendência de aumento até SE 39–40.

Rotulagem conservadora:
- SE 35–40 → positivo;
- SE 29–34 e SE 41–44 → transição / não rotulado.

### Episódio 3 — fim do ano
- nova elevação documentada nas SE 49–50;
- associação com circulação de Covid-19 e outros vírus respiratórios.

Rotulagem conservadora:
- SE 49–52 → positivo;
- SE 45–48 → transição / não rotulado.

Semanas fora dessas janelas não serão classificadas automaticamente como negativas
sem evidência externa suficiente.

In [422]:
ground_truth_mg_2024 = pd.DataFrame(
    {
        "epi_week": range(1, 53),
    }
)

ground_truth_mg_2024["region"] = "MG"
ground_truth_mg_2024["year"] = 2024

ground_truth_mg_2024["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth_mg_2024.index,
    dtype="Int64",
)

ground_truth_mg_2024["evidence_level"] = pd.NA
ground_truth_mg_2024["episode_id"] = pd.NA

In [423]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(14, 21)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_001"

In [424]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(35, 40)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_002"

In [425]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(49, 52)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_003"

In [426]:
negative_weeks = [
    *range(1, 9),
    *range(43, 46),
]

negative_mask = (
    ground_truth_mg_2024[
        "epi_week"
    ].isin(negative_weeks)
)

ground_truth_mg_2024.loc[
    negative_mask,
    "ground_truth",
] = 0

ground_truth_mg_2024.loc[
    negative_mask,
    "evidence_level",
] = "MODERATE"

In [427]:
mg_2024_evaluation = (
    ground_truth_mg_2024.loc[
        ground_truth_mg_2024[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_mg_2024[
            [
                "epi_week",
                "ratio_to_median",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

In [428]:
mg_2024_evaluation[
    "sentinela_positive"
] = (
    mg_2024_evaluation[
        "signal_detected"
    ]
)

mg_2024_evaluation[
    "ground_truth"
] = (
    mg_2024_evaluation[
        "ground_truth"
    ].astype(int)
)

mg_2024_evaluation[
    "outcome"
] = (
    mg_2024_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [429]:
display(
    mg_2024_evaluation[
        [
            "epi_week",
            "ground_truth",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
            "outcome",
        ]
    ]
)

display(
    mg_2024_evaluation[
        "outcome"
    ].value_counts()
)

,epi_week,ground_truth,ratio_to_median,signal_detected,alert_confirmed,outcome
0,1,0,0.321308,False,False,TN
1,2,0,0.413793,False,False,TN
2,3,0,0.629771,False,False,TN
3,4,0,0.606195,False,False,TN
4,5,0,0.720238,False,False,TN
5,6,0,0.593137,False,False,TN
6,7,0,0.811836,False,False,TN
7,8,0,0.989346,False,False,TN
8,14,1,1.269027,True,False,TP
9,15,1,1.237383,True,True,TP


outcome
TP    18
TN    11
Name: count, dtype: int64

## 20. Avaliação agregada do modelo candidato

A configuração candidata foi congelada em:

- baseline histórico excluindo 2020 e 2021;
- `SIGNAL` quando observado/mediana histórica >= 1,15;
- `ALERT` após 2 semanas consecutivas com `SIGNAL`.

A partir deste ponto, nenhum parâmetro será alterado.

Serão agregadas as avaliações independentes de:

- MG/2024;
- MG/2025;
- SP/2025;
- SP/2026.

O objetivo é estimar o desempenho global do modelo candidato antes de ampliar a validação para novas regiões.

In [430]:
eval_mg_2024 = (
    mg_2024_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "signal_detected",
        ]
    ]
    .copy()
)

eval_mg_2024["sentinela_positive"] = (
    eval_mg_2024["signal_detected"]
)

In [431]:
eval_sp_2025 = (
    sp_2025_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "sentinela_positive",
        ]
    ]
    .copy()
)

In [432]:
candidate_predictions_all = (
    predict_persistent_ratio(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [433]:
def predict_signal_and_alert(
    cases: list[dict],
    baseline_years_by_case: dict,
    ratio_threshold: float = 1.15,
    persistence: int = 2,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]
        key = (region, year)

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=case["historical_weekly"],
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["signal_detected"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["alert_confirmed"] = (
            evaluated["signal_detected"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [434]:
candidate_predictions_all = (
    predict_signal_and_alert(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [435]:
eval_base_cases = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

eval_base_cases["sentinela_positive"] = (
    eval_base_cases["signal_detected"]
)

In [436]:
aggregate_evaluation = pd.concat(
    [
        eval_base_cases[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
        eval_sp_2025[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
        eval_mg_2024[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
    ],
    ignore_index=True,
)

In [437]:
aggregate_evaluation["ground_truth"] = (
    aggregate_evaluation[
        "ground_truth"
    ].astype(int)
)

aggregate_evaluation["outcome"] = (
    aggregate_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [438]:
display(
    pd.crosstab(
        [
            aggregate_evaluation["region"],
            aggregate_evaluation["year"],
        ],
        aggregate_evaluation["outcome"],
    )
)

outcome      FN  TN  TP
region year            
MG     2024   0  11  18
       2025   0   0   8
SP     2025   1  11   8
       2026   1   8   8

In [439]:
aggregate_counts = (
    aggregate_evaluation[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(aggregate_counts)

outcome
TP    42
FP     0
FN     2
TN    30
Name: count, dtype: int64

In [440]:
TP = int(aggregate_counts["TP"])
FP = int(aggregate_counts["FP"])
FN = int(aggregate_counts["FN"])
TN = int(aggregate_counts["TN"])

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
ppv = TP / (TP + FP)
npv = TN / (TN + FN)

f1 = (
    2 * TP
    / (
        2 * TP
        + FP
        + FN
    )
)

balanced_accuracy = (
    sensitivity
    + specificity
) / 2

In [441]:
aggregate_metrics = pd.DataFrame(
    [
        {
            "metric": "Sensitivity",
            "value": sensitivity,
        },
        {
            "metric": "Specificity",
            "value": specificity,
        },
        {
            "metric": "PPV",
            "value": ppv,
        },
        {
            "metric": "NPV",
            "value": npv,
        },
        {
            "metric": "F1",
            "value": f1,
        },
        {
            "metric": "Balanced Accuracy",
            "value": balanced_accuracy,
        },
    ]
)

display(aggregate_metrics)

,metric,value
0,Sensitivity,0.954545
1,Specificity,1.000000
2,PPV,1.000000
3,NPV,0.937500
4,F1,0.976744
5,Balanced Accuracy,0.977273


## 21. Desempenho agregado do modelo candidato

A avaliação agregada considerou quatro cenários independentes:

- MG/2024;
- MG/2025;
- SP/2025;
- SP/2026.

A configuração avaliada foi mantida fixa:

- exclusão de 2020 e 2021 do baseline;
- `SIGNAL` quando a razão observado/mediana histórica é >= 1,15;
- `ALERT` após duas semanas consecutivas com `SIGNAL`.

### Resultados agregados

- Sensibilidade: 95,45%
- Especificidade: 100,00%
- PPV: 100,00%
- NPV: 93,75%
- F1-score: 97,67%
- Balanced Accuracy: 97,73%

### Interpretação

O modelo candidato apresentou alta capacidade de detectar semanas
associadas a episódios epidemiológicos externos, mantendo ausência de
falsos positivos nas semanas negativas rotuladas.

A principal melhora em relação à regra original ocorreu na sensibilidade.

A regra original, baseada em Q75 e 1,5× a mediana histórica, apresentou
baixa sensibilidade para episódios de elevação moderada e persistente.

O modelo candidato mostrou melhor capacidade de detectar tanto:

- aumentos abruptos;
- elevações sustentadas de menor magnitude relativa.

### Limitações

As métricas ainda devem ser consideradas preliminares.

O conjunto de validação contém apenas duas regiões e quatro cenários
região/ano, e parte do ground truth foi construída por janelas
epidemiológicas derivadas de fontes externas.

Novas regiões e anos devem ser incorporados antes da adoção definitiva
dos parâmetros.

In [442]:
print("TP:", TP)
print("FP:", FP)
print("FN:", FN)
print("TN:", TN)
print("Total avaliado:", TP + FP + FN + TN)

TP: 42
FP: 0
FN: 2
TN: 30
Total avaliado: 74


In [443]:
alert_evaluation = aggregate_evaluation.copy()

In [444]:
alert_sp_2025 = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_sp_2025[
            [
                "epi_week",
                "alert_confirmed",
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

In [445]:
alert_mg_2024 = (
    mg_2024_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "alert_confirmed",
        ]
    ]
    .copy()
)

In [446]:
alert_base_cases = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "alert_confirmed",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

In [447]:
aggregate_alert_evaluation = pd.concat(
    [
        alert_base_cases,
        alert_sp_2025,
        alert_mg_2024,
    ],
    ignore_index=True,
)

In [448]:
aggregate_alert_evaluation[
    "sentinela_positive"
] = (
    aggregate_alert_evaluation[
        "alert_confirmed"
    ].astype(bool)
)

aggregate_alert_evaluation[
    "ground_truth"
] = (
    aggregate_alert_evaluation[
        "ground_truth"
    ].astype(int)
)

In [449]:
aggregate_alert_evaluation[
    "outcome"
] = (
    aggregate_alert_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [450]:
alert_counts = (
    aggregate_alert_evaluation[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(alert_counts)

outcome
TP    41
FP     0
FN     3
TN    30
Name: count, dtype: int64

## 22. Desempenho agregado do estado ALERT

O estado `ALERT` foi avaliado separadamente do estado `SIGNAL`.

A regra utilizada foi:

- `SIGNAL`: razão observado/mediana histórica >= 1,15;
- `ALERT`: confirmação do excesso relativo em pelo menos 2 semanas consecutivas.

### Matriz de confusão

- TP: 41
- FP: 0
- FN: 3
- TN: 30

### Métricas

- Sensibilidade: 93,18%
- Especificidade: 100,00%
- PPV: 100,00%
- NPV: 90,91%
- F1-score: 96,47%
- Balanced Accuracy: 96,59%

### Interpretação

A exigência de persistência reduziu discretamente a sensibilidade em
relação ao estado `SIGNAL`, como esperado.

Entretanto, o estado `ALERT` manteve especificidade e valor preditivo
positivo máximos no conjunto avaliado.

Esse comportamento é compatível com a finalidade operacional dos
dois estados:

- `SIGNAL` prioriza detecção precoce;
- `ALERT` prioriza confirmação de uma elevação sustentada.

As métricas permanecem preliminares e devem ser reavaliadas em novas
regiões e períodos independentes.

In [451]:
signal_vs_alert = pd.DataFrame(
    [
        {
            "state": "SIGNAL",
            "sensitivity": 0.954545,
            "specificity": 1.0,
            "ppv": 1.0,
            "npv": 0.937500,
            "f1": 0.976744,
            "balanced_accuracy": 0.977273,
        },
        {
            "state": "ALERT",
            "sensitivity": 41 / 44,
            "specificity": 30 / 30,
            "ppv": 41 / 41,
            "npv": 30 / 33,
            "f1": 82 / 85,
            "balanced_accuracy": (
                (41 / 44) + 1.0
            ) / 2,
        },
    ]
)

display(signal_vs_alert)

,state,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,SIGNAL,0.954545,1.0,1.0,0.937500,0.976744,0.977273
1,ALERT,0.931818,1.0,1.0,0.909091,0.964706,0.965909


In [452]:
display(
    aggregate_alert_evaluation.loc[
        aggregate_alert_evaluation[
            "outcome"
        ].eq("FN"),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "alert_confirmed",
        ],
    ]
)

,region,year,epi_week,ground_truth,alert_confirmed
18,SP,2026,28,1,False
25,SP,2025,17,1,False
53,MG,2024,14,1,False


### Análise dos falsos negativos do ALERT

Foram identificados três falsos negativos na avaliação agregada:

- MG/2024 — SE 14;
- SP/2025 — SE 17;
- SP/2026 — SE 28.

Os casos de MG/2024 e SP/2025 correspondem às semanas iniciais dos respectivos episódios.

Nessas semanas, o estado `SIGNAL` já havia sido ativado, porém ainda não existiam duas semanas consecutivas de excesso relativo para confirmação de `ALERT`.

Portanto, esses casos representam o atraso operacional esperado da regra de persistência, e não ausência de detecção.

O terceiro caso, SP/2026 na SE 28, corresponde à fase de resolução do episódio. Nessa semana, a razão observado/mediana histórica já havia caído para aproximadamente 1,015, abaixo do limiar operacional de 1,15.

Assim, nenhum dos falsos negativos ocorreu durante a fase central de um episódio ativo.

In [453]:
episode_ground_truth = pd.DataFrame(
    [
        {
            "episode_id": "MG2024_RESP_001",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 14,
            "end_epi_week": 21,
        },
        {
            "episode_id": "MG2024_RESP_002",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 35,
            "end_epi_week": 40,
        },
        {
            "episode_id": "MG2024_RESP_003",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 49,
            "end_epi_week": 52,
        },
        {
            "episode_id": "MG2025_RESP_001",
            "region": "MG",
            "year": 2025,
            "start_epi_week": 18,
            "end_epi_week": 25,
        },
        {
            "episode_id": "SP2025_RESP_001",
            "region": "SP",
            "year": 2025,
            "start_epi_week": 17,
            "end_epi_week": 25,
        },
        {
            "episode_id": "SP2026_RESP_001",
            "region": "SP",
            "year": 2026,
            "start_epi_week": 20,
            "end_epi_week": 28,
        },
    ]
)

In [454]:
predictions_mg_2024 = (
    validation_mg_2024[
        [
            "epi_week",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
    .copy()
)

predictions_mg_2024["region"] = "MG"
predictions_mg_2024["year"] = 2024


predictions_sp_2025 = (
    validation_sp_2025[
        [
            "epi_week",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
    .copy()
)

predictions_sp_2025["region"] = "SP"
predictions_sp_2025["year"] = 2025

In [455]:
all_candidate_predictions = pd.concat(
    [
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        predictions_mg_2024,
        predictions_sp_2025,
    ],
    ignore_index=True,
)

In [456]:
episode_detection_rows = []

for episode in episode_ground_truth.itertuples(
    index=False
):
    episode_predictions = (
        all_candidate_predictions.loc[
            all_candidate_predictions[
                "region"
            ].eq(episode.region)
            & all_candidate_predictions[
                "year"
            ].eq(episode.year)
            & all_candidate_predictions[
                "epi_week"
            ].between(
                episode.start_epi_week,
                episode.end_epi_week,
            )
        ]
        .sort_values("epi_week")
    )

    signal_rows = episode_predictions.loc[
        episode_predictions[
            "signal_detected"
        ]
    ]

    alert_rows = episode_predictions.loc[
        episode_predictions[
            "alert_confirmed"
        ]
    ]

    first_signal_week = (
        int(signal_rows.iloc[0]["epi_week"])
        if not signal_rows.empty
        else None
    )

    first_alert_week = (
        int(alert_rows.iloc[0]["epi_week"])
        if not alert_rows.empty
        else None
    )

    signal_delay_weeks = (
        first_signal_week
        - episode.start_epi_week
        if first_signal_week is not None
        else None
    )

    alert_delay_weeks = (
        first_alert_week
        - episode.start_epi_week
        if first_alert_week is not None
        else None
    )

    episode_detection_rows.append(
        {
            "episode_id":
                episode.episode_id,
            "region":
                episode.region,
            "year":
                episode.year,
            "start_epi_week":
                episode.start_epi_week,
            "end_epi_week":
                episode.end_epi_week,
            "first_signal_week":
                first_signal_week,
            "signal_delay_weeks":
                signal_delay_weeks,
            "first_alert_week":
                first_alert_week,
            "alert_delay_weeks":
                alert_delay_weeks,
        }
    )

episode_detection = pd.DataFrame(
    episode_detection_rows
)

display(episode_detection)

,episode_id,region,year,start_epi_week,end_epi_week,first_signal_week,signal_delay_weeks,first_alert_week,alert_delay_weeks
0,MG2024_RESP_001,MG,2024,14,21,14,0,15,1
1,MG2024_RESP_002,MG,2024,35,40,35,0,35,0
2,MG2024_RESP_003,MG,2024,49,52,49,0,49,0
3,MG2025_RESP_001,MG,2025,18,25,18,0,18,0
4,SP2025_RESP_001,SP,2025,17,25,17,0,18,1
5,SP2026_RESP_001,SP,2026,20,28,20,0,20,0


In [457]:
print(
    "Episódios avaliados:",
    len(episode_detection),
)

print(
    "Episódios detectados por SIGNAL:",
    episode_detection[
        "first_signal_week"
    ].notna().sum(),
)

print(
    "Episódios confirmados por ALERT:",
    episode_detection[
        "first_alert_week"
    ].notna().sum(),
)

print(
    "Atraso médio SIGNAL:",
    episode_detection[
        "signal_delay_weeks"
    ].mean(),
    "semanas",
)

print(
    "Atraso médio ALERT:",
    episode_detection[
        "alert_delay_weeks"
    ].mean(),
    "semanas",
)

Episódios avaliados: 6
Episódios detectados por SIGNAL: 6
Episódios confirmados por ALERT: 6
Atraso médio SIGNAL: 0.0 semanas
Atraso médio ALERT: 0.3333333333333333 semanas


## 23. Avaliação pseudo-prospectiva do tempo de detecção

A avaliação semanal confirmou que todos os episódios externos foram
detectados pelo modelo candidato.

Entretanto, essa análise utiliza contagens semanais consolidadas.

Para avaliar capacidade de alerta precoce, é necessário reconstruir
o estado do Sentinela em diferentes datas após o encerramento de
cada semana epidemiológica.

Nesta etapa serão utilizados:

- registros conhecidos até cada snapshot;
- curva histórica regional de completude;
- nowcast regional;
- baseline histórico disponível antes do ano avaliado;
- limiar congelado de 1,15× a mediana histórica;
- duas semanas consecutivas para confirmação de `ALERT`.

Serão estimados:

- data do primeiro `SIGNAL`;
- data do primeiro `ALERT`;
- atraso em dias em relação ao início externo do episódio;
- antecedência em relação a respostas institucionais, quando disponível.

In [458]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = (
    project_root
    / "data"
    / "raw"
)

historical_files = sorted(
    raw_data_dir.glob("INFLUD*.csv")
)

print(
    "Arquivos encontrados:",
    len(historical_files),
)

for path in historical_files:
    print(path.name)

Arquivos encontrados: 8
INFLUD19-23-03-2026.csv
INFLUD20-23-03-2026.csv
INFLUD21-23-03-2026.csv
INFLUD22-23-03-2026.csv
INFLUD23-23-03-2026.csv
INFLUD24-23-03-2026.csv
INFLUD25-28-09-2026.csv
INFLUD26-28-09-2026.csv


In [459]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

df_mg_2025_temporal = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    usecols=[
        "NU_NOTIFIC",
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_mg_2025_temporal["SG_UF"] = (
    df_mg_2025_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

mg_2025_temporal = (
    df_mg_2025_temporal.loc[
        df_mg_2025_temporal["SG_UF"].eq("MG")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    mg_2025_temporal[column] = (
        pd.to_datetime(
            mg_2025_temporal[column],
            errors="coerce",
            dayfirst=True,
            utc=True,
        )
        .dt.tz_localize(None)
    )

/tmp/ipykernel_57211/1769100398.py:39: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_57211/1769100398.py:39: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(


In [460]:
model_years_mg = [
    2023,
    2024,
    2025,
]

delay_frames_mg = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if year not in model_years_mg:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    mg = df.loc[
        df["SG_UF"].eq("MG")
    ].copy()

    mg["DT_SIN_PRI"] = pd.to_datetime(
        mg["DT_SIN_PRI"],
        errors="coerce",
        dayfirst=True,
    )

    mg["DT_NOTIFIC"] = pd.to_datetime(
        mg["DT_NOTIFIC"],
        errors="coerce",
        dayfirst=True,
    )

    mg["delay_notification_days"] = (
        mg["DT_NOTIFIC"]
        - mg["DT_SIN_PRI"]
    ).dt.days

    mg["source_year"] = year

    delay_frames_mg.append(
        mg[
            [
                "source_year",
                "delay_notification_days",
            ]
        ]
    )

delay_mg = pd.concat(
    delay_frames_mg,
    ignore_index=True,
)

delay_mg = delay_mg.loc[
    delay_mg[
        "delay_notification_days"
    ].notna()
    & delay_mg[
        "delay_notification_days"
    ].ge(0)
].copy()

print(
    "Registros válidos MG:",
    len(delay_mg),
)

print(
    "Anos:",
    sorted(
        delay_mg[
            "source_year"
        ].unique()
    ),
)

/tmp/ipykernel_57211/142571632.py:48: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_NOTIFIC"] = pd.to_datetime(
/tmp/ipykernel_57211/142571632.py:42: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_SIN_PRI"] = pd.to_datetime(


Registros válidos MG: 57015
Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]


/tmp/ipykernel_57211/142571632.py:42: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_SIN_PRI"] = pd.to_datetime(
/tmp/ipykernel_57211/142571632.py:48: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_NOTIFIC"] = pd.to_datetime(


In [461]:
mg_2025_delay_check = (
    mg_2025_temporal[
        "DT_NOTIFIC"
    ]
    - mg_2025_temporal[
        "DT_SIN_PRI"
    ]
).dt.days

mg_2025_delay_check = (
    mg_2025_delay_check.loc[
        mg_2025_delay_check.ge(0)
    ]
)

display(
    mg_2025_delay_check.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95,
        ]
    )
)

count    45655.000000
mean         8.226153
std         22.972289
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
80%          7.000000
90%         12.000000
95%         22.000000
max        384.000000
dtype: float64

In [462]:
def build_week_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    min_completeness: float = 0.80,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(0, max_lag + 1):
        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue

        completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        if completeness > 0:
            nowcast = (
                known_cases
                / completeness
            )
        else:
            nowcast = float("nan")

        ratio_to_median = (
            nowcast
            / historical_median
        )

        is_mature = (
            completeness
            >= min_completeness
        )

        signal_detected = (
            is_mature
            and ratio_to_median >= 1.15
        )

        rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "expected_completeness": completeness,
                "is_mature": is_mature,
                "nowcast": nowcast,
                "historical_median": historical_median,
                "ratio_to_median": ratio_to_median,
                "signal_detected": signal_detected,
            }
        )

    return pd.DataFrame(rows)

In [463]:
mg_2025_week_18_reference = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
    .loc[
        lambda df: df["epi_week"].eq(18)
    ]
    .iloc[0]
)

mg_2025_week_18_reference

epi_week                18.00
historical_median      659.00
historical_quantile    728.75
historical_years         4.00
Name: 17, dtype: float64

In [464]:
completion_model_mg = pd.DataFrame(
    {
        "lag_days": range(0, 29),
    }
)

completion_model_mg[
    "expected_completeness"
] = (
    completion_model_mg[
        "lag_days"
    ]
    .apply(
        lambda lag: (
            mg_2025_delay_check <= lag
        ).mean()
    )
)

completion_model_mg[
    "expected_completeness_pct"
] = (
    completion_model_mg[
        "expected_completeness"
    ]
    * 100
)

display(
    completion_model_mg.loc[
        completion_model_mg[
            "lag_days"
        ].isin(
            [0, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,expected_completeness_pct
0,0,0.066674,6.667397
7,7,0.803702,80.370168
14,14,0.920315,92.031541
21,21,0.949688,94.968788
28,28,0.959150,95.915015


In [465]:
MIN_COMPLETENESS = 0.80

mature_lag_mg = int(
    completion_model_mg.loc[
        completion_model_mg[
            "expected_completeness"
        ].ge(
            MIN_COMPLETENESS
        ),
        "lag_days",
    ].min()
)

print(
    "Primeiro lag maduro MG:",
    f"D+{mature_lag_mg}",
)

Primeiro lag maduro MG: D+7


In [466]:
mg_2025_week_18_calendar = ground_truth_base.loc[
    ground_truth_base["region"].eq("MG")
    & ground_truth_base["year"].eq(2025)
    & ground_truth_base["epi_week"].eq(18)
].iloc[0]

week_start = mg_2025_week_18_calendar["week_start"]
week_end = mg_2025_week_18_calendar["week_end"]

timeline_mg_2025_se18 = (
    build_week_nowcast_timeline(
        temporal_df=mg_2025_temporal,
        week_start=week_start,
        week_end=week_end,
        epi_week=18,
        historical_median=float(
            mg_2025_week_18_reference[
                "historical_median"
            ]
        ),
        completion_model=completion_model_mg,
        min_completeness=0.80,
    )
)

display(
    timeline_mg_2025_se18
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,18,0,2025-05-03,477,0.066674,False,7154.216491,659.0,10.856171,False
1,18,1,2025-05-04,582,0.182214,False,3194.038947,659.0,4.846797,False
2,18,2,2025-05-05,778,0.325244,False,2392.052663,659.0,3.629822,False
3,18,3,2025-05-06,957,0.470858,False,2032.461971,659.0,3.084161,False
4,18,4,2025-05-07,1096,0.589815,False,1858.210042,659.0,2.819742,False
5,18,5,2025-05-08,1204,0.677297,False,1777.654097,659.0,2.697502,False
6,18,6,2025-05-09,1280,0.740445,False,1728.691022,659.0,2.623203,False
7,18,7,2025-05-10,1306,0.803702,True,1624.981059,659.0,2.465829,True
8,18,8,2025-05-11,1315,0.838112,True,1569.002849,659.0,2.380884,True
9,18,9,2025-05-12,1349,0.859424,True,1569.656064,659.0,2.381876,True


In [467]:
completion_years_mg_2025 = [
    2019,
    2022,
    2023,
    2024,
]

In [468]:
delay_frames = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_mg_2025:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq("MG")
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                errors="coerce",
                dayfirst=True,
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df = df.loc[
        df["delay_days"].notna()
        & df["delay_days"].ge(0)
    ].copy()

    df["source_year"] = source_year

    delay_frames.append(
        df[
            [
                "source_year",
                "delay_days",
            ]
        ]
    )

delay_historical_mg_2025 = pd.concat(
    delay_frames,
    ignore_index=True,
)

/tmp/ipykernel_57211/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_57211/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_57211/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_57211/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_57211/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. 

In [469]:
display(
    delay_historical_mg_2025
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
            0.95,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,95%,max
source_year,,,,,,,,,,
2019,4035.0,7.745725,24.045400,0.0,5.0,8.0,9.0,14.0,20.0,753.0
2022,11843.0,105.824960,87.679130,0.0,90.0,155.0,180.0,238.0,276.0,949.0
2023,5388.0,108.791574,88.010105,0.0,92.0,175.0,183.0,240.0,271.0,700.0
2024,5972.0,106.882954,81.295476,0.0,93.0,155.0,180.0,229.9,266.0,572.0


In [470]:
def load_notification_delays(
    path: Path,
    region: str,
    source_year: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq(region)
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df["source_year"] = source_year

    return df

In [471]:
notification_delay_frames = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    delays = load_notification_delays(
        path=path,
        region="MG",
        source_year=source_year,
    )

    notification_delay_frames.append(
        delays
    )

notification_delays_all = pd.concat(
    notification_delay_frames,
    ignore_index=True,
)

valid_notification_delays = (
    notification_delays_all.loc[
        notification_delays_all["delay_days"].notna()
        & notification_delays_all["delay_days"].ge(0)
    ]
    .copy()
)

print(
    valid_notification_delays
    .groupby("source_year")["delay_days"]
    .median()
)

source_year
2019    5.0
2020    5.0
2021    7.0
2022    4.0
2023    3.0
2024    4.0
2025    4.0
2026    4.0
Name: delay_days, dtype: float64


In [472]:
valid_notification_delays = (
    notification_delays_all.loc[
        notification_delays_all[
            "delay_days"
        ].notna()
        & notification_delays_all[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [473]:
delay_evolution = (
    valid_notification_delays
    .groupby("source_year")
    .agg(
        records=(
            "delay_days",
            "size",
        ),
        median_delay=(
            "delay_days",
            "median",
        ),
        p75_delay=(
            "delay_days",
            lambda x: x.quantile(0.75),
        ),
        p80_delay=(
            "delay_days",
            lambda x: x.quantile(0.80),
        ),
        p90_delay=(
            "delay_days",
            lambda x: x.quantile(0.90),
        ),
        completeness_d7=(
            "delay_days",
            lambda x: x.le(7).mean(),
        ),
        completeness_d14=(
            "delay_days",
            lambda x: x.le(14).mean(),
        ),
        completeness_d28=(
            "delay_days",
            lambda x: x.le(28).mean(),
        ),
    )
    .reset_index()
)

display(delay_evolution)

,source_year,records,median_delay,p75_delay,p80_delay,p90_delay,completeness_d7,completeness_d14,completeness_d28
0,2019,4035,5.0,8.0,9.0,14.0,0.712763,0.910781,0.967286
1,2020,113781,5.0,9.0,10.0,14.0,0.681379,0.913931,0.974732
2,2021,209248,7.0,10.0,11.0,15.0,0.550280,0.897437,0.973811
3,2022,69332,4.0,7.0,8.0,11.0,0.799660,0.936898,0.976807
4,2023,25613,3.0,6.0,7.0,11.0,0.821146,0.929450,0.964784
5,2024,27271,4.0,7.0,7.0,12.0,0.808698,0.924865,0.965714
6,2025,45655,4.0,7.0,7.0,12.0,0.803702,0.920315,0.959150
7,2026,29087,4.0,6.0,7.0,11.0,0.818235,0.935882,0.975866


In [474]:
completion_years_mg_2025 = [
    2019,
    2022,
    2023,
    2024,
]

historical_delays_mg_2025 = (
    valid_notification_delays.loc[
        valid_notification_delays[
            "source_year"
        ].isin(completion_years_mg_2025)
    ]
    .copy()
)

In [475]:
for year, group in (
    historical_delays_mg_2025
    .groupby("source_year")
):
    print(
        year,
        "D+7:",
        round(
            group["delay_days"]
            .le(7)
            .mean(),
            4,
        ),
    )

2019 D+7: 0.7128
2022 D+7: 0.7997
2023 D+7: 0.8211
2024 D+7: 0.8087


In [476]:
completion_rows = []

for year, group in (
    historical_delays_mg_2025
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_mg_2025 = pd.DataFrame(
    completion_rows
)

completion_model_mg_2025 = (
    completion_by_year_mg_2025
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_mg_2025[
    "expected_completeness_pct"
] = (
    completion_model_mg_2025[
        "expected_completeness"
    ]
    * 100
)

In [477]:
display(
    completion_model_mg_2025.loc[
        completion_model_mg_2025[
            "lag_days"
        ].isin(
            [0, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.072077,0.035440,0.077848,7.207700
7,7,0.804179,0.712763,0.821146,80.417875
14,14,0.927158,0.910781,0.936898,92.715757
21,21,0.956770,0.956107,0.967749,95.676998
28,28,0.966500,0.964784,0.976807,96.650037


In [478]:
timeline_mg_2025_se18 = (
    build_week_nowcast_timeline(
        temporal_df=mg_2025_temporal,
        week_start=week_start,
        week_end=week_end,
        epi_week=18,
        historical_median=float(
            mg_2025_week_18_reference[
                "historical_median"
            ]
        ),
        completion_model=completion_model_mg_2025,
        min_completeness=0.80,
    )
)

display(timeline_mg_2025_se18)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,18,0,2025-05-03,477,0.072077,False,6617.922334,659.0,10.042371,False
1,18,1,2025-05-04,582,0.188388,False,3089.373974,659.0,4.687973,False
2,18,2,2025-05-05,778,0.328573,False,2367.818055,659.0,3.593047,False
3,18,3,2025-05-06,957,0.476074,False,2010.192921,659.0,3.050369,False
4,18,4,2025-05-07,1096,0.594256,False,1844.323528,659.0,2.798670,False
5,18,5,2025-05-08,1204,0.679427,False,1772.082618,659.0,2.689048,False
6,18,6,2025-05-09,1280,0.742235,False,1724.521114,659.0,2.616876,False
7,18,7,2025-05-10,1306,0.804179,True,1624.017055,659.0,2.464366,True
8,18,8,2025-05-11,1315,0.842383,True,1561.047624,659.0,2.368813,True
9,18,9,2025-05-12,1349,0.866479,True,1556.876221,659.0,2.362483,True


In [479]:
first_signal_mg_2025_se18 = (
    timeline_mg_2025_se18.loc[
        timeline_mg_2025_se18[
            "signal_detected"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

display(first_signal_mg_2025_se18)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
7,18,7,2025-05-10,1306,0.804179,True,1624.017055,659.0,2.464366,True


In [480]:
if not first_signal_mg_2025_se18.empty:
    row = first_signal_mg_2025_se18.iloc[0]

    print(
        "Primeiro SIGNAL operacional:",
        row["snapshot_date"].date(),
    )

    print(
        "Lag:",
        f"D+{int(row['lag_days'])}",
    )

    print(
        "Nowcast:",
        round(row["nowcast"], 1),
    )

    print(
        "Razão para mediana:",
        round(row["ratio_to_median"], 3),
    )

Primeiro SIGNAL operacional: 2025-05-10
Lag: D+7
Nowcast: 1624.0
Razão para mediana: 2.464


In [481]:
weeks_mg_2025 = pd.DataFrame(
    {
        "epi_week": range(17, 26),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_mg_2025["week_start"] = (
    first_week_start
    + pd.to_timedelta(
        (weeks_mg_2025["epi_week"] - 1) * 7,
        unit="D",
    )
)

weeks_mg_2025["week_end"] = (
    weeks_mg_2025["week_start"]
    + pd.Timedelta(days=6)
)

display(weeks_mg_2025)

,epi_week,week_start,week_end
0,17,2025-04-20,2025-04-26
1,18,2025-04-27,2025-05-03
2,19,2025-05-04,2025-05-10
3,20,2025-05-11,2025-05-17
4,21,2025-05-18,2025-05-24
5,22,2025-05-25,2025-05-31
6,23,2025-06-01,2025-06-07
7,24,2025-06-08,2025-06-14
8,25,2025-06-15,2025-06-21


In [482]:
reference_mg_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
)

reference_mg_2025 = (
    reference_mg_2025.loc[
        reference_mg_2025[
            "epi_week"
        ].between(17, 25)
    ]
    .copy()
)

display(reference_mg_2025)

,epi_week,historical_median,historical_quantile,historical_years
16,17,600.0,693.00,4
17,18,659.0,728.75,4
18,19,632.0,777.50,4
19,20,670.0,854.00,4
20,21,654.5,880.75,4
21,22,598.0,884.50,4
22,23,582.0,903.75,4
23,24,544.0,910.50,4
24,25,588.5,973.00,4


In [483]:
timeline_frames = []

for week in weeks_mg_2025.itertuples(
    index=False
):
    reference_row = (
        reference_mg_2025.loc[
            reference_mg_2025[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=mg_2025_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_mg_2025
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames.append(timeline)

timeline_mg_2025 = pd.concat(
    timeline_frames,
    ignore_index=True,
)

display(
    timeline_mg_2025.head()
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,17,0,2025-04-26,412,0.072077,False,5716.109018,600.0,9.526848,False
1,17,1,2025-04-27,483,0.188388,False,2563.861906,600.0,4.273103,False
2,17,2,2025-04-28,636,0.328573,False,1935.645608,600.0,3.226076,False
3,17,3,2025-04-29,727,0.476074,False,1527.074455,600.0,2.545124,False
4,17,4,2025-04-30,800,0.594256,False,1346.221553,600.0,2.243703,False


In [484]:
first_signal_by_week = (
    timeline_mg_2025.loc[
        timeline_mg_2025[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,17,7,2025-05-03,879,1093.040576,600.0,1.821734
1,18,7,2025-05-10,1306,1624.017055,659.0,2.464366
2,19,7,2025-05-17,1642,2041.834613,632.0,3.230751
3,20,7,2025-05-24,1908,2372.606847,670.0,3.541204
4,21,7,2025-05-31,2098,2608.872728,654.5,3.986055
5,22,7,2025-06-07,1945,2418.616518,598.0,4.044509
6,23,7,2025-06-14,2007,2495.713806,582.0,4.288168
7,24,7,2025-06-21,1608,1999.555456,544.0,3.675653
8,25,7,2025-06-28,1488,1850.334899,588.5,3.144154


In [485]:
weekly_signal_summary = (
    weeks_mg_2025[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary[
    "has_signal"
] = (
    weekly_signal_summary[
        "snapshot_date"
    ].notna()
)

display(weekly_signal_summary)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal
0,17,2025-04-20,2025-04-26,2025-05-03,7,1.821734,True
1,18,2025-04-27,2025-05-03,2025-05-10,7,2.464366,True
2,19,2025-05-04,2025-05-10,2025-05-17,7,3.230751,True
3,20,2025-05-11,2025-05-17,2025-05-24,7,3.541204,True
4,21,2025-05-18,2025-05-24,2025-05-31,7,3.986055,True
5,22,2025-05-25,2025-05-31,2025-06-07,7,4.044509,True
6,23,2025-06-01,2025-06-07,2025-06-14,7,4.288168,True
7,24,2025-06-08,2025-06-14,2025-06-21,7,3.675653,True
8,25,2025-06-15,2025-06-21,2025-06-28,7,3.144154,True


In [486]:
weekly_signal_summary[
    "previous_week_signal"
] = (
    weekly_signal_summary[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary[
    "alert_candidate"
] = (
    weekly_signal_summary[
        "has_signal"
    ]
    & weekly_signal_summary[
        "previous_week_signal"
    ]
)

In [487]:
weekly_signal_summary[
    "previous_signal_date"
] = (
    weekly_signal_summary[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary[
        "alert_candidate"
    ]
)

weekly_signal_summary.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

In [488]:
display(
    weekly_signal_summary[
        [
            "epi_week",
            "has_signal",
            "snapshot_date",
            "previous_week_signal",
            "alert_candidate",
            "alert_date",
        ]
    ]
)

,epi_week,has_signal,snapshot_date,previous_week_signal,alert_candidate,alert_date
0,17,True,2025-05-03,False,False,NaT
1,18,True,2025-05-10,True,True,2025-05-10
2,19,True,2025-05-17,True,True,2025-05-17
3,20,True,2025-05-24,True,True,2025-05-24
4,21,True,2025-05-31,True,True,2025-05-31
5,22,True,2025-06-07,True,True,2025-06-07
6,23,True,2025-06-14,True,True,2025-06-14
7,24,True,2025-06-21,True,True,2025-06-21
8,25,True,2025-06-28,True,True,2025-06-28


In [489]:
first_signal = (
    weekly_signal_summary.loc[
        weekly_signal_summary[
            "has_signal"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

In [490]:
first_alert = (
    weekly_signal_summary.loc[
        weekly_signal_summary[
            "alert_candidate"
        ]
    ]
    .sort_values("alert_date")
    .head(1)
)

In [491]:
print("=== PRIMEIRO SIGNAL ===")

display(
    first_signal[
        [
            "epi_week",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
        ]
    ]
)

print("\n=== PRIMEIRO ALERT ===")

display(
    first_alert[
        [
            "epi_week",
            "alert_date",
        ]
    ]
)

=== PRIMEIRO SIGNAL ===


,epi_week,snapshot_date,lag_days,ratio_to_median
0,17,2025-05-03,7,1.821734



=== PRIMEIRO ALERT ===


,epi_week,alert_date
1,18,2025-05-10


In [492]:
official_events = {
    "Decreto estadual": pd.Timestamp(
        "2025-05-02"
    ),
    "Ativação do COE": pd.Timestamp(
        "2025-05-08"
    ),
}

In [493]:
first_signal_date = (
    pd.Timestamp(
        first_signal[
            "snapshot_date"
        ].iloc[0]
    )
    if not first_signal.empty
    else pd.NaT
)

first_alert_date = (
    pd.Timestamp(
        first_alert[
            "alert_date"
        ].iloc[0]
    )
    if not first_alert.empty
    else pd.NaT
)

In [494]:
lead_time_rows = []

for event_name, event_date in (
    official_events.items()
):
    lead_time_rows.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "SIGNAL",
            "sentinela_date": first_signal_date,
            "lead_days": (
                event_date
                - first_signal_date
            ).days,
        }
    )

    lead_time_rows.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "ALERT",
            "sentinela_date": first_alert_date,
            "lead_days": (
                event_date
                - first_alert_date
            ).days,
        }
    )

lead_time_mg_2025 = pd.DataFrame(
    lead_time_rows
)

display(lead_time_mg_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days
0,Decreto estadual,2025-05-02,SIGNAL,2025-05-03,-1
1,Decreto estadual,2025-05-02,ALERT,2025-05-10,-8
2,Ativação do COE,2025-05-08,SIGNAL,2025-05-03,5
3,Ativação do COE,2025-05-08,ALERT,2025-05-10,-2


## 24. Lead time pseudo-prospectivo — MG/2025

A reconstrução pseudo-prospectiva utilizou exclusivamente informações
que estariam disponíveis até cada data de snapshot.

Foram mantidos fixos:

- baseline histórico anterior a 2025;
- exclusão dos anos pandêmicos do baseline epidemiológico;
- razão observado/mediana >= 1,15;
- completude mínima de 80%;
- duas semanas consecutivas para confirmação de `ALERT`.

### Resultado

A SE 17 atingiu os critérios operacionais em 03/05/2025,
quando alcançou maturidade suficiente dos dados.

Esse foi o primeiro `SIGNAL` pseudo-prospectivo.

A SE 18 atingiu os mesmos critérios em 10/05/2025.
Como a SE 17 também permanecia sinalizada, essa data representa a
primeira confirmação de `ALERT`.

### Comparação com respostas institucionais

- Decreto estadual de emergência: 02/05/2025
  - SIGNAL: 1 dia depois;
  - ALERT: 8 dias depois.

- Ativação do Centro de Operações de Emergência: 08/05/2025
  - SIGNAL: 5 dias antes;
  - ALERT: 2 dias depois.

### Interpretação

O estado `SIGNAL` apresentou capacidade de detecção próxima ao primeiro
marco institucional e antecedeu a ativação do COE.

O estado `ALERT`, por exigir persistência em duas semanas epidemiológicas
consecutivas, apresentou atraso adicional esperado.

Esses resultados apoiam a utilização de `SIGNAL` como mecanismo de
detecção precoce e `ALERT` como confirmação operacional.

In [495]:
lead_time_mg_2025["interpretation"] = (
    lead_time_mg_2025["lead_days"]
    .apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else f"{abs(days)} dias depois"
            )
        )
    )
)

display(lead_time_mg_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,Decreto estadual,2025-05-02,SIGNAL,2025-05-03,-1,1 dias depois
1,Decreto estadual,2025-05-02,ALERT,2025-05-10,-8,8 dias depois
2,Ativação do COE,2025-05-08,SIGNAL,2025-05-03,5,5 dias antes
3,Ativação do COE,2025-05-08,ALERT,2025-05-10,-2,2 dias depois


In [496]:
completion_years_sp_2025 = [
    2019,
    2022,
    2023,
    2024,
]

In [497]:
def load_notification_delays_sp(
    path: Path,
    source_year: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq("SP")
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df["source_year"] = source_year

    return df

In [498]:
delay_frames_sp = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_sp_2025:
        continue

    delays = load_notification_delays_sp(
        path=path,
        source_year=source_year,
    )

    delay_frames_sp.append(delays)

historical_delays_sp_2025 = pd.concat(
    delay_frames_sp,
    ignore_index=True,
)

historical_delays_sp_2025 = (
    historical_delays_sp_2025.loc[
        historical_delays_sp_2025[
            "delay_days"
        ].notna()
        & historical_delays_sp_2025[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [499]:
display(
    historical_delays_sp_2025
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,max
source_year,,,,,,,,,
2019,12331.0,9.048009,19.966451,0.0,5.0,8.0,10.0,17.0,597.0
2022,160930.0,7.628292,21.866400,0.0,4.0,8.0,9.0,14.0,1104.0
2023,78490.0,7.521506,21.935172,0.0,4.0,7.0,8.0,13.0,750.0
2024,69006.0,8.874025,22.105549,0.0,4.0,7.0,8.0,14.0,445.0


In [500]:
completion_rows_sp = []

for year, group in (
    historical_delays_sp_2025
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_sp.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_sp_2025 = pd.DataFrame(
    completion_rows_sp
)

completion_model_sp_2025 = (
    completion_by_year_sp_2025
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_sp_2025[
    "expected_completeness_pct"
] = (
    completion_model_sp_2025[
        "expected_completeness"
    ]
    * 100
)

In [501]:
display(
    completion_model_sp_2025.loc[
        completion_model_sp_2025[
            "lag_days"
        ].isin(
            [0, 7, 8, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.072473,0.032682,0.076073,7.247270
7,7,0.753612,0.722001,0.786024,75.361182
8,8,0.796093,0.763523,0.824729,79.609336
14,14,0.905568,0.875193,0.919098,90.556751
21,21,0.946681,0.924337,0.955752,94.668143
28,28,0.961084,0.944692,0.969308,96.108387


In [502]:
mature_rows_sp = (
    completion_model_sp_2025.loc[
        completion_model_sp_2025[
            "expected_completeness"
        ].ge(0.80)
    ]
)

mature_lag_sp = int(
    mature_rows_sp[
        "lag_days"
    ].min()
)

print(
    "Primeiro lag maduro SP/2025:",
    f"D+{mature_lag_sp}",
)

Primeiro lag maduro SP/2025: D+9


In [503]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

df_sp_2025_temporal = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_sp_2025_temporal["SG_UF"] = (
    df_sp_2025_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

sp_2025_temporal = (
    df_sp_2025_temporal.loc[
        df_sp_2025_temporal[
            "SG_UF"
        ].eq("SP")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    sp_2025_temporal[column] = (
        pd.to_datetime(
            sp_2025_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

sp_2025_temporal = (
    sp_2025_temporal.loc[
        sp_2025_temporal[
            "DT_SIN_PRI"
        ].notna()
        & sp_2025_temporal[
            "DT_NOTIFIC"
        ].notna()
    ]
    .copy()
)

print(
    "Registros SP/2025:",
    len(sp_2025_temporal),
)

Registros SP/2025: 79295


In [504]:
weeks_sp_2025 = pd.DataFrame(
    {
        "epi_week": range(16, 27),
    }
)

first_week_start_sp_2025 = pd.Timestamp(
    "2024-12-29"
)

weeks_sp_2025["week_start"] = (
    first_week_start_sp_2025
    + pd.to_timedelta(
        (
            weeks_sp_2025["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2025["week_end"] = (
    weeks_sp_2025["week_start"]
    + pd.Timedelta(days=6)
)

In [505]:
reference_sp_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
)

In [506]:
timeline_frames_sp = []

for week in weeks_sp_2025.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2025.loc[
            reference_sp_2025[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=sp_2025_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_sp_2025
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames_sp.append(timeline)

timeline_sp_2025 = pd.concat(
    timeline_frames_sp,
    ignore_index=True,
)

In [507]:
first_signal_by_week_sp = (
    timeline_sp_2025.loc[
        timeline_sp_2025[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week_sp[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,16,9,2025-04-28,1715,2077.494511,1796.0,1.156734
1,17,9,2025-05-05,2013,2438.481895,1711.5,1.424763
2,18,9,2025-05-12,2144,2597.170980,1755.0,1.479870
3,19,9,2025-05-19,2338,2832.176190,1592.5,1.778447
4,20,9,2025-05-26,2749,3330.048053,1676.0,1.986902
5,21,9,2025-06-02,2815,3409.998279,1618.5,2.106888
6,22,9,2025-06-09,2511,3041.742692,1531.5,1.986120
7,23,9,2025-06-16,2504,3033.263123,1573.5,1.927717
8,24,9,2025-06-23,2110,2555.984500,1530.5,1.670032
9,25,9,2025-06-30,2198,2662.584802,1607.5,1.656351


In [508]:
weekly_signal_summary_sp = (
    weeks_sp_2025[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week_sp[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary_sp[
    "has_signal"
] = (
    weekly_signal_summary_sp[
        "snapshot_date"
    ].notna()
)

weekly_signal_summary_sp[
    "previous_week_signal"
] = (
    weekly_signal_summary_sp[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary_sp[
    "alert_candidate"
] = (
    weekly_signal_summary_sp[
        "has_signal"
    ]
    & weekly_signal_summary_sp[
        "previous_week_signal"
    ]
)

weekly_signal_summary_sp[
    "previous_signal_date"
] = (
    weekly_signal_summary_sp[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary_sp[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary_sp[
        "alert_candidate"
    ]
)

weekly_signal_summary_sp.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary_sp.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

display(weekly_signal_summary_sp)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,16,2025-04-13,2025-04-19,2025-04-28,9,1.156734,True,False,False,NaT,NaT
1,17,2025-04-20,2025-04-26,2025-05-05,9,1.424763,True,True,True,2025-04-28,2025-05-05
2,18,2025-04-27,2025-05-03,2025-05-12,9,1.479870,True,True,True,2025-05-05,2025-05-12
3,19,2025-05-04,2025-05-10,2025-05-19,9,1.778447,True,True,True,2025-05-12,2025-05-19
4,20,2025-05-11,2025-05-17,2025-05-26,9,1.986902,True,True,True,2025-05-19,2025-05-26
5,21,2025-05-18,2025-05-24,2025-06-02,9,2.106888,True,True,True,2025-05-26,2025-06-02
6,22,2025-05-25,2025-05-31,2025-06-09,9,1.986120,True,True,True,2025-06-02,2025-06-09
7,23,2025-06-01,2025-06-07,2025-06-16,9,1.927717,True,True,True,2025-06-09,2025-06-16
8,24,2025-06-08,2025-06-14,2025-06-23,9,1.670032,True,True,True,2025-06-16,2025-06-23
9,25,2025-06-15,2025-06-21,2025-06-30,9,1.656351,True,True,True,2025-06-23,2025-06-30


In [509]:
first_signal_sp = (
    weekly_signal_summary_sp.loc[
        weekly_signal_summary_sp[
            "has_signal"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

first_alert_sp = (
    weekly_signal_summary_sp.loc[
        weekly_signal_summary_sp[
            "alert_candidate"
        ]
    ]
    .sort_values("alert_date")
    .head(1)
)

print("=== PRIMEIRO SIGNAL SP/2025 ===")
display(first_signal_sp)

print("\n=== PRIMEIRO ALERT SP/2025 ===")
display(first_alert_sp)

=== PRIMEIRO SIGNAL SP/2025 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,16,2025-04-13,2025-04-19,2025-04-28,9,1.156734,True,False,False,NaT,NaT



=== PRIMEIRO ALERT SP/2025 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
1,17,2025-04-20,2025-04-26,2025-05-05,9,1.424763,True,True,True,2025-04-28,2025-05-05


In [510]:
first_signal_sp_date = pd.Timestamp(
    first_signal_sp[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_date = pd.Timestamp(
    first_alert_sp[
        "alert_date"
    ].iloc[0]
)

print(
    "Primeiro SIGNAL SP/2025:",
    first_signal_sp_date.date(),
)

print(
    "Primeiro ALERT SP/2025:",
    first_alert_sp_date.date(),
)

Primeiro SIGNAL SP/2025: 2025-04-28
Primeiro ALERT SP/2025: 2025-05-05


## 25. Reconstrução pseudo-prospectiva — SP/2025

Mantendo os mesmos parâmetros definidos anteriormente:

- razão observado/mediana >= 1,15;
- completude mínima de 80%;
- `ALERT` após duas semanas consecutivas com `SIGNAL`;
- baseline construído apenas com anos anteriores ao ano avaliado;
- sem recalibração específica para São Paulo.

### Resultado

O primeiro `SIGNAL` operacional ocorreu em 28/04/2025, referente à SE 16.

Naquele momento:

- lag de maturidade: D+9;
- razão nowcast/mediana histórica: 1,157.

A SE 17 atingiu o critério operacional em 05/05/2025.

Como a SE 16 também havia sido sinalizada, essa foi a primeira data de
confirmação do estado `ALERT`.

### Interpretação preliminar

O sistema detectou uma elevação ainda durante a fase de transição
epidemiológica, antes do início da janela positiva forte utilizada no
ground truth externo.

Esse resultado sugere capacidade de antecipação do episódio, sem alteração
dos parâmetros previamente definidos.

In [511]:
official_events_sp_2025 = {
    "Incentivo excepcional para SRAG":
        pd.Timestamp("2025-05-06"),

    "Boletim estadual atualizado até SE20":
        pd.Timestamp("2025-05-12"),
}

In [512]:
first_signal_sp_date = pd.Timestamp(
    first_signal_sp[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_date = pd.Timestamp(
    first_alert_sp[
        "alert_date"
    ].iloc[0]
)

lead_time_rows_sp = []

for event_name, event_date in (
    official_events_sp_2025.items()
):
    lead_time_rows_sp.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "SIGNAL",
            "sentinela_date":
                first_signal_sp_date,
            "lead_days": (
                event_date
                - first_signal_sp_date
            ).days,
        }
    )

    lead_time_rows_sp.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "ALERT",
            "sentinela_date":
                first_alert_sp_date,
            "lead_days": (
                event_date
                - first_alert_sp_date
            ).days,
        }
    )

lead_time_sp_2025 = pd.DataFrame(
    lead_time_rows_sp
)

lead_time_sp_2025["interpretation"] = (
    lead_time_sp_2025[
        "lead_days"
    ].apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else (
                    f"{abs(days)} dias depois"
                )
            )
        )
    )
)

display(lead_time_sp_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,Incentivo excepcional para SRAG,2025-05-06,SIGNAL,2025-04-28,8,8 dias antes
1,Incentivo excepcional para SRAG,2025-05-06,ALERT,2025-05-05,1,1 dias antes
2,Boletim estadual atualizado até SE20,2025-05-12,SIGNAL,2025-04-28,14,14 dias antes
3,Boletim estadual atualizado até SE20,2025-05-12,ALERT,2025-05-05,7,7 dias antes


### Evidência preliminar de capacidade de alerta precoce

Na reconstrução pseudo-prospectiva de SP/2025, o Sentinela teria emitido
seu primeiro `SIGNAL` em 28/04/2025 e confirmado `ALERT` em 05/05/2025.

Essas datas antecedem marcos externos utilizados como referência:

- o `SIGNAL` antecedeu em 8 dias o incentivo excepcional relacionado à SRAG;
- o `ALERT` antecedeu esse marco em 1 dia;
- em relação ao boletim estadual de 12/05, o `SIGNAL` apresentou 14 dias
  de antecedência e o `ALERT`, 7 dias.

Os parâmetros não foram recalibrados especificamente para SP/2025.

Esse resultado fornece evidência preliminar de que o modelo pode identificar
uma elevação epidemiológica antes de sua consolidação em marcos externos,
embora novas regiões, anos e episódios ainda sejam necessários para estimar
formalmente a capacidade de alerta precoce.

In [513]:
completion_years_sp_2026 = [
    2019,
    2022,
    2023,
    2024,
    2025,
]

In [514]:
delay_frames_sp_2026 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_sp_2026:
        continue

    delays = load_notification_delays_sp(
        path=path,
        source_year=source_year,
    )

    delay_frames_sp_2026.append(delays)

historical_delays_sp_2026 = pd.concat(
    delay_frames_sp_2026,
    ignore_index=True,
)

historical_delays_sp_2026 = (
    historical_delays_sp_2026.loc[
        historical_delays_sp_2026[
            "delay_days"
        ].notna()
        & historical_delays_sp_2026[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [515]:
display(
    historical_delays_sp_2026
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,max
source_year,,,,,,,,,
2019,12331.0,9.048009,19.966451,0.0,5.0,8.0,10.0,17.0,597.0
2022,160930.0,7.628292,21.866400,0.0,4.0,8.0,9.0,14.0,1104.0
2023,78490.0,7.521506,21.935172,0.0,4.0,7.0,8.0,13.0,750.0
2024,69006.0,8.874025,22.105549,0.0,4.0,7.0,8.0,14.0,445.0
2025,79295.0,6.997730,16.288858,0.0,4.0,7.0,8.0,12.0,573.0


In [516]:
completion_rows_sp_2026 = []

for year, group in (
    historical_delays_sp_2026
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_sp_2026.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_sp_2026 = pd.DataFrame(
    completion_rows_sp_2026
)

completion_model_sp_2026 = (
    completion_by_year_sp_2026
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_sp_2026[
    "expected_completeness_pct"
] = (
    completion_model_sp_2026[
        "expected_completeness"
    ]
    * 100
)

In [517]:
display(
    completion_model_sp_2026.loc[
        completion_model_sp_2026[
            "lag_days"
        ].isin(
            [0, 7, 8, 9, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.070690,0.032682,0.076073,7.068951
7,7,0.759905,0.722001,0.791525,75.990494
8,8,0.802278,0.763523,0.829876,80.227806
9,9,0.831609,0.793123,0.855174,83.160885
14,14,0.905866,0.875193,0.921912,90.586590
21,21,0.951532,0.924337,0.956441,95.153172
28,28,0.967265,0.944692,0.969308,96.726527


In [518]:
mature_rows_sp_2026 = (
    completion_model_sp_2026.loc[
        completion_model_sp_2026[
            "expected_completeness"
        ].ge(0.80)
    ]
)

mature_lag_sp_2026 = int(
    mature_rows_sp_2026[
        "lag_days"
    ].min()
)

print(
    "Primeiro lag maduro SP/2026:",
    f"D+{mature_lag_sp_2026}",
)

Primeiro lag maduro SP/2026: D+8


In [519]:
path_2026 = next(
    path
    for path in historical_files
    if "INFLUD26" in path.name.upper()
)

df_sp_2026_temporal = pd.read_csv(
    path_2026,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_sp_2026_temporal["SG_UF"] = (
    df_sp_2026_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

sp_2026_temporal = (
    df_sp_2026_temporal.loc[
        df_sp_2026_temporal[
            "SG_UF"
        ].eq("SP")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    sp_2026_temporal[column] = (
        pd.to_datetime(
            sp_2026_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

sp_2026_temporal = (
    sp_2026_temporal.loc[
        sp_2026_temporal[
            "DT_SIN_PRI"
        ].notna()
        & sp_2026_temporal[
            "DT_NOTIFIC"
        ].notna()
    ]
    .copy()
)

print(
    "Registros SP/2026:",
    len(sp_2026_temporal),
)

Registros SP/2026: 48978


In [520]:
weeks_sp_2026 = pd.DataFrame(
    {
        "epi_week": range(18, 31),
    }
)

first_week_start_sp_2026 = pd.Timestamp(
    "2025-12-28"
)

weeks_sp_2026["week_start"] = (
    first_week_start_sp_2026
    + pd.to_timedelta(
        (
            weeks_sp_2026["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2026["week_end"] = (
    weeks_sp_2026["week_start"]
    + pd.Timedelta(days=6)
)

display(weeks_sp_2026)

,epi_week,week_start,week_end
0,18,2026-04-26,2026-05-02
1,19,2026-05-03,2026-05-09
2,20,2026-05-10,2026-05-16
3,21,2026-05-17,2026-05-23
4,22,2026-05-24,2026-05-30
5,23,2026-05-31,2026-06-06
6,24,2026-06-07,2026-06-13
7,25,2026-06-14,2026-06-20
8,26,2026-06-21,2026-06-27
9,27,2026-06-28,2026-07-04


In [521]:
reference_sp_2026 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
            2025,
        ],
        quantile=0.75,
    )
)

In [522]:
timeline_frames_sp_2026 = []

for week in weeks_sp_2026.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2026.loc[
            reference_sp_2026[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=sp_2026_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_sp_2026
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames_sp_2026.append(
        timeline
    )

timeline_sp_2026 = pd.concat(
    timeline_frames_sp_2026,
    ignore_index=True,
)

In [523]:
first_signal_by_week_sp_2026 = (
    timeline_sp_2026.loc[
        timeline_sp_2026[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week_sp_2026[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,19,8,2026-05-17,1562,1946.955890,1688.0,1.153410
1,20,8,2026-05-24,1932,2408.142625,1869.0,1.288466
2,21,8,2026-05-31,2045,2548.991547,1872.0,1.361641
3,22,8,2026-06-07,1968,2453.014848,1692.0,1.449772
4,23,8,2026-06-14,2116,2637.489542,1574.0,1.675660
5,24,8,2026-06-21,2007,2501.626422,1629.0,1.535682
6,25,8,2026-06-28,1972,2458.000650,1637.0,1.501528
7,26,8,2026-07-05,1880,2343.327192,1527.0,1.534595
8,27,8,2026-07-12,1800,2243.611141,1457.0,1.539884
9,28,8,2026-07-19,1550,1931.998483,1206.0,1.601989


In [524]:
weekly_signal_summary_sp_2026 = (
    weeks_sp_2026[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week_sp_2026[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary_sp_2026[
    "has_signal"
] = (
    weekly_signal_summary_sp_2026[
        "snapshot_date"
    ].notna()
)

weekly_signal_summary_sp_2026[
    "previous_week_signal"
] = (
    weekly_signal_summary_sp_2026[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary_sp_2026[
    "alert_candidate"
] = (
    weekly_signal_summary_sp_2026[
        "has_signal"
    ]
    & weekly_signal_summary_sp_2026[
        "previous_week_signal"
    ]
)

weekly_signal_summary_sp_2026[
    "previous_signal_date"
] = (
    weekly_signal_summary_sp_2026[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary_sp_2026[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary_sp_2026[
        "alert_candidate"
    ]
)

weekly_signal_summary_sp_2026.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary_sp_2026.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

display(
    weekly_signal_summary_sp_2026
)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,18,2026-04-26,2026-05-02,NaT,NaN,NaN,False,False,False,NaT,NaT
1,19,2026-05-03,2026-05-09,2026-05-17,8.0,1.153410,True,False,False,NaT,NaT
2,20,2026-05-10,2026-05-16,2026-05-24,8.0,1.288466,True,True,True,2026-05-17,2026-05-24
3,21,2026-05-17,2026-05-23,2026-05-31,8.0,1.361641,True,True,True,2026-05-24,2026-05-31
4,22,2026-05-24,2026-05-30,2026-06-07,8.0,1.449772,True,True,True,2026-05-31,2026-06-07
5,23,2026-05-31,2026-06-06,2026-06-14,8.0,1.675660,True,True,True,2026-06-07,2026-06-14
6,24,2026-06-07,2026-06-13,2026-06-21,8.0,1.535682,True,True,True,2026-06-14,2026-06-21
7,25,2026-06-14,2026-06-20,2026-06-28,8.0,1.501528,True,True,True,2026-06-21,2026-06-28
8,26,2026-06-21,2026-06-27,2026-07-05,8.0,1.534595,True,True,True,2026-06-28,2026-07-05
9,27,2026-06-28,2026-07-04,2026-07-12,8.0,1.539884,True,True,True,2026-07-05,2026-07-12


In [525]:
first_signal_sp_2026 = (
    weekly_signal_summary_sp_2026.loc[
        weekly_signal_summary_sp_2026[
            "has_signal"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(1)
)

first_alert_sp_2026 = (
    weekly_signal_summary_sp_2026.loc[
        weekly_signal_summary_sp_2026[
            "alert_candidate"
        ]
    ]
    .sort_values(
        "alert_date"
    )
    .head(1)
)

print(
    "=== PRIMEIRO SIGNAL SP/2026 ==="
)

display(first_signal_sp_2026)

print(
    "\n=== PRIMEIRO ALERT SP/2026 ==="
)

display(first_alert_sp_2026)

=== PRIMEIRO SIGNAL SP/2026 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
1,19,2026-05-03,2026-05-09,2026-05-17,8.0,1.15341,True,False,False,NaT,NaT



=== PRIMEIRO ALERT SP/2026 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
2,20,2026-05-10,2026-05-16,2026-05-24,8.0,1.288466,True,True,True,2026-05-17,2026-05-24


In [526]:
official_events_sp_2026 = {
    "InfoGripe indica crescimento em SP":
        pd.Timestamp("2026-05-14"),

    "Ampliação vacinação influenza SP":
        pd.Timestamp("2026-05-18"),

    "Informe nacional SE20":
        pd.Timestamp("2026-05-23"),
}

In [527]:
first_signal_sp_2026_date = pd.Timestamp(
    first_signal_sp_2026[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_2026_date = pd.Timestamp(
    first_alert_sp_2026[
        "alert_date"
    ].iloc[0]
)

lead_time_rows_sp_2026 = []

for event_name, event_date in (
    official_events_sp_2026.items()
):
    for state, sentinela_date in [
        (
            "SIGNAL",
            first_signal_sp_2026_date,
        ),
        (
            "ALERT",
            first_alert_sp_2026_date,
        ),
    ]:
        lead_days = (
            event_date
            - sentinela_date
        ).days

        lead_time_rows_sp_2026.append(
            {
                "reference_event":
                    event_name,
                "official_date":
                    event_date,
                "sentinela_state":
                    state,
                "sentinela_date":
                    sentinela_date,
                "lead_days":
                    lead_days,
            }
        )

lead_time_sp_2026 = pd.DataFrame(
    lead_time_rows_sp_2026
)

lead_time_sp_2026[
    "interpretation"
] = (
    lead_time_sp_2026[
        "lead_days"
    ].apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else (
                    f"{abs(days)} dias depois"
                )
            )
        )
    )
)

display(lead_time_sp_2026)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,InfoGripe indica crescimento em SP,2026-05-14,SIGNAL,2026-05-17,-3,3 dias depois
1,InfoGripe indica crescimento em SP,2026-05-14,ALERT,2026-05-24,-10,10 dias depois
2,Ampliação vacinação influenza SP,2026-05-18,SIGNAL,2026-05-17,1,1 dias antes
3,Ampliação vacinação influenza SP,2026-05-18,ALERT,2026-05-24,-6,6 dias depois
4,Informe nacional SE20,2026-05-23,SIGNAL,2026-05-17,6,6 dias antes
5,Informe nacional SE20,2026-05-23,ALERT,2026-05-24,-1,1 dias depois


In [528]:
completion_uncertainty_sp_2026 = (
    completion_by_year_sp_2026
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=lambda x: x.quantile(0.25),
        completeness_q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

display(
    completion_uncertainty_sp_2026.loc[
        completion_uncertainty_sp_2026[
            "lag_days"
        ].between(0, 14)
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.070690,0.070496,0.074256
1,1,0.182839,0.181824,0.187654
2,2,0.303567,0.299177,0.319289
3,3,0.433195,0.422731,0.456132
4,4,0.541329,0.535417,0.576165
5,5,0.626974,0.623395,0.664607
6,6,0.697881,0.687535,0.729481
7,7,0.759905,0.747319,0.786024
8,8,0.802278,0.789909,0.824729
9,9,0.831609,0.819418,0.850529


In [529]:
def build_uncertainty_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    ratio_threshold: float = 1.15,
    mature_completeness: float = 0.80,
    min_early_completeness: float = 0.30,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(max_lag + 1):
        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue

        expected = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        q75 = float(
            completion_row[
                "completeness_q75"
            ].iloc[0]
        )

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        if expected <= 0 or q75 <= 0:
            continue

        nowcast = (
            known_cases
            / expected
        )

        conservative_nowcast = (
            known_cases
            / q75
        )

        ratio = (
            nowcast
            / historical_median
        )

        conservative_ratio = (
            conservative_nowcast
            / historical_median
        )

        is_mature = (
            expected
            >= mature_completeness
        )

        has_minimum_information = (
            expected
            >= min_early_completeness
        )

        # Regra:
        # - se maduro, usa estimativa central;
        # - se ainda imaturo, exige que até
        #   o cenário conservador ultrapasse
        #   o threshold.
        if is_mature:
            signal_detected = (
                ratio >= ratio_threshold
            )

            signal_mode = (
                "MATURE"
                if signal_detected
                else "NONE"
            )

        elif has_minimum_information:
            signal_detected = (
                conservative_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "EARLY_CONSERVATIVE"
                if signal_detected
                else "NONE"
            )

        else:
            signal_detected = False
            signal_mode = "INSUFFICIENT_DATA"

        rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "expected_completeness": expected,
                "completeness_q75": q75,
                "is_mature": is_mature,
                "nowcast": nowcast,
                "conservative_nowcast":
                    conservative_nowcast,
                "historical_median":
                    historical_median,
                "ratio_to_median": ratio,
                "conservative_ratio":
                    conservative_ratio,
                "signal_detected":
                    signal_detected,
                "signal_mode":
                    signal_mode,
            }
        )

    return pd.DataFrame(rows)

In [530]:
week_19 = weeks_sp_2026.loc[
    weeks_sp_2026[
        "epi_week"
    ].eq(19)
].iloc[0]

reference_week_19 = (
    reference_sp_2026.loc[
        reference_sp_2026[
            "epi_week"
        ].eq(19),
        "historical_median",
    ]
    .iloc[0]
)

In [531]:
timeline_sp_2026_se19_v2 = (
    build_uncertainty_nowcast_timeline(
        temporal_df=sp_2026_temporal,
        week_start=week_19["week_start"],
        week_end=week_19["week_end"],
        epi_week=19,
        historical_median=float(
            reference_week_19
        ),
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ratio_threshold=1.15,
        mature_completeness=0.80,
        min_early_completeness=0.30,
    )
)

display(
    timeline_sp_2026_se19_v2[
        [
            "lag_days",
            "snapshot_date",
            "known_cases",
            "expected_completeness",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
        ]
    ]
)

,lag_days,snapshot_date,known_cases,expected_completeness,ratio_to_median,conservative_ratio,signal_mode,signal_detected
0,0,2026-05-09,723,0.070690,6.059139,5.768129,INSUFFICIENT_DATA,False
1,1,2026-05-10,807,0.182839,2.614760,2.547675,INSUFFICIENT_DATA,False
2,2,2026-05-11,1086,0.303567,2.119352,2.014994,EARLY_CONSERVATIVE,True
3,3,2026-05-12,1248,0.433195,1.706708,1.620882,EARLY_CONSERVATIVE,True
4,4,2026-05-13,1368,0.541329,1.497107,1.406588,EARLY_CONSERVATIVE,True
5,5,2026-05-14,1463,0.626974,1.382363,1.304088,EARLY_CONSERVATIVE,True
6,6,2026-05-15,1544,0.697881,1.310670,1.253893,EARLY_CONSERVATIVE,True
7,7,2026-05-16,1551,0.759905,1.209150,1.168971,EARLY_CONSERVATIVE,True
8,8,2026-05-17,1562,0.802278,1.153410,1.122011,MATURE,True
9,9,2026-05-18,1599,0.831609,1.139087,1.113748,NONE,False


In [532]:
first_early_signal = (
    timeline_sp_2026_se19_v2.loc[
        timeline_sp_2026_se19_v2[
            "signal_detected"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

display(first_early_signal)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,completeness_q75,is_mature,nowcast,conservative_nowcast,historical_median,ratio_to_median,conservative_ratio,signal_detected,signal_mode
2,19,2,2026-05-11,1086,0.303567,0.319289,False,3577.466686,3401.310135,1688.0,2.119352,2.014994,True,EARLY_CONSERVATIVE


In [533]:
def get_completion_at_age(
    completion_model: pd.DataFrame,
    age_days: int,
    column: str,
) -> float:
    age_days = max(age_days, 0)

    max_lag = int(
        completion_model["lag_days"].max()
    )

    age_days = min(
        age_days,
        max_lag,
    )

    row = completion_model.loc[
        completion_model[
            "lag_days"
        ].eq(age_days)
    ]

    if row.empty:
        return float("nan")

    return float(
        row[column].iloc[0]
    )

In [534]:
def calculate_age_adjusted_nowcast(
    week_cases: pd.DataFrame,
    snapshot_date: pd.Timestamp,
    completion_model: pd.DataFrame,
    completion_column: str,
) -> float:
    known_cases = week_cases.loc[
        week_cases[
            "DT_NOTIFIC"
        ].le(snapshot_date)
    ].copy()

    if known_cases.empty:
        return 0.0

    known_cases["age_days"] = (
        snapshot_date
        - known_cases["DT_SIN_PRI"]
    ).dt.days

    completion_by_age = {
        age: get_completion_at_age(
            completion_model=completion_model,
            age_days=int(age),
            column=completion_column,
        )
        for age in known_cases["age_days"].unique()
    }

    known_cases["completion_probability"] = (
        known_cases["age_days"].map(completion_by_age)
    )

    known_cases = known_cases.loc[
        known_cases[
            "completion_probability"
        ].notna()
        & known_cases[
            "completion_probability"
        ].gt(0)
    ]

    return (
        1
        / known_cases[
            "completion_probability"
        ]
    ).sum()

In [535]:
def build_age_adjusted_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    ratio_threshold: float = 1.15,
    min_early_completeness: float = 0.30,
    mature_completeness: float = 0.80,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df[
            "DT_SIN_PRI"
        ].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(
        max_lag + 1
    ):
        snapshot_date = (
            week_end
            + pd.Timedelta(
                days=lag
            )
        )

        known = week_cases.loc[
            week_cases[
                "DT_NOTIFIC"
            ].le(snapshot_date)
        ]

        known_cases = len(known)

        central_nowcast = (
            calculate_age_adjusted_nowcast(
                week_cases=week_cases,
                snapshot_date=snapshot_date,
                completion_model=completion_model,
                completion_column=(
                    "expected_completeness"
                ),
            )
        )

        conservative_nowcast = (
            calculate_age_adjusted_nowcast(
                week_cases=week_cases,
                snapshot_date=snapshot_date,
                completion_model=completion_model,
                completion_column=(
                    "completeness_q75"
                ),
            )
        )

        central_ratio = (
            central_nowcast
            / historical_median
        )

        conservative_ratio = (
            conservative_nowcast
            / historical_median
        )

        completion_row = (
            completion_model.loc[
                completion_model[
                    "lag_days"
                ].eq(lag)
            ]
        )

        expected_completion = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        if (
            expected_completion
            >= mature_completeness
        ):
            signal_detected = (
                central_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "MATURE"
                if signal_detected
                else "NONE"
            )

        elif (
            expected_completion
            >= min_early_completeness
        ):
            signal_detected = (
                conservative_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "EARLY_CONSERVATIVE"
                if signal_detected
                else "NONE"
            )

        else:
            signal_detected = False
            signal_mode = (
                "INSUFFICIENT_DATA"
            )

        rows.append(
            {
                "epi_week":
                    epi_week,
                "lag_days":
                    lag,
                "snapshot_date":
                    snapshot_date,
                "known_cases":
                    known_cases,
                "expected_completeness":
                    expected_completion,
                "age_adjusted_nowcast":
                    central_nowcast,
                "conservative_nowcast":
                    conservative_nowcast,
                "historical_median":
                    historical_median,
                "ratio_to_median":
                    central_ratio,
                "conservative_ratio":
                    conservative_ratio,
                "signal_detected":
                    signal_detected,
                "signal_mode":
                    signal_mode,
            }
        )

    return pd.DataFrame(rows)

In [536]:
timeline_sp_2026_se19_v3 = (
    build_age_adjusted_nowcast_timeline(
        temporal_df=sp_2026_temporal,
        week_start=week_19[
            "week_start"
        ],
        week_end=week_19[
            "week_end"
        ],
        epi_week=19,
        historical_median=float(
            reference_week_19
        ),
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ratio_threshold=1.15,
        mature_completeness=0.80,
        min_early_completeness=0.30,
    )
)

display(
    timeline_sp_2026_se19_v3[
        [
            "lag_days",
            "snapshot_date",
            "known_cases",
            "expected_completeness",
            "age_adjusted_nowcast",
            "conservative_nowcast",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
        ]
    ]
)

,lag_days,snapshot_date,known_cases,expected_completeness,age_adjusted_nowcast,conservative_nowcast,ratio_to_median,conservative_ratio,signal_mode,signal_detected
0,0,2026-05-09,723,0.070690,1699.547527,1618.366131,1.006841,0.958748,INSUFFICIENT_DATA,False
1,1,2026-05-10,807,0.182839,1548.903258,1479.767022,0.917597,0.876639,INSUFFICIENT_DATA,False
2,2,2026-05-11,1086,0.303567,1837.197309,1753.499269,1.088387,1.038803,NONE,False
3,3,2026-05-12,1248,0.433195,1887.431231,1806.949582,1.118146,1.070468,NONE,False
4,4,2026-05-13,1368,0.541329,1899.289623,1824.871057,1.125172,1.081085,NONE,False
5,5,2026-05-14,1463,0.626974,1902.521863,1839.349898,1.127086,1.089662,NONE,False
6,6,2026-05-15,1544,0.697881,1910.987319,1859.233623,1.132101,1.101442,NONE,False
7,7,2026-05-16,1551,0.759905,1848.766498,1806.761022,1.095241,1.070356,NONE,False
8,8,2026-05-17,1562,0.802278,1813.598323,1777.688054,1.074407,1.053133,NONE,False
9,9,2026-05-18,1599,0.831609,1820.175750,1788.337912,1.078303,1.059442,NONE,False


In [537]:
display(
    timeline_sp_2026_se19_v3.loc[
        timeline_sp_2026_se19_v3[
            "signal_detected"
        ]
    ].head(1)
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,age_adjusted_nowcast,conservative_nowcast,historical_median,ratio_to_median,conservative_ratio,signal_detected,signal_mode


In [538]:
weeks_sp_2026_v3 = pd.DataFrame(
    {
        "epi_week": range(15, 38),
    }
)

first_week_start_sp_2026 = pd.Timestamp(
    "2025-12-28"
)

weeks_sp_2026_v3["week_start"] = (
    first_week_start_sp_2026
    + pd.to_timedelta(
        (
            weeks_sp_2026_v3["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2026_v3["week_end"] = (
    weeks_sp_2026_v3["week_start"]
    + pd.Timedelta(days=6)
)

In [539]:
timeline_frames_v3 = []

for week in weeks_sp_2026_v3.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2026.loc[
            reference_sp_2026[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_age_adjusted_nowcast_timeline(
            temporal_df=sp_2026_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_uncertainty_sp_2026
            ),
            ratio_threshold=1.15,
            mature_completeness=0.80,
            min_early_completeness=0.30,
        )
    )

    timeline_frames_v3.append(
        timeline
    )

timeline_sp_2026_v3 = pd.concat(
    timeline_frames_v3,
    ignore_index=True,
)

In [540]:
first_signal_v3 = (
    timeline_sp_2026_v3.loc[
        timeline_sp_2026_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_v3[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "expected_completeness",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,expected_completeness,ratio_to_median,conservative_ratio,signal_mode
0,20,2,2026-05-18,0.303567,1.224398,1.168706,EARLY_CONSERVATIVE
1,21,2,2026-05-25,0.303567,1.230247,1.174377,EARLY_CONSERVATIVE
2,22,2,2026-06-01,0.303567,1.436357,1.371251,EARLY_CONSERVATIVE
3,23,2,2026-06-08,0.303567,1.444234,1.378293,EARLY_CONSERVATIVE
4,24,2,2026-06-15,0.303567,1.463082,1.396960,EARLY_CONSERVATIVE
5,25,2,2026-06-22,0.303567,1.344299,1.283813,EARLY_CONSERVATIVE
6,26,2,2026-06-29,0.303567,1.415843,1.351444,EARLY_CONSERVATIVE
7,27,2,2026-07-06,0.303567,1.480455,1.412560,EARLY_CONSERVATIVE
8,28,2,2026-07-13,0.303567,1.428030,1.363218,EARLY_CONSERVATIVE
9,29,3,2026-07-21,0.433195,1.231045,1.179974,EARLY_CONSERVATIVE


In [541]:
evaluation_sp_2026_v3 = (
    weeks_sp_2026_v3[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_v3[
            [
                "epi_week",
                "lag_days",
                "snapshot_date",
                "ratio_to_median",
                "conservative_ratio",
                "signal_mode",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

evaluation_sp_2026_v3[
    "signal_detected"
] = (
    evaluation_sp_2026_v3[
        "snapshot_date"
    ].notna()
)

In [542]:
evaluation_sp_2026_v3[
    "ground_truth"
] = pd.Series(
    pd.NA,
    index=evaluation_sp_2026_v3.index,
    dtype="Int64",
)

# Negativo forte antes do episódio
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(15, 16),
    "ground_truth",
] = 0

# Positivo forte
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(20, 28),
    "ground_truth",
] = 1

# Negativo forte depois do episódio
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(32, 37),
    "ground_truth",
] = 0

In [543]:
def classify_early_outcome(row):
    if pd.isna(row["ground_truth"]):
        return "UNLABELED"

    truth = int(row["ground_truth"])
    predicted = bool(
        row["signal_detected"]
    )

    if truth == 1 and predicted:
        return "TP"

    if truth == 1 and not predicted:
        return "FN"

    if truth == 0 and predicted:
        return "FP"

    return "TN"


evaluation_sp_2026_v3[
    "outcome"
] = (
    evaluation_sp_2026_v3.apply(
        classify_early_outcome,
        axis=1,
    )
)

In [544]:
display(
    evaluation_sp_2026_v3[
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,ratio_to_median,conservative_ratio,signal_mode,signal_detected,outcome
0,15,0,NaT,NaN,NaN,NaN,NaN,False,TN
1,16,0,NaT,NaN,NaN,NaN,NaN,False,TN
2,17,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
3,18,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
4,19,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
5,20,1,2026-05-18,2.0,1.224398,1.168706,EARLY_CONSERVATIVE,True,TP
6,21,1,2026-05-25,2.0,1.230247,1.174377,EARLY_CONSERVATIVE,True,TP
7,22,1,2026-06-01,2.0,1.436357,1.371251,EARLY_CONSERVATIVE,True,TP
8,23,1,2026-06-08,2.0,1.444234,1.378293,EARLY_CONSERVATIVE,True,TP
9,24,1,2026-06-15,2.0,1.463082,1.396960,EARLY_CONSERVATIVE,True,TP


In [545]:
display(
    evaluation_sp_2026_v3.loc[
        evaluation_sp_2026_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    9
TN    8
Name: count, dtype: int64

In [546]:
positive_detection_sp_2026 = (
    evaluation_sp_2026_v3.loc[
        evaluation_sp_2026_v3[
            "ground_truth"
        ].eq(1)
    ][
        [
            "epi_week",
            "week_end",
            "snapshot_date",
            "lag_days",
            "signal_mode",
            "conservative_ratio",
        ]
    ]
)

display(positive_detection_sp_2026)

,epi_week,week_end,snapshot_date,lag_days,signal_mode,conservative_ratio
5,20,2026-05-16,2026-05-18,2.0,EARLY_CONSERVATIVE,1.168706
6,21,2026-05-23,2026-05-25,2.0,EARLY_CONSERVATIVE,1.174377
7,22,2026-05-30,2026-06-01,2.0,EARLY_CONSERVATIVE,1.371251
8,23,2026-06-06,2026-06-08,2.0,EARLY_CONSERVATIVE,1.378293
9,24,2026-06-13,2026-06-15,2.0,EARLY_CONSERVATIVE,1.396960
10,25,2026-06-20,2026-06-22,2.0,EARLY_CONSERVATIVE,1.283813
11,26,2026-06-27,2026-06-29,2.0,EARLY_CONSERVATIVE,1.351444
12,27,2026-07-04,2026-07-06,2.0,EARLY_CONSERVATIVE,1.412560
13,28,2026-07-11,2026-07-13,2.0,EARLY_CONSERVATIVE,1.363218


## 26. Nowcast ajustado pela idade do registro — SP/2026

A correção do nowcast passou a considerar individualmente o tempo
decorrido entre o início dos sintomas de cada registro e a data do
snapshot.

Essa abordagem evita tratar todos os registros de uma semana epidemiológica
como se possuíssem o mesmo atraso de notificação.

### Resultado

Nas semanas rotuladas de SP/2026:

- TP: 9
- FP: 0
- FN: 0
- TN: 8

O modelo não sinalizou as semanas negativas fortes SE15–16 e SE32–37.

A semana SE19, previamente sinalizada pelo nowcast simplificado,
deixou corretamente de atingir o limiar após a correção por idade,
permanecendo compatível com uma fase de transição.

Todas as semanas positivas SE20–28 foram identificadas através do modo
`EARLY_CONSERVATIVE`.

O primeiro sinal de cada uma ocorreu em D+2, quando a completude histórica
esperada era de aproximadamente 30%.

Apesar da baixa completude, o limite conservador do nowcast permaneceu
acima do limiar de 1,15 em todas as semanas positivas.

Esse comportamento sugere que a utilização explícita da incerteza permite
antecipar a detecção sem aumentar falsos sinais no conjunto avaliado.

In [547]:
def evaluate_early_signal_case(
    temporal_df: pd.DataFrame,
    weeks_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    completion_model: pd.DataFrame,
    ground_truth_by_week: dict[int, int],
    ratio_threshold: float = 1.15,
    min_early_completeness: float = 0.30,
    mature_completeness: float = 0.80,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    timeline_frames = []

    for week in weeks_df.itertuples(
        index=False
    ):
        reference_row = (
            reference_df.loc[
                reference_df[
                    "epi_week"
                ].eq(week.epi_week)
            ]
        )

        if reference_row.empty:
            continue

        historical_median = float(
            reference_row[
                "historical_median"
            ].iloc[0]
        )

        timeline = (
            build_age_adjusted_nowcast_timeline(
                temporal_df=temporal_df,
                week_start=week.week_start,
                week_end=week.week_end,
                epi_week=week.epi_week,
                historical_median=(
                    historical_median
                ),
                completion_model=(
                    completion_model
                ),
                ratio_threshold=(
                    ratio_threshold
                ),
                min_early_completeness=(
                    min_early_completeness
                ),
                mature_completeness=(
                    mature_completeness
                ),
            )
        )

        timeline_frames.append(
            timeline
        )

    timeline_all = pd.concat(
        timeline_frames,
        ignore_index=True,
    )

    first_signal = (
        timeline_all.loc[
            timeline_all[
                "signal_detected"
            ]
        ]
        .sort_values(
            [
                "epi_week",
                "snapshot_date",
            ]
        )
        .groupby(
            "epi_week",
            as_index=False,
        )
        .first()
    )

    evaluation = (
        weeks_df[
            [
                "epi_week",
                "week_start",
                "week_end",
            ]
        ]
        .merge(
            first_signal[
                [
                    "epi_week",
                    "lag_days",
                    "snapshot_date",
                    "ratio_to_median",
                    "conservative_ratio",
                    "signal_mode",
                ]
            ],
            on="epi_week",
            how="left",
        )
    )

    evaluation[
        "signal_detected"
    ] = (
        evaluation[
            "snapshot_date"
        ].notna()
    )

    evaluation[
        "ground_truth"
    ] = (
        evaluation[
            "epi_week"
        ]
        .map(
            ground_truth_by_week
        )
        .astype("Int64")
    )

    evaluation[
        "outcome"
    ] = (
        evaluation.apply(
            classify_early_outcome,
            axis=1,
        )
    )

    return (
        timeline_all,
        evaluation,
    )

In [548]:
ground_truth_sp_2026_map = {
    15: 0,
    16: 0,

    20: 1,
    21: 1,
    22: 1,
    23: 1,
    24: 1,
    25: 1,
    26: 1,
    27: 1,
    28: 1,

    32: 0,
    33: 0,
    34: 0,
    35: 0,
    36: 0,
    37: 0,
}

timeline_test, evaluation_test = (
    evaluate_early_signal_case(
        temporal_df=sp_2026_temporal,
        weeks_df=weeks_sp_2026_v3,
        reference_df=reference_sp_2026,
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ground_truth_by_week=(
            ground_truth_sp_2026_map
        ),
    )
)

print(
    evaluation_test.loc[
        evaluation_test[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    9
TN    8
Name: count, dtype: int64


In [549]:
completion_uncertainty_sp_2025 = (
    completion_by_year_sp_2025
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=(
            lambda x:
                x.quantile(0.25)
        ),
        completeness_q75=(
            lambda x:
                x.quantile(0.75)
        ),
    )
    .reset_index()
)

In [550]:
weeks_sp_2025_v3 = pd.DataFrame(
    {
        "epi_week": range(1, 41),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_sp_2025_v3[
    "week_start"
] = (
    first_week_start
    + pd.to_timedelta(
        (
            weeks_sp_2025_v3[
                "epi_week"
            ]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2025_v3[
    "week_end"
] = (
    weeks_sp_2025_v3[
        "week_start"
    ]
    + pd.Timedelta(days=6)
)

In [551]:
ground_truth_sp_2025_map = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [552]:
timeline_sp_2025_v3, evaluation_sp_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=sp_2025_temporal,
        weeks_df=weeks_sp_2025_v3,
        reference_df=reference_sp_2025,
        completion_model=(
            completion_uncertainty_sp_2025
        ),
        ground_truth_by_week=(
            ground_truth_sp_2025_map
        ),
    )
)

In [553]:
display(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
16,17,1,2025-04-28,2.0,1.246167,EARLY_CONSERVATIVE,True,TP
17,18,1,2025-05-05,2.0,1.204644,EARLY_CONSERVATIVE,True,TP
18,19,1,2025-05-12,2.0,1.552929,EARLY_CONSERVATIVE,True,TP
19,20,1,2025-05-19,2.0,1.721005,EARLY_CONSERVATIVE,True,TP
20,21,1,2025-05-26,2.0,1.928592,EARLY_CONSERVATIVE,True,TP
21,22,1,2025-06-02,2.0,1.871216,EARLY_CONSERVATIVE,True,TP
22,23,1,2025-06-09,2.0,1.879823,EARLY_CONSERVATIVE,True,TP
23,24,1,2025-06-16,2.0,1.586937,EARLY_CONSERVATIVE,True,TP
24,25,1,2025-06-23,2.0,1.501169,EARLY_CONSERVATIVE,True,TP
29,30,0,2025-07-28,2.0,1.206609,EARLY_CONSERVATIVE,True,FP


In [554]:
print(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TN    10
TP     9
FP     1
Name: count, dtype: int64


In [555]:
display(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(10)
)

,epi_week,week_start,week_end,lag_days,snapshot_date,ratio_to_median,conservative_ratio,signal_mode,signal_detected,ground_truth,outcome
16,17,2025-04-20,2025-04-26,2.0,2025-04-28,1.272197,1.246167,EARLY_CONSERVATIVE,True,1,TP
17,18,2025-04-27,2025-05-03,2.0,2025-05-05,1.230040,1.204644,EARLY_CONSERVATIVE,True,1,TP
18,19,2025-05-04,2025-05-10,2.0,2025-05-12,1.585130,1.552929,EARLY_CONSERVATIVE,True,1,TP
19,20,2025-05-11,2025-05-17,2.0,2025-05-19,1.757212,1.721005,EARLY_CONSERVATIVE,True,1,TP
20,21,2025-05-18,2025-05-24,2.0,2025-05-26,1.968923,1.928592,EARLY_CONSERVATIVE,True,1,TP
21,22,2025-05-25,2025-05-31,2.0,2025-06-02,1.910381,1.871216,EARLY_CONSERVATIVE,True,1,TP
22,23,2025-06-01,2025-06-07,2.0,2025-06-09,1.918792,1.879823,EARLY_CONSERVATIVE,True,1,TP
23,24,2025-06-08,2025-06-14,2.0,2025-06-16,1.620246,1.586937,EARLY_CONSERVATIVE,True,1,TP
24,25,2025-06-15,2025-06-21,2.0,2025-06-23,1.532431,1.501169,EARLY_CONSERVATIVE,True,1,TP
25,26,2025-06-22,2025-06-28,2.0,2025-06-30,1.389987,1.361642,EARLY_CONSERVATIVE,True,<NA>,UNLABELED


In [556]:
completion_uncertainty_mg_2025 = (
    completion_by_year_mg_2025
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=lambda x: x.quantile(0.25),
        completeness_q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

In [557]:
weeks_mg_2025_v3 = pd.DataFrame(
    {
        "epi_week": range(1, 41),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_mg_2025_v3[
    "week_start"
] = (
    first_week_start
    + pd.to_timedelta(
        (
            weeks_mg_2025_v3[
                "epi_week"
            ]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_mg_2025_v3[
    "week_end"
] = (
    weeks_mg_2025_v3[
        "week_start"
    ]
    + pd.Timedelta(days=6)
)

In [558]:
ground_truth_mg_2025_map = (
    ground_truth.loc[
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
        & ground_truth[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [559]:
timeline_mg_2025_v3, evaluation_mg_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=mg_2025_temporal,
        weeks_df=weeks_mg_2025_v3,
        reference_df=reference_mg_2025,
        completion_model=(
            completion_uncertainty_mg_2025
        ),
        ground_truth_by_week=(
            ground_truth_mg_2025_map
        ),
        ratio_threshold=1.15,
        min_early_completeness=0.30,
        mature_completeness=0.80,
    )
)

In [560]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
17,18,1,2025-05-05,2.0,1.862618,EARLY_CONSERVATIVE,True,TP
18,19,1,2025-05-12,2.0,2.575234,EARLY_CONSERVATIVE,True,TP
19,20,1,2025-05-19,2.0,2.982193,EARLY_CONSERVATIVE,True,TP
20,21,1,2025-05-26,2.0,3.284841,EARLY_CONSERVATIVE,True,TP
21,22,1,2025-06-02,2.0,3.497126,EARLY_CONSERVATIVE,True,TP
22,23,1,2025-06-09,2.0,3.685245,EARLY_CONSERVATIVE,True,TP
23,24,1,2025-06-16,2.0,3.250442,EARLY_CONSERVATIVE,True,TP
24,25,1,2025-06-23,2.0,2.620404,EARLY_CONSERVATIVE,True,TP


In [561]:
print(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    8
Name: count, dtype: int64


In [562]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(10)
)

,epi_week,week_start,week_end,lag_days,snapshot_date,ratio_to_median,conservative_ratio,signal_mode,signal_detected,ground_truth,outcome
16,17,2025-04-20,2025-04-26,2.0,2025-04-28,1.634392,1.599876,EARLY_CONSERVATIVE,True,<NA>,UNLABELED
17,18,2025-04-27,2025-05-03,2.0,2025-05-05,1.907510,1.862618,EARLY_CONSERVATIVE,True,1,TP
18,19,2025-05-04,2025-05-10,2.0,2025-05-12,2.632161,2.575234,EARLY_CONSERVATIVE,True,1,TP
19,20,2025-05-11,2025-05-17,2.0,2025-05-19,3.050462,2.982193,EARLY_CONSERVATIVE,True,1,TP
20,21,2025-05-18,2025-05-24,2.0,2025-05-26,3.357593,3.284841,EARLY_CONSERVATIVE,True,1,TP
21,22,2025-05-25,2025-05-31,2.0,2025-06-02,3.570340,3.497126,EARLY_CONSERVATIVE,True,1,TP
22,23,2025-06-01,2025-06-07,2.0,2025-06-09,3.759457,3.685245,EARLY_CONSERVATIVE,True,1,TP
23,24,2025-06-08,2025-06-14,2.0,2025-06-16,3.320740,3.250442,EARLY_CONSERVATIVE,True,1,TP
24,25,2025-06-15,2025-06-21,2.0,2025-06-23,2.677731,2.620404,EARLY_CONSERVATIVE,True,1,TP


In [563]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "signal_detected"
        ]
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,ratio_to_median,conservative_ratio,signal_mode,outcome
16,17,<NA>,2025-04-28,2.0,1.634392,1.599876,EARLY_CONSERVATIVE,UNLABELED
17,18,1,2025-05-05,2.0,1.907510,1.862618,EARLY_CONSERVATIVE,TP
18,19,1,2025-05-12,2.0,2.632161,2.575234,EARLY_CONSERVATIVE,TP
19,20,1,2025-05-19,2.0,3.050462,2.982193,EARLY_CONSERVATIVE,TP
20,21,1,2025-05-26,2.0,3.357593,3.284841,EARLY_CONSERVATIVE,TP
21,22,1,2025-06-02,2.0,3.570340,3.497126,EARLY_CONSERVATIVE,TP
22,23,1,2025-06-09,2.0,3.759457,3.685245,EARLY_CONSERVATIVE,TP
23,24,1,2025-06-16,2.0,3.320740,3.250442,EARLY_CONSERVATIVE,TP
24,25,1,2025-06-23,2.0,2.677731,2.620404,EARLY_CONSERVATIVE,TP


In [564]:
print(
    evaluation_mg_2025_v3[
        "outcome"
    ].value_counts()
)

outcome
UNLABELED    32
TP            8
Name: count, dtype: int64


In [565]:
negative_weeks_before = list(
    range(1, 11)
)

positive_weeks = list(
    range(18, 26)
)

negative_weeks_after = list(
    range(33, 41)
)

In [566]:
ground_truth_mg_2025_v2 = (
    ground_truth.loc[
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
    ]
    .copy()
)

ground_truth_mg_2025_v2[
    "ground_truth"
] = pd.Series(
    pd.NA,
    index=ground_truth_mg_2025_v2.index,
    dtype="Int64",
)

ground_truth_mg_2025_v2[
    "evidence_level"
] = "UNLABELED"

In [567]:
mask_negative_before = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        negative_weeks_before
    )
)

ground_truth_mg_2025_v2.loc[
    mask_negative_before,
    "ground_truth",
] = 0

ground_truth_mg_2025_v2.loc[
    mask_negative_before,
    "evidence_level",
] = "MODERATE"

In [568]:
mask_positive = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        positive_weeks
    )
)

ground_truth_mg_2025_v2.loc[
    mask_positive,
    "ground_truth",
] = 1

ground_truth_mg_2025_v2.loc[
    mask_positive,
    "evidence_level",
] = "STRONG"

In [569]:
mask_negative_after = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        negative_weeks_after
    )
)

ground_truth_mg_2025_v2.loc[
    mask_negative_after,
    "ground_truth",
] = 0

ground_truth_mg_2025_v2.loc[
    mask_negative_after,
    "evidence_level",
] = "STRONG"

In [570]:
display(
    ground_truth_mg_2025_v2[
        [
            "epi_week",
            "ground_truth",
            "evidence_level",
        ]
    ]
)

,epi_week,ground_truth,evidence_level
0,1,0,MODERATE
1,2,0,MODERATE
2,3,0,MODERATE
3,4,0,MODERATE
4,5,0,MODERATE
5,6,0,MODERATE
6,7,0,MODERATE
7,8,0,MODERATE
8,9,0,MODERATE
9,10,0,MODERATE


In [571]:
ground_truth_mg_2025_map = (
    ground_truth_mg_2025_v2.loc[
        ground_truth_mg_2025_v2[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [572]:
timeline_mg_2025_v3, evaluation_mg_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=mg_2025_temporal,
        weeks_df=weeks_mg_2025_v3,
        reference_df=reference_mg_2025,
        completion_model=(
            completion_uncertainty_mg_2025
        ),
        ground_truth_by_week=(
            ground_truth_mg_2025_map
        ),
        ratio_threshold=1.15,
        min_early_completeness=0.30,
        mature_completeness=0.80,
    )
)

In [573]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

print(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
0,1,0,NaT,NaN,NaN,NaN,False,TN
1,2,0,NaT,NaN,NaN,NaN,False,TN
2,3,0,NaT,NaN,NaN,NaN,False,TN
3,4,0,NaT,NaN,NaN,NaN,False,TN
4,5,0,NaT,NaN,NaN,NaN,False,TN
5,6,0,NaT,NaN,NaN,NaN,False,TN
6,7,0,NaT,NaN,NaN,NaN,False,TN
7,8,0,NaT,NaN,NaN,NaN,False,TN
8,9,0,NaT,NaN,NaN,NaN,False,TN
9,10,0,NaT,NaN,NaN,NaN,False,TN


outcome
TN    18
TP     8
Name: count, dtype: int64


In [574]:
SENTINELA_V3 = {
    "ratio_threshold": 1.15,
    "min_early_completeness": 0.30,
    "mature_completeness": 0.80,
    "uncertainty_bound": "q75",
    "nowcast": "age_adjusted",
}

In [575]:
def build_weekly_history_for_region(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        source_year = int(
            f"20{name.split('INFLUD')[1][:2]}"
        )

        if source_year not in years:
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df = df.loc[
            df["DT_SIN_PRI"].notna()
        ].copy()

        #
        # Importante:
        # usamos calendário epidemiológico
        # no padrão domingo-sábado.
        #
        df["epi_week"] = (
            df["DT_SIN_PRI"]
            .dt.strftime("%U")
            .astype(int)
            + 1
        )

        #
        # Mantém apenas as semanas
        # epidemiológicas válidas.
        #
        df = df.loc[
            df["epi_week"].between(
                1,
                53,
            )
        ].copy()

        weekly = (
            df.groupby(
                "epi_week",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size": "record_count",
                }
            )
        )

        weekly["source_year"] = (
            source_year
        )

        weekly["region"] = region

        frames.append(weekly)

    if not frames:
        raise ValueError(
            f"Nenhum dado encontrado "
            f"para {region} nos anos {years}."
        )

    result = pd.concat(
        frames,
        ignore_index=True,
    )

    result = (
        result[
            [
                "source_year",
                "epi_week",
                "record_count",
                "region",
            ]
        ]
        .sort_values(
            [
                "source_year",
                "epi_week",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [576]:
def get_epi_week(
    date: pd.Timestamp,
    epi_year: int,
) -> int | None:
    first_sunday = pd.Timestamp(
        f"{epi_year - 1}-12-28"
    )

    delta_days = (
        date.normalize()
        - first_sunday
    ).days

    epi_week = (
        delta_days // 7
        + 1
    )

    if 1 <= epi_week <= 53:
        return epi_week

    return None

In [577]:
def build_epi_calendar_for_year(
    year: int,
) -> pd.DataFrame:
    jan4 = pd.Timestamp(
        year=year,
        month=1,
        day=4,
    )

    #
    # Domingo imediatamente anterior
    # ou igual a 4 de janeiro.
    #
    days_since_sunday = (
        jan4.dayofweek + 1
    ) % 7

    first_week_start = (
        jan4
        - pd.Timedelta(
            days=days_since_sunday
        )
    )

    weeks = []

    for epi_week in range(1, 54):
        week_start = (
            first_week_start
            + pd.Timedelta(
                days=(epi_week - 1) * 7
            )
        )

        week_end = (
            week_start
            + pd.Timedelta(days=6)
        )

        weeks.append(
            {
                "epi_year": year,
                "epi_week": epi_week,
                "week_start": week_start,
                "week_end": week_end,
            }
        )

    return pd.DataFrame(weeks)

In [578]:
def assign_epi_week_from_calendar(
    dates: pd.Series,
    year: int,
) -> pd.Series:
    calendar = (
        build_epi_calendar_for_year(
            year
        )
    )

    result = pd.Series(
        pd.NA,
        index=dates.index,
        dtype="Int64",
    )

    for row in calendar.itertuples(
        index=False
    ):
        mask = dates.between(
            row.week_start,
            row.week_end,
        )

        result.loc[mask] = (
            row.epi_week
        )

    return result

In [579]:
def build_weekly_history_for_region(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        source_year = int(
            f"20{name.split('INFLUD')[1][:2]}"
        )

        if source_year not in years:
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df = df.loc[
            df["DT_SIN_PRI"].notna()
        ].copy()

        df["epi_week"] = (
            assign_epi_week_from_calendar(
                dates=df["DT_SIN_PRI"],
                year=source_year,
            )
        )

        df = df.loc[
            df["epi_week"].notna()
        ].copy()

        weekly = (
            df.groupby(
                "epi_week",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size":
                        "record_count",
                }
            )
        )

        weekly[
            "source_year"
        ] = source_year

        weekly[
            "region"
        ] = region

        frames.append(
            weekly
        )

    if not frames:
        raise ValueError(
            f"Nenhum dado encontrado "
            f"para região {region}."
        )

    return (
        pd.concat(
            frames,
            ignore_index=True,
        )
        [
            [
                "source_year",
                "epi_week",
                "record_count",
                "region",
            ]
        ]
        .sort_values(
            [
                "source_year",
                "epi_week",
            ]
        )
        .reset_index(drop=True)
    )

In [580]:
historical_pr = (
    build_weekly_history_for_region(
        historical_files=historical_files,
        region="PR",
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
    )
)

In [581]:
display(
    historical_pr.head(20)
)

,source_year,epi_week,record_count,region
0,2019,1,33,PR
1,2019,2,48,PR
2,2019,3,37,PR
3,2019,4,42,PR
4,2019,5,35,PR
5,2019,6,25,PR
6,2019,7,42,PR
7,2019,8,44,PR
8,2019,9,64,PR
9,2019,10,61,PR


In [582]:
print(
    historical_pr[
        "source_year"
    ].value_counts().sort_index()
)

source_year
2019    52
2022    52
2023    52
2024    52
Name: count, dtype: int64


In [583]:
summary_pr = (
    historical_pr
    .groupby("source_year")
    .agg(
        weeks=(
            "epi_week",
            "nunique",
        ),
        total_cases=(
            "record_count",
            "sum",
        ),
        first_week=(
            "epi_week",
            "min",
        ),
        last_week=(
            "epi_week",
            "max",
        ),
    )
)

display(summary_pr)

,weeks,total_cases,first_week,last_week
source_year,,,,
2019,52,6805,1,52
2022,52,55377,1,52
2023,52,28153,1,52
2024,52,26531,1,52


In [584]:
assert historical_pr[
    "source_year"
].isin(
    [
        2019,
        2022,
        2023,
        2024,
    ]
).all()

assert historical_pr[
    "epi_week"
].between(
    1,
    53,
).all()

assert (
    historical_pr[
        [
            "source_year",
            "epi_week",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

print(
    "✓ Histórico PR construído."
)

✓ Histórico PR construído.


In [585]:
historical_pr = historical_pr.rename(
    columns={
        "source_year": "epi_year",
    }
)

In [586]:
baseline_years_pr_2024 = [
    2019,
    2022,
    2023,
]

reference_pr_2024 = (
    build_reference_selected_years(
        historical_weekly=historical_pr,
        years=baseline_years_pr_2024,
        quantile=0.75,
    )
)

In [587]:
completion_years_pr_2024 = [
    2019,
    2022,
    2023,
]

In [588]:
delay_frames_pr_2024 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_pr_2024:
        continue

    delays = load_notification_delays(
        path=path,
        region="PR",
        source_year=source_year,
    )

    delay_frames_pr_2024.append(delays)

historical_delays_pr_2024 = pd.concat(
    delay_frames_pr_2024,
    ignore_index=True,
)

historical_delays_pr_2024 = (
    historical_delays_pr_2024.loc[
        historical_delays_pr_2024[
            "delay_days"
        ].notna()
        & historical_delays_pr_2024[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [589]:
delay_frames_pr_2024 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_pr_2024:
        continue

    delays = load_notification_delays(
        path=path,
        region="PR",
        source_year=source_year,
    )

    delay_frames_pr_2024.append(delays)

historical_delays_pr_2024 = pd.concat(
    delay_frames_pr_2024,
    ignore_index=True,
)

historical_delays_pr_2024 = (
    historical_delays_pr_2024.loc[
        historical_delays_pr_2024[
            "delay_days"
        ].notna()
        & historical_delays_pr_2024[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [590]:
completion_rows_pr_2024 = []

for year, group in (
    historical_delays_pr_2024
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_pr_2024.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_pr_2024 = pd.DataFrame(
    completion_rows_pr_2024
)

display(
    completion_by_year_pr_2024.head()
)

,source_year,lag_days,completeness
0,2019,0,0.053343
1,2019,1,0.184423
2,2019,2,0.333578
3,2019,3,0.481264
4,2019,4,0.611609


In [591]:
completion_uncertainty_pr_2024 = (
    completion_by_year_pr_2024
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=(
            lambda x: x.quantile(0.25)
        ),
        completeness_q75=(
            lambda x: x.quantile(0.75)
        ),
    )
    .reset_index()
)

In [592]:
display(
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "lag_days"
        ].isin(
            [0, 2, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.068376,0.060860,0.078837
2,2,0.333578,0.332526,0.339968
7,7,0.821937,0.796950,0.841534
14,14,0.945938,0.926036,0.957466
21,21,0.975598,0.957109,0.981700
28,28,0.983803,0.966828,0.987787


In [593]:
path_2024 = next(
    path
    for path in historical_files
    if "INFLUD24" in path.name.upper()
)

df_pr_2024_temporal = pd.read_csv(
    path_2024,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_pr_2024_temporal["SG_UF"] = (
    df_pr_2024_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

pr_2024_temporal = (
    df_pr_2024_temporal.loc[
        df_pr_2024_temporal[
            "SG_UF"
        ].eq("PR")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    pr_2024_temporal[column] = (
        pd.to_datetime(
            pr_2024_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

In [594]:
weeks_pr_2024 = pd.DataFrame(
    {
        "epi_week": range(1, 27),
    }
)

first_week_start_pr_2024 = pd.Timestamp(
    "2023-12-31"
)

weeks_pr_2024["week_start"] = (
    first_week_start_pr_2024
    + pd.to_timedelta(
        (
            weeks_pr_2024["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_pr_2024["week_end"] = (
    weeks_pr_2024["week_start"]
    + pd.Timedelta(days=6)
)

In [595]:
ground_truth_pr_2024_map = {
    # Negativos externos fortes
    1: 0,
    2: 0,
    3: 0,
    4: 0,

    # Positivos externos fortes
    13: 1,
    14: 1,
    15: 1,
    16: 1,
    17: 1,
    18: 1,
    19: 1,
    20: 1,
    21: 1,
}

In [596]:
timeline_pr_2024_v3, evaluation_pr_2024_v3 = (
    evaluate_early_signal_case(
        temporal_df=pr_2024_temporal,
        weeks_df=weeks_pr_2024,
        reference_df=reference_pr_2024,
        completion_model=completion_uncertainty_pr_2024,
        ground_truth_by_week=ground_truth_pr_2024_map,
    )
)


In [597]:
display(
    evaluation_pr_2024_v3.loc[
        evaluation_pr_2024_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

print(
    evaluation_pr_2024_v3.loc[
        evaluation_pr_2024_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
0,1,0,NaT,NaN,NaN,NaN,False,TN
1,2,0,NaT,NaN,NaN,NaN,False,TN
2,3,0,NaT,NaN,NaN,NaN,False,TN
3,4,0,NaT,NaN,NaN,NaN,False,TN
12,13,1,NaT,NaN,NaN,NaN,False,FN
13,14,1,NaT,NaN,NaN,NaN,False,FN
14,15,1,NaT,NaN,NaN,NaN,False,FN
15,16,1,NaT,NaN,NaN,NaN,False,FN
16,17,1,NaT,NaN,NaN,NaN,False,FN
17,18,1,NaT,NaN,NaN,NaN,False,FN


outcome
FN    9
TN    4
Name: count, dtype: int64


In [598]:
pr_2024_weekly_observed = (
    pr_2024_temporal
    .assign(
        epi_week=assign_epi_week_from_calendar(
            dates=pr_2024_temporal["DT_SIN_PRI"],
            year=2024,
        )
    )
    .dropna(
        subset=["epi_week"]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "record_count",
        }
    )
)

pr_2024_weekly_observed[
    "epi_week"
] = (
    pr_2024_weekly_observed[
        "epi_week"
    ].astype(int)
)

pr_2024_diagnostic = (
    pr_2024_weekly_observed
    .merge(
        reference_pr_2024[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

pr_2024_diagnostic[
    "final_ratio"
] = (
    pr_2024_diagnostic[
        "record_count"
    ]
    / pr_2024_diagnostic[
        "historical_median"
    ]
)

display(
    pr_2024_diagnostic.loc[
        pr_2024_diagnostic[
            "epi_week"
        ].between(1, 26)
    ]
)

,epi_week,record_count,historical_median,final_ratio
0,1,292,632.0,0.462025
1,2,280,469.0,0.597015
2,3,245,388.0,0.631443
3,4,224,346.0,0.647399
4,5,247,350.0,0.705714
5,6,281,415.0,0.677108
6,7,289,634.0,0.455836
7,8,438,625.0,0.700800
8,9,475,670.0,0.708955
9,10,492,635.0,0.774803


In [599]:
display(
    pr_2024_diagnostic.loc[
        pr_2024_diagnostic[
            "epi_week"
        ].between(13, 21),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "final_ratio",
        ],
    ]
)

,epi_week,record_count,historical_median,final_ratio
12,13,419,677.0,0.618907
13,14,498,680.0,0.732353
14,15,535,788.0,0.678934
15,16,605,746.0,0.810992
16,17,650,682.0,0.953079
17,18,635,751.0,0.845539
18,19,608,639.0,0.951487
19,20,582,783.0,0.743295
20,21,681,688.0,0.989826


In [600]:
pr_2024_max_ratio = (
    timeline_pr_2024_v3
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        max_ratio=(
            "ratio_to_median",
            "max",
        ),
        max_conservative_ratio=(
            "conservative_ratio",
            "max",
        ),
    )
)

display(
    pr_2024_max_ratio.loc[
        pr_2024_max_ratio[
            "epi_week"
        ].between(13, 21)
    ]
)

,epi_week,max_ratio,max_conservative_ratio
12,13,0.624270,0.620269
13,14,0.735752,0.725245
14,15,0.686062,0.678001
15,16,0.816547,0.812957
16,17,0.995113,0.976799
17,18,0.859134,0.843923
18,19,0.964500,0.959128
19,20,0.754423,0.747379
20,21,1.007979,0.990934


In [601]:
display(
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "lag_days"
        ].between(0, 14)
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.068376,0.060860,0.078837
1,1,0.197919,0.191171,0.200644
2,2,0.333578,0.332526,0.339968
3,3,0.481264,0.471621,0.486254
4,4,0.606117,0.588204,0.608863
5,5,0.696871,0.674266,0.707142
6,6,0.761446,0.737830,0.775359
7,7,0.821937,0.796950,0.841534
8,8,0.861400,0.836717,0.881104
9,9,0.885376,0.861671,0.902571


In [602]:
first_30 = (
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "expected_completeness"
        ].ge(0.30)
    ]
)

first_80 = (
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "expected_completeness"
        ].ge(0.80)
    ]
)

print(
    "Primeiro >= 30%:",
    (
        int(first_30["lag_days"].min())
        if not first_30.empty
        else None
    ),
)

print(
    "Primeiro >= 80%:",
    (
        int(first_80["lag_days"].min())
        if not first_80.empty
        else None
    ),
)

Primeiro >= 30%: 2
Primeiro >= 80%: 7


In [603]:
import numpy as np

pr_trend = (
    pr_2024_diagnostic[
        [
            "epi_week",
            "record_count",
            "historical_median",
            "final_ratio",
        ]
    ]
    .copy()
)

pr_trend["log_count"] = np.log1p(
    pr_trend["record_count"]
)

In [604]:
def rolling_log_slope(
    series: pd.Series,
    window: int = 4,
) -> pd.Series:
    values = series.to_numpy(
        dtype=float
    )

    result = np.full(
        len(values),
        np.nan,
    )

    x = np.arange(
        window,
        dtype=float,
    )

    for i in range(
        window - 1,
        len(values),
    ):
        y = values[
            i - window + 1:
            i + 1
        ]

        slope = np.polyfit(
            x,
            y,
            1,
        )[0]

        result[i] = slope

    return pd.Series(
        result,
        index=series.index,
    )


pr_trend["slope_4w"] = (
    rolling_log_slope(
        pr_trend["log_count"],
        window=4,
    )
)

In [605]:
pr_trend[
    "weekly_growth_rate"
] = (
    np.exp(
        pr_trend["slope_4w"]
    )
    - 1
)

In [606]:
display(
    pr_trend.loc[
        pr_trend[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "slope_4w",
            "weekly_growth_rate",
        ],
    ]
)

,epi_week,record_count,final_ratio,slope_4w,weekly_growth_rate
0,1,292,0.462025,NaN,NaN
1,2,280,0.597015,NaN,NaN
2,3,245,0.631443,NaN,NaN
3,4,224,0.647399,-0.092524,-0.088373
4,5,247,0.705714,-0.046401,-0.045341
5,6,281,0.677108,0.050705,0.052013
6,7,289,0.455836,0.088982,0.093061
7,8,438,0.700800,0.174119,0.190197
8,9,475,0.708955,0.198515,0.219590
9,10,492,0.774803,0.167280,0.182086


In [607]:
pr_trend[
    "week_over_week_growth"
] = (
    pr_trend[
        "record_count"
    ]
    .pct_change()
)

pr_trend[
    "positive_growth"
] = (
    pr_trend[
        "week_over_week_growth"
    ] > 0
)

pr_trend[
    "positive_growth_3of4"
] = (
    pr_trend[
        "positive_growth"
    ]
    .rolling(
        4,
        min_periods=4,
    )
    .sum()
    >= 3
)

In [608]:
display(
    pr_trend.loc[
        pr_trend[
            "epi_week"
        ].between(8, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4
7,8,438,0.700800,0.190197,True
8,9,475,0.708955,0.219590,True
9,10,492,0.774803,0.182086,True
10,11,490,0.768025,0.037789,True
11,12,474,0.679083,-0.001037,False
12,13,419,0.618907,-0.050091,False
13,14,498,0.732353,-0.007430,False
14,15,535,0.678934,0.054937,False
15,16,605,0.810992,0.124277,True
16,17,650,0.953079,0.096415,True


In [609]:
def add_trend_features(
    weekly_df: pd.DataFrame,
    window: int = 4,
) -> pd.DataFrame:
    df = weekly_df.sort_values(
        "epi_week"
    ).copy()

    df["log_count"] = np.log1p(
        df["record_count"]
    )

    df["slope_4w"] = (
        rolling_log_slope(
            df["log_count"],
            window=window,
        )
    )

    df["weekly_growth_rate"] = (
        np.exp(df["slope_4w"]) - 1
    )

    df["week_over_week_growth"] = (
        df["record_count"]
        .pct_change()
    )

    df["positive_growth"] = (
        df["week_over_week_growth"] > 0
    )

    df["positive_growth_3of4"] = (
        df["positive_growth"]
        .rolling(
            window,
            min_periods=window,
        )
        .sum()
        >= 3
    )

    return df

In [610]:
historical_pr_trend_frames = []

for year in [
    2019,
    2022,
    2023,
]:
    yearly = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].eq(year)
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    yearly = add_trend_features(
        yearly,
        window=4,
    )

    historical_pr_trend_frames.append(
        yearly
    )

historical_pr_trends = pd.concat(
    historical_pr_trend_frames,
    ignore_index=True,
)

In [611]:
trend_reference_pr_2024 = (
    historical_pr_trends
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        historical_slope_median=(
            "slope_4w",
            "median",
        ),
        historical_slope_q75=(
            "slope_4w",
            lambda x:
                x.quantile(0.75),
        ),
        historical_slope_q90=(
            "slope_4w",
            lambda x:
                x.quantile(0.90),
        ),
        historical_slope_max=(
            "slope_4w",
            "max",
        ),
    )
)

In [612]:
pr_2024_trend_eval = (
    pr_trend
    .merge(
        trend_reference_pr_2024,
        on="epi_week",
        how="left",
    )
)

In [613]:
pr_2024_trend_eval[
    "slope_excess_q90"
] = (
    pr_2024_trend_eval[
        "slope_4w"
    ]
    - pr_2024_trend_eval[
        "historical_slope_q90"
    ]
)

In [614]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
            "slope_4w",
            "historical_slope_q90",
            "slope_excess_q90",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4,slope_4w,historical_slope_q90,slope_excess_q90
0,1,292,0.462025,NaN,False,NaN,NaN,NaN
1,2,280,0.597015,NaN,False,NaN,NaN,NaN
2,3,245,0.631443,NaN,False,NaN,NaN,NaN
3,4,224,0.647399,-0.088373,False,-0.092524,0.063564,-0.156088
4,5,247,0.705714,-0.045341,False,-0.046401,-0.048873,0.002472
5,6,281,0.677108,0.052013,False,0.050705,-0.009301,0.060006
6,7,289,0.455836,0.093061,True,0.088982,0.152116,-0.063134
7,8,438,0.700800,0.190197,True,0.174119,0.196141,-0.022023
8,9,475,0.708955,0.219590,True,0.198515,0.251946,-0.053431
9,10,492,0.774803,0.182086,True,0.167280,0.118725,0.048555


In [615]:
pr_2024_trend_eval[
    "trend_signal"
] = (
    pr_2024_trend_eval[
        "positive_growth_3of4"
    ]
    & (
        pr_2024_trend_eval[
            "slope_4w"
        ]
        >
        pr_2024_trend_eval[
            "historical_slope_q90"
        ]
    )
)

In [616]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
            "trend_signal",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4,trend_signal
0,1,292,0.462025,NaN,False,False
1,2,280,0.597015,NaN,False,False
2,3,245,0.631443,NaN,False,False
3,4,224,0.647399,-0.088373,False,False
4,5,247,0.705714,-0.045341,False,False
5,6,281,0.677108,0.052013,False,False
6,7,289,0.455836,0.093061,True,False
7,8,438,0.700800,0.190197,True,False
8,9,475,0.708955,0.219590,True,False
9,10,492,0.774803,0.182086,True,True


In [617]:
pr_2024_trend_eval[
    "log_ratio"
] = np.log(
    pr_2024_trend_eval[
        "final_ratio"
    ]
)

In [618]:
pr_2024_trend_eval[
    "ratio_slope_4w"
] = rolling_log_slope(
    pr_2024_trend_eval[
        "log_ratio"
    ],
    window=4,
)

In [619]:
pr_2024_trend_eval[
    "relative_growth_rate"
] = (
    np.exp(
        pr_2024_trend_eval[
            "ratio_slope_4w"
        ]
    )
    - 1
)

In [620]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "ratio_slope_4w",
            "relative_growth_rate",
        ],
    ]
)

,epi_week,record_count,final_ratio,ratio_slope_4w,relative_growth_rate
5,6,281,0.677108,0.029572,0.030013
6,7,289,0.455836,-0.109387,-0.103616
7,8,438,0.700800,-0.041666,-0.040810
8,9,475,0.708955,0.056797,0.058441
9,10,492,0.774803,0.160300,0.173863
10,11,490,0.768025,0.036362,0.037031
11,12,474,0.679083,-0.013793,-0.013699
12,13,419,0.618907,-0.079704,-0.076610
13,14,498,0.732353,-0.023547,-0.023272
14,15,535,0.678934,0.016765,0.016906


In [621]:
pr_history_years = [
    2019,
    2022,
    2023,
]

In [622]:
historical_ratio_trend_frames = []

for target_year in pr_history_years:
    baseline_years = [
        year
        for year in pr_history_years
        if year != target_year
    ]

    target = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].eq(target_year)
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    baseline = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].isin(
                baseline_years
            )
        ]
        .groupby(
            "epi_week",
            as_index=False,
        )
        .agg(
            historical_median=(
                "record_count",
                "median",
            )
        )
    )

    target = target.merge(
        baseline,
        on="epi_week",
        how="left",
    )

    target["ratio"] = (
        target["record_count"]
        / target[
            "historical_median"
        ]
    )

    target["log_ratio"] = (
        np.log(target["ratio"])
    )

    target["ratio_slope_4w"] = (
        rolling_log_slope(
            target["log_ratio"],
            window=4,
        )
    )

    historical_ratio_trend_frames.append(
        target
    )

historical_ratio_trends = pd.concat(
    historical_ratio_trend_frames,
    ignore_index=True,
)

In [623]:
ratio_trend_reference_pr = (
    historical_ratio_trends
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        ratio_slope_median=(
            "ratio_slope_4w",
            "median",
        ),
        ratio_slope_q75=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.75),
        ),
        ratio_slope_q90=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.90),
        ),
        ratio_slope_max=(
            "ratio_slope_4w",
            "max",
        ),
    )
)

In [624]:
pr_2024_trend_v2 = (
    pr_2024_trend_eval
    .merge(
        ratio_trend_reference_pr,
        on="epi_week",
        how="left",
    )
)

In [625]:
pr_2024_trend_v2[
    "trend_signal_v2"
] = (
    pr_2024_trend_v2[
        "ratio_slope_4w"
    ]
    >
    pr_2024_trend_v2[
        "ratio_slope_q90"
    ]
)

In [626]:
pr_2024_trend_v2[
    "trend_signal_v2"
] = (
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ]
        >
        pr_2024_trend_v2[
            "ratio_slope_q90"
        ]
    )
    &
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ] > 0
    )
)

In [627]:
display(
    pr_2024_trend_v2.loc[
        pr_2024_trend_v2[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "ratio_slope_4w",
            "ratio_slope_q90",
            "trend_signal_v2",
        ],
    ]
)

,epi_week,record_count,final_ratio,ratio_slope_4w,ratio_slope_q90,trend_signal_v2
5,6,281,0.677108,0.029572,0.148106,False
6,7,289,0.455836,-0.109387,0.334611,False
7,8,438,0.700800,-0.041666,0.406444,False
8,9,475,0.708955,0.056797,0.387962,False
9,10,492,0.774803,0.160300,0.276777,False
10,11,490,0.768025,0.036362,0.182304,False
11,12,474,0.679083,-0.013793,0.113195,False
12,13,419,0.618907,-0.079704,0.164356,False
13,14,498,0.732353,-0.023547,0.200871,False
14,15,535,0.678934,0.016765,0.128977,False


In [628]:
ground_truth_pr_2024_v2 = {
    1: 0,
    2: 0,
    3: 0,
    4: 0,

    23: 1,
    24: 1,
}

In [629]:
evaluation_pr_2024_v4 = (
    pr_2024_trend_v2[
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "trend_signal_v2",
        ]
    ]
    .copy()
)

evaluation_pr_2024_v4[
    "ground_truth"
] = (
    evaluation_pr_2024_v4[
        "epi_week"
    ]
    .map(
        ground_truth_pr_2024_v2
    )
    .astype("Int64")
)

evaluation_pr_2024_v4[
    "magnitude_signal"
] = (
    evaluation_pr_2024_v4[
        "final_ratio"
    ] >= 1.15
)

evaluation_pr_2024_v4[
    "combined_signal"
] = (
    evaluation_pr_2024_v4[
        "magnitude_signal"
    ]
    |
    evaluation_pr_2024_v4[
        "trend_signal_v2"
    ]
)

In [630]:
def classify_combined(row):
    if pd.isna(
        row["ground_truth"]
    ):
        return "UNLABELED"

    truth = int(
        row["ground_truth"]
    )

    predicted = bool(
        row["combined_signal"]
    )

    if truth == 1 and predicted:
        return "TP"

    if truth == 1 and not predicted:
        return "FN"

    if truth == 0 and predicted:
        return "FP"

    return "TN"


evaluation_pr_2024_v4[
    "outcome"
] = (
    evaluation_pr_2024_v4.apply(
        classify_combined,
        axis=1,
    )
)

In [631]:
display(
    evaluation_pr_2024_v4.loc[
        evaluation_pr_2024_v4[
            "ground_truth"
        ].notna()
    ]
)

print(
    evaluation_pr_2024_v4.loc[
        evaluation_pr_2024_v4[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

,epi_week,record_count,final_ratio,trend_signal_v2,ground_truth,magnitude_signal,combined_signal,outcome
0,1,292,0.462025,False,0,False,False,TN
1,2,280,0.597015,False,0,False,False,TN
2,3,245,0.631443,False,0,False,False,TN
3,4,224,0.647399,False,0,False,False,TN
22,23,725,1.160000,True,1,True,True,TP
23,24,724,1.309222,True,1,True,True,TP


outcome
TN    4
TP    2
Name: count, dtype: int64


In [632]:
ground_truth_pr_2024_final = {}

# Negativos fortes
for week in range(1, 5):
    ground_truth_pr_2024_final[week] = 0

# Positivos: crescimento externo documentado
for week in range(13, 22):
    ground_truth_pr_2024_final[week] = 1

In [633]:
pr_2024_evidence = pd.DataFrame(
    [
        {
            "start_week": 1,
            "end_week": 4,
            "ground_truth": 0,
            "evidence_level": "STRONG",
            "reason": (
                "Período inicial usado como bloco "
                "negativo independente."
            ),
        },
        {
            "start_week": 13,
            "end_week": 17,
            "ground_truth": 1,
            "evidence_level": "MODERATE",
            "reason": (
                "InfoGripe já identifica Paraná "
                "com crescimento de SRAG na "
                "tendência de longo prazo."
            ),
        },
        {
            "start_week": 18,
            "end_week": 21,
            "ground_truth": 1,
            "evidence_level": "STRONG",
            "reason": (
                "Crescimento de SRAG no Paraná "
                "confirmado repetidamente por "
                "atualizações externas."
            ),
        },
    ]
)

display(pr_2024_evidence)

,start_week,end_week,ground_truth,evidence_level,reason
0,1,4,0,STRONG,Período inicial usado como bloco negativo inde...
1,13,17,1,MODERATE,InfoGripe já identifica Paraná com crescimento...
2,18,21,1,STRONG,Crescimento de SRAG no Paraná confirmado repet...


In [634]:
pr_2024_change = (
    pr_2024_diagnostic[
        [
            "epi_week",
            "record_count",
            "historical_median",
            "final_ratio",
        ]
    ]
    .copy()
)

pr_2024_change["log_ratio"] = np.log(
    pr_2024_change["final_ratio"]
)

In [635]:
pr_2024_change["delta_log_ratio"] = (
    pr_2024_change["log_ratio"]
    .diff()
)

In [636]:
def positive_cusum(
    series: pd.Series,
    drift: float = 0.0,
) -> pd.Series:
    values = (
        series
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    result = np.zeros(
        len(values),
        dtype=float,
    )

    running = 0.0

    for i, value in enumerate(values):
        running = max(
            0.0,
            running + value - drift,
        )

        result[i] = running

    return pd.Series(
        result,
        index=series.index,
    )

In [637]:
pr_2024_change["cusum_positive"] = (
    positive_cusum(
        pr_2024_change[
            "delta_log_ratio"
        ],
        drift=0.0,
    )
)

In [638]:
display(
    pr_2024_change.loc[
        pr_2024_change[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "delta_log_ratio",
            "cusum_positive",
        ],
    ]
)

,epi_week,record_count,final_ratio,delta_log_ratio,cusum_positive
5,6,281,0.677108,-0.041379,0.382212
6,7,289,0.455836,-0.395698,0.000000
7,8,438,0.700800,0.430090,0.430090
8,9,475,0.708955,0.011570,0.441659
9,10,492,0.774803,0.088817,0.530476
10,11,490,0.768025,-0.008787,0.521689
11,12,474,0.679083,-0.123079,0.398610
12,13,419,0.618907,-0.092789,0.305822
13,14,498,0.732353,0.168308,0.474130
14,15,535,0.678934,-0.075739,0.398391


In [639]:
def build_cusum_for_year(
    historical_weekly: pd.DataFrame,
    target_year: int,
    baseline_years: list[int],
) -> pd.DataFrame:
    target = (
        historical_weekly.loc[
            historical_weekly["epi_year"].eq(
                target_year
            )
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    baseline = (
        historical_weekly.loc[
            historical_weekly["epi_year"].isin(
                baseline_years
            )
        ]
        .groupby(
            "epi_week",
            as_index=False,
        )
        .agg(
            historical_median=(
                "record_count",
                "median",
            )
        )
    )

    target = target.merge(
        baseline,
        on="epi_week",
        how="left",
    )

    target["ratio"] = (
        target["record_count"]
        / target["historical_median"]
    )

    target["log_ratio"] = np.log(
        target["ratio"]
    )

    target["delta_log_ratio"] = (
        target["log_ratio"]
        .diff()
    )

    target["cusum_positive"] = (
        positive_cusum(
            target["delta_log_ratio"],
            drift=0.0,
        )
    )

    return target

In [640]:
pr_history_years = [
    2019,
    2022,
    2023,
]

historical_cusum_frames = []

for target_year in pr_history_years:
    baseline_years = [
        year
        for year in pr_history_years
        if year != target_year
    ]

    yearly_cusum = build_cusum_for_year(
        historical_weekly=historical_pr,
        target_year=target_year,
        baseline_years=baseline_years,
    )

    historical_cusum_frames.append(
        yearly_cusum
    )

historical_pr_cusum = pd.concat(
    historical_cusum_frames,
    ignore_index=True,
)

In [641]:
display(
    historical_pr_cusum.head(20)
)

,epi_year,epi_week,record_count,historical_median,ratio,log_ratio,delta_log_ratio,cusum_positive
0,2019,1,33,1344.5,0.024544,-3.707270,NaN,0.000000
1,2019,2,48,1402.5,0.034225,-3.374811,0.332459,0.332459
2,2019,3,37,1570.0,0.023567,-3.747913,-0.373102,0.000000
3,2019,4,42,1395.5,0.030097,-3.503338,0.244575,0.244575
4,2019,5,35,1234.5,0.028352,-3.563073,-0.059735,0.184840
5,2019,6,25,1042.0,0.023992,-3.730021,-0.166948,0.017892
6,2019,7,42,992.0,0.042339,-3.162053,0.567968,0.585859
7,2019,8,44,824.5,0.053366,-2.930588,0.231466,0.817325
8,2019,9,64,766.0,0.083551,-2.482299,0.448288,1.265614
9,2019,10,61,638.0,0.095611,-2.347464,0.134835,1.400449


In [642]:
def build_local_cusum_reference(
    historical_cusum: pd.DataFrame,
    window_radius: int = 2,
) -> pd.DataFrame:
    rows = []

    for epi_week in range(1, 53):
        low = max(
            1,
            epi_week - window_radius,
        )

        high = min(
            52,
            epi_week + window_radius,
        )

        values = (
            historical_cusum.loc[
                historical_cusum[
                    "epi_week"
                ].between(
                    low,
                    high,
                ),
                "cusum_positive",
            ]
            .dropna()
        )

        if values.empty:
            continue

        rows.append(
            {
                "epi_week": epi_week,
                "historical_cusum_median":
                    values.median(),
                "historical_cusum_q90":
                    values.quantile(0.90),
                "historical_cusum_q95":
                    values.quantile(0.95),
                "historical_cusum_max":
                    values.max(),
                "historical_samples":
                    len(values),
            }
        )

    return pd.DataFrame(rows)

In [643]:
cusum_reference_pr = (
    build_local_cusum_reference(
        historical_cusum=(
            historical_pr_cusum
        ),
        window_radius=2,
    )
)

In [644]:
pr_2024_cusum_eval = (
    pr_2024_change
    .merge(
        cusum_reference_pr,
        on="epi_week",
        how="left",
    )
)

In [645]:
display(
    pr_2024_cusum_eval.loc[
        pr_2024_cusum_eval[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "final_ratio",
            "cusum_positive",
            "historical_cusum_median",
            "historical_cusum_q90",
            "historical_cusum_q95",
            "historical_cusum_max",
            "historical_samples",
        ],
    ]
)

,epi_week,final_ratio,cusum_positive,historical_cusum_median,historical_cusum_q90,historical_cusum_q95,historical_cusum_max,historical_samples
5,6,0.677108,0.382212,0.244575,1.038521,1.261182,1.436642,15
6,7,0.455836,0.000000,0.565864,1.368231,1.500300,1.648837,15
7,8,0.700800,0.430090,0.585859,1.563959,1.715823,1.872125,15
8,9,0.708955,0.441659,1.185985,1.782809,1.907769,1.990939,15
9,10,0.774803,0.530476,1.295868,1.942078,1.989381,1.990939,15
10,11,0.768025,0.521689,1.400449,1.983298,1.989381,1.990939,15
11,12,0.679083,0.398610,1.666897,1.985058,1.989381,1.990939,15
12,13,0.618907,0.305822,1.669137,1.990049,2.040060,2.154677,15
13,14,0.732353,0.474130,1.669137,2.088292,2.160157,2.172942,15
14,15,0.678934,0.398391,1.666897,2.165636,2.221628,2.335229,15


In [646]:
def rolling_positive_cusum(
    series: pd.Series,
    window: int = 4,
) -> pd.Series:
    values = (
        series
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    result = np.full(
        len(values),
        np.nan,
    )

    for i in range(
        window - 1,
        len(values),
    ):
        local = values[
            i - window + 1:
            i + 1
        ]

        running = 0.0

        for value in local:
            running = max(
                0.0,
                running + value,
            )

        result[i] = running

    return pd.Series(
        result,
        index=series.index,
    )

In [647]:
pr_2024_change[
    "local_cusum_4w"
] = rolling_positive_cusum(
    pr_2024_change[
        "delta_log_ratio"
    ],
    window=4,
)

In [648]:
display(
    pr_2024_change.loc[
        pr_2024_change[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "final_ratio",
            "delta_log_ratio",
            "local_cusum_4w",
        ],
    ]
)

,epi_week,final_ratio,delta_log_ratio,local_cusum_4w
5,6,0.677108,-0.041379,0.125889
6,7,0.455836,-0.395698,0.000000
7,8,0.700800,0.430090,0.430090
8,9,0.708955,0.011570,0.441659
9,10,0.774803,0.088817,0.530476
10,11,0.768025,-0.008787,0.521689
11,12,0.679083,-0.123079,0.000000
12,13,0.618907,-0.092789,0.000000
13,14,0.732353,0.168308,0.168308
14,15,0.678934,-0.075739,0.092569


In [649]:
historical_local_cusum_frames = []

for target_year in [
    2019,
    2022,
    2023,
]:
    baseline_years = [
        year
        for year in [
            2019,
            2022,
            2023,
        ]
        if year != target_year
    ]

    target = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].eq(target_year)
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    baseline = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].isin(baseline_years)
        ]
        .groupby(
            "epi_week",
            as_index=False,
        )
        .agg(
            historical_median=(
                "record_count",
                "median",
            )
        )
    )

    target = target.merge(
        baseline,
        on="epi_week",
        how="left",
    )

    target["ratio"] = (
        target["record_count"]
        / target["historical_median"]
    )

    target["log_ratio"] = np.log(
        target["ratio"]
    )

    target["delta_log_ratio"] = (
        target["log_ratio"]
        .diff()
    )

    target["local_cusum_4w"] = (
        rolling_positive_cusum(
            target[
                "delta_log_ratio"
            ],
            window=4,
        )
    )

    historical_local_cusum_frames.append(
        target
    )

historical_pr_local_cusum = pd.concat(
    historical_local_cusum_frames,
    ignore_index=True,
)

In [658]:
def build_local_cusum_reference_v2(
    historical_df: pd.DataFrame,
    radius: int = 2,
) -> pd.DataFrame:
    rows = []

    for week in range(1, 53):
        low = max(
            1,
            week - radius,
        )

        high = min(
            52,
            week + radius,
        )

        values = (
            historical_df.loc[
                historical_df[
                    "epi_week"
                ].between(
                    low,
                    high,
                ),
                "local_cusum_4w",
            ]
            .dropna()
        )

        if values.empty:
            continue

        rows.append(
            {
                "epi_week": week,
                "local_cusum_median":
                    values.median(),
                "local_cusum_q90":
                    values.quantile(0.90),
                "local_cusum_q95":
                    values.quantile(0.95),
                "local_cusum_max":
                    values.max(),
                "historical_samples":
                    len(values),
            }
        )

    return pd.DataFrame(rows)

In [659]:
local_cusum_reference_pr = (
    build_local_cusum_reference_v2(
        historical_df=(
            historical_pr_local_cusum
        ),
        radius=2,
    )
)

In [660]:
pr_2024_local_cusum_eval = (
    pr_2024_change
    .merge(
        local_cusum_reference_pr,
        on="epi_week",
        how="left",
    )
)

In [661]:
display(
    pr_2024_local_cusum_eval.loc[
        pr_2024_local_cusum_eval[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "final_ratio",
            "delta_log_ratio",
            "local_cusum_4w",
            "local_cusum_median",
            "local_cusum_q90",
            "local_cusum_q95",
            "local_cusum_max",
        ],
    ]
)

,epi_week,final_ratio,delta_log_ratio,local_cusum_4w,local_cusum_median,local_cusum_q90,local_cusum_q95,local_cusum_max
5,6,0.677108,-0.041379,0.125889,0.244575,1.031365,1.261170,1.436602
6,7,0.455836,-0.395698,0.000000,0.565864,1.361050,1.453687,1.493552
7,8,0.700800,0.430090,0.430090,0.585859,1.414984,1.453687,1.493552
8,9,0.708955,0.011570,0.441659,0.799434,1.414984,1.453687,1.493552
9,10,0.774803,0.088817,0.530476,0.799434,1.414984,1.453687,1.493552
10,11,0.768025,-0.008787,0.521689,0.709560,1.352039,1.415856,1.493552
11,12,0.679083,-0.123079,0.000000,0.552072,1.124481,1.329150,1.382557
12,13,0.618907,-0.092789,0.000000,0.241835,0.833068,0.853910,0.858809
13,14,0.732353,0.168308,0.168308,0.241835,0.794911,0.853910,0.858809
14,15,0.678934,-0.075739,0.092569,0.241835,0.699218,0.754335,0.858809


In [654]:
ratio_trend_reference_pr = (
    historical_ratio_trends
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        ratio_slope_median=(
            "ratio_slope_4w",
            "median",
        ),
        ratio_slope_q75=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.75),
        ),
        ratio_slope_q90=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.90),
        ),
        ratio_slope_max=(
            "ratio_slope_4w",
            "max",
        ),
    )
)

In [655]:
pr_2024_trend_v2 = (
    pr_2024_trend_eval
    .merge(
        ratio_trend_reference_pr,
        on="epi_week",
        how="left"
    )
)

In [656]:
pr_2024_trend_v2[
    "trend_signal_v2"
] = (
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ]
        >
        pr_2024_trend_v2[
            "ratio_slope_q90"
        ]
    )
    &
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ] > 0
    )
)

In [657]:
display(
    pr_2024_trend_v2.loc[
        pr_2024_trend_v2[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "ratio_slope_4w",
            "ratio_slope_q90",
            "trend_signal_v2",
        ],
    ]
)

,epi_week,record_count,final_ratio,ratio_slope_4w,ratio_slope_q90,trend_signal_v2
5,6,281,0.677108,0.029572,0.148106,False
6,7,289,0.455836,-0.109387,0.334611,False
7,8,438,0.700800,-0.041666,0.406444,False
8,9,475,0.708955,0.056797,0.387962,False
9,10,492,0.774803,0.160300,0.276777,False
10,11,490,0.768025,0.036362,0.182304,False
11,12,474,0.679083,-0.013793,0.113195,False
12,13,419,0.618907,-0.079704,0.164356,False
13,14,498,0.732353,-0.023547,0.200871,False
14,15,535,0.678934,0.016765,0.128977,False


In [663]:
age_columns = [
    column
    for column in pd.read_csv(
        path_2024,
        sep=";",
        encoding="latin-1",
        nrows=1,
    ).columns
    if (
        "IDADE" in column.upper()
        or "NU_IDADE" in column.upper()
        or "TP_IDADE" in column.upper()
    )
]

print(age_columns)

['NU_IDADE_N', 'TP_IDADE', 'COD_IDADE', 'OBESIDADE']


In [664]:
pr_2024_age = pd.read_csv(
    path_2024,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "NU_IDADE_N",
        "TP_IDADE",
    ],
    dtype="string",
    low_memory=False,
)

pr_2024_age["SG_UF"] = (
    pr_2024_age["SG_UF"]
    .str.strip()
    .str.upper()
)

pr_2024_age = (
    pr_2024_age.loc[
        pr_2024_age["SG_UF"].eq("PR")
    ]
    .copy()
)

pr_2024_age["DT_SIN_PRI"] = (
    pd.to_datetime(
        pr_2024_age["DT_SIN_PRI"],
        errors="coerce",
        utc=True,
    )
    .dt.tz_localize(None)
)

In [665]:
display(
    pr_2024_age[
        [
            "NU_IDADE_N",
            "TP_IDADE",
        ]
    ].head(20)
)

print(
    pr_2024_age[
        "TP_IDADE"
    ].value_counts(
        dropna=False
    )
)

,NU_IDADE_N,TP_IDADE
0,82,3
6,1,3
45,19,3
50,1,3
74,34,3
78,2,3
97,9,3
194,25,3
270,1,3
295,8,2


TP_IDADE
3    21819
2     4421
1      291
Name: count, dtype: Int64


In [666]:
def convert_age_to_years(
    age_value,
    age_type,
):
    if pd.isna(age_value) or pd.isna(age_type):
        return np.nan

    age_value = float(age_value)
    age_type = int(age_type)

    if age_type == 1:
        return age_value / 365.25

    if age_type == 2:
        return age_value / 12.0

    if age_type == 3:
        return age_value

    return np.nan

In [667]:
pr_2024_age[
    "age_years"
] = pr_2024_age.apply(
    lambda row: convert_age_to_years(
        row["NU_IDADE_N"],
        row["TP_IDADE"]
    ),
    axis=1,
)

In [668]:
display(
    pr_2024_age[
        [
            "NU_IDADE_N",
            "TP_IDADE",
            "age_years",
        ]
    ].head(30)
)

,NU_IDADE_N,TP_IDADE,age_years
0,82,3,82.000000
6,1,3,1.000000
45,19,3,19.000000
50,1,3,1.000000
74,34,3,34.000000
78,2,3,2.000000
97,9,3,9.000000
194,25,3,25.000000
270,1,3,1.000000
295,8,2,0.666667


In [669]:
def classify_age_group(age):
    if pd.isna(age):
        return pd.NA

    if age < 5:
        return "0-4"

    if age < 12:
        return "5-11"

    if age < 18:
        return "12-17"

    if age < 40:
        return "18-39"

    if age < 60:
        return "40-59"

    return "60+"

In [670]:
pr_2024_age[
    "age_group"
] = (
    pr_2024_age[
        "age_years"
    ]
    .apply(
        classify_age_group
    )
)

In [671]:
print(
    pr_2024_age[
        "age_group"
    ].value_counts(
        dropna=False
    )
)

age_group
60+      9328
0-4      9170
5-11     2699
40-59    2636
18-39    2072
12-17     626
Name: count, dtype: int64


In [672]:
pr_2024_age[
    "epi_week"
] = assign_epi_week_from_calendar(
    dates=pr_2024_age[
        "DT_SIN_PRI"
    ],
    year=2024,
)

In [673]:
pr_2024_age_weekly = (
    pr_2024_age.loc[
        pr_2024_age[
            "epi_week"
        ].notna()
        & pr_2024_age[
            "age_group"
        ].notna()
    ]
    .groupby(
        [
            "epi_week",
            "age_group", 
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "record_count",
        }
    )
)

In [674]:
display(
    pr_2024_age_weekly.loc[
        pr_2024_age_weekly[
            "epi_week"
        ].between(
            10,
            25,
        )
    ]
)

,epi_week,age_group,record_count
54,10,0-4,138
55,10,12-17,9
56,10,18-39,35
57,10,40-59,56
58,10,5-11,48
...,...,...,...
145,25,12-17,17
146,25,18-39,44
147,25,40-59,55
148,25,5-11,71


In [675]:
def build_weekly_age_history(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        source_year = int(
            f"20{name.split('INFLUD')[1][:2]}"
        )

        if source_year not in years:
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
                "NU_IDADE_N",
                "TP_IDADE",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df["NU_IDADE_N"] = pd.to_numeric(
            df["NU_IDADE_N"],
            errors="coerce",
        )

        df["TP_IDADE"] = pd.to_numeric(
            df["TP_IDADE"],
            errors="coerce",
        )

        df["age_years"] = df.apply(
            lambda row: convert_age_to_years(
                row["NU_IDADE_N"],
                row["TP_IDADE"],
            ),
            axis=1,
        )

        df["age_group"] = (
            df["age_years"]
            .apply(
                classify_age_group
            )
        )

        df["epi_week"] = (
            assign_epi_week_from_calendar(
                dates=df["DT_SIN_PRI"],
                year=source_year,
            )
        )

        weekly = (
            df.loc[
                df["epi_week"].notna()
                & df["age_group"].notna()
            ]
            .groupby(
                [
                    "epi_week",
                    "age_group",
                ],
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size": "record_count",
                }
            )
        )

        weekly["epi_year"] = (
            source_year
        )

        weekly["region"] = region

        frames.append(
            weekly
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [676]:
historical_pr_age = (
    build_weekly_age_history(
        historical_files=historical_files,
        region="PR",
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
    )
)

In [677]:
reference_pr_age_2024 = (
    historical_pr_age.loc[
        historical_pr_age[
            "epi_year"
        ].isin(
            [
                2019,
                2022,
                2023,
            ]
        )
    ]
    .groupby(
        [
            "epi_week",
            "age_group",
        ],
        as_index=False,
    )
    .agg(
        historical_median=(
            "record_count",
            "median",
        )
    )
)

In [679]:
pr_2024_age_eval = (
    historical_pr_age.loc[
        historical_pr_age[
            "epi_year"
        ].eq(2024)
    ]
    .merge(
        reference_pr_age_2024,
        on=[
            "epi_week",
            "age_group",
        ],
        how="left",
    )
)

In [680]:
pr_2024_age_eval[
    "ratio_to_median"
] = (
    pr_2024_age_eval[
        "record_count"
    ]
    / pr_2024_age_eval[
        "historical_median"
    ]
)

In [681]:
display(
    pr_2024_age_eval.loc[
        pr_2024_age_eval[
            "epi_week"
        ].between(
            13,
            24,
        )
    ][
        [
            "epi_week",
            "age_group",
            "record_count",
            "historical_median",
            "ratio_to_median",
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "age_group",
        ]
    )
)

,epi_week,age_group,record_count,historical_median,ratio_to_median
72,13,0-4,185,283.0,0.653710
73,13,12-17,11,8.0,1.375000
74,13,18-39,30,30.0,1.000000
75,13,40-59,24,58.0,0.413793
76,13,5-11,39,40.0,0.975000
...,...,...,...,...,...
139,24,12-17,23,11.0,2.090909
140,24,18-39,66,37.0,1.783784
141,24,40-59,73,64.0,1.140625
142,24,5-11,71,40.0,1.775000


In [682]:
age_ratio_pivot = (
    pr_2024_age_eval.loc[
        pr_2024_age_eval[
            "epi_week"
        ].between(
            13,
            24,
        )
    ]
    .pivot(
        index="epi_week",
        columns="age_group",
        values="ratio_to_median",
    )
)

display(age_ratio_pivot)

age_group,0-4,12-17,18-39,40-59,5-11,60+
epi_week,,,,,,
13,0.653710,1.375000,1.000000,0.413793,0.975000,0.807453
14,0.643646,1.500000,0.935484,0.660377,1.021739,0.910959
15,0.641026,0.866667,0.945946,0.460526,1.533333,0.627358
16,0.836207,1.454545,0.698113,0.712329,0.949153,0.776650
17,1.003165,2.428571,0.951220,0.493333,1.925000,0.835897
18,1.043771,2.000000,0.692308,0.800000,0.761194,0.720339
19,1.152610,1.000000,1.297297,0.820896,0.932432,0.686567
20,0.871886,0.727273,0.554054,0.516484,0.770270,0.792793
21,1.157692,0.761905,1.127660,0.767857,0.908046,0.870968


In [683]:
AGE_RATIO_THRESHOLD = 1.50
MIN_CASES = 20

In [684]:
pr_2024_age_eval[
    "age_signal_raw"
] = (
    (
        pr_2024_age_eval[
            "ratio_to_median"
        ] >= AGE_RATIO_THRESHOLD
    )
    &
    (
        pr_2024_age_eval[
            "record_count"
        ] >= MIN_CASES
    )
)

In [686]:
pr_2024_age_eval = (
    pr_2024_age_eval
    .sort_values(
        [
            "age_group",
            "epi_week",
        ]
    )
    .copy()
)

pr_2024_age_eval[
    "age_signal_previous_week"
] = (
    pr_2024_age_eval
    .groupby(
        "age_group"
    )[
        "age_signal_raw"
    ]
    .shift(1)
    .fillna(False)
)

pr_2024_age_eval[
    "age_signal_persistent"
] = (
    pr_2024_age_eval[
        "age_signal_raw"
    ]
    &
    pr_2024_age_eval[
        "age_signal_previous_week"
    ]
)

In [687]:
display(
    pr_2024_age_eval.loc[
        pr_2024_age_eval[
            "epi_week"
        ].between(10, 24),
        [
            "epi_week",
            "age_group",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "age_signal_raw",
            "age_signal_persistent",
        ],
    ]
    .sort_values(
        [
            "epi_week",
            "age_group",
        ]
    )
)

,epi_week,age_group,record_count,historical_median,ratio_to_median,age_signal_raw,age_signal_persistent
54,10,0-4,138,150.0,0.920000,False,False
55,10,12-17,9,8.0,1.125000,False,False
56,10,18-39,35,36.0,0.972222,False,False
57,10,40-59,56,80.0,0.700000,False,False
58,10,5-11,48,25.0,1.920000,True,False
...,...,...,...,...,...,...,...
139,24,12-17,23,11.0,2.090909,True,False
140,24,18-39,66,37.0,1.783784,True,False
141,24,40-59,73,64.0,1.140625,False,False
142,24,5-11,71,40.0,1.775000,True,False


In [688]:
age_week_summary = (
    pr_2024_age_eval
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        elevated_age_groups=(
            "age_signal_raw",
            "sum",
        ),
        persistent_age_groups=(
            "age_signal_persistent",
            "sum",
        ),
        max_age_ratio=(
            "ratio_to_median",
            "max",
        ),
    )
)

In [689]:
display(
    age_week_summary.loc[
        age_week_summary[
            "epi_week"
        ].between(10, 24)
    ]
)

,epi_week,elevated_age_groups,persistent_age_groups,max_age_ratio
9,10,1,0,1.920000
10,11,0,0,1.433333
11,12,0,0,1.020408
12,13,0,0,1.375000
13,14,1,0,1.500000
14,15,1,0,1.533333
15,16,0,0,1.454545
16,17,1,0,2.428571
17,18,0,0,2.000000
18,19,0,0,1.297297


In [693]:
pr_age_history_years = [
    2019,
    2022,
    2023,
]

historical_age_ratio_frames = []

for target_year in pr_age_history_years:
    baseline_years = [
        year
        for year in pr_age_history_years
        if year != target_year
    ]

    target = (
        historical_pr_age.loc[
            historical_pr_age[
                "epi_year"
            ].eq(target_year)
        ][
            [
                "epi_year",
                "epi_week",
                "age_group",
                "record_count",
            ]
        ]
        .copy()
    )

    baseline = (
        historical_pr_age.loc[
            historical_pr_age[
                "epi_year"
            ].isin(baseline_years)
        ]
        .groupby(
            [
                "epi_week",
                "age_group",
            ],
            as_index=False,
        )
        .agg(
            historical_median=(
                "record_count",
                "median",
            )
        )
    )

    target = target.merge(
        baseline,
        on=[
            "epi_week",
            "age_group",
        ],
        how="left",
    )

    target[
        "ratio_to_median"
    ] = (
        target[
            "record_count"
        ]
        / target[
            "historical_median"
        ]
    )

    historical_age_ratio_frames.append(
        target
    )

historical_pr_age_ratios = pd.concat(
    historical_age_ratio_frames,
    ignore_index=True,
)

In [694]:
def build_age_ratio_reference(
    historical_df: pd.DataFrame,
    radius: int = 2,
) -> pd.DataFrame:
    rows = []

    age_groups = (
        historical_df[
            "age_group"
        ]
        .dropna()
        .unique()
    )

    for age_group in age_groups:
        group_df = (
           historical_df.loc[
                historical_df[
                    "age_group"
                ].eq(age_group)
            ] 
        )

        for epi_week in range(1, 53):
            low = max(
                1,
                epi_week - radius,
            )

            high = min(
                52,
                epi_week + radius,
            )

            values = (
                group_df.loc[
                    group_df[
                        "epi_week"
                    ].between(
                        low,
                        high,
                    ),
                    "ratio_to_median",
                ]
                .replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .dropna()
            )

            if values.empty:
                continue

            rows.append(
                {
                    "epi_week":
                        epi_week,
                    "age_group":
                        age_group,
                    "age_ratio_median":
                        values.median(),
                    "age_ratio_q90":
                        values.quantile(
                            0.90
                        ),
                    "age_ratio_q95":
                        values.quantile(
                            0.95
                        ),
                    "age_ratio_max":
                        values.max(),
                    "historical_samples":
                        len(values),
                }
            )

    return pd.DataFrame(rows)

In [696]:
age_ratio_reference_pr = (
    build_age_ratio_reference(
        historical_df=historical_pr_age_ratios,
        radius=2,
    )
)

display(
    age_ratio_reference_pr.head(20)
)

,epi_week,age_group,age_ratio_median,age_ratio_q90,age_ratio_q95,age_ratio_max,historical_samples
0,1,0-4,0.857143,3.801419,4.311821,4.822222,9
1,2,0-4,0.780994,4.371464,4.624737,4.822222,12
2,3,0-4,0.704846,4.520147,4.637364,4.822222,15
3,4,0-4,0.704846,4.520147,4.637364,4.822222,15
4,5,0-4,0.704846,4.520147,4.637364,4.822222,15
5,6,0-4,1.277567,3.673728,4.491652,4.558140,15
6,7,0-4,1.277567,2.593823,3.231763,4.558140,15
7,8,0-4,1.277567,2.593823,2.706311,2.806630,15
8,9,0-4,1.156463,2.749305,2.979566,3.383085,15
9,10,0-4,1.070295,2.824081,2.999925,3.383085,15


In [697]:
pr_2024_age_eval_v2 = (
    pr_2024_age_eval
    .merge(
        age_ratio_reference_pr,
        on=[
            "epi_week",
            "age_group",
        ],
        how="left",
    )
)

In [698]:
pr_2024_age_eval_v2[
    "age_anomaly_q95"
] = (
    pr_2024_age_eval_v2[
        "ratio_to_median"
    ]
    >
    pr_2024_age_eval_v2[
        "age_ratio_q95"
    ]
)

In [699]:
display(
    pr_2024_age_eval_v2.loc[
        pr_2024_age_eval_v2[
            "epi_week"
        ].between(10, 24),
        [
            "epi_week",
            "age_group",
            "record_count",
            "ratio_to_median",
            "age_ratio_q90",
            "age_ratio_q95",
            "age_anomaly_q95",
        ],
    ]
    .sort_values(
        [
            "epi_week",
            "age_group",
        ]
    )
)

,epi_week,age_group,record_count,ratio_to_median,age_ratio_q90,age_ratio_q95,age_anomaly_q95
9,10,0-4,138,0.920000,2.824081,2.999925,False
61,10,12-17,9,1.125000,2.262857,2.612698,False
113,10,18-39,35,0.972222,3.112453,3.202069,False
165,10,40-59,56,0.700000,3.348107,3.946232,False
217,10,5-11,48,1.920000,4.718118,5.010953,False
...,...,...,...,...,...,...,...
75,24,12-17,23,2.090909,2.857143,3.177778,False
127,24,18-39,66,1.783784,2.728889,2.976667,False
179,24,40-59,73,1.140625,3.095520,3.259051,False
231,24,5-11,71,1.775000,2.377396,2.725695,False


In [700]:
age_anomaly_summary = (
    pr_2024_age_eval_v2
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        anomalous_age_groups=(
            "age_anomaly_q95",
            "sum",
        ),
        max_age_ratio=(
            "ratio_to_median",
            "max",
        ),
    )
)

display(
    age_anomaly_summary.loc[
        age_anomaly_summary[
            "epi_week"
        ].between(10, 24)
    ]
)

,epi_week,anomalous_age_groups,max_age_ratio
9,10,0,1.920000
10,11,0,1.433333
11,12,0,1.020408
12,13,0,1.375000
13,14,0,1.500000
14,15,0,1.533333
15,16,0,1.454545
16,17,0,2.428571
17,18,0,2.000000
18,19,0,1.297297


In [701]:
header_2024 = pd.read_csv(
    path_2024,
    sep=";",
    encoding="latin-1",
    nrows=0,
)

etiology_keywords = [
    "PCR",
    "FLU",
    "INFLU",
    "VSR",
    "RSV",
    "SARS",
    "COVID",
    "CLASSI_FIN",
]

etiology_columns = [
    column
    for column in header_2024.columns
    if any(
        keyword in column.upper()
        for keyword in etiology_keywords
    )
]

print(etiology_columns)

['PCR_RESUL', 'DT_PCR', 'POS_PCRFLU', 'TP_FLU_PCR', 'PCR_FLUASU', 'FLUASU_OUT', 'PCR_FLUBLI', 'FLUBLI_OUT', 'POS_PCROUT', 'PCR_VSR', 'PCR_PARA1', 'PCR_PARA2', 'PCR_PARA3', 'PCR_PARA4', 'PCR_ADENO', 'PCR_METAP', 'PCR_BOCA', 'PCR_RINO', 'PCR_OUTRO', 'DS_PCR_OUT', 'CLASSI_FIN', 'PCR_SARS2', 'POS_AN_FLU', 'TP_FLU_AN', 'AN_SARS2', 'AN_VSR']


In [702]:
for column in etiology_columns:
    print(
        "\n",
        "=" * 60,
        "\n",
        column,
    )

    sample = pd.read_csv(
        path_2024,
        sep=";",
        encoding="latin-1",
        usecols=["SG_UF", column],
        dtype="string",
        low_memory=False,
    )

    sample["SG_UF"] = (
        sample["SG_UF"]
        .str.strip()
        .str.upper()
    )

    print(
        sample.loc[
            sample["SG_UF"].eq("PR"),
            column,
        ]
        .value_counts(
            dropna=False
        )
        .head(15)
    )


 PCR_RESUL
PCR_RESUL
2       11336
1        9929
4        3629
<NA>      949
5         661
9          21
3           6
Name: count, dtype: Int64

 DT_PCR
DT_PCR
<NA>                        5241
2024-06-24T00:00:00.000Z     277
2024-06-14T00:00:00.000Z     183
2024-09-19T00:00:00.000Z     176
2024-06-04T00:00:00.000Z     163
2024-06-19T00:00:00.000Z     162
2024-06-10T00:00:00.000Z     157
2025-01-09T00:00:00.000Z     157
2024-08-27T00:00:00.000Z     153
2024-11-19T00:00:00.000Z     152
2024-09-30T00:00:00.000Z     151
2024-07-26T00:00:00.000Z     151
2024-08-23T00:00:00.000Z     145
2024-10-10T00:00:00.000Z     143
2024-05-06T00:00:00.000Z     142
Name: count, dtype: Int64

 POS_PCRFLU
POS_PCRFLU
<NA>    17236
2        7203
1        2079
9          13
Name: count, dtype: Int64

 TP_FLU_PCR
TP_FLU_PCR
<NA>    24452
1        1781
2         298
Name: count, dtype: Int64

 PCR_FLUASU
PCR_FLUASU
<NA>    24750
2         782
1         657
3         340
5           2
Name: count, dtype: Int64

In [703]:
pr_2024_etiology = pd.read_csv(
    path_2024,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "PCR_RESUL",
        "POS_PCRFLU",
        "TP_FLU_PCR",
        "PCR_VSR",
        "PCR_SARS2",
        "PCR_RINO",
        "PCR_METAP",
        "CLASSI_FIN",
    ],
    dtype="string",
    low_memory=False,
)

pr_2024_etiology["SG_UF"] = (
    pr_2024_etiology["SG_UF"]
    .str.strip()
    .str.upper()
)

pr_2024_etiology = (
    pr_2024_etiology.loc[
        pr_2024_etiology[
            "SG_UF"
        ].eq("PR")
    ]
    .copy()
)

pr_2024_etiology["DT_SIN_PRI"] = (
    pd.to_datetime(
        pr_2024_etiology[
            "DT_SIN_PRI"
        ],
        errors="coerce",
        utc=True,
    )
    .dt.tz_localize(None)
)

pr_2024_etiology["epi_week"] = (
    assign_epi_week_from_calendar(
        dates=pr_2024_etiology[
            "DT_SIN_PRI"
        ],
        year=2024,
    )
)


In [704]:
for column in [
    "PCR_RESUL",
    "POS_PCRFLU",
    "TP_FLU_PCR",
    "PCR_VSR",
    "PCR_SARS2",
    "PCR_RINO",
    "PCR_METAP",
    "CLASSI_FIN",
]:
    pr_2024_etiology[column] = (
        pd.to_numeric(
            pr_2024_etiology[column],
            errors="coerce",
        )
    )

In [705]:
pr_2024_etiology[
    "flu_pcr_tested"
] = (
    pr_2024_etiology[
        "POS_PCRFLU"
    ].isin([1, 2])
)

pr_2024_etiology[
    "flu_pcr_positive"
] = (
    pr_2024_etiology[
        "POS_PCRFLU"
    ].eq(1)
)

In [706]:
pr_2024_flu_weekly = (
    pr_2024_etiology.loc[
        pr_2024_etiology[
            "epi_week"
        ].notna()
    ]
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        srag_cases=(
            "epi_week",
            "size",
        ),
        flu_tests=(
            "flu_pcr_tested",
            "sum",
        ),
        flu_positive=(
            "flu_pcr_positive",
            "sum",
        ),
    )
)

In [707]:
pr_2024_flu_weekly[
    "flu_positivity"
] = (
    pr_2024_flu_weekly[
        "flu_positive"
    ]
    / pr_2024_flu_weekly[
        "flu_tests"
    ]
)

In [708]:
display(
    pr_2024_flu_weekly.loc[
        pr_2024_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,srag_cases,flu_tests,flu_positive,flu_positivity
7,8,438,140,12,0.085714
8,9,475,173,12,0.069364
9,10,492,156,19,0.121795
10,11,490,164,13,0.079268
11,12,474,171,26,0.152047
12,13,419,160,30,0.1875
13,14,498,202,49,0.242574
14,15,535,230,52,0.226087
15,16,605,273,76,0.278388
16,17,650,297,92,0.309764


In [709]:
pr_2024_etiology[
    "flu_type_1"
] = (
    pr_2024_etiology[
        "TP_FLU_PCR"
    ].eq(1)
)

pr_2024_etiology[
    "flu_type_2"
] = (
    pr_2024_etiology[
        "TP_FLU_PCR"
    ].eq(2)
)

pr_2024_flu_types = (
    pr_2024_etiology.loc[
        pr_2024_etiology[
            "epi_week"
        ].notna()
    ]
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        flu_type_1=(
            "flu_type_1",
            "sum",
        ),
        flu_type_2=(
            "flu_type_2",
            "sum",
        ),
    )
)

display(
    pr_2024_flu_types.loc[
        pr_2024_flu_types[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,flu_type_1,flu_type_2
7,8,12,0
8,9,11,1
9,10,18,1
10,11,12,1
11,12,26,0
12,13,29,1
13,14,49,0
14,15,52,0
15,16,76,0
16,17,92,0


In [710]:
for pathogen in [
    "PCR_VSR",
    "PCR_SARS2",
    "PCR_RINO",
    "PCR_METAP",
]:
    pr_2024_etiology[
        f"{pathogen}_positive"
    ] = (
        pr_2024_etiology[
            pathogen
        ].eq(1)
    )

In [711]:
pr_2024_pathogens_weekly = (
    pr_2024_etiology.loc[
        pr_2024_etiology[
            "epi_week"
        ].notna()
    ]
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        vsr_positive=(
            "PCR_VSR_positive",
            "sum",
        ),
        sars2_positive=(
            "PCR_SARS2_positive",
            "sum",
        ),
        rino_positive=(
            "PCR_RINO_positive",
            "sum",
        ),
        metap_positive=(
            "PCR_METAP_positive",
            "sum",
        ),
    )
)

display(
    pr_2024_pathogens_weekly.loc[
        pr_2024_pathogens_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,vsr_positive,sars2_positive,rino_positive,metap_positive
7,8,19,77,39,1
8,9,34,84,74,1
9,10,29,66,52,1
10,11,55,63,56,3
11,12,68,39,54,0
12,13,74,20,47,3
13,14,105,13,63,2
14,15,139,9,69,4
15,16,163,11,69,1
16,17,182,4,66,2


In [712]:
def build_historical_flu_positivity(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        matched_year = None

        for year in years:
            if f"INFLUD{str(year)[-2:]}" in name:
                matched_year = year
                break

        if matched_year is None:
            continue

        header = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            nrows=0,
        )

        required = {
            "SG_UF",
            "DT_SIN_PRI",
            "POS_PCRFLU",
        }

        if not required.issubset(
            header.columns
        ):
            print(
                f"{matched_year}: "
                "campos de influenza ausentes"
            )
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
                "POS_PCRFLU",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df["POS_PCRFLU"] = pd.to_numeric(
            df["POS_PCRFLU"],
            errors="coerce",
        )

        df["epi_week"] = (
            assign_epi_week_from_calendar(
                dates=df["DT_SIN_PRI"],
                year=matched_year,
            )
        )

        df["flu_tested"] = (
            df["POS_PCRFLU"]
            .isin([1, 2])
        )

        df["flu_positive"] = (
            df["POS_PCRFLU"]
            .eq(1)
        )

        weekly = (
            df.loc[
                df["epi_week"].notna()
            ]
            .groupby(
                "epi_week",
                as_index=False,
            )
            .agg(
                flu_tests=(
                    "flu_tested",
                    "sum",
                ),
                flu_positive=(
                    "flu_positive",
                    "sum",
                ),
            )
        )

        weekly["flu_positivity"] = (
            weekly["flu_positive"]
            / weekly["flu_tests"]
        )

        weekly["epi_year"] = (
            matched_year
        )

        frames.append(weekly)

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [713]:
historical_pr_flu = (
    build_historical_flu_positivity(
        historical_files=historical_files,
        region="PR",
        years=[
            2019,
            2022,
            2023,
        ],
    )
)

display(
    historical_pr_flu.head(20)
)

,epi_week,flu_tests,flu_positive,flu_positivity,epi_year
0,1,8,1,0.125,2019
1,2,16,2,0.125,2019
2,3,11,1,0.090909,2019
3,4,7,2,0.285714,2019
4,5,5,0,0.0,2019
5,6,6,0,0.0,2019
6,7,8,2,0.25,2019
7,8,13,0,0.0,2019
8,9,22,2,0.090909,2019
9,10,21,2,0.095238,2019


In [714]:
display(
    historical_pr_flu
    .groupby("epi_year")
    .agg(
        total_tests=(
            "flu_tests",
            "sum",
        ),
        total_positive=(
            "flu_positive",
            "sum",
        ),
        median_weekly_positivity=(
            "flu_positivity",
            "median",
        ),
    )
)

,total_tests,total_positive,median_weekly_positivity
epi_year,,,
2019,2977,770,0.207912
2022,17825,1825,0.062461
2023,8888,883,0.038762


In [715]:
flu_reference_pr_recent = (
    historical_pr_flu.loc[
        historical_pr_flu[
            "epi_year"
        ].isin([2022, 2023])
    ]
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        historical_flu_median=(
            "flu_positivity",
            "median",
        ),
        historical_flu_min=(
            "flu_positivity",
            "min",
        ),
        historical_flu_max=(
            "flu_positivity",
            "max",
        ),
        historical_years=(
            "epi_year",
            "nunique",
        ),
    )
)

In [716]:
pr_2024_flu_eval_recent = (
    pr_2024_flu_weekly
    .merge(
        flu_reference_pr_recent,
        on="epi_week",
        how="left",
    )
)

pr_2024_flu_eval_recent[
    "positivity_ratio"
] = (
    pr_2024_flu_eval_recent[
        "flu_positivity"
    ]
    / pr_2024_flu_eval_recent[
        "historical_flu_median"
    ]
)

In [717]:
pr_2024_flu_eval_recent[
    "above_both_recent_years"
] = (
    pr_2024_flu_eval_recent[
        "flu_positivity"
    ]
    >
    pr_2024_flu_eval_recent[
        "historical_flu_max"
    ]
)

In [718]:
display(
    pr_2024_flu_eval_recent.loc[
        pr_2024_flu_eval_recent[
            "epi_week"
        ].between(8, 26),
        [
            "epi_week",
            "flu_tests",
            "flu_positive",
            "flu_positivity",
            "historical_flu_median",
            "historical_flu_min",
            "historical_flu_max",
            "positivity_ratio",
            "above_both_recent_years",
        ],
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,historical_flu_median,historical_flu_min,historical_flu_max,positivity_ratio,above_both_recent_years
7,8,140,12,0.085714,0.030152,0.01227,0.048035,2.842698,True
8,9,173,12,0.069364,0.042694,0.030303,0.055085,1.624686,True
9,10,156,19,0.121795,0.027009,0.007143,0.046875,4.50943,True
10,11,164,13,0.079268,0.029978,0.011628,0.048327,2.644258,True
11,12,171,26,0.152047,0.056883,0.022857,0.090909,2.672969,True
12,13,160,30,0.1875,0.060343,0.014563,0.106122,3.107248,True
13,14,202,49,0.242574,0.081838,0.035857,0.12782,2.964076,True
14,15,230,52,0.226087,0.090237,0.011194,0.169279,2.505493,True
15,16,273,76,0.278388,0.08477,0.008955,0.160584,3.284059,True
16,17,297,92,0.309764,0.10309,0.024024,0.182156,3.004793,True


In [719]:
flu_reference_pr_all = (
    historical_pr_flu
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        historical_flu_median_all=(
            "flu_positivity",
            "median",
        ),
        historical_flu_max_all=(
            "flu_positivity",
            "max",
        ),
    )
)

pr_2024_flu_sensitivity = (
    pr_2024_flu_weekly
    .merge(
        flu_reference_pr_all,
        on="epi_week",
        how="left",
    )
)

pr_2024_flu_sensitivity[
    "ratio_all_years"
] = (
    pr_2024_flu_sensitivity[
        "flu_positivity"
    ]
    / pr_2024_flu_sensitivity[
        "historical_flu_median_all"
    ]
)

In [720]:
import numpy as np
import pandas as pd


def bernoulli_log_likelihood(
    positives: int,
    tests: int,
) -> float:
    if tests <= 0:
        return np.nan

    p = positives / tests

    eps = 1e-12

    p = np.clip(
        p,
        eps,
        1 - eps,
    )

    negatives = tests - positives

    return (
        positives * np.log(p)
        + negatives * np.log(1 - p)
    )

In [721]:
flu_cp = (
    pr_2024_flu_weekly.loc[
        pr_2024_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
    .copy()
)

results = []

min_weeks_each_side = 3

total_positive = int(
    flu_cp["flu_positive"].sum()
)

total_tests = int(
    flu_cp["flu_tests"].sum()
)

null_ll = bernoulli_log_likelihood(
    total_positive,
    total_tests,
)

for split_idx in range(
    min_weeks_each_side,
    len(flu_cp) - min_weeks_each_side + 1,
):
    left = flu_cp.iloc[:split_idx]
    right = flu_cp.iloc[split_idx:]

    left_positive = int(
        left["flu_positive"].sum()
    )

    left_tests = int(
        left["flu_tests"].sum()
    )

    right_positive = int(
        right["flu_positive"].sum()
    )

    right_tests = int(
        right["flu_tests"].sum()
    )

    alternative_ll = (
        bernoulli_log_likelihood(
            left_positive,
            left_tests,
        )
        +
        bernoulli_log_likelihood(
            right_positive,
            right_tests,
        )
    )

    llr = 2 * (
        alternative_ll - null_ll
    )

    results.append(
        {
            "split_after_week":
                int(
                    left[
                        "epi_week"
                    ].iloc[-1]
                ),
            "next_week":
                int(
                    right[
                        "epi_week"
                    ].iloc[0]
                ),
            "left_positivity":
                left_positive
                / left_tests,
            "right_positivity":
                right_positive
                / right_tests,
            "llr":
                llr,
        }
    )

flu_change_points = (
    pd.DataFrame(results)
    .sort_values(
        "llr",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    flu_change_points.head(10)
)

,split_after_week,next_week,left_positivity,right_positivity,llr
0,12,13,0.101990,0.271400,118.959612
1,13,14,0.116183,0.275428,116.902318
2,11,12,0.088468,0.265830,111.121005
3,14,15,0.138079,0.277547,98.348350
4,15,16,0.152579,0.281627,91.214706
5,10,11,0.091684,0.257837,75.698823
6,16,17,0.173158,0.281963,68.425323
7,17,18,0.193795,0.278421,42.379829
8,18,19,0.203953,0.278127,32.406436
9,19,20,0.210526,0.279146,26.943224


In [722]:
best_change = (
    flu_change_points.iloc[0]
)

print(
    "Melhor mudança de regime:"
)

print(
    f"entre SE"
    f"{int(best_change['split_after_week'])} "
    f"e SE"
    f"{int(best_change['next_week'])}"
)

print(
    "Positividade antes:",
    f"{best_change['left_positivity']:.1%}",
)

print(
    "Positividade depois:",
    f"{best_change['right_positivity']:.1%}",
)

print(
    "LLR:",
    round(
        best_change["llr"],
        3,
    ),
)

Melhor mudança de regime:
entre SE12 e SE13
Positividade antes: 10.2%
Positividade depois: 27.1%
LLR: 118.96


In [723]:
flu_cp[
    "positivity_ma3"
] = (
    flu_cp[
        "flu_positivity"
    ]
    .rolling(
        window=3,
        center=True,
        min_periods=2,
    )
    .mean()
)

display(
    flu_cp[
        [
            "epi_week",
            "flu_tests",
            "flu_positive",
            "flu_positivity",
            "positivity_ma3",
        ]
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,positivity_ma3
0,8,140,12,0.085714,0.077539
1,9,173,12,0.069364,0.092291
2,10,156,19,0.121795,0.090142
3,11,164,13,0.079268,0.117703
4,12,171,26,0.152047,0.139605
5,13,160,30,0.1875,0.194040
6,14,202,49,0.242574,0.218720
7,15,230,52,0.226087,0.249016
8,16,273,76,0.278388,0.271413
9,17,297,92,0.309764,0.289641


In [1080]:
def detect_flu_change_point(
    weekly_df: pd.DataFrame,
    week_start: int = 8,
    week_end: int = 26,
    min_weeks_each_side: int = 3,
):
    data = (
        weekly_df.loc[
            weekly_df["epi_week"].between(
                week_start,
                week_end,
            )
        ]
        .sort_values("epi_week")
        .reset_index(drop=True)
        .copy()
    )

    # Não há semanas suficientes para formar os dois lados
    if len(data) < (2 * min_weeks_each_side):
        return pd.DataFrame(
            columns=[
                "split_after_week",
                "next_week",
                "left_positivity",
                "right_positivity",
                "llr",
            ]
        )

    total_positive = int(
        data["flu_positive"].sum()
    )

    total_tests = int(
        data["flu_tests"].sum()
    )

    # Nenhuma observação disponível no período inteiro
    if total_tests <= 0:
        return pd.DataFrame(
            columns=[
                "split_after_week",
                "next_week",
                "left_positivity",
                "right_positivity",
                "llr",
            ]
        )

    null_ll = bernoulli_log_likelihood(
        total_positive,
        total_tests,
    )

    results = []

    for split_idx in range(
        min_weeks_each_side,
        len(data) - min_weeks_each_side + 1,
    ):
        left = data.iloc[:split_idx]
        right = data.iloc[split_idx:]

        left_positive = int(
            left["flu_positive"].sum()
        )

        left_tests = int(
            left["flu_tests"].sum()
        )

        right_positive = int(
            right["flu_positive"].sum()
        )

        right_tests = int(
            right["flu_tests"].sum()
        )

        # Um dos lados não possui nenhum teste.
        # Portanto, esse split não é estatisticamente avaliável.
        if left_tests <= 0 or right_tests <= 0:
            continue

        alternative_ll = (
            bernoulli_log_likelihood(
                left_positive,
                left_tests,
            )
            +
            bernoulli_log_likelihood(
                right_positive,
                right_tests,
            )
        )

        llr = 2 * (
            alternative_ll - null_ll
        )

        results.append(
            {
                "split_after_week": int(
                    left[
                        "epi_week"
                    ].iloc[-1]
                ),
                "next_week": int(
                    right[
                        "epi_week"
                    ].iloc[0]
                ),
                "left_positivity": (
                    left_positive
                    / left_tests
                ),
                "right_positivity": (
                    right_positive
                    / right_tests
                ),
                "llr": llr,
            }
        )

    # Todos os splits podem ter sido inválidos
    if not results:
        return pd.DataFrame(
            columns=[
                "split_after_week",
                "next_week",
                "left_positivity",
                "right_positivity",
                "llr",
            ]
        )

    return (
        pd.DataFrame(results)
        .sort_values(
            "llr",
            ascending=False,
        )
        .reset_index(drop=True)
    )

In [725]:
historical_change_points = []

for year in [
    2019,
    2022,
    2023,
]:
    yearly = (
        historical_pr_flu.loc[
            historical_pr_flu[
                "epi_year"
            ].eq(year)
        ]
        .copy()
    )

    result = detect_flu_change_point(
        weekly_df=yearly,
        week_start=8,
        week_end=26,
    )

    best = result.iloc[0]

    historical_change_points.append(
        {
            "epi_year": year,
            "split_after_week":
                int(
                    best[
                        "split_after_week"
                    ]
                ),
            "next_week":
                int(
                    best[
                        "next_week"
                    ]
                ),
            "left_positivity":
                best[
                    "left_positivity"
                ],
            "right_positivity":
                best[
                    "right_positivity"
                ],
            "llr":
                best["llr"],
        }
    )

historical_change_points = (
    pd.DataFrame(
        historical_change_points
    )
)

display(
    historical_change_points
)

,epi_year,split_after_week,next_week,left_positivity,right_positivity,llr
0,2019,14,15,0.084906,0.315077,58.845799
1,2022,20,21,0.026600,0.146477,316.154765
2,2023,14,15,0.076701,0.210800,161.638263


In [726]:
best_2024 = (
    flu_change_points.iloc[0]
)

comparison_change_points = pd.concat(
    [
        historical_change_points,
        pd.DataFrame(
            [
                {
                    "epi_year": 2024,
                    "split_after_week":
                        int(
                            best_2024[
                                "split_after_week"
                            ]
                        ),
                    "next_week":
                        int(
                            best_2024[
                                "next_week"
                            ]
                        ),
                    "left_positivity":
                        best_2024[
                            "left_positivity"
                        ],
                    "right_positivity":
                        best_2024[
                            "right_positivity"
                        ],
                    "llr":
                        best_2024[
                            "llr"
                        ],
                }
            ]
        ),
    ],
    ignore_index=True,
)

display(
    comparison_change_points
)

,epi_year,split_after_week,next_week,left_positivity,right_positivity,llr
0,2019,14,15,0.084906,0.315077,58.845799
1,2022,20,21,0.026600,0.146477,316.154765
2,2023,14,15,0.076701,0.210800,161.638263
3,2024,12,13,0.101990,0.271400,118.959612


In [727]:
prospective_results = []

for current_week in range(
    13,
    27,
):
    partial = (
        pr_2024_flu_weekly.loc[
            pr_2024_flu_weekly[
                "epi_week"
            ].between(
                8,
                current_week,
            )
        ]
        .copy()
    )

    if len(partial) < 6:
        continue

    cp_result = detect_flu_change_point(
        weekly_df=partial,
        week_start=8,
        week_end=current_week,
        min_weeks_each_side=3,
    )

    if cp_result.empty:
        continue

    best = cp_result.iloc[0]

    prospective_results.append(
        {
            "data_available_until":
                current_week,
            "detected_split_after":
                int(
                    best[
                        "split_after_week"
                    ]
                ),
            "detected_change_week":
                int(
                    best[
                        "next_week"
                    ]
                ),
            "left_positivity":
                best[
                    "left_positivity"
                ],
            "right_positivity":
                best[
                    "right_positivity"
                ],
            "llr":
                best["llr"],
        }
    )

prospective_flu_cp = pd.DataFrame(
    prospective_results
)

display(
    prospective_flu_cp
)

,data_available_until,detected_split_after,detected_change_week,left_positivity,right_positivity,llr
0,13,10,11,0.091684,0.139394,5.389799
1,14,11,12,0.088468,0.196998,28.737570
2,15,11,12,0.088468,0.205767,38.458209
3,16,12,13,0.101990,0.239306,56.631187
4,17,12,13,0.101990,0.257315,78.336684
5,18,12,13,0.101990,0.261603,87.720122
6,19,12,13,0.101990,0.262905,93.179553
7,20,12,13,0.101990,0.268641,102.190639
8,21,12,13,0.101990,0.265250,102.027290
9,22,12,13,0.101990,0.267704,107.426050


In [728]:
prospective_flu_cp[
    "matches_final_change"
] = (
    prospective_flu_cp[
        "detected_split_after"
    ].eq(12)
    &
    prospective_flu_cp[
        "detected_change_week"
    ].eq(13)
)

display(
    prospective_flu_cp
)

,data_available_until,detected_split_after,detected_change_week,left_positivity,right_positivity,llr,matches_final_change
0,13,10,11,0.091684,0.139394,5.389799,False
1,14,11,12,0.088468,0.196998,28.737570,False
2,15,11,12,0.088468,0.205767,38.458209,False
3,16,12,13,0.101990,0.239306,56.631187,True
4,17,12,13,0.101990,0.257315,78.336684,True
5,18,12,13,0.101990,0.261603,87.720122,True
6,19,12,13,0.101990,0.262905,93.179553,True
7,20,12,13,0.101990,0.268641,102.190639,True
8,21,12,13,0.101990,0.265250,102.027290,True
9,22,12,13,0.101990,0.267704,107.426050,True


In [729]:
first_stable = (
    prospective_flu_cp.loc[
        prospective_flu_cp[
            "matches_final_change"
        ]
    ]
)

display(first_stable)

,data_available_until,detected_split_after,detected_change_week,left_positivity,right_positivity,llr,matches_final_change
3,16,12,13,0.10199,0.239306,56.631187,True
4,17,12,13,0.10199,0.257315,78.336684,True
5,18,12,13,0.10199,0.261603,87.720122,True
6,19,12,13,0.10199,0.262905,93.179553,True
7,20,12,13,0.10199,0.268641,102.190639,True
8,21,12,13,0.10199,0.265250,102.027290,True
9,22,12,13,0.10199,0.267704,107.426050,True
10,23,12,13,0.10199,0.269175,111.542143,True
11,24,12,13,0.10199,0.271743,116.674842,True
12,25,12,13,0.10199,0.271597,118.097585,True


In [730]:
def build_prospective_change_detection(
    weekly_df: pd.DataFrame,
    final_split_after: int,
    final_change_week: int,
    week_start: int = 8,
    week_end: int = 26,
    min_weeks_each_side: int = 3,
) -> pd.DataFrame:
    rows = []

    for current_week in range(
        week_start + (2 * min_weeks_each_side) - 1,
        week_end + 1,
    ):
        partial = (
            weekly_df.loc[
                weekly_df[
                    "epi_week"
                ].between(
                    week_start,
                    current_week,
                )
            ]
            .copy()
        )

        result = detect_flu_change_point(
            weekly_df=partial,
            week_start=week_start,
            week_end=current_week,
            min_weeks_each_side=min_weeks_each_side,
        )

        if result.empty:
            continue

        best = result.iloc[0]

        rows.append(
            {
                "data_available_until":
                    current_week,
                "detected_split_after":
                    int(
                        best[
                            "split_after_week"
                        ]
                    ),
                "detected_change_week":
                    int(
                        best[
                            "next_week"
                        ]
                    ),
                "left_positivity":
                    best[
                        "left_positivity"
                    ],
                "right_positivity":
                    best[
                        "right_positivity"
                    ],
                "llr":
                    best[
                        "llr"
                    ],
                "matches_final_change":
                    (
                        int(
                            best[
                                "split_after_week"
                            ]
                        )
                        == final_split_after
                        and
                        int(
                            best[
                                "next_week"
                            ]
                        )
                        == final_change_week
                    ),
            }
        )

    return pd.DataFrame(rows)

In [735]:
prospective_summary = []

for year in [
    2019,
    2022,
    2023,
    2024,
]:
    if year == 2024:
        yearly = pr_2024_flu_weekly.copy()
    else:
        yearly = (
            historical_pr_flu.loc[
                historical_pr_flu[
                    "epi_year"
                ].eq(year)
            ]
            .copy()
        )

    final_result = (
        detect_flu_change_point(
            weekly_df=yearly,
            week_start=8,
            week_end=26,
        )
    )

    final_best = (
        final_result.iloc[0]
    )

    final_split = int(
        final_best[
            "split_after_week"
        ]
    )

    final_change = int(
        final_best[
            "next_week"
        ]
    )

    prospective = (
        build_prospective_change_detection(
            weekly_df=yearly,
            final_split_after=final_split,
            final_change_week=final_change,
            week_start=8,
            week_end=26,
        )
    )

    matches = (
        prospective.loc[
            prospective[
                "matches_final_change"
            ]
        ]
    )

    first_match_week = (
        first_stable_match_week(
            prospective
        )
    )

    prospective_summary.append(
        {
            "epi_year": year,
            "final_split_after":
                final_split,
            "final_change_week":
                final_change,
            "final_llr":
                final_best["llr"],
            "first_matching_detection_week":
                first_match_week,
            "detection_delay_weeks":
                (
                    first_match_week
                    - final_change
                    if pd.notna(
                        first_match_week
                    )
                    else np.nan
                ),
        }
    )

prospective_summary = (
    pd.DataFrame(
        prospective_summary
    )
)

display(
    prospective_summary
)

,epi_year,final_split_after,final_change_week,final_llr,first_matching_detection_week,detection_delay_weeks
0,2019,14,15,58.845799,17,2
1,2022,20,21,316.154765,23,2
2,2023,14,15,161.638263,19,4
3,2024,12,13,118.959612,16,3


In [734]:
def first_stable_match_week(
    prospective_df: pd.DataFrame,
):
    data = (
        prospective_df
        .sort_values(
            "data_available_until"
        )
        .reset_index(drop=True)
    )

    for i in range(len(data)):
        remaining = (
            data.loc[
                i:,
                "matches_final_change",
            ]
        )

        if remaining.all():
            return int(
                data.loc[
                    i,
                    "data_available_until",
                ]
            )

    return np.nan

In [736]:
change_timing_summary = (
    prospective_summary[
        [
            "epi_year",
            "final_change_week",
            "first_matching_detection_week",
            "detection_delay_weeks",
            "final_llr",
        ]
    ]
    .copy()
)

reference_change_weeks = (
    change_timing_summary.loc[
        change_timing_summary[
            "epi_year"
        ].isin([2019, 2022, 2023]),
        "final_change_week",
    ]
)

historical_median_change_week = (
    reference_change_weeks.median()
)

change_timing_summary[
    "weeks_earlier_than_historical_median"
] = (
    historical_median_change_week
    - change_timing_summary[
        "final_change_week"
    ]
)

display(change_timing_summary)

print(
    "Mediana histórica da mudança:",
    historical_median_change_week,
)

,epi_year,final_change_week,first_matching_detection_week,detection_delay_weeks,final_llr,weeks_earlier_than_historical_median
0,2019,15,17,2,58.845799,0.0
1,2022,21,23,2,316.154765,-6.0
2,2023,15,19,4,161.638263,0.0
3,2024,13,16,3,118.959612,2.0


Mediana histórica da mudança: 15.0


In [737]:
pr_2024_shift_result = {
    "region": "PR",
    "year": 2024,
    "pathogen": "influenza",
    "change_week": 13,
    "stable_detection_week": 16,
    "detection_delay_weeks": 3,
    "pre_change_positivity": 0.101990,
    "post_change_positivity": 0.271400,
    "final_llr": 118.959612,
}

pd.DataFrame(
    [pr_2024_shift_result]
)

,region,year,pathogen,change_week,stable_detection_week,detection_delay_weeks,pre_change_positivity,post_change_positivity,final_llr
0,PR,2024,influenza,13,16,3,0.10199,0.2714,118.959612


In [ ]:
def build_flu_weekly_for_state_year(
    path,
    region: str,
    year: int,
) -> pd.DataFrame:

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "POS_PCRFLU",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    return build_flu_weekly_from_df(
        df=df,
        region=region,
        year=year,
    )

In [971]:
def build_flu_weekly_from_df(
    df: pd.DataFrame,
    region: str,
    year: int,
) -> pd.DataFrame:

    state_df = df.loc[
        df["SG_UF"].eq(region)
    ].copy()

    state_df["DT_SIN_PRI"] = (
        pd.to_datetime(
            state_df["DT_SIN_PRI"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    state_df["POS_PCRFLU"] = pd.to_numeric(
        state_df["POS_PCRFLU"],
        errors="coerce",
    )

    state_df["epi_week"] = (
        assign_epi_week_from_calendar(
            dates=state_df["DT_SIN_PRI"],
            year=year,
        )
    )

    state_df["flu_tested"] = (
        state_df["POS_PCRFLU"]
        .isin([1, 2])
    )

    state_df["flu_positive"] = (
        state_df["POS_PCRFLU"]
        .eq(1)
    )

    weekly = (
        state_df.loc[
            state_df["epi_week"].notna()
        ]
        .groupby(
            "epi_week",
            as_index=False,
        )
        .agg(
            flu_tests=(
                "flu_tested",
                "sum",
            ),
            flu_positive=(
                "flu_positive",
                "sum",
            ),
        )
    )

    weekly["flu_positivity"] = (
        weekly["flu_positive"]
        / weekly["flu_tests"]
    )

    weekly["region"] = region
    weekly["epi_year"] = year

    return weekly

In [972]:
raw_year_cache = {}


def load_influd_year_once(
    year: int,
) -> pd.DataFrame:

    if year in raw_year_cache:
        return raw_year_cache[year]

    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}"
        in p.name.upper()
    )

    print(
        f"Lendo INFLUD{str(year)[-2:]}..."
    )

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "POS_PCRFLU",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    raw_year_cache[year] = df

    return df

In [739]:
path_2024 = next(
    path
    for path in historical_files
    if "INFLUD24" in path.name.upper()
)

In [740]:
sc_2024_flu_weekly = (
    build_flu_weekly_for_state_year(
        path=path_2024,
        region="SC",
        year=2024,
    )
)

display(
    sc_2024_flu_weekly.loc[
        sc_2024_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,region,epi_year
7,8,113,24,0.212389,SC,2024
8,9,135,31,0.22963,SC,2024
9,10,172,52,0.302326,SC,2024
10,11,194,45,0.231959,SC,2024
11,12,153,37,0.24183,SC,2024
12,13,182,43,0.236264,SC,2024
13,14,165,41,0.248485,SC,2024
14,15,202,59,0.292079,SC,2024
15,16,231,66,0.285714,SC,2024
16,17,233,62,0.266094,SC,2024


In [742]:
sc_2024_change = (
    detect_flu_change_point(
        weekly_df=sc_2024_flu_weekly,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

display(
    sc_2024_change.head(10)
)

,split_after_week,next_week,left_positivity,right_positivity,llr
0,21,22,0.264695,0.220118,6.879969
1,22,23,0.262883,0.214286,6.750388
2,20,21,0.265828,0.226236,6.190702
3,23,24,0.260376,0.208333,5.622748
4,19,20,0.265369,0.233466,4.372331
5,18,19,0.265779,0.237903,3.534411
6,14,15,0.245063,0.258118,0.687519
7,13,14,0.244468,0.257499,0.624263
8,17,18,0.258427,0.249424,0.376014
9,12,13,0.246415,0.256093,0.297902


In [743]:
best_sc_2024 = (
    sc_2024_change.iloc[0]
)

print(
    "SC/2024"
)

print(
    "Mudança final:",
    f"SE{int(best_sc_2024['split_after_week'])}"
    f"→SE{int(best_sc_2024['next_week'])}",
)

print(
    "Positividade antes:",
    f"{best_sc_2024['left_positivity']:.1%}",
)

print(
    "Positividade depois:",
    f"{best_sc_2024['right_positivity']:.1%}",
)

print(
    "LLR:",
    round(
        best_sc_2024["llr"],
        3,
    ),
)

SC/2024
Mudança final: SE21→SE22
Positividade antes: 26.5%
Positividade depois: 22.0%
LLR: 6.88


In [744]:
sc_2024_prospective = (
    build_prospective_change_detection(
        weekly_df=sc_2024_flu_weekly,
        final_split_after=int(
            best_sc_2024[
                "split_after_week"
            ]
        ),
        final_change_week=int(
            best_sc_2024[
                "next_week"
            ]
        ),
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

display(
    sc_2024_prospective
)

,data_available_until,detected_split_after,detected_change_week,left_positivity,right_positivity,llr,matches_final_change
0,13,10,11,0.254762,0.236295,0.431509,False
1,14,10,11,0.254762,0.239193,0.341651,False
2,15,12,13,0.246415,0.260474,0.334625,False
3,16,13,14,0.244468,0.277592,2.094267,False
4,17,14,15,0.245063,0.280781,2.755732,False
5,18,14,15,0.245063,0.291028,5.421590,False
6,19,14,15,0.245063,0.285092,4.650430,False
7,20,14,15,0.245063,0.282963,4.508324,False
8,21,14,15,0.245063,0.278741,3.803537,False
9,22,14,15,0.245063,0.274175,2.999738,False


In [745]:
sc_2024_stable_week = (
    first_stable_match_week(
        sc_2024_prospective
    )
)

print(
    "Primeira detecção estável:",
    sc_2024_stable_week,
)

print(
    "Atraso:",
    (
        sc_2024_stable_week
        - int(
            best_sc_2024[
                "next_week"
            ]
        )
    ),
    "semanas",
)

Primeira detecção estável: 26
Atraso: 4 semanas


In [746]:
external_shift_validation = pd.DataFrame(
    [
        {
            "region": "PR",
            "year": 2024,
            "change_week": 13,
            "stable_detection_week": 16,
            "detection_delay_weeks": 3,
            "pre_positivity": 0.101990,
            "post_positivity": 0.271400,
            "llr": 118.959612,
        },
        {
            "region": "SC",
            "year": 2024,
            "change_week": int(
                best_sc_2024[
                    "next_week"
                ]
            ),
            "stable_detection_week":
                sc_2024_stable_week,
            "detection_delay_weeks":
                (
                    sc_2024_stable_week
                    - int(
                        best_sc_2024[
                            "next_week"
                        ]
                    )
                ),
            "pre_positivity":
                best_sc_2024[
                    "left_positivity"
                ],
            "post_positivity":
                best_sc_2024[
                    "right_positivity"
                ],
            "llr":
                best_sc_2024[
                    "llr"
                ],
        },
    ]
)

display(
    external_shift_validation
)

,region,year,change_week,stable_detection_week,detection_delay_weeks,pre_positivity,post_positivity,llr
0,PR,2024,13,16,3,0.101990,0.271400,118.959612
1,SC,2024,22,26,4,0.264695,0.220118,6.879969


In [747]:
sc_2024_shift_result = {
    "region": "SC",
    "year": 2024,
    "change_week": 22,
    "stable_detection_week": 26,
    "detection_delay_weeks": 4,
    "pre_positivity": 0.264695,
    "post_positivity": 0.220118,
    "absolute_change": (
        0.220118 - 0.264695
    ),
    "direction": "DOWN",
    "llr": 6.879969,
    "supports_upward_shift": False,
}

In [748]:
pr_2024_shift_result = {
    "region": "PR",
    "year": 2024,
    "change_week": 13,
    "stable_detection_week": 16,
    "detection_delay_weeks": 3,
    "pre_positivity": 0.101990,
    "post_positivity": 0.271400,
    "absolute_change": (
        0.271400 - 0.101990
    ),
    "direction": "UP",
    "llr": 118.959612,
    "supports_upward_shift": True,
}

In [749]:
def evaluate_shift_state_year(
    path,
    region: str,
    year: int,
):
    weekly = (
        build_flu_weekly_for_state_year(
            path=path,
            region=region,
            year=year,
        )
    )

    change_points = (
        detect_flu_change_point(
            weekly_df=weekly,
            week_start=8,
            week_end=26,
            min_weeks_each_side=3,
        )
    )

    best = change_points.iloc[0]

    change_week = int(
        best["next_week"]
    )

    prospective = (
        build_prospective_change_detection(
            weekly_df=weekly,
            final_split_after=int(
                best["split_after_week"]
            ),
            final_change_week=change_week,
            week_start=8,
            week_end=26,
            min_weeks_each_side=3,
        )
    )

    stable_week = (
        first_stable_match_week(
            prospective
        )
    )

    pre = float(
        best["left_positivity"]
    )

    post = float(
        best["right_positivity"]
    )

    return {
        "region": region,
        "year": year,
        "change_week": change_week,
        "stable_detection_week":
            stable_week,
        "detection_delay_weeks":
            (
                stable_week
                - change_week
                if pd.notna(stable_week)
                else np.nan
            ),
        "pre_positivity": pre,
        "post_positivity": post,
        "absolute_change":
            post - pre,
        "relative_change":
            (
                post / pre
                if pre > 0
                else np.nan
            ),
        "direction":
            (
                "UP"
                if post > pre
                else "DOWN"
            ),
        "llr": float(
            best["llr"]
        ),
    }

In [750]:
external_cases = [
    ("RS", 2024),
    ("RS", 2025),
    ("SC", 2025),
    ("PR", 2025),
]

In [751]:
external_results = []

for region, year in external_cases:
    path = next(
        path
        for path in historical_files
        if (
            f"INFLUD{str(year)[-2:]}"
            in path.name.upper()
        )
    )

    result = (
        evaluate_shift_state_year(
            path=path,
            region=region,
            year=year,
        )
    )

    external_results.append(
        result
    )

external_shift_results = (
    pd.DataFrame(
        external_results
    )
)

display(
    external_shift_results
)

,region,year,change_week,stable_detection_week,detection_delay_weeks,pre_positivity,post_positivity,absolute_change,relative_change,direction,llr
0,RS,2024,13,15,2,0.197368,0.363316,0.165948,1.840803,UP,65.521958
1,RS,2025,17,19,2,0.142145,0.434714,0.292569,3.058252,UP,275.940028
2,SC,2025,14,24,10,0.127119,0.314200,0.187081,2.471707,UP,101.826543
3,PR,2025,18,21,3,0.104772,0.408275,0.303503,3.896785,UP,653.704299


In [752]:
validation_shift_all = (
    pd.concat(
        [
            pd.DataFrame(
                [
                    {
                        "region": "PR",
                        "year": 2024,
                        "change_week": 13,
                        "stable_detection_week": 16,
                        "detection_delay_weeks": 3,
                        "pre_positivity": 0.101990,
                        "post_positivity": 0.271400,
                        "absolute_change": (
                            0.271400
                            - 0.101990
                        ),
                        "relative_change": (
                            0.271400
                            / 0.101990
                        ),
                        "direction": "UP",
                        "llr": 118.959612,
                    },
                    {
                        "region": "SC",
                        "year": 2024,
                        "change_week": 22,
                        "stable_detection_week": 26,
                        "detection_delay_weeks": 4,
                        "pre_positivity": 0.264695,
                        "post_positivity": 0.220118,
                        "absolute_change": (
                            0.220118
                            - 0.264695
                        ),
                        "relative_change": (
                            0.220118
                            / 0.264695
                        ),
                        "direction": "DOWN",
                        "llr": 6.879969,
                    },
                ]
            ),
            external_shift_results,
        ],
        ignore_index=True,
    )
)

display(
    validation_shift_all
)

,region,year,change_week,stable_detection_week,detection_delay_weeks,pre_positivity,post_positivity,absolute_change,relative_change,direction,llr
0,PR,2024,13,16,3,0.101990,0.271400,0.169410,2.661045,UP,118.959612
1,SC,2024,22,26,4,0.264695,0.220118,-0.044577,0.831591,DOWN,6.879969
2,RS,2024,13,15,2,0.197368,0.363316,0.165948,1.840803,UP,65.521958
3,RS,2025,17,19,2,0.142145,0.434714,0.292569,3.058252,UP,275.940028
4,SC,2025,14,24,10,0.127119,0.314200,0.187081,2.471707,UP,101.826543
5,PR,2025,18,21,3,0.104772,0.408275,0.303503,3.896785,UP,653.704299


In [753]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

sc_2025_flu_weekly = (
    build_flu_weekly_for_state_year(
        path=path_2025,
        region="SC",
        year=2025,
    )
)

sc_2025_final = (
    detect_flu_change_point(
        weekly_df=sc_2025_flu_weekly,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

best_sc_2025 = sc_2025_final.iloc[0]

final_split_sc_2025 = int(
    best_sc_2025["split_after_week"]
)

final_change_sc_2025 = int(
    best_sc_2025["next_week"]
)

print(
    "Quebra final:",
    f"SE{final_split_sc_2025}"
    f"→SE{final_change_sc_2025}",
)

print(
    "LLR final:",
    round(best_sc_2025["llr"], 3),
)

Quebra final: SE13→SE14
LLR final: 101.827


In [754]:
sc_2025_prospective = (
    build_prospective_change_detection(
        weekly_df=sc_2025_flu_weekly,
        final_split_after=final_split_sc_2025,
        final_change_week=final_change_sc_2025,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

sc_2025_prospective[
    "absolute_change"
] = (
    sc_2025_prospective[
        "right_positivity"
    ]
    -
    sc_2025_prospective[
        "left_positivity"
    ]
)

sc_2025_prospective[
    "relative_change"
] = (
    sc_2025_prospective[
        "right_positivity"
    ]
    /
    sc_2025_prospective[
        "left_positivity"
    ]
)

sc_2025_prospective[
    "direction"
] = np.where(
    sc_2025_prospective[
        "absolute_change"
    ] > 0,
    "UP",
    "DOWN",
)

In [755]:
display(
    sc_2025_prospective[
        [
            "data_available_until",
            "detected_split_after",
            "detected_change_week",
            "left_positivity",
            "right_positivity",
            "absolute_change",
            "relative_change",
            "direction",
            "llr",
            "matches_final_change",
        ]
    ]
)

,data_available_until,detected_split_after,detected_change_week,left_positivity,right_positivity,absolute_change,relative_change,direction,llr,matches_final_change
0,13,10,11,0.085202,0.152589,0.067387,1.790908,UP,5.967482,False
1,14,11,12,0.087248,0.190578,0.103330,2.184319,UP,16.206950,False
2,15,11,12,0.087248,0.218341,0.131092,2.502519,UP,27.125869,False
3,16,13,14,0.127119,0.272727,0.145609,2.145455,UP,41.251511,True
4,17,13,14,0.127119,0.316894,0.189775,2.492900,UP,76.136551,True
5,18,13,14,0.127119,0.329188,0.202069,2.589612,UP,94.404116,True
6,19,14,15,0.150327,0.344242,0.193916,2.289960,UP,104.531734,False
7,20,14,15,0.150327,0.353387,0.203060,2.350792,UP,120.758583,False
8,21,14,15,0.150327,0.354272,0.203946,2.356681,UP,126.963206,False
9,22,14,15,0.150327,0.347096,0.196770,2.308945,UP,122.750783,False


In [756]:
final_change = 14

sc_2025_prospective[
    "near_final_change"
] = (
    (
        sc_2025_prospective[
            "detected_change_week"
        ]
        - final_change
    )
    .abs()
    <= 1
)

display(
    sc_2025_prospective[
        [
            "data_available_until",
            "detected_change_week",
            "direction",
            "llr",
            "near_final_change",
            "matches_final_change",
        ]
    ]
)

,data_available_until,detected_change_week,direction,llr,near_final_change,matches_final_change
0,13,11,UP,5.967482,False,False
1,14,12,UP,16.206950,False,False
2,15,12,UP,27.125869,False,False
3,16,14,UP,41.251511,True,True
4,17,14,UP,76.136551,True,True
5,18,14,UP,94.404116,True,True
6,19,15,UP,104.531734,True,False
7,20,15,UP,120.758583,True,False
8,21,15,UP,126.963206,True,False
9,22,15,UP,122.750783,True,False


In [757]:
def first_stable_true_week(
    df: pd.DataFrame,
    column: str,
):
    data = (
        df
        .sort_values(
            "data_available_until"
        )
        .reset_index(drop=True)
    )

    for i in range(len(data)):
        if data.loc[i:, column].all():
            return int(
                data.loc[
                    i,
                    "data_available_until"
                ]
            )

    return np.nan


stable_near_final_week = (
    first_stable_true_week(
        sc_2025_prospective,
        "near_final_change",
    )
)

print(
    "Estável em ±1 semana a partir da SE:",
    stable_near_final_week,
)

Estável em ±1 semana a partir da SE: 16


In [758]:
def summarize_shift_operationally(
    weekly_df: pd.DataFrame,
    region: str,
    year: int,
    week_start: int = 8,
    week_end: int = 26,
    min_weeks_each_side: int = 3,
):
    final_result = detect_flu_change_point(
        weekly_df=weekly_df,
        week_start=week_start,
        week_end=week_end,
        min_weeks_each_side=min_weeks_each_side,
    )

    best = final_result.iloc[0]

    final_split = int(
        best["split_after_week"]
    )

    final_change = int(
        best["next_week"]
    )

    prospective = (
        build_prospective_change_detection(
            weekly_df=weekly_df,
            final_split_after=final_split,
            final_change_week=final_change,
            week_start=week_start,
            week_end=week_end,
            min_weeks_each_side=min_weeks_each_side,
        )
    )

    prospective["absolute_change"] = (
        prospective["right_positivity"]
        - prospective["left_positivity"]
    )

    prospective["direction"] = np.where(
        prospective["absolute_change"] > 0,
        "UP",
        "DOWN",
    )

    prospective["near_final_change"] = (
        (
            prospective["detected_change_week"]
            - final_change
        )
        .abs()
        <= 1
    )

    first_up_week = (
        prospective.loc[
            prospective["direction"].eq("UP"),
            "data_available_until",
        ]
        .min()
    )

    first_final_match_week = (
        prospective.loc[
            prospective["matches_final_change"],
            "data_available_until",
        ]
        .min()
    )

    stable_near_final_week = (
        first_stable_true_week(
            prospective,
            "near_final_change",
        )
    )

    stable_final_week = (
        first_stable_match_week(
            prospective
        )
    )

    pre = float(
        best["left_positivity"]
    )

    post = float(
        best["right_positivity"]
    )

    return {
        "region": region,
        "year": year,
        "change_week": final_change,
        "direction": (
            "UP"
            if post > pre
            else "DOWN"
        ),
        "pre_positivity": pre,
        "post_positivity": post,
        "absolute_change": post - pre,
        "relative_change": (
            post / pre
            if pre > 0
            else np.nan
        ),
        "llr": float(
            best["llr"]
        ),
        "first_up_week":
            first_up_week,
        "first_final_match_week":
            first_final_match_week,
        "stable_near_final_week":
            stable_near_final_week,
        "stable_final_week":
            stable_final_week,
        "near_final_delay_weeks": (
            stable_near_final_week
            - final_change
            if pd.notna(
                stable_near_final_week
            )
            else np.nan
        ),
        "exact_final_delay_weeks": (
            stable_final_week
            - final_change
            if pd.notna(
                stable_final_week
            )
            else np.nan
        ),
    }

In [759]:
cases = [
    ("PR", 2024),
    ("SC", 2024),
    ("RS", 2024),
    ("RS", 2025),
    ("SC", 2025),
    ("PR", 2025),
]

operational_results = []

for region, year in cases:
    path = next(
        path
        for path in historical_files
        if (
            f"INFLUD{str(year)[-2:]}"
            in path.name.upper()
        )
    )

    weekly = (
        build_flu_weekly_for_state_year(
            path=path,
            region=region,
            year=year,
        )
    )

    result = (
        summarize_shift_operationally(
            weekly_df=weekly,
            region=region,
            year=year,
        )
    )

    operational_results.append(
        result
    )

shift_operational_validation = (
    pd.DataFrame(
        operational_results
    )
)

display(
    shift_operational_validation
)

,region,year,change_week,direction,pre_positivity,post_positivity,absolute_change,relative_change,llr,first_up_week,first_final_match_week,stable_near_final_week,stable_final_week,near_final_delay_weeks,exact_final_delay_weeks
0,PR,2024,13,UP,0.101990,0.271400,0.169410,2.661043,118.959612,13,16,14,16,1,3
1,SC,2024,22,DOWN,0.264695,0.220118,-0.044577,0.831593,6.879969,15,24,24,26,2,4
2,RS,2024,13,UP,0.197368,0.363316,0.165948,1.840803,65.521958,13,15,14,15,1,2
3,RS,2025,17,UP,0.142145,0.434714,0.292569,3.058252,275.940028,13,19,18,19,1,2
4,SC,2025,14,UP,0.127119,0.314200,0.187081,2.471707,101.826543,13,16,16,24,2,10
5,PR,2025,18,UP,0.104772,0.408275,0.303503,3.896785,653.704299,13,21,21,21,3,3


In [760]:
display(
    shift_operational_validation[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "absolute_change",
            "llr",
            "first_up_week",
            "stable_near_final_week",
            "near_final_delay_weeks",
            "stable_final_week",
            "exact_final_delay_weeks",
        ]
    ]
)

,region,year,change_week,direction,absolute_change,llr,first_up_week,stable_near_final_week,near_final_delay_weeks,stable_final_week,exact_final_delay_weeks
0,PR,2024,13,UP,0.169410,118.959612,13,14,1,16,3
1,SC,2024,22,DOWN,-0.044577,6.879969,15,24,2,26,4
2,RS,2024,13,UP,0.165948,65.521958,13,14,1,15,2
3,RS,2025,17,UP,0.292569,275.940028,13,18,1,19,2
4,SC,2025,14,UP,0.187081,101.826543,13,16,2,24,10
5,PR,2025,18,UP,0.303503,653.704299,13,21,3,21,3


In [761]:
up_cases = (
    shift_operational_validation.loc[
        shift_operational_validation[
            "direction"
        ].eq("UP")
    ]
    .copy()
)

shift_summary = pd.DataFrame(
    {
        "metric": [
            "N UP",
            "Median absolute change",
            "Min absolute change",
            "Median LLR",
            "Min LLR",
            "Median near-final delay",
            "Max near-final delay",
            "Median exact-final delay",
            "Max exact-final delay",
        ],
        "value": [
            len(up_cases),
            up_cases[
                "absolute_change"
            ].median(),
            up_cases[
                "absolute_change"
            ].min(),
            up_cases[
                "llr"
            ].median(),
            up_cases[
                "llr"
            ].min(),
            up_cases[
                "near_final_delay_weeks"
            ].median(),
            up_cases[
                "near_final_delay_weeks"
            ].max(),
            up_cases[
                "exact_final_delay_weeks"
            ].median(),
            up_cases[
                "exact_final_delay_weeks"
            ].max(),
        ],
    }
)

display(shift_summary)

,metric,value
0,N UP,5.000000
1,Median absolute change,0.187081
2,Min absolute change,0.165948
3,Median LLR,118.959612
4,Min LLR,65.521958
5,Median near-final delay,1.000000
6,Max near-final delay,3.000000
7,Median exact-final delay,3.000000
8,Max exact-final delay,10.000000


In [762]:
shift_validation_report = (
    shift_operational_validation[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "pre_positivity",
            "post_positivity",
            "absolute_change",
            "relative_change",
            "llr",
            "stable_near_final_week",
            "near_final_delay_weeks",
        ]
    ]
    .copy()
)

display(shift_validation_report)

,region,year,change_week,direction,pre_positivity,post_positivity,absolute_change,relative_change,llr,stable_near_final_week,near_final_delay_weeks
0,PR,2024,13,UP,0.101990,0.271400,0.169410,2.661043,118.959612,14,1
1,SC,2024,22,DOWN,0.264695,0.220118,-0.044577,0.831593,6.879969,24,2
2,RS,2024,13,UP,0.197368,0.363316,0.165948,1.840803,65.521958,14,1
3,RS,2025,17,UP,0.142145,0.434714,0.292569,3.058252,275.940028,18,1
4,SC,2025,14,UP,0.127119,0.314200,0.187081,2.471707,101.826543,16,2
5,PR,2025,18,UP,0.104772,0.408275,0.303503,3.896785,653.704299,21,3


In [763]:
external_ground_truth = pd.DataFrame(
    [
        {
            "region": "PR",
            "year": 2024,
            "external_label": pd.NA,
            "external_status": "UNLABELED",
            "notes": (
                "SHIFT forte em SE13, "
                "mas ainda falta fonte externa "
                "independente com janela temporal suficiente."
            ),
        },
        {
            "region": "SC",
            "year": 2024,
            "external_label": 0,
            "external_status": "NEGATIVE_CANDIDATE",
            "notes": (
                "Detector encontrou mudança DOWN fraca "
                "e não há, até aqui, evidência externa "
                "de episódio ascendente equivalente."
            ),
        },
        {
            "region": "RS",
            "year": 2024,
            "external_label": 1,
            "external_status": "POSITIVE",
            "notes": (
                "2024 teve forte carga de SRAG por influenza; "
                "a temporalidade exata da SE13 ainda deve "
                "ser conferida."
            ),
        },
        {
            "region": "RS",
            "year": 2025,
            "external_label": 1,
            "external_status": "POSITIVE",
            "notes": (
                "2025 superou 2024 em hospitalizações "
                "por influenza no RS."
            ),
        },
        {
            "region": "SC",
            "year": 2025,
            "external_label": pd.NA,
            "external_status": "POSITIVE_CANDIDATE",
            "notes": (
                "Boletins estaduais acompanham exatamente "
                "a janela da SE14; falta extrair a afirmação "
                "epidemiológica do documento."
            ),
        },
        {
            "region": "PR",
            "year": 2025,
            "external_label": 1,
            "external_status": "POSITIVE",
            "notes": (
                "Estado reportou ascendência de Influenza A "
                "e forte aumento de SRAG entre SE18 e SE22."
            ),
        },
    ]
)

display(external_ground_truth)

,region,year,external_label,external_status,notes
0,PR,2024,<NA>,UNLABELED,"SHIFT forte em SE13, mas ainda falta fonte ext..."
1,SC,2024,0,NEGATIVE_CANDIDATE,Detector encontrou mudança DOWN fraca e não há...
2,RS,2024,1,POSITIVE,2024 teve forte carga de SRAG por influenza; a...
3,RS,2025,1,POSITIVE,2025 superou 2024 em hospitalizações por influ...
4,SC,2025,<NA>,POSITIVE_CANDIDATE,Boletins estaduais acompanham exatamente a jan...
5,PR,2025,1,POSITIVE,Estado reportou ascendência de Influenza A e f...


In [764]:
shift_external_eval = (
    shift_operational_validation
    .merge(
        external_ground_truth[
            [
                "region",
                "year",
                "external_label",
                "external_status",
                "notes",
            ]
        ],
        on=[
            "region",
            "year",
        ],
        how="left",
    )
)

display(
    shift_external_eval[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "absolute_change",
            "llr",
            "near_final_delay_weeks",
            "external_status",
            "external_label",
        ]
    ]
)

,region,year,change_week,direction,absolute_change,llr,near_final_delay_weeks,external_status,external_label
0,PR,2024,13,UP,0.169410,118.959612,1,UNLABELED,<NA>
1,SC,2024,22,DOWN,-0.044577,6.879969,2,NEGATIVE_CANDIDATE,0
2,RS,2024,13,UP,0.165948,65.521958,1,POSITIVE,1
3,RS,2025,17,UP,0.292569,275.940028,1,POSITIVE,1
4,SC,2025,14,UP,0.187081,101.826543,2,POSITIVE_CANDIDATE,<NA>
5,PR,2025,18,UP,0.303503,653.704299,3,POSITIVE,1


In [765]:
external_ground_truth.loc[
    (
        external_ground_truth["region"].eq("SC")
        &
        external_ground_truth["year"].eq(2025)
    ),
    [
        "external_label",
        "external_status",
        "notes",
    ],
] = [
    1,
    "POSITIVE",
    (
        "Vigilância estadual descreve aumento expressivo "
        "de influenza a partir da SE12; influenza torna-se "
        "predominante entre SE17 e SE23."
    ),
]

external_ground_truth.loc[
    (
        external_ground_truth["region"].eq("PR")
        &
        external_ground_truth["year"].eq(2024)
    ),
    [
        "external_label",
        "external_status",
        "notes",
    ],
] = [
    pd.NA,
    "UNLABELED",
    (
        "Há evidência oficial de circulação importante de "
        "influenza em 2024, mas ainda não há fonte independente "
        "suficientemente precisa para rotular a janela SE13."
    ),
]

display(external_ground_truth)

,region,year,external_label,external_status,notes
0,PR,2024,<NA>,UNLABELED,Há evidência oficial de circulação importante ...
1,SC,2024,0,NEGATIVE_CANDIDATE,Detector encontrou mudança DOWN fraca e não há...
2,RS,2024,1,POSITIVE,2024 teve forte carga de SRAG por influenza; a...
3,RS,2025,1,POSITIVE,2025 superou 2024 em hospitalizações por influ...
4,SC,2025,1,POSITIVE,Vigilância estadual descreve aumento expressiv...
5,PR,2025,1,POSITIVE,Estado reportou ascendência de Influenza A e f...


In [766]:
shift_external_eval = (
    shift_operational_validation
    .merge(
        external_ground_truth[
            [
                "region",
                "year",
                "external_label",
                "external_status",
                "notes",
            ]
        ],
        on=["region", "year"],
        how="left",
    )
)

display(
    shift_external_eval[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "absolute_change",
            "llr",
            "near_final_delay_weeks",
            "external_status",
            "external_label",
        ]
    ]
)

,region,year,change_week,direction,absolute_change,llr,near_final_delay_weeks,external_status,external_label
0,PR,2024,13,UP,0.169410,118.959612,1,UNLABELED,<NA>
1,SC,2024,22,DOWN,-0.044577,6.879969,2,NEGATIVE_CANDIDATE,0
2,RS,2024,13,UP,0.165948,65.521958,1,POSITIVE,1
3,RS,2025,17,UP,0.292569,275.940028,1,POSITIVE,1
4,SC,2025,14,UP,0.187081,101.826543,2,POSITIVE,1
5,PR,2025,18,UP,0.303503,653.704299,3,POSITIVE,1


In [767]:
path_2024 = next(
    path
    for path in historical_files
    if "INFLUD24" in path.name.upper()
)

rj_2024_flu_weekly = (
    build_flu_weekly_for_state_year(
        path=path_2024,
        region="RJ",
        year=2024,
    )
)

display(
    rj_2024_flu_weekly.loc[
        rj_2024_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,region,epi_year
7,8,69,18,0.26087,RJ,2024
8,9,68,19,0.279412,RJ,2024
9,10,81,35,0.432099,RJ,2024
10,11,85,28,0.329412,RJ,2024
11,12,94,32,0.340426,RJ,2024
12,13,77,20,0.25974,RJ,2024
13,14,110,29,0.263636,RJ,2024
14,15,160,34,0.2125,RJ,2024
15,16,166,36,0.216867,RJ,2024
16,17,142,37,0.260563,RJ,2024


In [768]:
rj_2024_shift = (
    summarize_shift_operationally(
        weekly_df=rj_2024_flu_weekly,
        region="RJ",
        year=2024,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

pd.DataFrame(
    [rj_2024_shift]
)

,region,year,change_week,direction,pre_positivity,post_positivity,absolute_change,relative_change,llr,first_up_week,first_final_match_week,stable_near_final_week,stable_final_week,near_final_delay_weeks,exact_final_delay_weeks
0,RJ,2024,18,DOWN,0.273764,0.104497,-0.169267,0.381706,122.019839,NaN,20,20,20,2,2


In [769]:
rj_2024_final = detect_flu_change_point(
    weekly_df=rj_2024_flu_weekly,
    week_start=8,
    week_end=26,
    min_weeks_each_side=3,
)

best_rj_2024 = rj_2024_final.iloc[0]

rj_2024_prospective = (
    build_prospective_change_detection(
        weekly_df=rj_2024_flu_weekly,
        final_split_after=int(
            best_rj_2024["split_after_week"]
        ),
        final_change_week=int(
            best_rj_2024["next_week"]
        ),
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

rj_2024_prospective[
    "absolute_change"
] = (
    rj_2024_prospective[
        "right_positivity"
    ]
    -
    rj_2024_prospective[
        "left_positivity"
    ]
)

rj_2024_prospective[
    "direction"
] = np.where(
    rj_2024_prospective[
        "absolute_change"
    ] > 0,
    "UP",
    "DOWN",
)

final_change_rj = int(
    best_rj_2024["next_week"]
)

rj_2024_prospective[
    "near_final_change"
] = (
    (
        rj_2024_prospective[
            "detected_change_week"
        ]
        - final_change_rj
    )
    .abs()
    <= 1
)

display(
    rj_2024_prospective[
        [
            "data_available_until",
            "detected_change_week",
            "left_positivity",
            "right_positivity",
            "absolute_change",
            "direction",
            "llr",
            "matches_final_change",
            "near_final_change",
        ]
    ]
)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr,matches_final_change,near_final_change
0,13,11,0.330275,0.312500,-0.017775,DOWN,0.170651,False,False
1,14,12,0.330033,0.288256,-0.041777,DOWN,1.191524,False,False
2,15,13,0.332494,0.239193,-0.093301,DOWN,7.905325,False,False
3,16,13,0.332494,0.231969,-0.100525,DOWN,11.262176,False,False
4,17,13,0.332494,0.238168,-0.094326,DOWN,10.913509,False,False
5,18,13,0.332494,0.225527,-0.106967,DOWN,15.420044,False,False
6,19,15,0.309932,0.196891,-0.113040,DOWN,22.761541,False,False
7,20,18,0.273764,0.132505,-0.141259,DOWN,40.235034,True,True
8,21,18,0.273764,0.123853,-0.149911,DOWN,56.807725,True,True
9,22,18,0.273764,0.117720,-0.156044,DOWN,71.307151,True,True


In [770]:
shift_geographic_validation = pd.concat(
    [
        shift_operational_validation,
        pd.DataFrame(
            [rj_2024_shift]
        ),
    ],
    ignore_index=True,
)

display(
    shift_geographic_validation[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "absolute_change",
            "relative_change",
            "llr",
            "stable_near_final_week",
            "near_final_delay_weeks",
            "stable_final_week",
        ]
    ]
)

,region,year,change_week,direction,absolute_change,relative_change,llr,stable_near_final_week,near_final_delay_weeks,stable_final_week
0,PR,2024,13,UP,0.169410,2.661043,118.959612,14,1,16
1,SC,2024,22,DOWN,-0.044577,0.831593,6.879969,24,2,26
2,RS,2024,13,UP,0.165948,1.840803,65.521958,14,1,15
3,RS,2025,17,UP,0.292569,3.058252,275.940028,18,1,19
4,SC,2025,14,UP,0.187081,2.471707,101.826543,16,2,24
5,PR,2025,18,UP,0.303503,3.896785,653.704299,21,3,21
6,RJ,2024,18,DOWN,-0.169267,0.381706,122.019839,20,2,20


In [771]:
rj_2024_ground_truth = {
    "region": "RJ",
    "year": 2024,
    "external_label": 0,
    "external_status": "NEGATIVE_FOR_UP_SHIFT",
    "notes": (
        "Fontes oficiais mostram circulação importante de Influenza A "
        "antes e durante o período, mas não uma nova mudança ascendente "
        "em torno da SE18. O algoritmo detectou mudança forte DOWN."
    ),
}

In [772]:
external_ground_truth = pd.concat(
    [
        external_ground_truth,
        pd.DataFrame(
            [rj_2024_ground_truth]
        ),
    ],
    ignore_index=True,
)

In [773]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

rj_2025_flu_weekly = (
    build_flu_weekly_for_state_year(
        path=path_2025,
        region="RJ",
        year=2025,
    )
)

display(
    rj_2025_flu_weekly.loc[
        rj_2025_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,region,epi_year
7,8,62,3,0.048387,RJ,2025
8,9,51,1,0.019608,RJ,2025
9,10,59,2,0.033898,RJ,2025
10,11,80,5,0.0625,RJ,2025
11,12,98,5,0.05102,RJ,2025
12,13,114,8,0.070175,RJ,2025
13,14,156,12,0.076923,RJ,2025
14,15,172,11,0.063953,RJ,2025
15,16,177,24,0.135593,RJ,2025
16,17,256,45,0.175781,RJ,2025


In [774]:
rj_2025_shift = (
    summarize_shift_operationally(
        weekly_df=rj_2025_flu_weekly,
        region="RJ",
        year=2025,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

display(
    pd.DataFrame(
        [rj_2025_shift]
    )
)

,region,year,change_week,direction,pre_positivity,post_positivity,absolute_change,relative_change,llr,first_up_week,first_final_match_week,stable_near_final_week,stable_final_week,near_final_delay_weeks,exact_final_delay_weeks
0,RJ,2025,24,DOWN,0.178908,0.051963,-0.126945,0.290445,105.185071,13,26,26,26,2,2


In [775]:
rj_2025_final = (
    detect_flu_change_point(
        weekly_df=rj_2025_flu_weekly,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

best_rj_2025 = (
    rj_2025_final.iloc[0]
)

final_change_rj_2025 = int(
    best_rj_2025[
        "next_week"
    ]
)

rj_2025_prospective = (
    build_prospective_change_detection(
        weekly_df=rj_2025_flu_weekly,
        final_split_after=int(
            best_rj_2025[
                "split_after_week"
            ]
        ),
        final_change_week=final_change_rj_2025,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

rj_2025_prospective[
    "absolute_change"
] = (
    rj_2025_prospective[
        "right_positivity"
    ]
    -
    rj_2025_prospective[
        "left_positivity"
    ]
)

rj_2025_prospective[
    "direction"
] = np.where(
    rj_2025_prospective[
        "absolute_change"
    ] > 0,
    "UP",
    "DOWN",
)

rj_2025_prospective[
    "near_final_change"
] = (
    (
        rj_2025_prospective[
            "detected_change_week"
        ]
        - final_change_rj_2025
    )
    .abs()
    <= 1
)

display(
    rj_2025_prospective[
        [
            "data_available_until",
            "detected_change_week",
            "left_positivity",
            "right_positivity",
            "absolute_change",
            "direction",
            "llr",
            "matches_final_change",
            "near_final_change",
        ]
    ]
)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr,matches_final_change,near_final_change
0,13,11,0.034884,0.061644,0.026760,UP,1.670845,False,False
1,14,11,0.034884,0.066964,0.032081,UP,2.576985,False,False
2,15,11,0.034884,0.066129,0.031245,UP,2.640870,False,False
3,16,13,0.045714,0.088853,0.043139,UP,6.560165,False,False
4,17,15,0.058065,0.132231,0.074167,UP,20.071515,False,False
5,18,16,0.059343,0.176389,0.117045,UP,52.076399,False,False
6,19,16,0.059343,0.197588,0.138245,UP,79.745505,False,False
7,20,16,0.059343,0.208955,0.149612,UP,100.859689,False,False
8,21,16,0.059343,0.217546,0.158202,UP,116.998533,False,False
9,22,16,0.059343,0.219178,0.159835,UP,123.256631,False,False


In [776]:
shift_geographic_validation = pd.concat(
    [
        shift_geographic_validation,
        pd.DataFrame(
            [rj_2025_shift]
        ),
    ],
    ignore_index=True,
)

display(
    shift_geographic_validation[
        [
            "region",
            "year",
            "change_week",
            "direction",
            "absolute_change",
            "relative_change",
            "llr",
            "stable_near_final_week",
            "near_final_delay_weeks",
            "stable_final_week",
        ]
    ]
)

,region,year,change_week,direction,absolute_change,relative_change,llr,stable_near_final_week,near_final_delay_weeks,stable_final_week
0,PR,2024,13,UP,0.169410,2.661043,118.959612,14,1,16
1,SC,2024,22,DOWN,-0.044577,0.831593,6.879969,24,2,26
2,RS,2024,13,UP,0.165948,1.840803,65.521958,14,1,15
3,RS,2025,17,UP,0.292569,3.058252,275.940028,18,1,19
4,SC,2025,14,UP,0.187081,2.471707,101.826543,16,2,24
5,PR,2025,18,UP,0.303503,3.896785,653.704299,21,3,21
6,RJ,2024,18,DOWN,-0.169267,0.381706,122.019839,20,2,20
7,RJ,2025,24,DOWN,-0.126945,0.290445,105.185071,26,2,26


In [777]:
rj_2025_event_history = (
    rj_2025_prospective[
        [
            "data_available_until",
            "detected_change_week",
            "left_positivity",
            "right_positivity",
            "absolute_change",
            "direction",
            "llr",
        ]
    ]
    .copy()
)

display(rj_2025_event_history)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr
0,13,11,0.034884,0.061644,0.026760,UP,1.670845
1,14,11,0.034884,0.066964,0.032081,UP,2.576985
2,15,11,0.034884,0.066129,0.031245,UP,2.640870
3,16,13,0.045714,0.088853,0.043139,UP,6.560165
4,17,15,0.058065,0.132231,0.074167,UP,20.071515
5,18,16,0.059343,0.176389,0.117045,UP,52.076399
6,19,16,0.059343,0.197588,0.138245,UP,79.745505
7,20,16,0.059343,0.208955,0.149612,UP,100.859689
8,21,16,0.059343,0.217546,0.158202,UP,116.998533
9,22,16,0.059343,0.219178,0.159835,UP,123.256631


In [778]:
rj_2025_event_history[
    "direction_changed"
] = (
    rj_2025_event_history[
        "direction"
    ]
    .ne(
        rj_2025_event_history[
            "direction"
        ].shift()
    )
)

display(
    rj_2025_event_history
)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr,direction_changed
0,13,11,0.034884,0.061644,0.026760,UP,1.670845,True
1,14,11,0.034884,0.066964,0.032081,UP,2.576985,False
2,15,11,0.034884,0.066129,0.031245,UP,2.640870,False
3,16,13,0.045714,0.088853,0.043139,UP,6.560165,False
4,17,15,0.058065,0.132231,0.074167,UP,20.071515,False
5,18,16,0.059343,0.176389,0.117045,UP,52.076399,False
6,19,16,0.059343,0.197588,0.138245,UP,79.745505,False
7,20,16,0.059343,0.208955,0.149612,UP,100.859689,False
8,21,16,0.059343,0.217546,0.158202,UP,116.998533,False
9,22,16,0.059343,0.219178,0.159835,UP,123.256631,False


In [779]:
rj_2025_event_history[
    "regime_id"
] = (
    rj_2025_event_history[
        "direction_changed"
    ]
    .cumsum()
)

rj_2025_regimes = (
    rj_2025_event_history
    .groupby(
        [
            "regime_id",
            "direction",
        ],
        as_index=False,
    )
    .agg(
        first_available_week=(
            "data_available_until",
            "min",
        ),
        last_available_week=(
            "data_available_until",
            "max",
        ),
        median_change_week=(
            "detected_change_week",
            "median",
        ),
        max_llr=(
            "llr",
            "max",
        ),
        max_absolute_change=(
            "absolute_change",
            lambda x: x.abs().max(),
        ),
    )
)

display(rj_2025_regimes)

,regime_id,direction,first_available_week,last_available_week,median_change_week,max_llr,max_absolute_change
0,1,UP,13,25,16.0,123.256631,0.159835
1,2,DOWN,26,26,24.0,105.185071,0.126945


In [780]:
up_peak = (
    rj_2025_event_history.loc[
        rj_2025_event_history[
            "direction"
        ].eq("UP")
    ]
    .sort_values(
        "llr",
        ascending=False,
    )
    .head(1)
)

down_peak = (
    rj_2025_event_history.loc[
        rj_2025_event_history[
            "direction"
        ].eq("DOWN")
    ]
    .sort_values(
        "llr",
        ascending=False,
    )
    .head(1)
)

display(up_peak)
display(down_peak)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr,direction_changed,regime_id
9,22,16,0.059343,0.219178,0.159835,UP,123.256631,False,1


,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,direction,llr,direction_changed,regime_id
13,26,24,0.178908,0.051963,-0.126945,DOWN,105.185071,True,2


In [781]:
rj_2025_shift_events = pd.DataFrame(
    [
        {
            "region": "RJ",
            "year": 2025,
            "event_type": "SHIFT_UP",
            "change_week": 16,
            "peak_evidence_week": 22,
            "absolute_change": 0.159835,
            "llr": 123.256631,
            "external_status": "CONFIRMED",
            "external_window": "May 2025",
            "notes": (
                "SES-RJ reported increased influenza/H1N1 "
                "activity in late May."
            ),
        },
        {
            "region": "RJ",
            "year": 2025,
            "event_type": "SHIFT_DOWN",
            "change_week": 24,
            "peak_evidence_week": 26,
            "absolute_change": -0.126945,
            "llr": 105.185071,
            "external_status": "CONFIRMED",
            "external_window": "SE24 onward",
            "notes": (
                "SES-RJ reported decline beginning in SE24."
            ),
        },
    ]
)

display(rj_2025_shift_events)

,region,year,event_type,change_week,peak_evidence_week,absolute_change,llr,external_status,external_window,notes
0,RJ,2025,SHIFT_UP,16,22,0.159835,123.256631,CONFIRMED,May 2025,SES-RJ reported increased influenza/H1N1 activ...
1,RJ,2025,SHIFT_DOWN,24,26,-0.126945,105.185071,CONFIRMED,SE24 onward,SES-RJ reported decline beginning in SE24.


In [782]:
def extract_shift_events_from_history(
    prospective_df: pd.DataFrame,
) -> pd.DataFrame:
    history = (
        prospective_df
        .sort_values("data_available_until")
        .reset_index(drop=True)
        .copy()
    )

    history["direction_changed"] = (
        history["direction"]
        .ne(
            history["direction"].shift()
        )
    )

    history["regime_id"] = (
        history["direction_changed"]
        .cumsum()
    )

    events = []

    for (
        regime_id,
        direction,
    ), group in history.groupby(
        [
            "regime_id",
            "direction",
        ],
        sort=True,
    ):
        strongest = (
            group.sort_values(
                "llr",
                ascending=False,
            )
            .iloc[0]
        )

        events.append(
            {
                "regime_id":
                    int(regime_id),

                "direction":
                    direction,

                "first_observed_week":
                    int(
                        group[
                            "data_available_until"
                        ].min()
                    ),

                "last_observed_week":
                    int(
                        group[
                            "data_available_until"
                        ].max()
                    ),

                "representative_change_week":
                    int(
                        round(
                            group[
                                "detected_change_week"
                            ].median()
                        )
                    ),

                "first_change_week":
                    int(
                        group[
                            "detected_change_week"
                        ].iloc[0]
                    ),

                "last_change_week":
                    int(
                        group[
                            "detected_change_week"
                        ].iloc[-1]
                    ),

                "peak_evidence_week":
                    int(
                        strongest[
                            "data_available_until"
                        ]
                    ),

                "peak_llr":
                    float(
                        strongest["llr"]
                    ),

                "peak_absolute_change":
                    float(
                        strongest[
                            "absolute_change"
                        ]
                    ),

                "peak_left_positivity":
                    float(
                        strongest[
                            "left_positivity"
                        ]
                    ),

                "peak_right_positivity":
                    float(
                        strongest[
                            "right_positivity"
                        ]
                    ),

                "duration_snapshots":
                    len(group),
            }
        )

    return pd.DataFrame(events)

In [783]:
rj_2025_events = (
    extract_shift_events_from_history(
        rj_2025_event_history
    )
)

display(rj_2025_events)

,regime_id,direction,first_observed_week,last_observed_week,representative_change_week,first_change_week,last_change_week,peak_evidence_week,peak_llr,peak_absolute_change,peak_left_positivity,peak_right_positivity,duration_snapshots
0,1,UP,13,25,16,11,16,22,123.256631,0.159835,0.059343,0.219178,13
1,2,DOWN,26,26,24,24,24,26,105.185071,-0.126945,0.178908,0.051963,1


In [784]:
def build_shift_event_history(
    weekly_df: pd.DataFrame,
    week_start: int = 8,
    week_end: int = 26,
    min_weeks_each_side: int = 3,
) -> pd.DataFrame:
    rows = []

    first_available_week = (
        week_start
        + (2 * min_weeks_each_side)
        - 1
    )

    for current_week in range(
        first_available_week,
        week_end + 1,
    ):
        partial = (
            weekly_df.loc[
                weekly_df[
                    "epi_week"
                ].between(
                    week_start,
                    current_week,
                )
            ]
            .copy()
        )

        result = (
            detect_flu_change_point(
                weekly_df=partial,
                week_start=week_start,
                week_end=current_week,
                min_weeks_each_side=min_weeks_each_side,
            )
        )

        if result.empty:
            continue

        best = result.iloc[0]

        left = float(
            best["left_positivity"]
        )

        right = float(
            best["right_positivity"]
        )

        absolute_change = (
            right - left
        )

        rows.append(
            {
                "data_available_until":
                    current_week,

                "detected_change_week":
                    int(
                        best["next_week"]
                    ),

                "detected_split_after":
                    int(
                        best[
                            "split_after_week"
                        ]
                    ),

                "left_positivity":
                    left,

                "right_positivity":
                    right,

                "absolute_change":
                    absolute_change,

                "relative_change":
                    (
                        right / left
                        if left > 0
                        else np.nan
                    ),

                "direction":
                    (
                        "UP"
                        if absolute_change > 0
                        else "DOWN"
                    ),

                "llr":
                    float(
                        best["llr"]
                    ),
            }
        )

    return pd.DataFrame(rows)

In [785]:
cases = [
    ("PR", 2024),
    ("SC", 2024),
    ("RS", 2024),
    ("RJ", 2024),
    ("PR", 2025),
    ("SC", 2025),
    ("RS", 2025),
    ("RJ", 2025),
]

all_shift_events = []

for region, year in cases:
    path = next(
        path
        for path in historical_files
        if (
            f"INFLUD{str(year)[-2:]}"
            in path.name.upper()
        )
    )

    weekly = (
        build_flu_weekly_for_state_year(
            path=path,
            region=region,
            year=year,
        )
    )

    history = (
        build_shift_event_history(
            weekly_df=weekly,
            week_start=8,
            week_end=26,
            min_weeks_each_side=3,
        )
    )

    events = (
        extract_shift_events_from_history(
            history
        )
    )

    events["region"] = region
    events["year"] = year

    all_shift_events.append(events)

shift_events_all = pd.concat(
    all_shift_events,
    ignore_index=True,
)

display(
    shift_events_all[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "first_observed_week",
            "last_observed_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
        ]
    ]
)

,region,year,regime_id,direction,representative_change_week,first_observed_week,last_observed_week,peak_evidence_week,peak_llr,peak_absolute_change,duration_snapshots
0,PR,2024,1,UP,13,13,26,26,118.959612,0.169410,14
1,SC,2024,1,DOWN,11,13,14,13,0.431509,-0.018467,2
2,SC,2024,2,UP,15,15,23,18,5.421590,0.045966,9
3,SC,2024,3,DOWN,22,24,26,25,7.781389,-0.056444,3
4,RS,2024,1,UP,13,13,26,23,73.026589,0.180247,14
5,RJ,2024,1,DOWN,16,13,26,26,122.019839,-0.169267,14
6,PR,2025,1,UP,16,13,26,24,688.183542,0.323666,14
7,SC,2025,1,UP,14,13,26,21,126.963206,0.203946,14
8,RS,2025,1,UP,17,13,26,23,332.537921,0.337454,14
9,RJ,2025,1,UP,16,13,25,22,123.256631,0.159835,13


In [786]:
display(
    shift_events_all.sort_values(
        [
            "region",
            "year",
            "first_observed_week",
        ]
    )
)

,regime_id,direction,first_observed_week,last_observed_week,representative_change_week,first_change_week,last_change_week,peak_evidence_week,peak_llr,peak_absolute_change,peak_left_positivity,peak_right_positivity,duration_snapshots,region,year
0,1,UP,13,26,13,11,13,26,118.959612,0.169410,0.101990,0.271400,14,PR,2024
6,1,UP,13,26,16,11,18,24,688.183542,0.323666,0.104772,0.428438,14,PR,2025
5,1,DOWN,13,26,16,11,18,26,122.019839,-0.169267,0.273764,0.104497,14,RJ,2024
9,1,UP,13,25,16,11,16,22,123.256631,0.159835,0.059343,0.219178,13,RJ,2025
10,2,DOWN,26,26,24,24,24,26,105.185071,-0.126945,0.178908,0.051963,1,RJ,2025
4,1,UP,13,26,13,11,13,23,73.026589,0.180247,0.197368,0.377616,14,RS,2024
8,1,UP,13,26,17,11,17,23,332.537921,0.337454,0.142145,0.479599,14,RS,2025
1,1,DOWN,13,14,11,11,11,13,0.431509,-0.018467,0.254762,0.236295,2,SC,2024
2,2,UP,15,23,15,13,15,18,5.421590,0.045966,0.245063,0.291028,9,SC,2024
3,3,DOWN,24,26,22,22,22,25,7.781389,-0.056444,0.262883,0.206439,3,SC,2024


In [787]:
shift_events_all[
    "persistent"
] = (
    shift_events_all[
        "duration_snapshots"
    ] >= 2
)

In [788]:
shift_events_all[
    "right_censored"
] = (
    shift_events_all[
        "last_observed_week"
    ].eq(26)
    &
    shift_events_all[
        "first_observed_week"
    ].eq(
        shift_events_all[
            "last_observed_week"
        ]
    )
)

display(
    shift_events_all[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,PR,2024,UP,13,118.959612,0.169410,14,False
1,SC,2024,DOWN,11,0.431509,-0.018467,2,False
2,SC,2024,UP,15,5.421590,0.045966,9,False
3,SC,2024,DOWN,22,7.781389,-0.056444,3,False
4,RS,2024,UP,13,73.026589,0.180247,14,False
5,RJ,2024,DOWN,16,122.019839,-0.169267,14,False
6,PR,2025,UP,16,688.183542,0.323666,14,False
7,SC,2025,UP,14,126.963206,0.203946,14,False
8,RS,2025,UP,17,332.537921,0.337454,14,False
9,RJ,2025,UP,16,123.256631,0.159835,13,False


In [789]:
shift_event_catalog = (
    shift_events_all[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "first_observed_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "peak_left_positivity",
            "peak_right_positivity",
            "duration_snapshots",
            "right_censored",
        ]
    ]
    .sort_values(
        [
            "region",
            "year",
            "first_observed_week",
        ]
    )
    .reset_index(drop=True)
)

display(shift_event_catalog)

,region,year,regime_id,direction,representative_change_week,first_observed_week,peak_evidence_week,peak_llr,peak_absolute_change,peak_left_positivity,peak_right_positivity,duration_snapshots,right_censored
0,PR,2024,1,UP,13,13,26,118.959612,0.169410,0.101990,0.271400,14,False
1,PR,2025,1,UP,16,13,24,688.183542,0.323666,0.104772,0.428438,14,False
2,RJ,2024,1,DOWN,16,13,26,122.019839,-0.169267,0.273764,0.104497,14,False
3,RJ,2025,1,UP,16,13,22,123.256631,0.159835,0.059343,0.219178,13,False
4,RJ,2025,2,DOWN,24,26,26,105.185071,-0.126945,0.178908,0.051963,1,True
5,RS,2024,1,UP,13,13,23,73.026589,0.180247,0.197368,0.377616,14,False
6,RS,2025,1,UP,17,13,23,332.537921,0.337454,0.142145,0.479599,14,False
7,SC,2024,1,DOWN,11,13,13,0.431509,-0.018467,0.254762,0.236295,2,False
8,SC,2024,2,UP,15,15,18,5.421590,0.045966,0.245063,0.291028,9,False
9,SC,2024,3,DOWN,22,24,25,7.781389,-0.056444,0.262883,0.206439,3,False


In [790]:
shift_event_catalog[
    "peak_relative_change"
] = (
    shift_event_catalog[
        "peak_right_positivity"
    ]
    /
    shift_event_catalog[
        "peak_left_positivity"
    ]
)

display(
    shift_event_catalog[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_absolute_change",
            "peak_relative_change",
            "peak_llr",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_absolute_change,peak_relative_change,peak_llr,duration_snapshots,right_censored
0,PR,2024,UP,13,0.169410,2.661043,118.959612,14,False
1,PR,2025,UP,16,0.323666,4.089232,688.183542,14,False
2,RJ,2024,DOWN,16,-0.169267,0.381706,122.019839,14,False
3,RJ,2025,UP,16,0.159835,3.693384,123.256631,13,False
4,RJ,2025,DOWN,24,-0.126945,0.290445,105.185071,1,True
5,RS,2024,UP,13,0.180247,1.913252,73.026589,14,False
6,RS,2025,UP,17,0.337454,3.374019,332.537921,14,False
7,SC,2024,DOWN,11,-0.018467,0.927513,0.431509,2,False
8,SC,2024,UP,15,0.045966,1.187567,5.421590,9,False
9,SC,2024,DOWN,22,-0.056444,0.785290,7.781389,3,False


In [791]:
shift_event_ground_truth = (
    shift_event_catalog[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
        ]
    ]
    .copy()
)

shift_event_ground_truth[
    "external_label"
] = pd.NA

shift_event_ground_truth[
    "external_status"
] = "UNLABELED"

shift_event_ground_truth[
    "external_reference_week"
] = pd.NA

shift_event_ground_truth[
    "notes"
] = ""

display(
    shift_event_ground_truth
)

,region,year,regime_id,direction,representative_change_week,external_label,external_status,external_reference_week,notes
0,PR,2024,1,UP,13,<NA>,UNLABELED,<NA>,
1,PR,2025,1,UP,16,<NA>,UNLABELED,<NA>,
2,RJ,2024,1,DOWN,16,<NA>,UNLABELED,<NA>,
3,RJ,2025,1,UP,16,<NA>,UNLABELED,<NA>,
4,RJ,2025,2,DOWN,24,<NA>,UNLABELED,<NA>,
5,RS,2024,1,UP,13,<NA>,UNLABELED,<NA>,
6,RS,2025,1,UP,17,<NA>,UNLABELED,<NA>,
7,SC,2024,1,DOWN,11,<NA>,UNLABELED,<NA>,
8,SC,2024,2,UP,15,<NA>,UNLABELED,<NA>,
9,SC,2024,3,DOWN,22,<NA>,UNLABELED,<NA>,


In [792]:
updates = [
    {
        "region": "PR",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 18,
        "notes": (
            "Sesa informou fase de ascendência de Influenza A "
            "em maio de 2025; entre SE18 e SE22 houve aumento "
            "importante de hospitalizações por SRAG."
        ),
    },
    {
        "region": "SC",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 12,
        "notes": (
            "Boletins estaduais descrevem aumento expressivo "
            "de influenza a partir de aproximadamente SE12."
        ),
    },
    {
        "region": "RJ",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 16,
        "notes": (
            "SES-RJ reportou aumento de influenza/H1N1 durante "
            "maio; trajetória é compatível com SHIFT_UP."
        ),
    },
    {
        "region": "RJ",
        "year": 2025,
        "regime_id": 2,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 24,
        "notes": (
            "SES-RJ posteriormente reportou recuo da circulação "
            "respiratória a partir da SE24."
        ),
    },
]

In [793]:
{
    "region": "RJ",
    "year": 2024,
    "regime_id": 1,
    "external_label": pd.NA,
    "external_status": "UNLABELED",
    "external_reference_week": pd.NA,
    "notes": (
        "Fontes oficiais confirmam forte circulação de Influenza A "
        "antes da janela; não há evidência textual suficientemente "
        "precisa para confirmar a mudança DOWN em SE16."
    ),
}

{'region': 'RJ',
 'year': 2024,
 'regime_id': 1,
 'external_label': <NA>,
 'external_status': 'UNLABELED',
 'external_reference_week': <NA>,
 'notes': 'Fontes oficiais confirmam forte circulação de Influenza A antes da janela; não há evidência textual suficientemente precisa para confirmar a mudança DOWN em SE16.'}

In [794]:
for update in updates:
    mask = (
        shift_event_ground_truth["region"].eq(
            update["region"]
        )
        &
        shift_event_ground_truth["year"].eq(
            update["year"]
        )
        &
        shift_event_ground_truth["regime_id"].eq(
            update["regime_id"]
        )
    )

    for column in [
        "external_label",
        "external_status",
        "external_reference_week",
        "notes",
    ]:
        shift_event_ground_truth.loc[
            mask,
            column,
        ] = update[column]

In [795]:
confirmed_events = (
    shift_event_ground_truth.loc[
        shift_event_ground_truth[
            "external_status"
        ].eq("CONFIRMED")
    ]
    .copy()
)

confirmed_events[
    "temporal_error_weeks"
] = (
    confirmed_events[
        "representative_change_week"
    ]
    -
    confirmed_events[
        "external_reference_week"
    ].astype(int)
)

display(
    confirmed_events[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "temporal_error_weeks",
            "external_status",
        ]
    ]
)

,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks,external_status
1,PR,2025,UP,16,18,-2,CONFIRMED
3,RJ,2025,UP,16,16,0,CONFIRMED
4,RJ,2025,DOWN,24,24,0,CONFIRMED
10,SC,2025,UP,14,12,2,CONFIRMED


In [796]:
print(
    "Eventos confirmados:",
    len(confirmed_events),
)

print(
    "Erro temporal absoluto mediano:",
    confirmed_events[
        "temporal_error_weeks"
    ]
    .abs()
    .median(),
    "semana(s)",
)

print(
    "Erro temporal absoluto máximo:",
    confirmed_events[
        "temporal_error_weeks"
    ]
    .abs()
    .max(),
    "semana(s)",
)

Eventos confirmados: 4
Erro temporal absoluto mediano: 1.0 semana(s)
Erro temporal absoluto máximo: 2 semana(s)


In [797]:
updates_rs = [
    {
        "region": "RS",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 21,
        "notes": (
            "Fonte oficial do RS mostra forte aumento de SRAG por influenza "
            "em 2025; na SE21 foram 283 hospitalizações por influenza, "
            "contra 106 em 2024."
        ),
    },
    {
        "region": "RS",
        "year": 2024,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED_NO_PRECISE_WEEK",
        "external_reference_week": pd.NA,
        "notes": (
            "2024 teve 2.326 casos de SRAG por influenza e 288 óbitos, "
            "maior carga anual recente no RS, mas ainda sem semana externa "
            "precisa o bastante para comparar com a SE13."
        ),
    },
]

for update in updates_rs:
    mask = (
        shift_event_ground_truth["region"].eq(update["region"])
        &
        shift_event_ground_truth["year"].eq(update["year"])
        &
        shift_event_ground_truth["regime_id"].eq(update["regime_id"])
    )

    for column in [
        "external_label",
        "external_status",
        "external_reference_week",
        "notes",
    ]:
        shift_event_ground_truth.loc[
            mask,
            column,
        ] = update[column]

In [798]:
confirmed_with_week = (
    shift_event_ground_truth.loc[
        shift_event_ground_truth[
            "external_status"
        ].eq("CONFIRMED")
        &
        shift_event_ground_truth[
            "external_reference_week"
        ].notna()
    ]
    .copy()
)

confirmed_with_week[
    "external_reference_week"
] = (
    confirmed_with_week[
        "external_reference_week"
    ].astype(int)
)

confirmed_with_week[
    "temporal_error_weeks"
] = (
    confirmed_with_week[
        "representative_change_week"
    ]
    -
    confirmed_with_week[
        "external_reference_week"
    ]
)

display(
    confirmed_with_week[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "temporal_error_weeks",
        ]
    ]
)

,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks
1,PR,2025,UP,16,18,-2
3,RJ,2025,UP,16,16,0
4,RJ,2025,DOWN,24,24,0
6,RS,2025,UP,17,21,-4
10,SC,2025,UP,14,12,2


In [799]:
print("N com semana externa:", len(confirmed_with_week))

print(
    "Erro absoluto mediano:",
    confirmed_with_week[
        "temporal_error_weeks"
    ].abs().median(),
)

print(
    "Erro absoluto máximo:",
    confirmed_with_week[
        "temporal_error_weeks"
    ].abs().max(),
)

N com semana externa: 5
Erro absoluto mediano: 2.0
Erro absoluto máximo: 4


In [800]:
confirmed_with_week[
    "lead_lag_weeks"
] = (
    confirmed_with_week[
        "external_reference_week"
    ]
    -
    confirmed_with_week[
        "representative_change_week"
    ]
)

In [801]:
display(
    confirmed_with_week[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "temporal_error_weeks",
            "lead_lag_weeks",
        ]
    ]
)

,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks,lead_lag_weeks
1,PR,2025,UP,16,18,-2,2
3,RJ,2025,UP,16,16,0,0
4,RJ,2025,DOWN,24,24,0,0
6,RS,2025,UP,17,21,-4,4
10,SC,2025,UP,14,12,2,-2


In [802]:
validation_summary = pd.DataFrame(
    {
        "metric": [
            "confirmed_events",
            "median_absolute_error_weeks",
            "max_absolute_error_weeks",
            "within_1_week",
            "within_2_weeks",
            "within_4_weeks",
            "detected_before_external",
            "same_week",
            "detected_after_external",
        ],
        "value": [
            len(confirmed_with_week),

            confirmed_with_week[
                "temporal_error_weeks"
            ].abs().median(),

            confirmed_with_week[
                "temporal_error_weeks"
            ].abs().max(),

            (
                confirmed_with_week[
                    "temporal_error_weeks"
                ].abs() <= 1
            ).sum(),

            (
                confirmed_with_week[
                    "temporal_error_weeks"
                ].abs() <= 2
            ).sum(),

            (
                confirmed_with_week[
                    "temporal_error_weeks"
                ].abs() <= 4
            ).sum(),

            (
                confirmed_with_week[
                    "lead_lag_weeks"
                ] > 0
            ).sum(),

            (
                confirmed_with_week[
                    "lead_lag_weeks"
                ] == 0
            ).sum(),

            (
                confirmed_with_week[
                    "lead_lag_weeks"
                ] < 0
            ).sum(),
        ],
    }
)

display(validation_summary)

,metric,value
0,confirmed_events,5.0
1,median_absolute_error_weeks,2.0
2,max_absolute_error_weeks,4.0
3,within_1_week,2.0
4,within_2_weeks,4.0
5,within_4_weeks,5.0
6,detected_before_external,2.0
7,same_week,2.0
8,detected_after_external,1.0


In [803]:
shift_external_validation_report = (
    confirmed_with_week[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "lead_lag_weeks",
            "temporal_error_weeks",
        ]
    ]
    .sort_values(
        [
            "year",
            "region",
        ]
    )
    .reset_index(drop=True)
)

display(
    shift_external_validation_report
)

,region,year,direction,representative_change_week,external_reference_week,lead_lag_weeks,temporal_error_weeks
0,PR,2025,UP,16,18,2,-2
1,RJ,2025,UP,16,16,0,0
2,RJ,2025,DOWN,24,24,0,0
3,RS,2025,UP,17,21,4,-4
4,SC,2025,UP,14,12,-2,2


In [804]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

ba_2025_flu_weekly = (
    build_flu_weekly_for_state_year(
        path=path_2025,
        region="BA",
        year=2025,
    )
)

display(
    ba_2025_flu_weekly.loc[
        ba_2025_flu_weekly[
            "epi_week"
        ].between(8, 26)
    ]
)

,epi_week,flu_tests,flu_positive,flu_positivity,region,epi_year
7,8,44,1,0.022727,BA,2025
8,9,45,3,0.066667,BA,2025
9,10,62,5,0.080645,BA,2025
10,11,69,15,0.217391,BA,2025
11,12,82,15,0.182927,BA,2025
12,13,107,16,0.149533,BA,2025
13,14,140,32,0.228571,BA,2025
14,15,120,35,0.291667,BA,2025
15,16,162,48,0.296296,BA,2025
16,17,136,24,0.176471,BA,2025


In [805]:
ba_2025_history = (
    build_shift_event_history(
        weekly_df=ba_2025_flu_weekly,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )
)

display(
    ba_2025_history[
        [
            "data_available_until",
            "detected_change_week",
            "left_positivity",
            "right_positivity",
            "absolute_change",
            "relative_change",
            "direction",
            "llr",
        ]
    ]
)

,data_available_until,detected_change_week,left_positivity,right_positivity,absolute_change,relative_change,direction,llr
0,13,11,0.059603,0.178295,0.118692,2.991387,UP,12.836992
1,14,11,0.059603,0.195980,0.136377,3.288107,UP,17.904751
2,15,11,0.059603,0.218147,0.158544,3.660017,UP,23.841184
3,16,11,0.059603,0.236765,0.177162,3.972386,UP,29.525321
4,17,11,0.059603,0.226716,0.167113,3.803785,UP,27.653358
5,18,11,0.059603,0.234531,0.174928,3.934908,UP,30.341841
6,19,11,0.059603,0.243536,0.183934,4.085998,UP,33.311936
7,20,11,0.059603,0.244039,0.184437,4.094437,UP,33.916157
8,21,11,0.059603,0.236397,0.176794,3.966210,UP,32.123604
9,22,11,0.059603,0.234651,0.175049,3.936929,UP,31.927570


In [807]:
ba_2025_events = (
    extract_shift_events_from_history(
        ba_2025_history
    )
)

ba_2025_events["region"] = "BA"
ba_2025_events["year"] = 2025

display(
    ba_2025_events[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "first_observed_week",
            "last_observed_week",
            "representative_change_week",
            "first_change_week",
            "last_change_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "peak_left_positivity",
            "peak_right_positivity",
            "duration_snapshots",
        ]
    ]
)

,region,year,regime_id,direction,first_observed_week,last_observed_week,representative_change_week,first_change_week,last_change_week,peak_evidence_week,peak_llr,peak_absolute_change,peak_left_positivity,peak_right_positivity,duration_snapshots
0,BA,2025,1,UP,13,25,11,11,11,20,33.916157,0.184437,0.059603,0.244039,13
1,BA,2025,2,DOWN,26,26,24,24,24,26,34.841122,-0.091669,0.218672,0.127004,1


In [808]:
ba_2025_events[
    "peak_relative_change"
] = (
    ba_2025_events[
        "peak_right_positivity"
    ]
    /
    ba_2025_events[
        "peak_left_positivity"
    ]
)

display(
    ba_2025_events[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_absolute_change",
            "peak_relative_change",
            "peak_llr",
            "duration_snapshots",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_absolute_change,peak_relative_change,peak_llr,duration_snapshots
0,BA,2025,UP,11,0.184437,4.094437,33.916157,13
1,BA,2025,DOWN,24,-0.091669,0.580795,34.841122,1


In [809]:
ba_2025_events[
    "right_censored"
] = (
    ba_2025_events[
        "last_observed_week"
    ].eq(26)
    &
    ba_2025_events[
        "first_observed_week"
    ].eq(
        ba_2025_events[
            "last_observed_week"
        ]
    )
)

display(
    ba_2025_events[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,BA,2025,UP,11,33.916157,0.184437,13,False
1,BA,2025,DOWN,24,34.841122,-0.091669,1,True


In [810]:
shift_events_all = pd.concat(
    [
        shift_events_all,
        ba_2025_events,
    ],
    ignore_index=True,
)

In [811]:
ba_2025_blind_result = (
    ba_2025_events[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
    .copy()
)

ba_2025_blind_result[
    "external_status"
] = "NOT_CHECKED_YET"

display(
    ba_2025_blind_result
)

,region,year,regime_id,direction,representative_change_week,peak_evidence_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored,external_status
0,BA,2025,1,UP,11,20,33.916157,0.184437,13,False,NOT_CHECKED_YET
1,BA,2025,2,DOWN,24,26,34.841122,-0.091669,1,True,NOT_CHECKED_YET


In [812]:
updates_ba = [
    {
        "region": "BA",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 14,
        "notes": (
            "Boletins da SESAB mostram aumento da circulação respiratória "
            "e de influenza principalmente a partir de aproximadamente SE14, "
            "com maior concentração entre SE14 e SE24 e pico por volta "
            "de SE19-SE23."
        ),
    },
    {
        "region": "BA",
        "year": 2025,
        "regime_id": 2,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 24,
        "notes": (
            "Boletins da SESAB mostram pico de SRAG por influenza "
            "aproximadamente entre SE19 e SE23, seguido de queda "
            "a partir da região da SE24."
        ),
    },
]

In [813]:
for update in updates_ba:
    mask = (
        shift_event_ground_truth["region"].eq(update["region"])
        &
        shift_event_ground_truth["year"].eq(update["year"])
        &
        shift_event_ground_truth["regime_id"].eq(update["regime_id"])
    )

    for column in [
        "external_label",
        "external_status",
        "external_reference_week",
        "notes",
    ]:
        shift_event_ground_truth.loc[
            mask,
            column,
        ] = update[column]

In [814]:
confirmed_with_week = (
    shift_event_ground_truth.loc[
        shift_event_ground_truth[
            "external_status"
        ].eq("CONFIRMED")
        &
        shift_event_ground_truth[
            "external_reference_week"
        ].notna()
    ]
    .copy()
)

confirmed_with_week[
    "external_reference_week"
] = (
    confirmed_with_week[
        "external_reference_week"
    ].astype(int)
)

confirmed_with_week[
    "temporal_error_weeks"
] = (
    confirmed_with_week[
        "representative_change_week"
    ]
    -
    confirmed_with_week[
        "external_reference_week"
    ]
)

confirmed_with_week[
    "lead_lag_weeks"
] = (
    -confirmed_with_week[
        "temporal_error_weeks"
    ]
)

In [815]:
abs_error = (
    confirmed_with_week[
        "temporal_error_weeks"
    ].abs()
)

print(
    "Eventos confirmados:",
    len(confirmed_with_week),
)

print(
    "Erro absoluto mediano:",
    abs_error.median(),
)

print(
    "Erro absoluto máximo:",
    abs_error.max(),
)

print(
    "Dentro de ±1 semana:",
    (abs_error <= 1).sum(),
    "/",
    len(abs_error),
)

print(
    "Dentro de ±2 semanas:",
    (abs_error <= 2).sum(),
    "/",
    len(abs_error),
)

print(
    "Dentro de ±3 semanas:",
    (abs_error <= 3).sum(),
    "/",
    len(abs_error),
)

print(
    "Dentro de ±4 semanas:",
    (abs_error <= 4).sum(),
    "/",
    len(abs_error),
)

Eventos confirmados: 5
Erro absoluto mediano: 2.0
Erro absoluto máximo: 4
Dentro de ±1 semana: 2 / 5
Dentro de ±2 semanas: 4 / 5
Dentro de ±3 semanas: 4 / 5
Dentro de ±4 semanas: 5 / 5


In [816]:
shift_event_catalog = (
    shift_events_all[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "peak_left_positivity",
            "peak_right_positivity",
            "duration_snapshots",
            "right_censored",
        ]
    ]
    .drop_duplicates(
        subset=[
            "region",
            "year",
            "regime_id",
        ]
    )
    .sort_values(
        [
            "region",
            "year",
            "regime_id",
        ]
    )
    .reset_index(drop=True)
)

In [817]:
old_ground_truth = (
    shift_event_ground_truth.copy()
)

shift_event_ground_truth = (
    shift_event_catalog[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
        ]
    ]
    .merge(
        old_ground_truth[
            [
                "region",
                "year",
                "regime_id",
                "external_label",
                "external_status",
                "external_reference_week",
                "notes",
            ]
        ],
        on=[
            "region",
            "year",
            "regime_id",
        ],
        how="left",
    )
)

shift_event_ground_truth[
    "external_status"
] = (
    shift_event_ground_truth[
        "external_status"
    ]
    .fillna("UNLABELED")
)

shift_event_ground_truth[
    "notes"
] = (
    shift_event_ground_truth[
        "notes"
    ]
    .fillna("")
)

In [818]:
for update in updates_ba:
    mask = (
        shift_event_ground_truth[
            "region"
        ].eq(update["region"])
        &
        shift_event_ground_truth[
            "year"
        ].eq(update["year"])
        &
        shift_event_ground_truth[
            "regime_id"
        ].eq(update["regime_id"])
    )

    for column in [
        "external_label",
        "external_status",
        "external_reference_week",
        "notes",
    ]:
        shift_event_ground_truth.loc[
            mask,
            column,
        ] = update[column]

In [819]:
display(
    shift_event_ground_truth.loc[
        shift_event_ground_truth[
            "region"
        ].eq("BA")
    ]
)

,region,year,regime_id,direction,representative_change_week,external_label,external_status,external_reference_week,notes
0,BA,2025,1,UP,11,1,CONFIRMED,14,Boletins da SESAB mostram aumento da circulaçã...
1,BA,2025,2,DOWN,24,1,CONFIRMED,24,Boletins da SESAB mostram pico de SRAG por inf...


In [820]:
shift_final_validation = (
    shift_event_catalog
    .merge(
        shift_event_ground_truth[
            [
                "region",
                "year",
                "regime_id",
                "external_label",
                "external_status",
                "external_reference_week",
                "notes",
            ]
        ],
        on=[
            "region",
            "year",
            "regime_id",
        ],
        how="left",
    )
)

In [821]:
shift_final_validation[
    "peak_relative_change"
] = (
    shift_final_validation[
        "peak_right_positivity"
    ]
    /
    shift_final_validation[
        "peak_left_positivity"
    ]
)

In [822]:
mask_confirmed = (
    shift_final_validation[
        "external_status"
    ].eq("CONFIRMED")
    &
    shift_final_validation[
        "external_reference_week"
    ].notna()
)

shift_final_validation.loc[
    mask_confirmed,
    "external_reference_week",
] = (
    shift_final_validation.loc[
        mask_confirmed,
        "external_reference_week",
    ]
    .astype(int)
)

shift_final_validation[
    "temporal_error_weeks"
] = pd.NA

shift_final_validation.loc[
    mask_confirmed,
    "temporal_error_weeks",
] = (
    shift_final_validation.loc[
        mask_confirmed,
        "representative_change_week",
    ]
    -
    shift_final_validation.loc[
        mask_confirmed,
        "external_reference_week",
    ]
)

shift_final_validation[
    "absolute_temporal_error_weeks"
] = (
    pd.to_numeric(
        shift_final_validation[
            "temporal_error_weeks"
        ],
        errors="coerce",
    )
    .abs()
)

In [823]:
shift_final_validation[
    "lead_lag_weeks"
] = (
    -pd.to_numeric(
        shift_final_validation[
            "temporal_error_weeks"
        ],
        errors="coerce",
    )
)

In [824]:
confirmed_shift_events = (
    shift_final_validation.loc[
        mask_confirmed
    ]
    .copy()
)

display(
    confirmed_shift_events[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "temporal_error_weeks",
            "absolute_temporal_error_weeks",
            "lead_lag_weeks",
            "peak_absolute_change",
            "peak_relative_change",
            "peak_llr",
            "duration_snapshots",
            "right_censored",
        ]
    ]
    .sort_values(
        [
            "year",
            "region",
            "representative_change_week",
        ]
    )
)

,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks,absolute_temporal_error_weeks,lead_lag_weeks,peak_absolute_change,peak_relative_change,peak_llr,duration_snapshots,right_censored
0,BA,2025,UP,11,14,-3,3.0,3.0,0.184437,4.094437,33.916157,13,False
1,BA,2025,DOWN,24,24,0,0.0,-0.0,-0.091669,0.580795,34.841122,1,True
3,PR,2025,UP,16,18,-2,2.0,2.0,0.323666,4.089232,688.183542,14,False
5,RJ,2025,UP,16,16,0,0.0,-0.0,0.159835,3.693384,123.256631,13,False
6,RJ,2025,DOWN,24,24,0,0.0,-0.0,-0.126945,0.290445,105.185071,1,True
8,RS,2025,UP,17,21,-4,4.0,4.0,0.337454,3.374019,332.537921,14,False
12,SC,2025,UP,14,12,2,2.0,-2.0,0.203946,2.356681,126.963206,14,False


In [825]:
abs_error = (
    confirmed_shift_events[
        "absolute_temporal_error_weeks"
    ]
)

summary_shift = pd.DataFrame(
    {
        "metric": [
            "confirmed_events",
            "median_abs_error_weeks",
            "mean_abs_error_weeks",
            "max_abs_error_weeks",
            "within_1_week",
            "within_2_weeks",
            "within_3_weeks",
            "within_4_weeks",
            "detected_before_external",
            "same_week",
            "detected_after_external",
            "up_events",
            "down_events",
        ],
        "value": [
            len(confirmed_shift_events),

            abs_error.median(),

            abs_error.mean(),

            abs_error.max(),

            (abs_error <= 1).sum(),

            (abs_error <= 2).sum(),

            (abs_error <= 3).sum(),

            (abs_error <= 4).sum(),

            (
                confirmed_shift_events[
                    "lead_lag_weeks"
                ] > 0
            ).sum(),

            (
                confirmed_shift_events[
                    "lead_lag_weeks"
                ] == 0
            ).sum(),

            (
                confirmed_shift_events[
                    "lead_lag_weeks"
                ] < 0
            ).sum(),

            confirmed_shift_events[
                "direction"
            ].eq("UP").sum(),

            confirmed_shift_events[
                "direction"
            ].eq("DOWN").sum(),
        ],
    }
)

display(summary_shift)

,metric,value
0,confirmed_events,7.000000
1,median_abs_error_weeks,2.000000
2,mean_abs_error_weeks,1.571429
3,max_abs_error_weeks,4.000000
4,within_1_week,3.000000
5,within_2_weeks,5.000000
6,within_3_weeks,6.000000
7,within_4_weeks,7.000000
8,detected_before_external,3.000000
9,same_week,3.000000


In [826]:
n = len(
    confirmed_shift_events
)

for weeks in [1, 2, 3, 4]:
    count = (
        abs_error <= weeks
    ).sum()

    print(
        f"Dentro de ±{weeks} semana(s): "
        f"{count}/{n} "
        f"({count / n:.1%})"
    )

Dentro de ±1 semana(s): 3/7 (42.9%)
Dentro de ±2 semana(s): 5/7 (71.4%)
Dentro de ±3 semana(s): 6/7 (85.7%)
Dentro de ±4 semana(s): 7/7 (100.0%)


In [827]:
direction_summary = (
    confirmed_shift_events
    .groupby(
        "direction",
        as_index=False,
    )
    .agg(
        events=(
            "direction",
            "size",
        ),
        median_abs_error=(
            "absolute_temporal_error_weeks",
            "median",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
        median_abs_change=(
            "peak_absolute_change",
            lambda x: x.abs().median(),
        ),
        median_duration=(
            "duration_snapshots",
            "median",
        ),
    )
)

display(direction_summary)

,direction,events,median_abs_error,median_llr,median_abs_change,median_duration
0,DOWN,2,0.0,70.013096,0.109307,1.0
1,UP,5,2.0,126.963206,0.203946,14.0


In [828]:
SHIFT_VERSION = "V4_FROZEN"

SHIFT_CONFIG = {
    "week_start": 8,
    "week_end": 26,
    "min_weeks_each_side": 3,
}

HOLDOUT_CASES = [
    ("PE", 2024),
    ("PE", 2025),
    ("CE", 2024),
    ("CE", 2025),
    ("GO", 2024),
    ("GO", 2025),
]

In [829]:
holdout_events = []

for region, year in HOLDOUT_CASES:
    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}" in p.name.upper()
    )

    weekly = build_flu_weekly_for_state_year(
        path=path,
        region=region,
        year=year,
    )

    history = build_shift_event_history(
        weekly_df=weekly,
        week_start=SHIFT_CONFIG["week_start"],
        week_end=SHIFT_CONFIG["week_end"],
        min_weeks_each_side=SHIFT_CONFIG[
            "min_weeks_each_side"
        ],
    )

    events = extract_shift_events_from_history(
        history
    )

    if events.empty:
        holdout_events.append(
            pd.DataFrame(
                [{
                    "region": region,
                    "year": year,
                    "regime_id": pd.NA,
                    "direction": pd.NA,
                    "representative_change_week": pd.NA,
                    "peak_llr": pd.NA,
                    "peak_absolute_change": pd.NA,
                    "duration_snapshots": 0,
                    "no_event": True,
                }]
            )
        )
        continue

    events["region"] = region
    events["year"] = year
    events["no_event"] = False

    holdout_events.append(events)

shift_holdout_blind = pd.concat(
    holdout_events,
    ignore_index=True,
)

display(
    shift_holdout_blind[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "no_event",
        ]
    ]
)

,region,year,regime_id,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,no_event
0,PE,2024,1,DOWN,16,73.767578,-0.267811,14,False
1,PE,2025,1,UP,13,14.041013,0.187747,6,False
2,PE,2025,2,DOWN,18,26.714464,-0.121286,8,False
3,CE,2024,1,UP,12,22.197936,0.184787,4,False
4,CE,2024,2,DOWN,19,938.381214,-0.690104,10,False
5,CE,2025,1,UP,14,65.387365,0.158593,14,False
6,GO,2024,1,UP,14,94.616864,0.203941,14,False
7,GO,2025,1,UP,16,339.761872,0.275547,14,False


In [830]:
shift_holdout_blind.to_csv(
    "../data/processed/"
    "shift_holdout_blind_predictions.csv",
    index=False,
)

print(
    "Predições cegas congeladas:",
    len(shift_holdout_blind),
)

Predições cegas congeladas: 8


In [831]:
from datetime import datetime, timezone

blind_timestamp = datetime.now(
    timezone.utc
).isoformat()

print(
    "SHIFT:",
    SHIFT_VERSION,
)

print(
    "Frozen at:",
    blind_timestamp,
)

SHIFT: V4_FROZEN
Frozen at: 2026-10-07T21:19:50.602828+00:00


In [832]:
holdout_ground_truth = pd.DataFrame(
    columns=[
        "region",
        "year",
        "event_direction",
        "reference_week",
        "status",
        "source",
        "notes",
    ]
)

In [833]:
MATCH_WINDOW_WEEKS = 4

In [834]:
def event_matches(
    predicted_direction,
    predicted_week,
    external_direction,
    external_week,
    max_distance=4,
):
    if predicted_direction != external_direction:
        return False

    return (
        abs(
            int(predicted_week)
            - int(external_week)
        )
        <= max_distance
    )

In [835]:
HOLDOUT_CASES = [
    ("PE", 2024),
    ("PE", 2025),
    ("CE", 2024),
    ("CE", 2025),
    ("GO", 2024),
    ("GO", 2025),
]

In [836]:
holdout_events = []

for region, year in HOLDOUT_CASES:
    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}" in p.name.upper()
    )

    weekly = build_flu_weekly_for_state_year(
        path=path,
        region=region,
        year=year,
    )

    history = build_shift_event_history(
        weekly_df=weekly,
        week_start=8,
        week_end=26,
        min_weeks_each_side=3,
    )

    events = extract_shift_events_from_history(
        history
    )

    if events.empty:
        holdout_events.append(
            pd.DataFrame(
                [{
                    "region": region,
                    "year": year,
                    "regime_id": pd.NA,
                    "direction": pd.NA,
                    "first_observed_week": pd.NA,
                    "last_observed_week": pd.NA,
                    "representative_change_week": pd.NA,
                    "peak_evidence_week": pd.NA,
                    "peak_llr": pd.NA,
                    "peak_absolute_change": pd.NA,
                    "peak_left_positivity": pd.NA,
                    "peak_right_positivity": pd.NA,
                    "duration_snapshots": 0,
                    "no_event": True,
                    "right_censored": False,
                }]
            )
        )
        continue

    events["region"] = region
    events["year"] = year
    events["no_event"] = False

    events["right_censored"] = (
        events["last_observed_week"].eq(26)
        &
        events["first_observed_week"].eq(
            events["last_observed_week"]
        )
    )

    holdout_events.append(events)

shift_holdout_blind = pd.concat(
    holdout_events,
    ignore_index=True,
)

display(
    shift_holdout_blind[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "first_observed_week",
            "last_observed_week",
            "peak_evidence_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
            "no_event",
        ]
    ]
)

,region,year,regime_id,direction,representative_change_week,first_observed_week,last_observed_week,peak_evidence_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored,no_event
0,PE,2024,1,DOWN,16,13,26,26,73.767578,-0.267811,14,False,False
1,PE,2025,1,UP,13,13,18,15,14.041013,0.187747,6,False,False
2,PE,2025,2,DOWN,18,19,26,26,26.714464,-0.121286,8,False,False
3,CE,2024,1,UP,12,13,16,14,22.197936,0.184787,4,False,False
4,CE,2024,2,DOWN,19,17,26,26,938.381214,-0.690104,10,False,False
5,CE,2025,1,UP,14,13,26,21,65.387365,0.158593,14,False,False
6,GO,2024,1,UP,14,13,26,26,94.616864,0.203941,14,False,False
7,GO,2025,1,UP,16,13,26,26,339.761872,0.275547,14,False,False


In [837]:
shift_holdout_blind[
    "peak_relative_change"
] = (
    shift_holdout_blind[
        "peak_right_positivity"
    ]
    /
    shift_holdout_blind[
        "peak_left_positivity"
    ]
)

display(
    shift_holdout_blind[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_absolute_change",
            "peak_relative_change",
            "peak_llr",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_absolute_change,peak_relative_change,peak_llr,duration_snapshots,right_censored
0,PE,2024,DOWN,16,-0.267811,0.187488,73.767578,14,False
1,PE,2025,UP,13,0.187747,2.439394,14.041013,6,False
2,PE,2025,DOWN,18,-0.121286,0.453482,26.714464,8,False
3,CE,2024,UP,12,0.184787,1.257970,22.197936,4,False
4,CE,2024,DOWN,19,-0.690104,0.080600,938.381214,10,False
5,CE,2025,UP,14,0.158593,2.268741,65.387365,14,False
6,GO,2024,UP,14,0.203941,4.025126,94.616864,14,False
7,GO,2025,UP,16,0.275547,4.218084,339.761872,14,False


In [838]:
from datetime import datetime, timezone

SHIFT_VERSION = "V4_FROZEN"
MATCH_WINDOW_WEEKS = 4

blind_timestamp = datetime.now(
    timezone.utc
).isoformat()

shift_holdout_blind[
    "algorithm_version"
] = SHIFT_VERSION

shift_holdout_blind[
    "blind_frozen_at"
] = blind_timestamp

shift_holdout_blind[
    "external_status"
] = "NOT_CHECKED_YET"

display(
    shift_holdout_blind[
        [
            "region",
            "year",
            "regime_id",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
            "algorithm_version",
            "external_status",
        ]
    ]
)

,region,year,regime_id,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored,algorithm_version,external_status
0,PE,2024,1,DOWN,16,73.767578,-0.267811,14,False,V4_FROZEN,NOT_CHECKED_YET
1,PE,2025,1,UP,13,14.041013,0.187747,6,False,V4_FROZEN,NOT_CHECKED_YET
2,PE,2025,2,DOWN,18,26.714464,-0.121286,8,False,V4_FROZEN,NOT_CHECKED_YET
3,CE,2024,1,UP,12,22.197936,0.184787,4,False,V4_FROZEN,NOT_CHECKED_YET
4,CE,2024,2,DOWN,19,938.381214,-0.690104,10,False,V4_FROZEN,NOT_CHECKED_YET
5,CE,2025,1,UP,14,65.387365,0.158593,14,False,V4_FROZEN,NOT_CHECKED_YET
6,GO,2024,1,UP,14,94.616864,0.203941,14,False,V4_FROZEN,NOT_CHECKED_YET
7,GO,2025,1,UP,16,339.761872,0.275547,14,False,V4_FROZEN,NOT_CHECKED_YET


In [839]:
output_path = (
    "../data/processed/"
    "shift_holdout_blind_predictions.csv"
)

shift_holdout_blind.to_csv(
    output_path,
    index=False,
)

print("Salvo em:", output_path)
print("Versão:", SHIFT_VERSION)
print("Frozen at:", blind_timestamp)
print("Casos holdout:", HOLDOUT_CASES)

Salvo em: ../data/processed/shift_holdout_blind_predictions.csv
Versão: V4_FROZEN
Frozen at: 2026-10-07T21:22:31.837393+00:00
Casos holdout: [('PE', 2024), ('PE', 2025), ('CE', 2024), ('CE', 2025), ('GO', 2024), ('GO', 2025)]


In [840]:
holdout_case_summary = (
    shift_holdout_blind
    .groupby(
        [
            "region",
            "year",
        ],
        as_index=False,
    )
    .agg(
        detected_events=(
            "no_event",
            lambda x: (~x).sum(),
        ),
        directions=(
            "direction",
            lambda x: ", ".join(
                x.dropna().astype(str)
            ),
        ),
        max_llr=(
            "peak_llr",
            "max",
        ),
        max_abs_change=(
            "peak_absolute_change",
            lambda x: (
                pd.to_numeric(
                    x,
                    errors="coerce",
                )
                .abs()
                .max()
            ),
        ),
    )
)

display(holdout_case_summary)

,region,year,detected_events,directions,max_llr,max_abs_change
0,CE,2024,2,"UP, DOWN",938.381214,0.690104
1,CE,2025,1,UP,65.387365,0.158593
2,GO,2024,1,UP,94.616864,0.203941
3,GO,2025,1,UP,339.761872,0.275547
4,PE,2024,1,DOWN,73.767578,0.267811
5,PE,2025,2,"UP, DOWN",26.714464,0.187747


In [841]:
{
    "region": "CE",
    "year": 2024,
    "regime_id": 1,
    "external_status": "CONFIRMED",
    "external_reference_week": 15,
}

{
    "region": "CE",
    "year": 2024,
    "regime_id": 2,
    "external_status": "CONFIRMED",
    "external_reference_week": 18,
}

{'region': 'CE',
 'year': 2024,
 'regime_id': 2,
 'external_status': 'CONFIRMED',
 'external_reference_week': 18}

In [842]:
holdout_external_updates = [
    # CE 2024
    {
        "region": "CE",
        "year": 2024,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 15,
        "notes": (
            "SESA-CE: Influenza A atingiu pico na SE15/2024; "
            "evento SHIFT_UP detectado na SE12."
        ),
    },
    {
        "region": "CE",
        "year": 2024,
        "regime_id": 2,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 18,
        "notes": (
            "SESA-CE: diminuição progressiva da positividade "
            "de Influenza A a partir da SE18."
        ),
    },

    # CE 2025
    {
        "region": "CE",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 12,
        "notes": (
            "SESA-CE: incremento da circulação de Influenza "
            "a partir da SE12/2025."
        ),
    },

    # GO 2024
    {
        "region": "GO",
        "year": 2024,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED_NO_PRECISE_WEEK",
        "external_reference_week": pd.NA,
        "notes": (
            "SES-GO informou aumento de influenza no período, "
            "mas sem semana de início suficientemente precisa."
        ),
    },

    # GO 2025
    {
        "region": "GO",
        "year": 2025,
        "regime_id": 1,
        "external_label": 1,
        "external_status": "CONFIRMED",
        "external_reference_week": 13,
        "notes": (
            "SES-GO: aumento de influenza a partir do fim de março, "
            "com pico entre SE17-SE21 e sete semanas acima "
            "do limite superior esperado."
        ),
    },

    # PE 2024
    {
        "region": "PE",
        "year": 2024,
        "regime_id": 1,
        "external_label": pd.NA,
        "external_status": "UNLABELED",
        "external_reference_week": pd.NA,
        "notes": (
            "Fonte oficial mostra pico geral de SRAG na SE17, "
            "mas não permite confirmar queda específica de influenza."
        ),
    },

    # PE 2025
    {
        "region": "PE",
        "year": 2025,
        "regime_id": 1,
        "external_label": pd.NA,
        "external_status": "UNLABELED",
        "external_reference_week": pd.NA,
        "notes": (
            "Sem evidência externa específica suficiente para "
            "confirmar temporalmente o SHIFT_UP."
        ),
    },
    {
        "region": "PE",
        "year": 2025,
        "regime_id": 2,
        "external_label": pd.NA,
        "external_status": "UNLABELED",
        "external_reference_week": pd.NA,
        "notes": (
            "Sem evidência externa específica suficiente para "
            "confirmar temporalmente o SHIFT_DOWN."
        ),
    },
]

In [843]:
holdout_validation = (
    shift_holdout_blind.copy()
)

holdout_validation[
    "external_label"
] = pd.NA

holdout_validation[
    "external_status"
] = "UNLABELED"

holdout_validation[
    "external_reference_week"
] = pd.NA

holdout_validation[
    "notes"
] = ""

for update in holdout_external_updates:
    mask = (
        holdout_validation["region"].eq(
            update["region"]
        )
        &
        holdout_validation["year"].eq(
            update["year"]
        )
        &
        holdout_validation["regime_id"].eq(
            update["regime_id"]
        )
    )

    for column in [
        "external_label",
        "external_status",
        "external_reference_week",
        "notes",
    ]:
        holdout_validation.loc[
            mask,
            column,
        ] = update[column]

In [844]:
holdout_confirmed = (
    holdout_validation.loc[
        holdout_validation[
            "external_status"
        ].eq("CONFIRMED")
        &
        holdout_validation[
            "external_reference_week"
        ].notna()
    ]
    .copy()
)

holdout_confirmed[
    "external_reference_week"
] = (
    holdout_confirmed[
        "external_reference_week"
    ].astype(int)
)

holdout_confirmed[
    "temporal_error_weeks"
] = (
    holdout_confirmed[
        "representative_change_week"
    ]
    -
    holdout_confirmed[
        "external_reference_week"
    ]
)

holdout_confirmed[
    "absolute_temporal_error_weeks"
] = (
    holdout_confirmed[
        "temporal_error_weeks"
    ].abs()
)

display(
    holdout_confirmed[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "external_reference_week",
            "temporal_error_weeks",
            "peak_llr",
            "peak_absolute_change",
        ]
    ]
)

,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks,peak_llr,peak_absolute_change
3,CE,2024,UP,12,15,-3,22.197936,0.184787
4,CE,2024,DOWN,19,18,1,938.381214,-0.690104
5,CE,2025,UP,14,12,2,65.387365,0.158593
7,GO,2025,UP,16,13,3,339.761872,0.275547


In [845]:
n = len(holdout_confirmed)

print(
    "=== HOLDOUT CE/GO/PE ==="
)

print(
    "Eventos temporalmente confirmados:",
    n,
)

print(
    "Erro absoluto mediano:",
    holdout_confirmed[
        "absolute_temporal_error_weeks"
    ].median(),
)

print(
    "Erro absoluto médio:",
    holdout_confirmed[
        "absolute_temporal_error_weeks"
    ].mean(),
)

print(
    "Erro máximo:",
    holdout_confirmed[
        "absolute_temporal_error_weeks"
    ].max(),
)

for window in [1, 2, 3, 4]:
    hits = (
        holdout_confirmed[
            "absolute_temporal_error_weeks"
        ]
        <= window
    ).sum()

    print(
        f"±{window} semana(s): "
        f"{hits}/{n} "
        f"({hits / n:.1%})"
    )

=== HOLDOUT CE/GO/PE ===
Eventos temporalmente confirmados: 4
Erro absoluto mediano: 2.5
Erro absoluto médio: 2.25
Erro máximo: 3
±1 semana(s): 1/4 (25.0%)
±2 semana(s): 2/4 (50.0%)
±3 semana(s): 4/4 (100.0%)
±4 semana(s): 4/4 (100.0%)


In [846]:
development_external = (
    confirmed_shift_events
    .copy()
)

development_external[
    "dataset"
] = "DEVELOPMENT_EXTERNAL"

In [847]:
blind_holdout = (
    holdout_confirmed
    .copy()
)

blind_holdout[
    "dataset"
] = "BLIND_HOLDOUT"

In [848]:
validation_columns = [
    "dataset",
    "region",
    "year",
    "direction",
    "representative_change_week",
    "external_reference_week",
    "temporal_error_weeks",
    "absolute_temporal_error_weeks",
    "peak_llr",
    "peak_absolute_change",
    "duration_snapshots",
    "right_censored",
]

In [849]:
for col in validation_columns:
    if col not in development_external.columns:
        development_external[col] = pd.NA

    if col not in blind_holdout.columns:
        blind_holdout[col] = pd.NA

In [850]:
shift_validation_all = pd.concat(
    [
        development_external[
            validation_columns
        ],
        blind_holdout[
            validation_columns
        ],
    ],
    ignore_index=True,
)

display(
    shift_validation_all.sort_values(
        [
            "dataset",
            "year",
            "region",
            "representative_change_week",
        ]
    )
)

,dataset,region,year,direction,representative_change_week,external_reference_week,temporal_error_weeks,absolute_temporal_error_weeks,peak_llr,peak_absolute_change,duration_snapshots,right_censored
7,BLIND_HOLDOUT,CE,2024,UP,12,15,-3,3.0,22.197936,0.184787,4,False
8,BLIND_HOLDOUT,CE,2024,DOWN,19,18,1,1.0,938.381214,-0.690104,10,False
9,BLIND_HOLDOUT,CE,2025,UP,14,12,2,2.0,65.387365,0.158593,14,False
10,BLIND_HOLDOUT,GO,2025,UP,16,13,3,3.0,339.761872,0.275547,14,False
0,DEVELOPMENT_EXTERNAL,BA,2025,UP,11,14,-3,3.0,33.916157,0.184437,13,False
1,DEVELOPMENT_EXTERNAL,BA,2025,DOWN,24,24,0,0.0,34.841122,-0.091669,1,True
2,DEVELOPMENT_EXTERNAL,PR,2025,UP,16,18,-2,2.0,688.183542,0.323666,14,False
3,DEVELOPMENT_EXTERNAL,RJ,2025,UP,16,16,0,0.0,123.256631,0.159835,13,False
4,DEVELOPMENT_EXTERNAL,RJ,2025,DOWN,24,24,0,0.0,105.185071,-0.126945,1,True
5,DEVELOPMENT_EXTERNAL,RS,2025,UP,17,21,-4,4.0,332.537921,0.337454,14,False


In [851]:
validation_by_dataset = (
    shift_validation_all
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        events=(
            "region",
            "size",
        ),
        median_abs_error_weeks=(
            "absolute_temporal_error_weeks",
            "median",
        ),
        mean_abs_error_weeks=(
            "absolute_temporal_error_weeks",
            "mean",
        ),
        max_abs_error_weeks=(
            "absolute_temporal_error_weeks",
            "max",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
        median_abs_change=(
            "peak_absolute_change",
            lambda x: (
                pd.to_numeric(
                    x,
                    errors="coerce",
                )
                .abs()
                .median()
            ),
        ),
    )
)

display(validation_by_dataset)

,dataset,events,median_abs_error_weeks,mean_abs_error_weeks,max_abs_error_weeks,median_llr,median_abs_change
0,BLIND_HOLDOUT,4,2.5,2.250000,3.0,202.574618,0.230167
1,DEVELOPMENT_EXTERNAL,7,2.0,1.571429,4.0,123.256631,0.184437


In [852]:
rows = []

for dataset, group in (
    shift_validation_all
    .groupby("dataset")
):
    errors = (
        pd.to_numeric(
            group[
                "absolute_temporal_error_weeks"
            ],
            errors="coerce",
        )
        .dropna()
    )

    n = len(errors)

    rows.append(
        {
            "dataset": dataset,
            "events": n,
            "within_1_week":
                (errors <= 1).sum(),
            "within_1_week_pct":
                (errors <= 1).mean(),
            "within_2_weeks":
                (errors <= 2).sum(),
            "within_2_weeks_pct":
                (errors <= 2).mean(),
            "within_3_weeks":
                (errors <= 3).sum(),
            "within_3_weeks_pct":
                (errors <= 3).mean(),
            "within_4_weeks":
                (errors <= 4).sum(),
            "within_4_weeks_pct":
                (errors <= 4).mean(),
        }
    )

temporal_accuracy = pd.DataFrame(
    rows
)

display(temporal_accuracy)

,dataset,events,within_1_week,within_1_week_pct,within_2_weeks,within_2_weeks_pct,within_3_weeks,within_3_weeks_pct,within_4_weeks,within_4_weeks_pct
0,BLIND_HOLDOUT,4,1,0.250000,2,0.500000,4,1.000000,4,1.0
1,DEVELOPMENT_EXTERNAL,7,3,0.428571,5,0.714286,6,0.857143,7,1.0


In [853]:
errors_all = (
    pd.to_numeric(
        shift_validation_all[
            "absolute_temporal_error_weeks"
        ],
        errors="coerce",
    )
    .dropna()
)

print(
    "=== VALIDAÇÃO TOTAL ==="
)

print(
    "Eventos:",
    len(errors_all),
)

print(
    "Erro absoluto mediano:",
    errors_all.median(),
)

print(
    "Erro absoluto médio:",
    errors_all.mean(),
)

print(
    "Erro máximo:",
    errors_all.max(),
)

for weeks in [1, 2, 3, 4]:
    hits = (
        errors_all <= weeks
    ).sum()

    print(
        f"±{weeks} semana(s): "
        f"{hits}/{len(errors_all)} "
        f"({hits / len(errors_all):.1%})"
    )

=== VALIDAÇÃO TOTAL ===
Eventos: 11
Erro absoluto mediano: 2.0
Erro absoluto médio: 1.8181818181818181
Erro máximo: 4.0
±1 semana(s): 4/11 (36.4%)
±2 semana(s): 7/11 (63.6%)
±3 semana(s): 10/11 (90.9%)
±4 semana(s): 11/11 (100.0%)


In [854]:
display(
    validation_by_dataset[
        [
            "dataset",
            "events",
            "median_abs_error_weeks",
            "mean_abs_error_weeks",
            "max_abs_error_weeks",
        ]
    ]
)

,dataset,events,median_abs_error_weeks,mean_abs_error_weeks,max_abs_error_weeks
0,BLIND_HOLDOUT,4,2.5,2.250000,3.0
1,DEVELOPMENT_EXTERNAL,7,2.0,1.571429,4.0


In [855]:
VALIDATION_STATUS = {
    "algorithm": "SHIFT V4_FROZEN",
    "development_external_events": 7,
    "blind_holdout_events": 4,
    "holdout_states": ["PE", "CE", "GO"],
    "parameters_changed_after_freeze": False,
}

In [856]:
shift_validation_all.to_csv(
    "../data/processed/"
    "shift_external_validation_all.csv",
    index=False,
)

validation_by_dataset.to_csv(
    "../data/processed/"
    "shift_validation_summary_by_dataset.csv",
    index=False,
)

temporal_accuracy.to_csv(
    "../data/processed/"
    "shift_validation_temporal_accuracy.csv",
    index=False,
)

In [857]:
EVAL_CASES = [
    ("PE", 2024),
    ("PE", 2025),
    ("CE", 2024),
    ("CE", 2025),
    ("GO", 2024),
    ("GO", 2025),
]

eval_grid = pd.MultiIndex.from_product(
    [
        EVAL_CASES,
        range(8, 27),
    ],
    names=[
        "case",
        "epi_week",
    ],
).to_frame(
    index=False
)

eval_grid[
    ["region", "year"]
] = pd.DataFrame(
    eval_grid[
        "case"
    ].tolist(),
    index=eval_grid.index,
)

eval_grid = (
    eval_grid.drop(
        columns="case"
    )
    .loc[
        :,
        [
            "region",
            "year",
            "epi_week",
        ],
    ]
)

display(eval_grid.head())

,region,year,epi_week
0,PE,2024,8
1,PE,2024,9
2,PE,2024,10
3,PE,2024,11
4,PE,2024,12


In [858]:
eval_grid[
    "ground_truth"
] = pd.NA

eval_grid[
    "ground_truth_direction"
] = pd.NA

eval_grid[
    "ground_truth_status"
] = "UNLABELED"

eval_grid[
    "notes"
] = ""

In [859]:
external_event_windows = [
    {
        "region": "CE",
        "year": 2024,
        "direction": "UP",
        "start_week": 13,
        "end_week": 17,
        "reference_week": 15,
    },
    {
        "region": "CE",
        "year": 2024,
        "direction": "DOWN",
        "start_week": 18,
        "end_week": 21,
        "reference_week": 18,
    },
    {
        "region": "CE",
        "year": 2025,
        "direction": "UP",
        "start_week": 12,
        "end_week": 16,
        "reference_week": 12,
    },
    {
        "region": "GO",
        "year": 2025,
        "direction": "UP",
        "start_week": 13,
        "end_week": 21,
        "reference_week": 13,
    },
]

In [860]:
for event in external_event_windows:
    mask = (
        eval_grid[
            "region"
        ].eq(event["region"])
        &
        eval_grid[
            "year"
        ].eq(event["year"])
        &
        eval_grid[
            "epi_week"
        ].between(
            event["start_week"],
            event["end_week"],
        )
    )

    eval_grid.loc[
        mask,
        "ground_truth",
    ] = 1

    eval_grid.loc[
        mask,
        "ground_truth_direction",
    ] = event["direction"]

    eval_grid.loc[
        mask,
        "ground_truth_status",
    ] = "POSITIVE"

In [862]:
strong_negative_windows = [
    # preencher apenas com janelas sustentadas por fonte externa
]

In [863]:
for item in strong_negative_windows:
    mask = (
        eval_grid[
            "region"
        ].eq(item["region"])
        &
        eval_grid[
            "year"
        ].eq(item["year"])
        &
        eval_grid[
            "epi_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
        &
        eval_grid[
            "ground_truth"
        ].isna()
    )

    eval_grid.loc[
        mask,
        "ground_truth",
    ] = 0

    eval_grid.loc[
        mask,
        "ground_truth_status",
    ] = "NEGATIVE"

In [864]:
prediction_grid = (
    eval_grid[
        [
            "region",
            "year",
            "epi_week",
        ]
    ]
    .copy()
)

prediction_grid[
    "signal_detected"
] = False

prediction_grid[
    "predicted_direction"
] = pd.NA

In [865]:
for _, event in (
    shift_holdout_blind.loc[
        shift_holdout_blind[
            "no_event"
        ].eq(False)
    ]
    .iterrows()
):
    week = int(
        event[
            "representative_change_week"
        ]
    )

    mask = (
        prediction_grid[
            "region"
        ].eq(event["region"])
        &
        prediction_grid[
            "year"
        ].eq(event["year"])
        &
        prediction_grid[
            "epi_week"
        ].eq(week)
    )

    prediction_grid.loc[
        mask,
        "signal_detected",
    ] = True

    prediction_grid.loc[
        mask,
        "predicted_direction",
    ] = event["direction"]

In [866]:
weekly_eval = (
    eval_grid
    .merge(
        prediction_grid,
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="left",
    )
)

In [867]:
MATCH_WINDOW_WEEKS = 4

def event_detected(
    region,
    year,
    reference_week,
    direction,
):
    candidates = (
        shift_holdout_blind.loc[
            shift_holdout_blind[
                "region"
            ].eq(region)
            &
            shift_holdout_blind[
                "year"
            ].eq(year)
            &
            shift_holdout_blind[
                "direction"
            ].eq(direction)
        ]
    )

    if candidates.empty:
        return False

    return (
        candidates[
            "representative_change_week"
        ]
        .sub(reference_week)
        .abs()
        .le(MATCH_WINDOW_WEEKS)
        .any()
    )

In [868]:
event_eval_rows = []

for event in external_event_windows:
    detected = event_detected(
        region=event["region"],
        year=event["year"],
        reference_week=event[
            "reference_week"
        ],
        direction=event[
            "direction"
        ],
    )

    event_eval_rows.append(
        {
            **event,
            "detected": detected,
            "outcome": (
                "TP"
                if detected
                else "FN"
            ),
        }
    )

event_eval = pd.DataFrame(
    event_eval_rows
)

display(event_eval)

,region,year,direction,start_week,end_week,reference_week,detected,outcome
0,CE,2024,UP,13,17,15,True,TP
1,CE,2024,DOWN,18,21,18,True,TP
2,CE,2025,UP,12,16,12,True,TP
3,GO,2025,UP,13,21,13,True,TP


In [881]:
tp = (
    event_eval[
        "outcome"
    ].eq("TP")
).sum()

fn = (
    event_eval[
        "outcome"
    ].eq("FN")
).sum()

sensitivity = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else np.nan
)

print(
    "TP:",
    tp,
)

print(
    "FN:",
    fn,
)

print(
    "Sensibilidade:",
    f"{sensitivity:.1%}",
)

TP: 4
FN: 0
Sensibilidade: 100.0%


In [878]:
negative_eval_rows = []

for item in strong_negative_windows:
    predictions = (
        shift_holdout_blind.loc[
            shift_holdout_blind[
                "region"
            ].eq(item["region"])
            &
            shift_holdout_blind[
                "year"
            ].eq(item["year"])
            &
            shift_holdout_blind[
                "representative_change_week"
            ].between(
                item["start_week"],
                item["end_week"],
            )
        ]
    )

    fp = not predictions.empty

    negative_eval_rows.append(
        {
            **item,
            "false_positive": fp,
            "outcome": (
                "FP"
                if fp
                else "TN"
            ),
        }
    )

negative_eval = pd.DataFrame(
    negative_eval_rows
)

display(negative_eval)

""


In [879]:
negative_eval = pd.DataFrame(
    negative_eval_rows,
    columns=[
        "region",
        "year",
        "start_week",
        "end_week",
        "false_positive",
        "outcome",
    ],
)

In [880]:
if len(strong_negative_windows) == 0:
    print(
        "NEGATIVOS AINDA NÃO ROTULADOS."
    )
    print(
        "FPR e especificidade não devem "
        "ser calculados neste momento."
    )
else:
    negative_eval_rows = []

    for item in strong_negative_windows:
        predictions = (
            shift_holdout_blind.loc[
                shift_holdout_blind[
                    "region"
                ].eq(item["region"])
                &
                shift_holdout_blind[
                    "year"
                ].eq(item["year"])
                &
                shift_holdout_blind[
                    "representative_change_week"
                ].between(
                    item["start_week"],
                    item["end_week"],
                )
            ]
        )

        fp_detected = not predictions.empty

        negative_eval_rows.append(
            {
                **item,
                "false_positive":
                    fp_detected,
                "outcome":
                    (
                        "FP"
                        if fp_detected
                        else "TN"
                    ),
            }
        )

    negative_eval = pd.DataFrame(
        negative_eval_rows
    )

    display(negative_eval)

    fp = (
        negative_eval[
            "outcome"
        ].eq("FP")
    ).sum()

    tn = (
        negative_eval[
            "outcome"
        ].eq("TN")
    ).sum()

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    print("FP:", fp)
    print("TN:", tn)
    print("FPR:", f"{fpr:.1%}")
    print(
        "Especificidade:",
        f"{specificity:.1%}",
    )

NEGATIVOS AINDA NÃO ROTULADOS.
FPR e especificidade não devem ser calculados neste momento.


In [882]:
strong_negative_windows = [
    {
        "region": "CE",
        "year": 2024,
        "start_week": 27,
        "end_week": 34,
        "direction_to_test": "UP",
        "notes": (
            "SESA-CE informa queda progressiva de Influenza A "
            "após SE18 e ausência de identificação nas últimas "
            "quatro semanas do período analisado."
        ),
    },
    {
        "region": "CE",
        "year": 2025,
        "start_week": 24,
        "end_week": 29,
        "direction_to_test": "UP",
        "notes": (
            "Após pico de Influenza A na SE20, SESA-CE descreve "
            "declínio nas semanas seguintes."
        ),
    },
    {
        "region": "GO",
        "year": 2025,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "notes": (
            "Episódio oficial de aumento se concentra do fim de março "
            "até pico entre SE17-SE21; janela posterior usada como "
            "controle negativo para novo SHIFT_UP."
        ),
    },
]

In [883]:
strong_negative_windows = [
    {
        "region": "CE",
        "year": 2025,
        "start_week": 24,
        "end_week": 26,
        "direction_to_test": "UP",
        "notes": (
            "Pós-pico de Influenza A; boletim oficial descreve "
            "declínio após SE20."
        ),
    },
    {
        "region": "GO",
        "year": 2025,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "notes": (
            "Janela posterior ao pico oficial entre SE17-SE21."
        ),
    },
]

In [884]:
negative_eval_rows = []

for item in strong_negative_windows:
    predictions = shift_holdout_blind.loc[
        shift_holdout_blind["region"].eq(item["region"])
        &
        shift_holdout_blind["year"].eq(item["year"])
        &
        shift_holdout_blind["direction"].eq(
            item["direction_to_test"]
        )
        &
        shift_holdout_blind[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]

    false_positive = not predictions.empty

    negative_eval_rows.append(
        {
            **item,
            "false_positive": false_positive,
            "outcome": "FP" if false_positive else "TN",
        }
    )

negative_eval = pd.DataFrame(negative_eval_rows)

display(negative_eval)

,region,year,start_week,end_week,direction_to_test,notes,false_positive,outcome
0,CE,2025,24,26,UP,Pós-pico de Influenza A; boletim oficial descr...,False,TN
1,GO,2025,22,26,UP,Janela posterior ao pico oficial entre SE17-SE21.,False,TN


In [885]:
fp = negative_eval["outcome"].eq("FP").sum()
tn = negative_eval["outcome"].eq("TN").sum()

fpr = fp / (fp + tn)
specificity = tn / (tn + fp)

print("FP:", fp)
print("TN:", tn)
print("FPR:", f"{fpr:.1%}")
print("Especificidade:", f"{specificity:.1%}")

FP: 0
TN: 2
FPR: 0.0%
Especificidade: 100.0%


In [886]:
strong_negative_windows = [
    {
        "region": "CE",
        "year": 2024,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "Influenza A entrou em queda progressiva após SE18; "
            "janela tardia usada como negativa para novo SHIFT_UP."
        ),
    },
    {
        "region": "CE",
        "year": 2025,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "Após pico na SE20, boletim oficial descreve declínio "
            "da circulação de Influenza A."
        ),
    },
]

In [887]:
negative_eval_rows = []

for item in strong_negative_windows:
    predictions = shift_holdout_blind.loc[
        shift_holdout_blind["region"].eq(item["region"])
        &
        shift_holdout_blind["year"].eq(item["year"])
        &
        shift_holdout_blind["direction"].eq(
            item["direction_to_test"]
        )
        &
        shift_holdout_blind[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]

    false_positive = not predictions.empty

    negative_eval_rows.append(
        {
            **item,
            "predictions_in_window": len(predictions),
            "false_positive": false_positive,
            "outcome": "FP" if false_positive else "TN",
        }
    )

negative_eval = pd.DataFrame(negative_eval_rows)

display(negative_eval)

fp = negative_eval["outcome"].eq("FP").sum()
tn = negative_eval["outcome"].eq("TN").sum()

print("FP:", fp)
print("TN:", tn)

print(
    "FPR:",
    f"{fp / (fp + tn):.1%}"
)

print(
    "Especificidade:",
    f"{tn / (tn + fp):.1%}"
)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,predictions_in_window,false_positive,outcome
0,CE,2024,22,26,UP,0,STRONG_NEGATIVE,Influenza A entrou em queda progressiva após S...,0,False,TN
1,CE,2025,22,26,UP,0,STRONG_NEGATIVE,"Após pico na SE20, boletim oficial descreve de...",0,False,TN


FP: 0
TN: 2
FPR: 0.0%
Especificidade: 100.0%


In [888]:
NEGATIVE_VALIDATION_PANEL = [
    ("BA", 2024),
    ("CE", 2024),
    ("CE", 2025),
    ("ES", 2024),
    ("ES", 2025),
    ("GO", 2024),
    ("GO", 2025),
    ("MT", 2024),
    ("MT", 2025),
    ("PB", 2024),
    ("PB", 2025),
    ("PE", 2024),
    ("PE", 2025),
    ("PR", 2024),
    ("RJ", 2024),
    ("SC", 2024),
]

In [889]:
negative_panel_registry = pd.DataFrame(
    NEGATIVE_VALIDATION_PANEL,
    columns=[
        "region",
        "year",
    ],
)

negative_panel_registry[
    "algorithm_version"
] = "V4_FROZEN"

negative_panel_registry[
    "ground_truth_checked"
] = False

negative_panel_registry[
    "eligible_negative_window"
] = pd.NA

negative_panel_registry[
    "negative_start_week"
] = pd.NA

negative_panel_registry[
    "negative_end_week"
] = pd.NA

negative_panel_registry[
    "direction_to_test"
] = pd.NA

negative_panel_registry[
    "source_status"
] = "NOT_REVIEWED"

display(negative_panel_registry)

,region,year,algorithm_version,ground_truth_checked,eligible_negative_window,negative_start_week,negative_end_week,direction_to_test,source_status
0,BA,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
1,CE,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
2,CE,2025,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
3,ES,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
4,ES,2025,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
5,GO,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
6,GO,2025,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
7,MT,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
8,MT,2025,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED
9,PB,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED


In [890]:
negative_panel_registry.to_csv(
    "../data/processed/"
    "shift_negative_validation_panel_preregistered.csv",
    index=False,
)

In [891]:
NEGATIVE_LABEL_RULE = {
    "requires_official_source": True,
    "requires_weekly_or_near_weekly_timing": True,
    "requires_direction_specific_evidence": True,
    "absence_of_news_is_not_negative": True,
    "annual_summary_alone_is_not_enough": True,
}

In [895]:
strong_negative_windows.extend(
    [
        {
            "region": "PB",
            "year": 2024,
            "start_week": 24,
            "end_week": 26,
            "direction_to_test": "UP",
            "ground_truth": 0,
            "status": "STRONG_NEGATIVE",
            "notes": (
                "SES-PB: Influenza A detectada até SE23 e queda "
                "na detecção nas semanas epidemiológicas mais recentes."
            ),
        },
        {
            "region": "SC",
            "year": 2024,
            "start_week": 21,
            "end_week": 26,
            "direction_to_test": "UP",
            "ground_truth": 0,
            "status": "STRONG_NEGATIVE",
            "notes": (
                "DIVE-SC: pico de influenza em torno da SE18 "
                "e tendência de queda a partir da SE20/SE21."
            ),
        },
    ]
)

In [896]:
def run_frozen_shift_case(
    region: str,
    year: int,
) -> pd.DataFrame:
    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}"
        in p.name.upper()
    )

    weekly = (
        build_flu_weekly_for_state_year(
            path=path,
            region=region,
            year=year,
        )
    )

    history = (
        build_shift_event_history(
            weekly_df=weekly,
            week_start=8,
            week_end=26,
            min_weeks_each_side=3,
        )
    )

    events = (
        extract_shift_events_from_history(
            history
        )
    )

    if events.empty:
        return pd.DataFrame(
            [{
                "region": region,
                "year": year,
                "regime_id": pd.NA,
                "direction": pd.NA,
                "representative_change_week": pd.NA,
                "peak_llr": pd.NA,
                "peak_absolute_change": pd.NA,
                "duration_snapshots": 0,
                "right_censored": False,
                "no_event": True,
                "algorithm_version": "V4_FROZEN",
            }]
        )

    events["region"] = region
    events["year"] = year
    events["no_event"] = False
    events["algorithm_version"] = "V4_FROZEN"

    events["right_censored"] = (
        events["last_observed_week"].eq(26)
        &
        events["first_observed_week"].eq(
            events["last_observed_week"]
        )
    )

    return events

In [897]:
negative_eval_rows = []

for item in strong_negative_windows:
    predictions = shift_holdout_blind.loc[
        shift_holdout_blind["region"].eq(item["region"])
        &
        shift_holdout_blind["year"].eq(item["year"])
        &
        shift_holdout_blind["direction"].eq(
            item["direction_to_test"]
        )
        &
        shift_holdout_blind[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]

    false_positive = not predictions.empty

    negative_eval_rows.append(
        {
            **item,
            "predictions_in_window": len(predictions),
            "false_positive": false_positive,
            "outcome": (
                "FP"
                if false_positive
                else "TN"
            ),
        }
    )

negative_eval = pd.DataFrame(
    negative_eval_rows
)

display(
    negative_eval[
        [
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
            "predictions_in_window",
            "outcome",
        ]
    ]
)

fp = negative_eval["outcome"].eq("FP").sum()
tn = negative_eval["outcome"].eq("TN").sum()

print("FP:", fp)
print("TN:", tn)
print(
    "FPR:",
    f"{fp / (fp + tn):.1%}"
)
print(
    "Especificidade:",
    f"{tn / (tn + fp):.1%}"
)

,region,year,start_week,end_week,direction_to_test,predictions_in_window,outcome
0,CE,2024,22,26,UP,0,TN
1,CE,2025,22,26,UP,0,TN
2,PB,2024,24,26,UP,0,TN
3,SC,2024,21,26,UP,0,TN
4,PB,2024,24,26,UP,0,TN
5,SC,2024,21,26,UP,0,TN


FP: 0
TN: 6
FPR: 0.0%
Especificidade: 100.0%


In [899]:
extra_negative_cases = pd.concat(
    [
        run_frozen_shift_case(
            "PB",
            2024,
        ),
        run_frozen_shift_case(
            "SC",
            2024,
        ),
    ],
    ignore_index=True,
)

display(
    extra_negative_cases[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots
0,PB,2024,DOWN,11,0.731811,-0.082082,1
1,PB,2024,UP,12,4.106813,0.122351,4
2,PB,2024,DOWN,18,99.956165,-0.291716,9
3,SC,2024,DOWN,11,0.431509,-0.018467,2
4,SC,2024,UP,15,5.421590,0.045966,9
5,SC,2024,DOWN,22,7.781389,-0.056444,3


In [901]:
negative_eval = (
    negative_eval
    .drop_duplicates(
        subset=[
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
        ]
    )
    .reset_index(drop=True)
)

display(negative_eval)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,predictions_in_window,false_positive,outcome
0,CE,2024,22,26,UP,0,STRONG_NEGATIVE,Influenza A entrou em queda progressiva após S...,0,False,TN
1,CE,2025,22,26,UP,0,STRONG_NEGATIVE,"Após pico na SE20, boletim oficial descreve de...",0,False,TN
2,PB,2024,24,26,UP,0,STRONG_NEGATIVE,SES-PB: Influenza A detectada até SE23 e queda...,0,False,TN
3,SC,2024,21,26,UP,0,STRONG_NEGATIVE,DIVE-SC: pico de influenza em torno da SE18 e ...,0,False,TN


In [902]:
fp = (
    negative_eval[
        "outcome"
    ].eq("FP")
).sum()

tn = (
    negative_eval[
        "outcome"
    ].eq("TN")
).sum()

fpr = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else np.nan
)

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

print("FP:", fp)
print("TN:", tn)
print("FPR:", f"{fpr:.1%}")
print(
    "Especificidade:",
    f"{specificity:.1%}",
)

FP: 0
TN: 4
FPR: 0.0%
Especificidade: 100.0%


In [903]:
strong_negative_windows = [
    dict(t)
    for t in {
        tuple(sorted(item.items()))
        for item in strong_negative_windows
    }
]

In [904]:
strong_negative_df = (
    pd.DataFrame(
        strong_negative_windows
    )
    .drop_duplicates(
        subset=[
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
        ]
    )
    .reset_index(drop=True)
)

strong_negative_windows = (
    strong_negative_df
    .to_dict("records")
)

In [906]:
from statsmodels.stats.proportion import (
    proportion_confint,
)

tp = 4
fn = 0
tn = 4
fp = 0

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

sens_low, sens_high = proportion_confint(
    count=tp,
    nobs=tp + fn,
    alpha=0.05,
    method="wilson",
)

spec_low, spec_high = proportion_confint(
    count=tn,
    nobs=tn + fp,
    alpha=0.05,
    method="wilson",
)

print(
    f"Sensibilidade: "
    f"{sensitivity:.1%} "
    f"(IC95% "
    f"{sens_low:.1%}–"
    f"{sens_high:.1%})"
)

print(
    f"Especificidade: "
    f"{specificity:.1%} "
    f"(IC95% "
    f"{spec_low:.1%}–"
    f"{spec_high:.1%})"
)

Sensibilidade: 100.0% (IC95% 51.0%–100.0%)
Especificidade: 100.0% (IC95% 51.0%–100.0%)


In [907]:
strong_negative_windows.extend(
    [
        {
            "region": "BA",
            "year": 2024,
            "start_week": 20,
            "end_week": 26,
            "direction_to_test": "UP",
            "ground_truth": 0,
            "status": "STRONG_NEGATIVE",
            "notes": (
                "SESAB: SRAG por influenza teve pico precoce em torno da SE9 "
                "e declínio sustentado nas semanas seguintes; janela SE20-26 "
                "usada como negativa para novo SHIFT_UP."
            ),
        },
        {
            "region": "ES",
            "year": 2024,
            "start_week": 22,
            "end_week": 26,
            "direction_to_test": "UP",
            "ground_truth": 0,
            "status": "STRONG_NEGATIVE",
            "notes": (
                "SESA-ES: SRAG apresentou pico entre aproximadamente SE11-17 "
                "e queda progressiva depois; janela tardia usada como negativa "
                "para novo SHIFT_UP."
            ),
        },
        {
            "region": "PB",
            "year": 2025,
            "start_week": 24,
            "end_week": 26,
            "direction_to_test": "UP",
            "ground_truth": 0,
            "status": "STRONG_NEGATIVE_MODERATE",
            "notes": (
                "SES-PB: detecção de vírus respiratórios em 2025 ficou abaixo "
                "de 2024 e Influenza A apresentou redução importante; "
                "SE24-26 usada como controle negativo para novo SHIFT_UP."
            ),
        },
    ]
)

In [908]:
strong_negative_df = (
    pd.DataFrame(strong_negative_windows)
    .drop_duplicates(
        subset=[
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
        ]
    )
    .reset_index(drop=True)
)

strong_negative_windows = (
    strong_negative_df
    .to_dict("records")
)

display(strong_negative_df)

,direction_to_test,end_week,ground_truth,notes,region,start_week,status,year
0,UP,26,0,SES-PB: Influenza A detectada até SE23 e queda...,PB,24,STRONG_NEGATIVE,2024
1,UP,26,0,DIVE-SC: pico de influenza em torno da SE18 e ...,SC,21,STRONG_NEGATIVE,2024
2,UP,26,0,"Após pico na SE20, boletim oficial descreve de...",CE,22,STRONG_NEGATIVE,2025
3,UP,26,0,Influenza A entrou em queda progressiva após S...,CE,22,STRONG_NEGATIVE,2024
4,UP,26,0,SESAB: SRAG por influenza teve pico precoce em...,BA,20,STRONG_NEGATIVE,2024
5,UP,26,0,SESA-ES: SRAG apresentou pico entre aproximada...,ES,22,STRONG_NEGATIVE,2024
6,UP,26,0,SES-PB: detecção de vírus respiratórios em 202...,PB,24,STRONG_NEGATIVE_MODERATE,2025


In [909]:
extra_negative_cases_2 = pd.concat(
    [
        run_frozen_shift_case("BA", 2024),
        run_frozen_shift_case("ES", 2024),
        run_frozen_shift_case("PB", 2025),
    ],
    ignore_index=True,
)

display(
    extra_negative_cases_2[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots
0,BA,2024,DOWN,12,343.204164,-0.300070,14
1,ES,2024,UP,15,70.297040,0.251675,14
2,PB,2025,UP,12,30.753870,0.411538,14


In [910]:
negative_prediction_pool = pd.concat(
    [
        shift_holdout_blind,
        extra_negative_cases,
        extra_negative_cases_2,
    ],
    ignore_index=True,
)

In [911]:
negative_eval_rows = []

for item in strong_negative_windows:
    predictions = (
        negative_prediction_pool.loc[
            negative_prediction_pool["region"].eq(
                item["region"]
            )
            &
            negative_prediction_pool["year"].eq(
                item["year"]
            )
            &
            negative_prediction_pool["direction"].eq(
                item["direction_to_test"]
            )
            &
            negative_prediction_pool[
                "representative_change_week"
            ].between(
                item["start_week"],
                item["end_week"],
            )
        ]
    )

    false_positive = not predictions.empty

    negative_eval_rows.append(
        {
            **item,
            "predictions_in_window": len(predictions),
            "false_positive": false_positive,
            "outcome": (
                "FP"
                if false_positive
                else "TN"
            ),
        }
    )

negative_eval = (
    pd.DataFrame(negative_eval_rows)
    .drop_duplicates(
        subset=[
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
        ]
    )
)

display(
    negative_eval[
        [
            "region",
            "year",
            "start_week",
            "end_week",
            "direction_to_test",
            "status",
            "predictions_in_window",
            "outcome",
        ]
    ]
)

fp = negative_eval["outcome"].eq("FP").sum()
tn = negative_eval["outcome"].eq("TN").sum()

print("FP:", fp)
print("TN:", tn)
print(
    "FPR:",
    f"{fp/(fp+tn):.1%}",
)
print(
    "Especificidade:",
    f"{tn/(tn+fp):.1%}",
)

,region,year,start_week,end_week,direction_to_test,status,predictions_in_window,outcome
0,PB,2024,24,26,UP,STRONG_NEGATIVE,0,TN
1,SC,2024,21,26,UP,STRONG_NEGATIVE,0,TN
2,CE,2025,22,26,UP,STRONG_NEGATIVE,0,TN
3,CE,2024,22,26,UP,STRONG_NEGATIVE,0,TN
4,BA,2024,20,26,UP,STRONG_NEGATIVE,0,TN
5,ES,2024,22,26,UP,STRONG_NEGATIVE,0,TN
6,PB,2025,24,26,UP,STRONG_NEGATIVE_MODERATE,0,TN


FP: 0
TN: 7
FPR: 0.0%
Especificidade: 100.0%


In [912]:
from statsmodels.stats.proportion import proportion_confint

spec_low, spec_high = proportion_confint(
    count=tn,
    nobs=tn + fp,
    alpha=0.05,
    method="wilson",
)

print(
    f"Especificidade: "
    f"{tn/(tn+fp):.1%} "
    f"(IC95% {spec_low:.1%}–{spec_high:.1%})"
)

Especificidade: 100.0% (IC95% 64.6%–100.0%)


In [913]:
holdout_checkpoint = {
    "algorithm_version": "V4_FROZEN",
    "tp": 4,
    "fn": 0,
    "tn": 7,
    "fp": 0,
    "sensitivity": 1.0,
    "specificity": 1.0,
    "false_positive_rate": 0.0,
}

pd.DataFrame(
    [holdout_checkpoint]
).to_csv(
    "../data/processed/"
    "shift_v4_holdout_checkpoint.csv",
    index=False,
)

In [914]:
confusion = pd.DataFrame(
    {
        "Predito positivo": [4, 0],
        "Predito negativo": [0, 7],
    },
    index=[
        "Real positivo",
        "Real negativo",
    ],
)

display(confusion)

,Predito positivo,Predito negativo
Real positivo,4,0
Real negativo,0,7


In [915]:
validation_metrics = pd.DataFrame(
    {
        "metric": [
            "TP",
            "FN",
            "TN",
            "FP",
            "sensitivity",
            "specificity",
            "false_positive_rate",
        ],
        "value": [
            4,
            0,
            7,
            0,
            1.0,
            1.0,
            0.0,
        ],
    }
)

display(validation_metrics)

,metric,value
0,TP,4.0
1,FN,0.0
2,TN,7.0
3,FP,0.0
4,sensitivity,1.0
5,specificity,1.0
6,false_positive_rate,0.0


In [916]:
remaining_panel_cases = [
    ("ES", 2025),
    ("GO", 2024),
    ("MT", 2024),
    ("MT", 2025),
    ("PE", 2024),
    ("PE", 2025),
    ("PR", 2024),
    ("RJ", 2024),
]

remaining_predictions = pd.concat(
    [
        run_frozen_shift_case(region, year)
        for region, year
        in remaining_panel_cases
    ],
    ignore_index=True,
)

display(
    remaining_predictions[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,ES,2025,UP,16,31.206539,0.214877,14,False
1,GO,2024,UP,14,94.616864,0.203941,14,False
2,MT,2024,UP,11,18.116944,0.553995,14,False
3,MT,2025,DOWN,11,0.185072,-0.035065,1,False
4,MT,2025,UP,14,11.402316,0.185806,13,False
5,PE,2024,DOWN,16,73.767578,-0.267811,14,False
6,PE,2025,UP,13,14.041013,0.187747,6,False
7,PE,2025,DOWN,18,26.714464,-0.121286,8,False
8,PR,2024,UP,13,118.959612,0.169410,14,False
9,RJ,2024,DOWN,16,122.019839,-0.169267,14,False


In [917]:
remaining_predictions[
    "algorithm_version"
] = "V4_FROZEN"

remaining_predictions[
    "ground_truth_status"
] = "NOT_REVIEWED"

remaining_predictions.to_csv(
    "../data/processed/"
    "shift_v4_remaining_negative_panel_predictions.csv",
    index=False,
)

In [918]:
negative_audit = pd.DataFrame(
    remaining_panel_cases,
    columns=[
        "region",
        "year",
    ],
)

negative_audit[
    "eligible_negative"
] = pd.NA

negative_audit[
    "start_week"
] = pd.NA

negative_audit[
    "end_week"
] = pd.NA

negative_audit[
    "direction_to_test"
] = pd.NA

negative_audit[
    "evidence_strength"
] = "NOT_REVIEWED"

negative_audit[
    "notes"
] = ""

display(negative_audit)

,region,year,eligible_negative,start_week,end_week,direction_to_test,evidence_strength,notes
0,ES,2025,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
1,GO,2024,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
2,MT,2024,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
3,MT,2025,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
4,PE,2024,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
5,PE,2025,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
6,PR,2024,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,
7,RJ,2024,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,


In [919]:
def eligible_as_strong_negative(
    has_official_source: bool,
    has_week_specific_evidence: bool,
    clearly_post_event_or_stable: bool,
    direction_is_defined: bool,
):
    return all(
        [
            has_official_source,
            has_week_specific_evidence,
            clearly_post_event_or_stable,
            direction_is_defined,
        ]
    )

In [920]:
SECOND_NEGATIVE_PANEL = [
    {
        "region": "AL",
        "year": 2024,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "SESAU-AL documenta circulação destacada de influenza "
            "entre SE7 e SE20 em 2024. Janela SE22-26 pré-registrada "
            "como controle negativo para novo SHIFT_UP."
        ),
    },
]

In [921]:
second_negative_registry = pd.DataFrame(
    SECOND_NEGATIVE_PANEL
)

second_negative_registry[
    "algorithm_version"
] = "V4_FROZEN"

second_negative_registry[
    "prediction_checked"
] = False

second_negative_registry.to_csv(
    "../data/processed/"
    "shift_v4_second_negative_panel_preregistered.csv",
    index=False,
)

display(second_negative_registry)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,algorithm_version,prediction_checked
0,AL,2024,22,26,UP,0,STRONG_NEGATIVE,SESAU-AL documenta circulação destacada de inf...,V4_FROZEN,False


In [922]:
al_2024_prediction = run_frozen_shift_case(
    "AL",
    2024,
)

display(
    al_2024_prediction[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots
0,AL,2024,DOWN,16,40.400329,-0.353505,14


In [923]:
negative_prediction_pool_v2 = pd.concat(
    [
        negative_prediction_pool,
        al_2024_prediction,
    ],
    ignore_index=True,
)

item = SECOND_NEGATIVE_PANEL[0]

al_window_predictions = (
    negative_prediction_pool_v2.loc[
        negative_prediction_pool_v2[
            "region"
        ].eq(item["region"])
        &
        negative_prediction_pool_v2[
            "year"
        ].eq(item["year"])
        &
        negative_prediction_pool_v2[
            "direction"
        ].eq(item["direction_to_test"])
        &
        negative_prediction_pool_v2[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]
)

print(
    "AL/2024:",
    "FP"
    if not al_window_predictions.empty
    else "TN"
)

AL/2024: TN


In [924]:
NEXT_AUDIT_CASES = [
    ("MS", 2024),
    ("MS", 2025),
    ("DF", 2024),
    ("DF", 2025),
    ("SE", 2024),
    ("RN", 2024),
]

In [925]:
from statsmodels.stats.proportion import proportion_confint

spec_low, spec_high = proportion_confint(
    count=8,
    nobs=8,
    alpha=0.05,
    method="wilson",
)

print(
    f"Especificidade: 100.0% "
    f"(IC95% {spec_low:.1%}–{spec_high:.1%})"
)

Especificidade: 100.0% (IC95% 67.6%–100.0%)


In [926]:
THIRD_NEGATIVE_AUDIT = [
    ("MS", 2024),
    ("SE", 2024),
    ("RN", 2024),
    ("DF", 2024),
    ("AL", 2023),
    ("PB", 2023),
]

In [927]:
third_negative_audit = pd.DataFrame(
    THIRD_NEGATIVE_AUDIT,
    columns=[
        "region",
        "year",
    ],
)

third_negative_audit[
    "algorithm_version"
] = "V4_FROZEN"

third_negative_audit[
    "ground_truth_checked"
] = False

third_negative_audit[
    "eligible_negative_window"
] = pd.NA

third_negative_audit[
    "negative_start_week"
] = pd.NA

third_negative_audit[
    "negative_end_week"
] = pd.NA

third_negative_audit[
    "direction_to_test"
] = pd.NA

third_negative_audit[
    "evidence_strength"
] = "NOT_REVIEWED"

third_negative_audit[
    "prediction_checked"
] = False

display(third_negative_audit)

,region,year,algorithm_version,ground_truth_checked,eligible_negative_window,negative_start_week,negative_end_week,direction_to_test,evidence_strength,prediction_checked
0,MS,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
1,SE,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
2,RN,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
3,DF,2024,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
4,AL,2023,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
5,PB,2023,V4_FROZEN,False,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False


In [928]:
third_negative_audit.to_csv(
    "../data/processed/"
    "shift_v4_third_negative_audit_preregistered.csv",
    index=False,
)

In [929]:
NEGATIVE_ELIGIBILITY_RULE = {
    "official_source_required": True,
    "week_specific_evidence_required": True,
    "stable_or_post_event_required": True,
    "direction_must_be_defined": True,
    "window_must_be_within_se8_se26": True,
    "absence_of_news_is_not_negative": True,
}

In [930]:
FOURTH_NEGATIVE_PANEL = [
    {
        "region": "BA",
        "year": 2023,
        "start_week": 25,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "SESAB: maior ocorrência de SRAG por influenza entre SE10-SE24; "
            "após esse período houve redução significativa dos casos."
        ),
    },
    {
        "region": "PB",
        "year": 2023,
        "start_week": 23,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE_MODERATE",
        "notes": (
            "SES-PB: Influenza B predominou entre SE10-SE20 e a série "
            "semanal de vírus respiratórios declina após o pico em torno "
            "de SE18-SE20 até SE27."
        ),
    },
]

In [931]:
fourth_negative_registry = pd.DataFrame(
    FOURTH_NEGATIVE_PANEL
)

fourth_negative_registry[
    "algorithm_version"
] = "V4_FROZEN"

fourth_negative_registry[
    "prediction_checked"
] = False

fourth_negative_registry.to_csv(
    "../data/processed/"
    "shift_v4_fourth_negative_panel_preregistered.csv",
    index=False,
)

display(fourth_negative_registry)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,algorithm_version,prediction_checked
0,BA,2023,25,26,UP,0,STRONG_NEGATIVE,SESAB: maior ocorrência de SRAG por influenza ...,V4_FROZEN,False
1,PB,2023,23,26,UP,0,STRONG_NEGATIVE_MODERATE,SES-PB: Influenza B predominou entre SE10-SE20...,V4_FROZEN,False


In [932]:
fourth_negative_predictions = pd.concat(
    [
        run_frozen_shift_case("BA", 2023),
        run_frozen_shift_case("PB", 2023),
    ],
    ignore_index=True,
)

display(
    fourth_negative_predictions[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,BA,2023,UP,11,0.780457,0.039541,4,False
1,BA,2023,DOWN,18,51.787665,-0.120429,10,False
2,PB,2023,DOWN,12,0.174797,-0.051037,2,False
3,PB,2023,UP,14,5.234770,0.171828,4,False
4,PB,2023,DOWN,19,76.710537,-0.240570,8,False


In [933]:
fourth_negative_eval_rows = []

for item in FOURTH_NEGATIVE_PANEL:
    predictions = (
        fourth_negative_predictions.loc[
            fourth_negative_predictions[
                "region"
            ].eq(item["region"])
            &
            fourth_negative_predictions[
                "year"
            ].eq(item["year"])
            &
            fourth_negative_predictions[
                "direction"
            ].eq(item["direction_to_test"])
            &
            fourth_negative_predictions[
                "representative_change_week"
            ].between(
                item["start_week"],
                item["end_week"],
            )
        ]
    )

    false_positive = not predictions.empty

    fourth_negative_eval_rows.append(
        {
            **item,
            "predictions_in_window":
                len(predictions),
            "outcome":
                "FP"
                if false_positive
                else "TN",
        }
    )

fourth_negative_eval = pd.DataFrame(
    fourth_negative_eval_rows
)

display(fourth_negative_eval)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,predictions_in_window,outcome
0,BA,2023,25,26,UP,0,STRONG_NEGATIVE,SESAB: maior ocorrência de SRAG por influenza ...,0,TN
1,PB,2023,23,26,UP,0,STRONG_NEGATIVE_MODERATE,SES-PB: Influenza B predominou entre SE10-SE20...,0,TN


In [934]:
new_tn = (
    fourth_negative_eval[
        "outcome"
    ].eq("TN")
).sum()

new_fp = (
    fourth_negative_eval[
        "outcome"
    ].eq("FP")
).sum()

tn_total = 8 + new_tn
fp_total = 0 + new_fp

print("TN total:", tn_total)
print("FP total:", fp_total)

print(
    "Especificidade:",
    f"{tn_total/(tn_total+fp_total):.1%}"
)

TN total: 10
FP total: 0
Especificidade: 100.0%


In [935]:
FIFTH_NEGATIVE_AUDIT = [
    ("MA", 2024),
    ("TO", 2024),
    ("PA", 2024),
    ("RO", 2024),
    ("PI", 2024),
    ("AC", 2024),
]

In [936]:
fifth_negative_audit = pd.DataFrame(
    FIFTH_NEGATIVE_AUDIT,
    columns=["region", "year"],
)

fifth_negative_audit[
    "algorithm_version"
] = "V4_FROZEN"

fifth_negative_audit[
    "eligible_negative_window"
] = pd.NA

fifth_negative_audit[
    "negative_start_week"
] = pd.NA

fifth_negative_audit[
    "negative_end_week"
] = pd.NA

fifth_negative_audit[
    "direction_to_test"
] = pd.NA

fifth_negative_audit[
    "source_status"
] = "NOT_REVIEWED"

fifth_negative_audit[
    "prediction_checked"
] = False

fifth_negative_audit.to_csv(
    "../data/processed/"
    "shift_v4_fifth_negative_audit_preregistered.csv",
    index=False,
)

display(fifth_negative_audit)

,region,year,algorithm_version,eligible_negative_window,negative_start_week,negative_end_week,direction_to_test,source_status,prediction_checked
0,MA,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
1,TO,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
2,PA,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
3,RO,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
4,PI,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False
5,AC,2024,V4_FROZEN,<NA>,<NA>,<NA>,<NA>,NOT_REVIEWED,False


In [937]:
SIXTH_NEGATIVE_PANEL = [
    {
        "region": "AC",
        "year": 2024,
        "start_week": 24,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "SESAC: Influenza A identificada desde SE13; "
            "nas seis semanas finais do período até SE31, "
            "as notificações de SRAG apresentavam decréscimo. "
            "SE24-26 pré-registrada como controle negativo "
            "para novo SHIFT_UP."
        ),
    }
]

In [938]:
sixth_negative_registry = pd.DataFrame(
    SIXTH_NEGATIVE_PANEL
)

sixth_negative_registry[
    "algorithm_version"
] = "V4_FROZEN"

sixth_negative_registry[
    "prediction_checked"
] = False

sixth_negative_registry.to_csv(
    "../data/processed/"
    "shift_v4_sixth_negative_panel_preregistered.csv",
    index=False,
)

display(sixth_negative_registry)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,algorithm_version,prediction_checked
0,AC,2024,24,26,UP,0,STRONG_NEGATIVE,SESAC: Influenza A identificada desde SE13; na...,V4_FROZEN,False


In [939]:
ac_2024_prediction = run_frozen_shift_case(
    "AC",
    2024,
)

display(
    ac_2024_prediction[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,AC,2024,UP,13,47.112779,0.593816,9,False
1,AC,2024,DOWN,20,135.155071,-0.512098,5,False


In [940]:
item = SIXTH_NEGATIVE_PANEL[0]

ac_window_predictions = (
    ac_2024_prediction.loc[
        ac_2024_prediction[
            "direction"
        ].eq(
            item["direction_to_test"]
        )
        &
        ac_2024_prediction[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]
)

ac_outcome = (
    "FP"
    if not ac_window_predictions.empty
    else "TN"
)

print(
    "AC/2024:",
    ac_outcome,
)

AC/2024: TN


In [941]:
checkpoint_11tn = pd.DataFrame(
    [{
        "algorithm_version": "V4_FROZEN",
        "tp": 4,
        "fn": 0,
        "tn": 11,
        "fp": 0,
        "sensitivity": 1.0,
        "specificity": 1.0,
        "false_positive_rate": 0.0,
    }]
)

checkpoint_11tn.to_csv(
    "../data/processed/"
    "shift_v4_checkpoint_11tn.csv",
    index=False,
)

In [942]:
from statsmodels.stats.proportion import proportion_confint

spec_low, spec_high = proportion_confint(
    count=11,
    nobs=11,
    alpha=0.05,
    method="wilson",
)

print(
    f"Especificidade: 100.0% "
    f"(IC95% {spec_low:.1%}–{spec_high:.1%})"
)

Especificidade: 100.0% (IC95% 74.1%–100.0%)


In [943]:
TWELFTH_NEGATIVE_PANEL = [
    {
        "region": "RO",
        "year": 2024,
        "start_week": 22,
        "end_week": 26,
        "direction_to_test": "UP",
        "ground_truth": 0,
        "status": "STRONG_NEGATIVE",
        "notes": (
            "AGEVISA-RO: retrospectiva oficial descreve 2024 "
            "com notificações de SRAG baixas e relativamente estáveis, "
            "com aumento discreto concentrado entre SE15-SE20. "
            "SE22-26 pré-registrada como controle negativo "
            "para novo SHIFT_UP."
        ),
    }
]

In [944]:
twelfth_negative_registry = pd.DataFrame(
    TWELFTH_NEGATIVE_PANEL
)

twelfth_negative_registry[
    "algorithm_version"
] = "V4_FROZEN"

twelfth_negative_registry[
    "prediction_checked"
] = False

twelfth_negative_registry[
    "preregistered_before_prediction"
] = True

twelfth_negative_registry.to_csv(
    "../data/processed/"
    "shift_v4_twelfth_negative_preregistered.csv",
    index=False,
)

display(twelfth_negative_registry)

,region,year,start_week,end_week,direction_to_test,ground_truth,status,notes,algorithm_version,prediction_checked,preregistered_before_prediction
0,RO,2024,22,26,UP,0,STRONG_NEGATIVE,AGEVISA-RO: retrospectiva oficial descreve 202...,V4_FROZEN,False,True


In [945]:
ro_2024_prediction = run_frozen_shift_case(
    "RO",
    2024,
)

display(
    ro_2024_prediction[
        [
            "region",
            "year",
            "direction",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "duration_snapshots",
            "right_censored",
        ]
    ]
)

,region,year,direction,representative_change_week,peak_llr,peak_absolute_change,duration_snapshots,right_censored
0,RO,2024,UP,12,29.743984,0.409572,13,False
1,RO,2024,DOWN,20,18.128890,-0.185000,1,True


In [946]:
item = TWELFTH_NEGATIVE_PANEL[0]

ro_negative_predictions = (
    ro_2024_prediction.loc[
        ro_2024_prediction[
            "direction"
        ].eq(
            item["direction_to_test"]
        )
        &
        ro_2024_prediction[
            "representative_change_week"
        ].between(
            item["start_week"],
            item["end_week"],
        )
    ]
)

ro_outcome = (
    "FP"
    if not ro_negative_predictions.empty
    else "TN"
)

print(
    "RO/2024:",
    ro_outcome,
)

if not ro_negative_predictions.empty:
    print(
        "\nPredição que gerou FP:"
    )

    display(
        ro_negative_predictions[
            [
                "direction",
                "representative_change_week",
                "peak_llr",
                "peak_absolute_change",
                "duration_snapshots",
            ]
        ]
    )

RO/2024: TN


In [947]:
tn_before = 11
fp_before = 0

tn_total = (
    tn_before
    + int(ro_outcome == "TN")
)

fp_total = (
    fp_before
    + int(ro_outcome == "FP")
)

print(
    "=== PLACAR V4 FROZEN ==="
)

print("TP:", 4)
print("FN:", 0)
print("TN:", tn_total)
print("FP:", fp_total)

print(
    "Sensibilidade:",
    f"{4 / 4:.1%}",
)

print(
    "Especificidade:",
    f"{tn_total / (tn_total + fp_total):.1%}",
)

print(
    "FPR:",
    f"{fp_total / (tn_total + fp_total):.1%}",
)

=== PLACAR V4 FROZEN ===
TP: 4
FN: 0
TN: 12
FP: 0
Sensibilidade: 100.0%
Especificidade: 100.0%
FPR: 0.0%


In [948]:
from statsmodels.stats.proportion import (
    proportion_confint,
)

sens_low, sens_high = proportion_confint(
    count=4,
    nobs=4,
    alpha=0.05,
    method="wilson",
)

spec_low, spec_high = proportion_confint(
    count=tn_total,
    nobs=tn_total + fp_total,
    alpha=0.05,
    method="wilson",
)

print(
    "\nSensibilidade:",
    f"100.0% "
    f"(IC95% "
    f"{sens_low:.1%}–{sens_high:.1%})",
)

print(
    "Especificidade:",
    f"{tn_total/(tn_total+fp_total):.1%} "
    f"(IC95% "
    f"{spec_low:.1%}–{spec_high:.1%})",
)


Sensibilidade: 100.0% (IC95% 51.0%–100.0%)
Especificidade: 100.0% (IC95% 75.8%–100.0%)


In [949]:
FINAL_SHIFT_VERSION = "V4_FROZEN_VALIDATED"

FINAL_HOLDOUT_RESULTS = {
    "tp": 4,
    "fn": 0,
    "tn": 12,
    "fp": 0,
    "sensitivity": 1.0,
    "specificity": 1.0,
    "false_positive_rate": 0.0,
    "parameters_changed_after_freeze": False,
}

In [950]:
pd.DataFrame(
    [FINAL_HOLDOUT_RESULTS]
).to_csv(
    "../data/processed/"
    "shift_v4_frozen_validation_final.csv",
    index=False,
)

In [951]:
PSEUDO_PROSPECTIVE_VERSION = "V4_FROZEN"

PSEUDO_STATES = [
    "AC", "AL", "AM", "AP", "BA",
    "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB",
    "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE",
    "SP", "TO",
]

PSEUDO_YEARS = [
    2024,
    2025,
]

PSEUDO_WEEK_START = 8
PSEUDO_WEEK_END = 40

In [954]:
weekly_pr_2025 = build_flu_weekly_for_state_year(
    path=next(
        p for p in historical_files
        if "INFLUD25" in p.name.upper()
    ),
    region="PR",
    year=2025,
)

history_pr_2025 = build_shift_event_history(
    weekly_df=weekly_pr_2025,
    week_start=8,
    week_end=40,
    min_weeks_each_side=3,
)

print(history_pr_2025.columns.tolist())

display(
    history_pr_2025.head(20)
)

['data_available_until', 'detected_change_week', 'detected_split_after', 'left_positivity', 'right_positivity', 'absolute_change', 'relative_change', 'direction', 'llr']


,data_available_until,detected_change_week,detected_split_after,left_positivity,right_positivity,absolute_change,relative_change,direction,llr
0,13,11,10,0.018433,0.074236,0.055803,4.027293,UP,16.682584
1,14,11,10,0.018433,0.078275,0.059842,4.246406,UP,20.629784
2,15,11,10,0.018433,0.082840,0.064407,4.494083,UP,24.993756
3,16,12,11,0.025180,0.113660,0.088480,4.513928,UP,43.538877
4,17,13,12,0.034237,0.148841,0.114605,4.347408,UP,69.849309
5,18,16,15,0.060985,0.251566,0.190581,4.125033,UP,164.599130
6,19,16,15,0.060985,0.305874,0.244889,5.015556,UP,291.613207
7,20,16,15,0.060985,0.342787,0.281802,5.620833,UP,402.926354
8,21,18,17,0.104772,0.416628,0.311855,3.976505,UP,517.893965
9,22,18,17,0.104772,0.430162,0.325389,4.105678,UP,611.518056


In [955]:
def run_pseudo_prospective_case(
    region: str,
    year: int,
) -> pd.DataFrame:

    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}"
        in p.name.upper()
    )

    weekly = build_flu_weekly_for_state_year(
        path=path,
        region=region,
        year=year,
    )

    history = build_shift_event_history(
        weekly_df=weekly,
        week_start=8,
        week_end=40,
        min_weeks_each_side=3,
    )

    if history.empty:
        return pd.DataFrame()

    history = history.copy()

    history["region"] = region
    history["year"] = year

    return history

In [956]:
test_pr = run_pseudo_prospective_case(
    "PR",
    2025,
)

display(
    test_pr[
        [
            "region",
            "year",
            "data_available_until",
            "detected_change_week",
            "direction",
            "absolute_change",
            "relative_change",
            "llr",
        ]
    ].head(30)
)

,region,year,data_available_until,detected_change_week,direction,absolute_change,relative_change,llr
0,PR,2025,13,11,UP,0.055803,4.027293,16.682584
1,PR,2025,14,11,UP,0.059842,4.246406,20.629784
2,PR,2025,15,11,UP,0.064407,4.494083,24.993756
3,PR,2025,16,12,UP,0.088480,4.513928,43.538877
4,PR,2025,17,13,UP,0.114605,4.347408,69.849309
5,PR,2025,18,16,UP,0.190581,4.125033,164.599130
6,PR,2025,19,16,UP,0.244889,5.015556,291.613207
7,PR,2025,20,16,UP,0.281802,5.620833,402.926354
8,PR,2025,21,18,UP,0.311855,3.976505,517.893965
9,PR,2025,22,18,UP,0.325389,4.105678,611.518056


In [957]:
def build_operational_alerts_from_history(
    pseudo_df: pd.DataFrame,
    tolerance_weeks: int = 1,
    required_snapshots: int = 2,
) -> pd.DataFrame:

    alerts = []

    for (region, year), case_df in (
        pseudo_df
        .groupby(["region", "year"])
    ):
        case_df = (
            case_df
            .sort_values("data_available_until")
            .reset_index(drop=True)
        )

        # separa por direção para evitar
        # misturar UP e DOWN no mesmo regime
        for direction in ["UP", "DOWN"]:

            d = (
                case_df.loc[
                    case_df["direction"].eq(direction)
                ]
                .copy()
                .reset_index(drop=True)
            )

            if len(d) < required_snapshots:
                continue

            current_regime_id = 1
            last_alert_change_week = None

            for i in range(
                required_snapshots - 1,
                len(d),
            ):

                window = d.iloc[
                    i - required_snapshots + 1:
                    i + 1
                ]

                change_weeks = pd.to_numeric(
                    window["detected_change_week"],
                    errors="coerce",
                ).dropna()

                if len(change_weeks) < required_snapshots:
                    continue

                stable = (
                    change_weeks.max()
                    -
                    change_weeks.min()
                    <= tolerance_weeks
                )

                if not stable:
                    continue

                estimated_change_week = int(
                    round(
                        change_weeks.median()
                    )
                )

                # evita emitir várias vezes o mesmo regime
                if (
                    last_alert_change_week is not None
                    and abs(
                        estimated_change_week
                        -
                        last_alert_change_week
                    ) <= tolerance_weeks
                ):
                    continue

                alert_row = d.iloc[i]

                alerts.append(
                    {
                        "region": region,
                        "year": year,
                        "regime_id": current_regime_id,
                        "direction": direction,
                        "estimated_change_week":
                            estimated_change_week,
                        "alert_week":
                            int(
                                alert_row[
                                    "data_available_until"
                                ]
                            ),
                        "internal_detection_delay_weeks":
                            int(
                                alert_row[
                                    "data_available_until"
                                ]
                                -
                                estimated_change_week
                            ),
                        "llr_at_alert":
                            float(
                                alert_row["llr"]
                            ),
                        "absolute_change_at_alert":
                            float(
                                alert_row[
                                    "absolute_change"
                                ]
                            ),
                        "relative_change_at_alert":
                            float(
                                alert_row[
                                    "relative_change"
                                ]
                            ),
                    }
                )

                last_alert_change_week = (
                    estimated_change_week
                )

                current_regime_id += 1

    return pd.DataFrame(alerts)

In [958]:
pr_2025_alerts = (
    build_operational_alerts_from_history(
        test_pr
    )
)

display(pr_2025_alerts)

,region,year,regime_id,direction,estimated_change_week,alert_week,internal_detection_delay_weeks,llr_at_alert,absolute_change_at_alert,relative_change_at_alert
0,PR,2025,1,UP,11,14,3,20.629784,0.059842,4.246406
1,PR,2025,2,UP,16,19,3,291.613207,0.244889,5.015556
2,PR,2025,3,UP,18,22,4,611.518056,0.325389,4.105678
3,PR,2025,4,UP,16,28,12,576.663654,0.302820,5.965472
4,PR,2025,1,DOWN,29,35,6,501.972943,-0.237637,0.250425


In [959]:
def collapse_operational_alerts(
    alerts: pd.DataFrame,
    merge_gap_weeks: int = 4,
) -> pd.DataFrame:

    if alerts.empty:
        return alerts.copy()

    collapsed = []

    for (region, year, direction), group in (
        alerts
        .sort_values("alert_week")
        .groupby(
            ["region", "year", "direction"]
        )
    ):

        group = group.reset_index(drop=True)

        cluster = [group.iloc[0]]

        for i in range(1, len(group)):
            current = group.iloc[i]
            previous = cluster[-1]

            same_episode = (
                abs(
                    int(
                        current[
                            "estimated_change_week"
                        ]
                    )
                    -
                    int(
                        previous[
                            "estimated_change_week"
                        ]
                    )
                )
                <= merge_gap_weeks
            )

            if same_episode:
                cluster.append(current)
                continue

            cluster_df = pd.DataFrame(cluster)

            strongest = cluster_df.loc[
                cluster_df[
                    "llr_at_alert"
                ].idxmax()
            ]

            collapsed.append(
                {
                    "region": region,
                    "year": year,
                    "direction": direction,
                    "first_alert_week":
                        int(
                            cluster_df[
                                "alert_week"
                            ].min()
                        ),
                    "representative_change_week":
                        int(
                            round(
                                cluster_df[
                                    "estimated_change_week"
                                ].median()
                            )
                        ),
                    "peak_llr":
                        float(
                            strongest[
                                "llr_at_alert"
                            ]
                        ),
                    "peak_absolute_change":
                        float(
                            strongest[
                                "absolute_change_at_alert"
                            ]
                        ),
                    "alerts_in_regime":
                        len(cluster_df),
                }
            )

            cluster = [current]

        cluster_df = pd.DataFrame(cluster)

        strongest = cluster_df.loc[
            cluster_df[
                "llr_at_alert"
            ].idxmax()
        ]

        collapsed.append(
            {
                "region": region,
                "year": year,
                "direction": direction,
                "first_alert_week":
                    int(
                        cluster_df[
                            "alert_week"
                        ].min()
                    ),
                "representative_change_week":
                    int(
                        round(
                            cluster_df[
                                "estimated_change_week"
                            ].median()
                        )
                    ),
                "peak_llr":
                    float(
                        strongest[
                            "llr_at_alert"
                        ]
                    ),
                "peak_absolute_change":
                    float(
                        strongest[
                            "absolute_change_at_alert"
                        ]
                    ),
                "alerts_in_regime":
                    len(cluster_df),
            }
        )

    return pd.DataFrame(collapsed)

In [960]:
pr_2025_regimes = (
    collapse_operational_alerts(
        pr_2025_alerts
    )
)

display(pr_2025_regimes)

,region,year,direction,first_alert_week,representative_change_week,peak_llr,peak_absolute_change,alerts_in_regime
0,PR,2025,DOWN,35,29,501.972943,-0.237637,1
1,PR,2025,UP,14,11,20.629784,0.059842,1
2,PR,2025,UP,19,16,611.518056,0.325389,3


In [961]:
PILOT_CASES = [
    ("PR", 2025),
    ("RJ", 2025),
    ("RS", 2025),
    ("SC", 2025),
    ("CE", 2025),
]

pilot_results = []

for region, year in PILOT_CASES:
    print(f"Rodando {region}/{year}...")

    result = run_pseudo_prospective_case(
        region,
        year,
    )

    if not result.empty:
        pilot_results.append(result)

pseudo_pilot_raw = pd.concat(
    pilot_results,
    ignore_index=True,
)

pilot_alerts = (
    build_operational_alerts_from_history(
        pseudo_pilot_raw
    )
)

pilot_regimes = (
    collapse_operational_alerts(
        pilot_alerts
    )
)

display(
    pilot_regimes.sort_values(
        [
            "year",
            "first_alert_week",
            "region",
        ]
    )
)

Rodando PR/2025...
Rodando RJ/2025...
Rodando RS/2025...
Rodando SC/2025...
Rodando CE/2025...


,region,year,direction,first_alert_week,representative_change_week,peak_llr,peak_absolute_change,alerts_in_regime
1,CE,2025,UP,14,14,52.036726,0.160993,4
3,PR,2025,UP,14,11,20.629784,0.059842,1
6,RJ,2025,UP,14,11,2.576985,0.032081,1
9,RS,2025,UP,14,14,108.825412,0.245165,2
11,SC,2025,UP,14,13,76.136551,0.189775,2
7,RJ,2025,UP,18,16,52.076399,0.117045,1
4,PR,2025,UP,19,16,611.518056,0.325389,3
5,RJ,2025,DOWN,27,24,121.141657,-0.126226,1
10,SC,2025,DOWN,30,26,326.333822,-0.195229,2
0,CE,2025,DOWN,31,26,58.425022,-0.143790,1


In [962]:
def collapse_operational_alerts_v2(
    alerts: pd.DataFrame,
    max_alert_gap_weeks: int = 6,
) -> pd.DataFrame:

    if alerts.empty:
        return alerts.copy()

    collapsed = []

    for (region, year), case in (
        alerts
        .sort_values("alert_week")
        .groupby(["region", "year"])
    ):

        case = (
            case
            .sort_values("alert_week")
            .reset_index(drop=True)
        )

        clusters = []
        current_cluster = [case.iloc[0]]

        for i in range(1, len(case)):
            current = case.iloc[i]
            previous = current_cluster[-1]

            same_direction = (
                current["direction"]
                ==
                previous["direction"]
            )

            close_in_time = (
                int(current["alert_week"])
                -
                int(previous["alert_week"])
                <= max_alert_gap_weeks
            )

            if same_direction and close_in_time:
                current_cluster.append(current)

            else:
                clusters.append(
                    pd.DataFrame(
                        current_cluster
                    )
                )

                current_cluster = [current]

        clusters.append(
            pd.DataFrame(
                current_cluster
            )
        )

        for regime_id, cluster in enumerate(
            clusters,
            start=1,
        ):
            strongest = cluster.loc[
                cluster[
                    "llr_at_alert"
                ].idxmax()
            ]

            collapsed.append(
                {
                    "region": region,
                    "year": year,
                    "regime_id": regime_id,
                    "direction":
                        cluster[
                            "direction"
                        ].iloc[0],
                    "first_alert_week":
                        int(
                            cluster[
                                "alert_week"
                            ].min()
                        ),
                    "last_alert_week":
                        int(
                            cluster[
                                "alert_week"
                            ].max()
                        ),
                    "representative_change_week":
                        int(
                            round(
                                cluster[
                                    "estimated_change_week"
                                ].median()
                            )
                        ),
                    "first_estimated_change_week":
                        int(
                            cluster[
                                "estimated_change_week"
                            ].min()
                        ),
                    "last_estimated_change_week":
                        int(
                            cluster[
                                "estimated_change_week"
                            ].max()
                        ),
                    "peak_llr":
                        float(
                            strongest[
                                "llr_at_alert"
                            ]
                        ),
                    "peak_absolute_change":
                        float(
                            strongest[
                                "absolute_change_at_alert"
                            ]
                        ),
                    "alerts_in_regime":
                        len(cluster),
                }
            )

    return pd.DataFrame(
        collapsed
    )

In [963]:
pilot_regimes_v2 = (
    collapse_operational_alerts_v2(
        pilot_alerts
    )
)

display(
    pilot_regimes_v2.sort_values(
        [
            "year",
            "first_alert_week",
            "region",
        ]
    )
)

,region,year,regime_id,direction,first_alert_week,last_alert_week,representative_change_week,first_estimated_change_week,last_estimated_change_week,peak_llr,peak_absolute_change,alerts_in_regime
0,CE,2025,1,UP,14,25,14,12,16,52.036726,0.160993,4
2,PR,2025,1,UP,14,28,16,11,18,611.518056,0.325389,4
4,RJ,2025,1,UP,14,18,14,11,16,52.076399,0.117045,2
6,RS,2025,1,UP,14,18,14,12,16,108.825412,0.245165,2
8,SC,2025,1,UP,14,17,13,12,14,76.136551,0.189775,2
5,RJ,2025,2,DOWN,27,27,24,24,24,121.141657,-0.126226,1
9,SC,2025,2,DOWN,30,30,25,25,25,110.158262,-0.129604,1
1,CE,2025,2,DOWN,31,31,26,26,26,58.425022,-0.143790,1
7,RS,2025,2,DOWN,31,31,27,27,27,396.927304,-0.243370,1
3,PR,2025,2,DOWN,35,35,29,29,29,501.972943,-0.237637,1


In [964]:
def build_pseudo_prospective_regimes(
    pseudo_df: pd.DataFrame,
    tolerance_weeks: int = 1,
    required_snapshots: int = 2,
) -> pd.DataFrame:

    results = []

    for (region, year), case in (
        pseudo_df
        .sort_values("data_available_until")
        .groupby(["region", "year"])
    ):
        case = (
            case
            .sort_values("data_available_until")
            .reset_index(drop=True)
            .copy()
        )

        # Novo regime somente quando a direção realmente muda
        case["regime_id"] = (
            case["direction"]
            .ne(case["direction"].shift())
            .cumsum()
        )

        for regime_id, regime in case.groupby(
            "regime_id"
        ):
            regime = (
                regime
                .reset_index(drop=True)
            )

            direction = regime[
                "direction"
            ].iloc[0]

            if pd.isna(direction):
                continue

            first_alert = None

            # Procura o primeiro momento em que
            # a estimativa fica estável.
            for i in range(
                required_snapshots - 1,
                len(regime),
            ):
                window = regime.iloc[
                    i - required_snapshots + 1:
                    i + 1
                ]

                weeks = pd.to_numeric(
                    window[
                        "detected_change_week"
                    ],
                    errors="coerce",
                ).dropna()

                if len(weeks) < required_snapshots:
                    continue

                stable = (
                    weeks.max()
                    -
                    weeks.min()
                    <= tolerance_weeks
                )

                if stable:
                    row = regime.iloc[i]

                    first_alert = {
                        "alert_week":
                            int(
                                row[
                                    "data_available_until"
                                ]
                            ),

                        "change_week_at_alert":
                            int(
                                round(
                                    weeks.median()
                                )
                            ),

                        "llr_at_alert":
                            float(
                                row["llr"]
                            ),

                        "absolute_change_at_alert":
                            float(
                                row[
                                    "absolute_change"
                                ]
                            ),
                    }

                    break

            if first_alert is None:
                continue

            # Descrição retrospectiva do regime.
            # NÃO usar para calcular tempo do alerta.
            representative_change_week = int(
                round(
                    pd.to_numeric(
                        regime[
                            "detected_change_week"
                        ],
                        errors="coerce",
                    ).median()
                )
            )

            peak_row = regime.loc[
                regime["llr"].idxmax()
            ]

            results.append(
                {
                    "region": region,
                    "year": year,
                    "regime_id": int(regime_id),
                    "direction": direction,

                    "first_observed_week":
                        int(
                            regime[
                                "data_available_until"
                            ].min()
                        ),

                    "last_observed_week":
                        int(
                            regime[
                                "data_available_until"
                            ].max()
                        ),

                    "first_alert_week":
                        first_alert[
                            "alert_week"
                        ],

                    "change_week_at_alert":
                        first_alert[
                            "change_week_at_alert"
                        ],

                    "operational_delay_weeks":
                        (
                            first_alert[
                                "alert_week"
                            ]
                            -
                            first_alert[
                                "change_week_at_alert"
                            ]
                        ),

                    "representative_change_week":
                        representative_change_week,

                    "peak_llr":
                        float(
                            peak_row["llr"]
                        ),

                    "peak_absolute_change":
                        float(
                            peak_row[
                                "absolute_change"
                            ]
                        ),

                    "snapshots_in_regime":
                        len(regime),
                }
            )

    return pd.DataFrame(results)

In [965]:
pilot_regimes_v3 = (
    build_pseudo_prospective_regimes(
        pseudo_pilot_raw
    )
)

display(
    pilot_regimes_v3.sort_values(
        [
            "first_alert_week",
            "region",
        ]
    )
)

,region,year,regime_id,direction,first_observed_week,last_observed_week,first_alert_week,change_week_at_alert,operational_delay_weeks,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime
0,CE,2025,1,UP,13,29,14,12,2,14,65.387365,0.158593,17
2,PR,2025,1,UP,13,33,14,11,3,16,688.183542,0.323666,21
4,RJ,2025,1,UP,13,25,14,11,3,16,123.256631,0.159835,13
6,RS,2025,1,UP,13,29,14,12,2,17,332.537921,0.337454,17
8,SC,2025,1,UP,13,28,14,12,2,14,126.963206,0.203946,16
5,RJ,2025,2,DOWN,26,40,27,24,3,24,215.567058,-0.123651,15
9,SC,2025,2,DOWN,29,40,30,25,5,26,326.333822,-0.195229,12
1,CE,2025,2,DOWN,30,40,31,26,5,26,162.137769,-0.173747,11
7,RS,2025,2,DOWN,30,40,31,27,4,27,902.503467,-0.285758,11
3,PR,2025,2,DOWN,34,40,35,29,6,29,628.259663,-0.222683,7


In [966]:
display(
    pilot_regimes_v3[
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
        ]
    ]
)

,region,year,direction,change_week_at_alert,first_alert_week,operational_delay_weeks,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime
0,CE,2025,UP,12,14,2,14,65.387365,0.158593,17
1,CE,2025,DOWN,26,31,5,26,162.137769,-0.173747,11
2,PR,2025,UP,11,14,3,16,688.183542,0.323666,21
3,PR,2025,DOWN,29,35,6,29,628.259663,-0.222683,7
4,RJ,2025,UP,11,14,3,16,123.256631,0.159835,13
5,RJ,2025,DOWN,24,27,3,24,215.567058,-0.123651,15
6,RS,2025,UP,12,14,2,17,332.537921,0.337454,17
7,RS,2025,DOWN,27,31,4,27,902.503467,-0.285758,11
8,SC,2025,UP,12,14,2,14,126.963206,0.203946,16
9,SC,2025,DOWN,25,30,5,26,326.333822,-0.195229,12


In [967]:
pilot_summary = (
    pilot_regimes_v3
    .groupby(
        "direction",
        as_index=False,
    )
    .agg(
        states=(
            "region",
            "nunique",
        ),
        median_change_week=(
            "representative_change_week",
            "median",
        ),
        min_change_week=(
            "representative_change_week",
            "min",
        ),
        max_change_week=(
            "representative_change_week",
            "max",
        ),
        median_alert_week=(
            "first_alert_week",
            "median",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
        max_operational_delay=(
            "operational_delay_weeks",
            "max",
        ),
    )
)

display(pilot_summary)

,direction,states,median_change_week,min_change_week,max_change_week,median_alert_week,median_operational_delay,max_operational_delay
0,DOWN,5,26.0,24,29,31.0,5.0,6
1,UP,5,16.0,14,17,14.0,2.0,3


In [968]:
PSEUDO_PROTOCOL = {
    "algorithm_version": "V4_FROZEN",
    "operational_layer": "PSEUDO_V1_FROZEN",
    "stability_tolerance_weeks": 1,
    "required_stable_snapshots": 2,
    "week_start": 8,
    "week_end": 40,
    "future_information_allowed": False,
    "pilot_states": ["CE", "PR", "RJ", "RS", "SC"],
    "pilot_year": 2025,
    "parameters_changed_after_pilot": False,
}

In [969]:
pd.DataFrame(
    [PSEUDO_PROTOCOL]
).to_json(
    "../data/processed/"
    "shift_v4_pseudo_protocol_frozen.json",
    orient="records",
    indent=2,
)

In [970]:
raw_year_cache = {}

def load_influd_year_once(
    year: int,
) -> pd.DataFrame:

    if year in raw_year_cache:
        return raw_year_cache[year]

    path = next(
        p
        for p in historical_files
        if f"INFLUD{str(year)[-2:]}"
        in p.name.upper()
    )

    print(
        f"Lendo INFLUD{str(year)[-2:]} "
        "uma única vez..."
    )

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        low_memory=False,
    )

    raw_year_cache[year] = df

    return df

In [973]:
def run_pseudo_prospective_case_fast(
    region: str,
    year: int,
) -> pd.DataFrame:

    raw = load_influd_year_once(
        year
    )

    weekly = build_flu_weekly_from_df(
        df=raw,
        region=region,
        year=year,
    )

    history = build_shift_event_history(
        weekly_df=weekly,
        week_start=8,
        week_end=40,
        min_weeks_each_side=3,
    )

    if history.empty:
        return pd.DataFrame()

    history = history.copy()

    history["region"] = region
    history["year"] = year

    return history

In [974]:
BRAZIL_STATES = [
    "AC", "AL", "AM", "AP", "BA",
    "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB",
    "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE",
    "SP", "TO",
]

NATIONAL_YEARS = [
    2024,
    2025,
]

In [975]:
national_results = []
national_errors = []

for year in NATIONAL_YEARS:

    # força leitura única
    load_influd_year_once(year)

    for region in BRAZIL_STATES:

        print(
            f"Rodando {region}/{year}..."
        )

        try:
            result = (
                run_pseudo_prospective_case_fast(
                    region,
                    year,
                )
            )

            if not result.empty:
                national_results.append(
                    result
                )

        except Exception as exc:
            national_errors.append(
                {
                    "region": region,
                    "year": year,
                    "error_type":
                        type(exc).__name__,
                    "error":
                        str(exc),
                }
            )

            print(
                f"✗ {region}/{year}: "
                f"{type(exc).__name__}: {exc}"
            )

Lendo INFLUD24...
Rodando AC/2024...
Rodando AL/2024...
Rodando AM/2024...
Rodando AP/2024...
Rodando BA/2024...
Rodando CE/2024...
Rodando DF/2024...
Rodando ES/2024...
Rodando GO/2024...
Rodando MA/2024...
Rodando MG/2024...
Rodando MS/2024...
Rodando MT/2024...
Rodando PA/2024...
Rodando PB/2024...
Rodando PE/2024...
Rodando PI/2024...
Rodando PR/2024...
Rodando RJ/2024...
Rodando RN/2024...
Rodando RO/2024...
Rodando RR/2024...
Rodando RS/2024...
Rodando SC/2024...
Rodando SE/2024...
Rodando SP/2024...
Rodando TO/2024...
Lendo INFLUD25...
Rodando AC/2025...
Rodando AL/2025...
Rodando AM/2025...
Rodando AP/2025...
Rodando BA/2025...
Rodando CE/2025...
Rodando DF/2025...
Rodando ES/2025...
Rodando GO/2025...
Rodando MA/2025...
Rodando MG/2025...
Rodando MS/2025...
Rodando MT/2025...
Rodando PA/2025...
Rodando PB/2025...
Rodando PE/2025...
Rodando PI/2025...
Rodando PR/2025...
Rodando RJ/2025...
Rodando RN/2025...
Rodando RO/2025...
Rodando RR/2025...
Rodando RS/2025...
Rodando SC/202

In [976]:
pseudo_national_raw = pd.concat(
    national_results,
    ignore_index=True,
)

national_errors_df = pd.DataFrame(
    national_errors
)

print(
    "Snapshots:",
    len(pseudo_national_raw),
)

print(
    "Casos com erro:",
    len(national_errors_df),
)

display(national_errors_df)

Snapshots: 1512
Casos com erro: 0


""


In [977]:
pseudo_national_raw.to_csv(
    "../data/processed/"
    "shift_v4_pseudo_national_raw.csv",
    index=False,
)

In [978]:
pseudo_national_regimes = (
    build_pseudo_prospective_regimes(
        pseudo_national_raw,
        tolerance_weeks=1,
        required_snapshots=2,
    )
)

In [979]:
pseudo_national_regimes.to_csv(
    "../data/processed/"
    "shift_v4_pseudo_national_regimes_frozen.csv",
    index=False,
)

In [980]:
display(
    pseudo_national_regimes[
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
        ]
    ].sort_values(
        [
            "year",
            "first_alert_week",
            "region",
        ]
    )
)

,region,year,direction,change_week_at_alert,first_alert_week,operational_delay_weeks,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime
0,AC,2024,UP,11,14,3,13,47.112779,0.593816,9
3,AL,2024,DOWN,11,14,3,16,78.194010,-0.351495,28
6,AM,2024,DOWN,11,14,3,11,2.960652,-0.124029,6
10,AP,2024,UP,11,14,3,14,16.116769,0.268107,10
15,BA,2024,DOWN,12,14,2,14,516.304737,-0.268545,28
...,...,...,...,...,...,...,...,...,...,...
85,SE,2025,DOWN,31,34,3,31,50.539531,-0.142117,8
14,AP,2025,DOWN,25,35,10,25,55.224368,-0.156387,7
58,PR,2025,DOWN,29,35,6,29,628.259663,-0.222683,7
5,AL,2025,DOWN,30,36,6,30,59.675070,-0.292915,6


In [981]:
national_summary = (
    pseudo_national_regimes
    .groupby(
        [
            "year",
            "direction",
        ],
        as_index=False,
    )
    .agg(
        states=(
            "region",
            "nunique",
        ),
        regimes=(
            "region",
            "size",
        ),
        median_change_at_alert=(
            "change_week_at_alert",
            "median",
        ),
        median_alert_week=(
            "first_alert_week",
            "median",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
        max_operational_delay=(
            "operational_delay_weeks",
            "max",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
    )
)

display(national_summary)

,year,direction,states,regimes,median_change_at_alert,median_alert_week,median_operational_delay,max_operational_delay,median_llr
0,2024,DOWN,18,20,16.0,19.0,3.0,7,88.527020
1,2024,UP,21,22,12.0,14.0,2.0,28,26.236878
2,2025,DOWN,21,23,25.0,29.0,3.0,10,162.137769
3,2025,UP,27,27,12.0,14.0,2.0,3,65.387365


In [982]:
national_up_wave = (
    pseudo_national_regimes.loc[
        pseudo_national_regimes[
            "direction"
        ].eq("UP")
    ]
    .groupby(
        [
            "year",
            "first_alert_week",
        ],
        as_index=False,
    )
    .agg(
        states_alerting=(
            "region",
            "nunique",
        ),
        states=(
            "region",
            lambda x:
                ", ".join(
                    sorted(set(x))
                ),
        ),
    )
)

display(national_up_wave)

,year,first_alert_week,states_alerting,states
0,2024,14,18,"AC, AP, CE, DF, ES, GO, MA, MG, MS, MT, PA, PI..."
1,2024,15,2,"PB, TO"
2,2024,16,1,SC
3,2024,39,1,RR
4,2025,14,19,"AM, BA, CE, DF, ES, GO, MA, MS, PA, PB, PE, PR..."
5,2025,15,4,"AC, AL, MT, RR"
6,2025,17,3,"AP, MG, PI"
7,2025,18,1,SE


In [983]:
EARLIEST_OPERATIONAL_ALERT_WEEK = 14

pseudo_national_regimes[
    "left_censored"
] = (
    pseudo_national_regimes[
        "first_alert_week"
    ].eq(
        EARLIEST_OPERATIONAL_ALERT_WEEK
    )
)

left_censor_summary = (
    pseudo_national_regimes
    .groupby(
        [
            "year",
            "direction",
        ],
        as_index=False,
    )
    .agg(
        regimes=(
            "region",
            "size",
        ),
        left_censored_regimes=(
            "left_censored",
            "sum",
        ),
    )
)

left_censor_summary[
    "left_censored_pct"
] = (
    left_censor_summary[
        "left_censored_regimes"
    ]
    /
    left_censor_summary[
        "regimes"
    ]
)

display(left_censor_summary)

,year,direction,regimes,left_censored_regimes,left_censored_pct
0,2024,DOWN,20,7,0.350000
1,2024,UP,22,18,0.818182
2,2025,DOWN,23,1,0.043478
3,2025,UP,27,19,0.703704


In [984]:
display(
    pseudo_national_regimes.loc[
        pseudo_national_regimes[
            "left_censored"
        ]
    ][
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "peak_llr",
            "peak_absolute_change",
        ]
    ].sort_values(
        [
            "year",
            "direction",
            "region",
        ]
    )
)

,region,year,direction,change_week_at_alert,first_alert_week,operational_delay_weeks,peak_llr,peak_absolute_change
3,AL,2024,DOWN,11,14,3,78.194010,-0.351495
6,AM,2024,DOWN,11,14,3,2.960652,-0.124029
15,BA,2024,DOWN,12,14,2,516.304737,-0.268545
49,PE,2024,DOWN,12,14,2,101.529167,-0.202506
59,RJ,2024,DOWN,12,14,2,142.549902,-0.164729
77,SC,2024,DOWN,11,14,3,0.431509,-0.018467
82,SE,2024,DOWN,12,14,2,101.189777,-0.182393
0,AC,2024,UP,11,14,3,47.112779,0.593816
10,AP,2024,UP,11,14,3,16.116769,0.268107
18,CE,2024,UP,12,14,2,22.197936,0.184787


In [985]:
pseudo_national_regimes[
    "timing_status"
] = np.where(
    pseudo_national_regimes[
        "left_censored"
    ],
    "LEFT_CENSORED",
    "OBSERVED_ONSET",
)

In [986]:
national_uncensored = (
    pseudo_national_regimes.loc[
        ~pseudo_national_regimes[
            "left_censored"
        ]
    ]
    .copy()
)

uncensored_summary = (
    national_uncensored
    .groupby(
        [
            "year",
            "direction",
        ],
        as_index=False,
    )
    .agg(
        states=(
            "region",
            "nunique",
        ),
        regimes=(
            "region",
            "size",
        ),
        median_change_at_alert=(
            "change_week_at_alert",
            "median",
        ),
        median_alert_week=(
            "first_alert_week",
            "median",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
        max_operational_delay=(
            "operational_delay_weeks",
            "max",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
    )
)

display(uncensored_summary)

,year,direction,states,regimes,median_change_at_alert,median_alert_week,median_operational_delay,max_operational_delay,median_llr
0,2024,DOWN,13,13,20.0,24.0,3.0,7,84.524884
1,2024,UP,4,4,12.0,15.5,3.0,28,5.293559
2,2025,DOWN,21,22,25.0,29.5,3.5,10,173.626996
3,2025,UP,8,8,13.0,16.0,3.0,3,39.554221


In [987]:
display(
    pseudo_national_regimes.loc[
        pseudo_national_regimes[
            "operational_delay_weeks"
        ].ge(10)
    ][
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
        ]
    ].sort_values(
        "operational_delay_weeks",
        ascending=False,
    )
)

,region,year,direction,change_week_at_alert,first_alert_week,operational_delay_weeks,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime
72,RR,2024,UP,11,39,28,11,5.165529,0.097192,3
14,AP,2025,DOWN,25,35,10,25,55.224368,-0.156387,7


In [988]:
final_pseudo_summary = (
    pseudo_national_regimes
    .groupby(
        [
            "year",
            "direction",
        ]
    )
    .apply(
        lambda g: pd.Series(
            {
                "regimes": len(g),

                "left_censored_regimes":
                    g["left_censored"].sum(),

                "left_censored_pct":
                    g["left_censored"].mean(),

                "median_delay_uncensored":
                    g.loc[
                        ~g["left_censored"],
                        "operational_delay_weeks",
                    ].median(),

                "max_delay_uncensored":
                    g.loc[
                        ~g["left_censored"],
                        "operational_delay_weeks",
                    ].max(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

display(final_pseudo_summary)

,year,direction,regimes,left_censored_regimes,left_censored_pct,median_delay_uncensored,max_delay_uncensored
0,2024,DOWN,20.0,7.0,0.350000,3.0,7.0
1,2024,UP,22.0,18.0,0.818182,3.0,28.0
2,2025,DOWN,23.0,1.0,0.043478,3.5,10.0
3,2025,UP,27.0,19.0,0.703704,3.0,3.0


In [989]:
def inspect_case_quality(
    region: str,
    year: int,
) -> pd.DataFrame:

    raw = load_influd_year_once(year)

    weekly = build_flu_weekly_from_df(
        df=raw,
        region=region,
        year=year,
    )

    weekly = (
        weekly
        .sort_values("epi_week")
        .reset_index(drop=True)
    )

    all_weeks = pd.DataFrame(
        {
            "epi_week": range(8, 41)
        }
    )

    quality = all_weeks.merge(
        weekly[
            [
                "epi_week",
                "flu_tests",
                "flu_positive",
                "flu_positivity",
            ]
        ],
        on="epi_week",
        how="left",
    )

    quality["region"] = region
    quality["year"] = year

    return quality

In [990]:
rr_2024_quality = inspect_case_quality(
    "RR",
    2024,
)

ap_2025_quality = inspect_case_quality(
    "AP",
    2025,
)

display(rr_2024_quality)
display(ap_2025_quality)

,epi_week,flu_tests,flu_positive,flu_positivity,region,year
0,8,10,0,0.0,RR,2024
1,9,7,0,0.0,RR,2024
2,10,9,0,0.0,RR,2024
3,11,10,1,0.1,RR,2024
4,12,9,0,0.0,RR,2024
5,13,4,0,0.0,RR,2024
6,14,8,1,0.125,RR,2024
7,15,7,1,0.142857,RR,2024
8,16,5,0,0.0,RR,2024
9,17,7,2,0.285714,RR,2024


,epi_week,flu_tests,flu_positive,flu_positivity,region,year
0,8,4,1,0.25,AP,2025
1,9,16,0,0.0,AP,2025
2,10,7,0,0.0,AP,2025
3,11,21,0,0.0,AP,2025
4,12,23,0,0.0,AP,2025
5,13,21,0,0.0,AP,2025
6,14,27,1,0.037037,AP,2025
7,15,28,1,0.035714,AP,2025
8,16,35,1,0.028571,AP,2025
9,17,40,4,0.1,AP,2025


In [991]:
def summarize_quality(
    df: pd.DataFrame,
) -> pd.Series:

    observed = df[
        "flu_tests"
    ].notna()

    return pd.Series(
        {
            "weeks_expected": len(df),

            "weeks_observed":
                observed.sum(),

            "missing_weeks":
                (~observed).sum(),

            "total_tests":
                df["flu_tests"].sum(),

            "median_tests_per_week":
                df["flu_tests"].median(),

            "min_tests_per_week":
                df["flu_tests"].min(),

            "max_tests_per_week":
                df["flu_tests"].max(),

            "weeks_lt_10_tests":
                df["flu_tests"].lt(10).sum(),

            "weeks_lt_20_tests":
                df["flu_tests"].lt(20).sum(),
        }
    )

In [996]:
quality_summary = pd.DataFrame(
    {
        "RR_2024":
            summarize_quality(
                rr_2024_quality
            ),

        "AP_2025":
            summarize_quality(
                ap_2025_quality
            ),
    }
).T

display(quality_summary)

,weeks_expected,weeks_observed,missing_weeks,total_tests,median_tests_per_week,min_tests_per_week,max_tests_per_week,weeks_lt_10_tests,weeks_lt_20_tests
RR_2024,33.0,33.0,0.0,489.0,10.0,4.0,33.0,12.0,23.0
AP_2025,33.0,33.0,0.0,837.0,21.0,4.0,61.0,5.0,13.0


In [993]:
quality_rows = []

for year in [2024, 2025]:

    raw = load_influd_year_once(
        year
    )

    for region in BRAZIL_STATES:

        weekly = build_flu_weekly_from_df(
            df=raw,
            region=region,
            year=year,
        )

        target = weekly.loc[
            weekly["epi_week"].between(
                8,
                40,
            )
        ]

        quality_rows.append(
            {
                "region":
                    region,

                "year":
                    year,

                "weeks_observed":
                    target[
                        "epi_week"
                    ].nunique(),

                "total_flu_tests":
                    target[
                        "flu_tests"
                    ].sum(),

                "median_tests_week":
                    target[
                        "flu_tests"
                    ].median(),

                "weeks_lt_10_tests":
                    target[
                        "flu_tests"
                    ].lt(10).sum(),

                "weeks_lt_20_tests":
                    target[
                        "flu_tests"
                    ].lt(20).sum(),
            }
        )

national_data_quality = pd.DataFrame(
    quality_rows
)

display(
    national_data_quality.sort_values(
        "median_tests_week"
    )
)

,region,year,weeks_observed,total_flu_tests,median_tests_week,weeks_lt_10_tests,weeks_lt_20_tests
16,PI,2024,33,84,2.0,33,33
12,MT,2024,33,138,3.0,29,33
26,TO,2024,33,357,6.0,20,27
43,PI,2025,33,213,6.0,25,33
21,RR,2024,33,489,10.0,12,23
20,RO,2024,33,459,11.0,14,25
1,AL,2024,33,486,12.0,7,25
53,TO,2025,33,463,13.0,12,26
28,AL,2025,33,577,15.0,9,21
47,RO,2025,33,516,15.0,10,25


In [994]:
pseudo_national_with_quality = (
    pseudo_national_regimes
    .merge(
        national_data_quality,
        on=[
            "region",
            "year",
        ],
        how="left",
    )
)

In [995]:
display(
    pseudo_national_with_quality[
        [
            "region",
            "year",
            "direction",
            "operational_delay_weeks",
            "median_tests_week",
            "total_flu_tests",
            "weeks_lt_10_tests",
            "left_censored",
        ]
    ]
    .sort_values(
        "operational_delay_weeks",
        ascending=False,
    )
)

,region,year,direction,operational_delay_weeks,median_tests_week,total_flu_tests,weeks_lt_10_tests,left_censored
72,RR,2024,UP,28,10.0,489,12,False
14,AP,2025,DOWN,10,21.0,837,5,False
88,SP,2025,DOWN,9,387.0,18056,0,False
55,PI,2025,DOWN,7,6.0,213,25,False
67,RO,2024,DOWN,7,11.0,459,14,False
...,...,...,...,...,...,...,...,...
78,SC,2024,UP,2,172.0,5727,0,False
82,SE,2024,DOWN,2,23.0,1109,4,True
80,SC,2025,UP,2,175.0,7308,0,True
84,SE,2025,UP,2,31.0,1103,0,False


In [997]:
def classify_observability(row):

    median_tests = row["median_tests_week"]

    if median_tests < 10:
        return "VERY_LOW"

    if median_tests < 20:
        return "LOW"

    if median_tests < 50:
        return "MODERATE"

    return "HIGH"


pseudo_national_with_quality[
    "observability"
] = (
    pseudo_national_with_quality
    .apply(
        classify_observability,
        axis=1,
    )
)

In [998]:
observability_summary = (
    pseudo_national_with_quality.loc[
        ~pseudo_national_with_quality[
            "left_censored"
        ]
    ]
    .groupby(
        [
            "direction",
            "observability",
        ],
        as_index=False,
    )
    .agg(
        regimes=(
            "region",
            "size",
        ),
        median_delay=(
            "operational_delay_weeks",
            "median",
        ),
        mean_delay=(
            "operational_delay_weeks",
            "mean",
        ),
        max_delay=(
            "operational_delay_weeks",
            "max",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
    )
)

display(observability_summary)

,direction,observability,regimes,median_delay,mean_delay,max_delay,median_llr
0,DOWN,HIGH,11,4.0,4.272727,9,326.333822
1,DOWN,LOW,9,3.0,4.222222,7,48.915675
2,DOWN,MODERATE,12,3.0,3.333333,10,129.492927
3,DOWN,VERY_LOW,3,5.0,5.333333,7,19.984103
4,UP,HIGH,2,2.5,2.500000,3,321.872952
5,UP,LOW,3,3.0,11.333333,28,11.402316
6,UP,MODERATE,5,3.0,2.800000,3,31.151622
7,UP,VERY_LOW,2,3.0,3.000000,3,29.586567


In [999]:
uncensored_quality = (
    pseudo_national_with_quality.loc[
        ~pseudo_national_with_quality[
            "left_censored"
        ]
    ]
    .copy()
)

display(
    uncensored_quality[
        [
            "operational_delay_weeks",
            "median_tests_week",
            "total_flu_tests",
            "weeks_lt_10_tests",
        ]
    ].corr(
        method="spearman"
    )
)

,operational_delay_weeks,median_tests_week,total_flu_tests,weeks_lt_10_tests
operational_delay_weeks,1.000000,-0.189172,-0.165782,0.312349
median_tests_week,-0.189172,1.000000,0.982143,-0.916412
total_flu_tests,-0.165782,0.982143,1.000000,-0.915027
weeks_lt_10_tests,0.312349,-0.916412,-0.915027,1.000000


In [1000]:
quality_sensitivity = []

for min_median_tests in [
    0,
    10,
    20,
    50,
]:

    subset = (
        pseudo_national_with_quality.loc[
            (
                ~pseudo_national_with_quality[
                    "left_censored"
                ]
            )
            &
            (
                pseudo_national_with_quality[
                    "median_tests_week"
                ]
                >= min_median_tests
            )
        ]
    )

    quality_sensitivity.append(
        {
            "min_median_tests_week":
                min_median_tests,

            "regimes":
                len(subset),

            "states":
                subset[
                    "region"
                ].nunique(),

            "median_delay":
                subset[
                    "operational_delay_weeks"
                ].median(),

            "mean_delay":
                subset[
                    "operational_delay_weeks"
                ].mean(),

            "max_delay":
                subset[
                    "operational_delay_weeks"
                ].max(),

            "within_3_weeks":
                (
                    subset[
                        "operational_delay_weeks"
                    ]
                    <= 3
                ).mean(),

            "within_4_weeks":
                (
                    subset[
                        "operational_delay_weeks"
                    ]
                    <= 4
                ).mean(),
        }
    )

quality_sensitivity = pd.DataFrame(
    quality_sensitivity
)

display(quality_sensitivity)

,min_median_tests_week,regimes,states,median_delay,mean_delay,max_delay,within_3_weeks,within_4_weeks
0,0,47,25,3.0,4.255319,28,0.659574,0.765957
1,10,42,24,3.0,4.238095,28,0.690476,0.785714
2,20,30,20,3.0,3.533333,10,0.733333,0.833333
3,50,13,10,3.0,4.000000,9,0.538462,0.692308


In [1001]:
quality_direction = []

for direction in ["UP", "DOWN"]:
    for min_tests in [0, 10, 20, 50]:

        subset = (
            pseudo_national_with_quality.loc[
                (
                    ~pseudo_national_with_quality[
                        "left_censored"
                    ]
                )
                &
                pseudo_national_with_quality[
                    "direction"
                ].eq(direction)
                &
                (
                    pseudo_national_with_quality[
                        "median_tests_week"
                    ]
                    >= min_tests
                )
            ]
        )

        if subset.empty:
            continue

        quality_direction.append(
            {
                "direction": direction,
                "min_median_tests_week":
                    min_tests,

                "regimes":
                    len(subset),

                "states":
                    subset[
                        "region"
                    ].nunique(),

                "median_delay":
                    subset[
                        "operational_delay_weeks"
                    ].median(),

                "mean_delay":
                    subset[
                        "operational_delay_weeks"
                    ].mean(),

                "max_delay":
                    subset[
                        "operational_delay_weeks"
                    ].max(),

                "within_3_weeks":
                    subset[
                        "operational_delay_weeks"
                    ].le(3).mean(),

                "within_4_weeks":
                    subset[
                        "operational_delay_weeks"
                    ].le(4).mean(),
            }
        )

quality_direction = pd.DataFrame(
    quality_direction
)

display(quality_direction)

,direction,min_median_tests_week,regimes,states,median_delay,mean_delay,max_delay,within_3_weeks,within_4_weeks
0,UP,0,12,11,3.0,4.916667,28,0.916667,0.916667
1,UP,10,10,9,3.0,5.300000,28,0.900000,0.900000
2,UP,20,7,7,3.0,2.714286,3,1.000000,1.000000
3,UP,50,2,2,2.5,2.500000,3,1.000000,1.000000
4,DOWN,0,35,24,3.0,4.028571,10,0.571429,0.714286
5,DOWN,10,32,23,3.0,3.906250,10,0.625000,0.750000
6,DOWN,20,23,17,3.0,3.782609,10,0.652174,0.782609
7,DOWN,50,11,9,4.0,4.272727,9,0.454545,0.636364


In [1002]:
for direction in ["UP", "DOWN"]:

    subset = (
        pseudo_national_with_quality.loc[
            (
                ~pseudo_national_with_quality[
                    "left_censored"
                ]
            )
            &
            pseudo_national_with_quality[
                "direction"
            ].eq(direction)
        ]
    )

    print(
        "\n",
        "=" * 60,
        direction,
    )

    display(
        subset[
            [
                "operational_delay_weeks",
                "median_tests_week",
                "total_flu_tests",
                "weeks_lt_10_tests",
            ]
        ].corr(
            method="spearman"
        )
    )


 ============================================================ UP


,operational_delay_weeks,median_tests_week,total_flu_tests,weeks_lt_10_tests
operational_delay_weeks,1.000000,-0.568204,-0.515576,0.623424
median_tests_week,-0.568204,1.000000,0.947374,-0.957471
total_flu_tests,-0.515576,0.947374,1.000000,-0.939982
weeks_lt_10_tests,0.623424,-0.957471,-0.939982,1.000000



 ============================================================ DOWN


,operational_delay_weeks,median_tests_week,total_flu_tests,weeks_lt_10_tests
operational_delay_weeks,1.000000,-0.129645,-0.114085,0.299987
median_tests_week,-0.129645,1.000000,0.987109,-0.880835
total_flu_tests,-0.114085,0.987109,1.000000,-0.882970
weeks_lt_10_tests,0.299987,-0.880835,-0.882970,1.000000


In [1003]:
delay_distribution = (
    pseudo_national_with_quality.loc[
        ~pseudo_national_with_quality[
            "left_censored"
        ]
    ]
    .groupby(
        [
            "year",
            "direction",
        ],
        as_index=False,
    )
    .agg(
        regimes=(
            "region",
            "size",
        ),

        median_delay=(
            "operational_delay_weeks",
            "median",
        ),

        q75_delay=(
            "operational_delay_weeks",
            lambda x: x.quantile(0.75),
        ),

        q90_delay=(
            "operational_delay_weeks",
            lambda x: x.quantile(0.90),
        ),

        max_delay=(
            "operational_delay_weeks",
            "max",
        ),
    )
)

display(delay_distribution)

,year,direction,regimes,median_delay,q75_delay,q90_delay,max_delay
0,2024,DOWN,13,3.0,4.00,5.8,7
1,2024,UP,4,3.0,9.25,20.5,28
2,2025,DOWN,22,3.5,5.00,6.9,10
3,2025,UP,8,3.0,3.00,3.0,3


In [1004]:
VALIDATION_SAMPLE_PLAN = {
    "2024_UP": 5,
    "2024_DOWN": 5,
    "2025_UP": 5,
    "2025_DOWN": 5,
}

In [1005]:
validation_candidates = (
    pseudo_national_regimes.loc[
        ~pseudo_national_regimes[
            "left_censored"
        ]
    ]
    .copy()
)

validation_candidates[
    "stratum"
] = (
    validation_candidates[
        "year"
    ].astype(str)
    + "_"
    + validation_candidates[
        "direction"
    ]
)

validation_sample = (
    validation_candidates
    .groupby(
        "stratum",
        group_keys=False,
    )
    .apply(
        lambda g: g.sample(
            n=min(5, len(g)),
            random_state=20261007,
        )
    )
    .reset_index(drop=True)
)

In [1006]:
validation_sample[
    "external_status"
] = "NOT_CHECKED_YET"

validation_sample[
    "external_reference_week"
] = pd.NA

validation_sample[
    "external_direction"
] = pd.NA

validation_sample[
    "external_confirmed"
] = pd.NA

validation_sample.to_csv(
    "../data/processed/"
    "shift_v4_national_external_validation_sample_frozen.csv",
    index=False,
)

In [1007]:
external_validation_candidates = (
    pseudo_national_with_quality.loc[
        ~pseudo_national_with_quality[
            "left_censored"
        ]
    ]
    .copy()
)

external_validation_candidates[
    "stratum"
] = (
    external_validation_candidates[
        "year"
    ].astype(str)
    + "_"
    + external_validation_candidates[
        "direction"
    ]
)

print(
    external_validation_candidates[
        "stratum"
    ].value_counts()
)

stratum
2025_DOWN    22
2024_DOWN    13
2025_UP       8
2024_UP       4
Name: count, dtype: int64


In [1008]:
validation_pool = (
    external_validation_candidates
    .sort_values(
        [
            "stratum",
            "region",
            "year",
            "first_alert_week",
        ]
    )
    .drop_duplicates(
        subset=[
            "stratum",
            "region",
            "year",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

display(
    validation_pool[
        "stratum"
    ].value_counts()
)

stratum
2025_DOWN    21
2024_DOWN    13
2025_UP       8
2024_UP       4
Name: count, dtype: int64

In [1009]:
VALIDATION_RANDOM_SEED = 20261007
EVENTS_PER_STRATUM = 4

validation_sample_parts = []

for stratum, group in (
    validation_pool
    .groupby("stratum")
):

    if len(group) < EVENTS_PER_STRATUM:
        raise ValueError(
            f"{stratum} possui apenas "
            f"{len(group)} casos elegíveis."
        )

    sampled = group.sample(
        n=EVENTS_PER_STRATUM,
        random_state=VALIDATION_RANDOM_SEED,
        replace=False,
    )

    validation_sample_parts.append(
        sampled
    )

validation_sample = pd.concat(
    validation_sample_parts,
    ignore_index=True,
)

In [1010]:
print(
    validation_sample[
        "stratum"
    ].value_counts()
)

print(
    "\nTotal:",
    len(validation_sample),
)

stratum
2024_DOWN    4
2024_UP      4
2025_DOWN    4
2025_UP      4
Name: count, dtype: int64

Total: 16


In [1011]:
validation_sample_frozen = (
    validation_sample[
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "median_tests_week",
            "observability",
            "stratum",
        ]
    ]
    .copy()
)

validation_sample_frozen[
    "algorithm_version"
] = "V4_FROZEN"

validation_sample_frozen[
    "operational_version"
] = "PSEUDO_V1_FROZEN"

validation_sample_frozen[
    "external_checked"
] = False

validation_sample_frozen[
    "external_direction"
] = pd.NA

validation_sample_frozen[
    "external_reference_week"
] = pd.NA

validation_sample_frozen[
    "external_status"
] = "NOT_CHECKED_YET"

validation_sample_frozen[
    "external_source"
] = pd.NA

validation_sample_frozen[
    "external_notes"
] = pd.NA

In [1012]:
VALIDATION_PATH = (
    "../data/processed/"
    "shift_v4_national_external_validation_"
    "sample_frozen.csv"
)

validation_sample_frozen.to_csv(
    VALIDATION_PATH,
    index=False,
)

print(
    "Amostra congelada:",
    VALIDATION_PATH,
)

Amostra congelada: ../data/processed/shift_v4_national_external_validation_sample_frozen.csv


In [1013]:
validation_protocol = pd.DataFrame(
    [
        {
            "algorithm_version":
                "V4_FROZEN",

            "operational_version":
                "PSEUDO_V1_FROZEN",

            "sampling_seed":
                VALIDATION_RANDOM_SEED,

            "events_per_stratum":
                EVENTS_PER_STRATUM,

            "total_events":
                len(
                    validation_sample_frozen
                ),

            "exclude_left_censored":
                True,

            "max_one_event_per_region_year_stratum":
                True,

            "parameters_changed_after_sampling":
                False,
        }
    ]
)

validation_protocol.to_json(
    "../data/processed/"
    "shift_v4_national_external_validation_"
    "protocol_frozen.json",
    orient="records",
    indent=2,
)

In [1014]:
EXTERNAL_VALIDATION_RULES = {
    "direction_match_required": True,

    "temporal_match_primary_weeks": 3,

    "temporal_match_secondary_weeks": 4,

    "official_source_preferred": True,

    "week_specific_evidence_preferred": True,

    "absence_of_source_is_not_negative": True,

    "ambiguous_case_status": "UNRESOLVED",

    "no_algorithm_changes_allowed": True,
}

In [1018]:
display(
    validation_sample_frozen[
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "observability",
            "stratum",
            "external_status",
        ]
    ].sort_values(
        [
            "stratum",
            "region",
        ]
    )
)

,region,year,direction,change_week_at_alert,first_alert_week,observability,stratum,external_status
2,CE,2024,DOWN,16,18,HIGH,2024_DOWN,NOT_CHECKED_YET
0,MT,2024,DOWN,30,34,VERY_LOW,2024_DOWN,NOT_CHECKED_YET
3,RN,2024,DOWN,16,19,MODERATE,2024_DOWN,NOT_CHECKED_YET
1,RR,2024,DOWN,29,35,LOW,2024_DOWN,NOT_CHECKED_YET
7,PB,2024,UP,12,15,MODERATE,2024_UP,NOT_CHECKED_YET
5,RR,2024,UP,11,39,LOW,2024_UP,NOT_CHECKED_YET
4,SC,2024,UP,14,16,HIGH,2024_UP,NOT_CHECKED_YET
6,TO,2024,UP,12,15,VERY_LOW,2024_UP,NOT_CHECKED_YET
9,MS,2025,DOWN,21,24,HIGH,2025_DOWN,NOT_CHECKED_YET
8,RJ,2025,DOWN,24,27,HIGH,2025_DOWN,NOT_CHECKED_YET


In [1019]:
updates = [
    {
        "region": "CE",
        "year": 2024,
        "direction": "DOWN",
        "external_direction": "DOWN",
        "external_reference_week": 18,
        "external_status": "CONFIRMED",
        "external_checked": True,
        "external_notes": (
            "SESA-CE: Influenza A atingiu pico na SE15; "
            "a partir da SE18 houve diminuição progressiva "
            "da positividade."
        ),
    },
    {
        "region": "AL",
        "year": 2025,
        "direction": "UP",
        "external_direction": "UP",
        "external_reference_week": 11,
        "external_status": "CONFIRMED",
        "external_checked": True,
        "external_notes": (
            "SESAU-AL: aumento/intensificação da Influenza "
            "observado a partir da SE11, com destaque "
            "para SE16-17."
        ),
    },
    {
        "region": "MG",
        "year": 2025,
        "direction": "UP",
        "external_direction": "UP",
        "external_reference_week": pd.NA,
        "external_status": "UNRESOLVED",
        "external_checked": True,
        "external_notes": (
            "SES-MG confirma crescimento da circulação "
            "de Influenza A, mas a fonte consultada ainda "
            "não delimita com precisão suficiente a semana "
            "de início para avaliar erro temporal."
        ),
    },
]

In [1020]:
for update in updates:
    mask = (
        validation_sample_frozen["region"].eq(
            update["region"]
        )
        &
        validation_sample_frozen["year"].eq(
            update["year"]
        )
        &
        validation_sample_frozen["direction"].eq(
            update["direction"]
        )
    )

    for column, value in update.items():
        if column in [
            "region",
            "year",
            "direction",
        ]:
            continue

        validation_sample_frozen.loc[
            mask,
            column,
        ] = value

In [1023]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1025]:
display(
    validation_sample_frozen
    .sort_values(
        [
            "stratum",
            "region",
        ]
    )[
        [
            "region",
            "year",
            "direction",
            "change_week_at_alert",
            "external_reference_week",
            "temporal_error_weeks",
            "external_status",
        ]
    ]
)

,region,year,direction,change_week_at_alert,external_reference_week,temporal_error_weeks,external_status
2,CE,2024,DOWN,16,18,-2,CONFIRMED
0,MT,2024,DOWN,30,<NA>,<NA>,NOT_CHECKED_YET
3,RN,2024,DOWN,16,<NA>,<NA>,NOT_CHECKED_YET
1,RR,2024,DOWN,29,<NA>,<NA>,NOT_CHECKED_YET
7,PB,2024,UP,12,<NA>,<NA>,NOT_CHECKED_YET
5,RR,2024,UP,11,<NA>,<NA>,NOT_CHECKED_YET
4,SC,2024,UP,14,<NA>,<NA>,NOT_CHECKED_YET
6,TO,2024,UP,12,<NA>,<NA>,NOT_CHECKED_YET
9,MS,2025,DOWN,21,<NA>,<NA>,NOT_CHECKED_YET
8,RJ,2025,DOWN,24,<NA>,<NA>,NOT_CHECKED_YET


In [1026]:
rr_updates = [
    {
        "region": "RR",
        "year": 2024,
        "direction": "UP",
        "external_status": "UNRESOLVED",
        "external_checked": True,
        "external_notes": (
            "Relatório epidemiológico anual de Roraima indica "
            "circulação de influenza principalmente entre abril e junho, "
            "com pico em maio, mas sem resolução semanal suficiente "
            "para validar SE11."
        ),
    },
    {
        "region": "RR",
        "year": 2024,
        "direction": "DOWN",
        "external_status": "UNRESOLVED",
        "external_checked": True,
        "external_notes": (
            "Relatório anual confirma redução após o período de maior "
            "circulação, mas não fornece semana externa suficientemente "
            "precisa para validar SE29."
        ),
    },
    {
        "region": "PB",
        "year": 2024,
        "direction": "UP",
        "external_status": "UNRESOLVED",
        "external_checked": True,
        "external_notes": (
            "Boletim oficial até SE12 confirma circulação de Influenza A, "
            "mas não demonstra mudança ascendente temporalmente precisa "
            "o bastante para validar o onset em SE12."
        ),
    },
]

In [1027]:
for update in rr_updates:
    mask = (
        validation_sample_frozen["region"].eq(update["region"])
        &
        validation_sample_frozen["year"].eq(update["year"])
        &
        validation_sample_frozen["direction"].eq(update["direction"])
    )

    for column, value in update.items():
        if column in ["region", "year", "direction"]:
            continue

        validation_sample_frozen.loc[
            mask,
            column,
        ] = value

In [1028]:
update_sc = {
    "region": "SC",
    "year": 2024,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": 15,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "DIVE-SC: casos de SRAG por Influenza aumentaram "
        "de 37 na SE14 para 56 na SE15, 66 na SE16 "
        "e 78 na SE18, indicando ascensão a partir de SE15."
    ),
}

mask = (
    validation_sample_frozen["region"].eq(
        update_sc["region"]
    )
    &
    validation_sample_frozen["year"].eq(
        update_sc["year"]
    )
    &
    validation_sample_frozen["direction"].eq(
        update_sc["direction"]
    )
)

for column, value in update_sc.items():
    if column in [
        "region",
        "year",
        "direction",
    ]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1029]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1030]:
update_ms = {
    "region": "MS",
    "year": 2025,
    "direction": "DOWN",
    "external_direction": "DOWN",
    "external_reference_week": 23,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "SES-MS documentou crescimento progressivo de SRAG a partir "
        "da SE10. InfoGripe, na atualização referente à SE23, já "
        "identificava interrupção do crescimento/início de queda em MS, "
        "associada à redução de SRAG por Influenza A."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("MS")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("DOWN")
)

for column, value in update_ms.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1031]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1032]:
update_rj = {
    "region": "RJ",
    "year": 2025,
    "direction": "DOWN",
    "external_direction": "DOWN",
    "external_reference_week": 26,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "SES-RJ informou que a Influenza A/H1N1, após crescimento "
        "expressivo desde a segunda quinzena de abril, havia iniciado "
        "trajetória de queda no cenário da SE26. InfoGripe também "
        "classificou o RJ entre os estados com interrupção do crescimento "
        "ou queda de SRAG associada à Influenza A até a SE26."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("RJ")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("DOWN")
)

for column, value in update_rj.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1033]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1034]:
update_rs = {
    "region": "RS",
    "year": 2025,
    "direction": "DOWN",
    "external_direction": "DOWN",
    "external_reference_week": 26,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe: até a SE25-SE26 o Rio Grande do Sul já apresentava "
        "sinal de interrupção do crescimento ou queda dos casos de SRAG "
        "associados à Influenza A."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("RS")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("DOWN")
)

for column, value in update_rs.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1035]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1036]:
update_to = {
    "region": "TO",
    "year": 2025,
    "direction": "DOWN",
    "external_direction": "DOWN",
    "external_reference_week": 23,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe referente à SE23 classificou Tocantins com "
        "tendência de queda/estabilidade, incluindo diminuição "
        "dos casos associados à Influenza A. Na SE24, a tendência "
        "de queda permanecia presente."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("TO")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("DOWN")
)

for column, value in update_to.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1037]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1038]:
update_ac = {
    "region": "AC",
    "year": 2025,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe até SE14 confirma tendência de crescimento de SRAG no Acre. "
        "Posteriormente, o estado aparece com crescimento expressivo associado "
        "a influenza A e VSR, mas a evidência localizada não permite atribuir "
        "com precisão semanal a mudança exclusivamente à influenza A."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("AC")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("UP")
)

for column, value in update_ac.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[mask, column] = value

In [1039]:
update_rr_2025 = {
    "region": "RR",
    "year": 2025,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe até SE14 já mostrava crescimento de SRAG em Roraima. "
        "Até SE20, Roraima seguia em crescimento e havia evidência explícita "
        "de aumento de SRAG associado a influenza A e VSR. A fonte disponível, "
        "porém, não permite atribuir com precisão semanal à influenza A o onset "
        "próximo da SE12."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("RR")
    &
    validation_sample_frozen["year"].eq(2025)
    &
    validation_sample_frozen["direction"].eq("UP")
)

for column, value in update_rr_2025.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[mask, column] = value

In [1040]:
update_rn = {
    "region": "RN",
    "year": 2024,
    "direction": "DOWN",
    "external_direction": pd.NA,
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe referente à SE19 ainda indicava crescimento de SRAG "
        "na tendência de longo prazo no Rio Grande do Norte. "
        "A evidência localizada, porém, não é específica para influenza, "
        "portanto não permite classificar o caso como confirmação "
        "nem como mismatch etiológico."
    ),
}

In [1041]:
update_mt = {
    "region": "MT",
    "year": 2024,
    "direction": "DOWN",
    "external_direction": "DOWN",
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe referente à SE31 não classificava Mato Grosso entre "
        "os estados com crescimento de SRAG de longo prazo, o que é "
        "compatível com estabilização/queda. A fonte localizada não "
        "fornece evidência específica de influenza nem semana de reversão "
        "precisa o suficiente para confirmação temporal."
    ),
}

In [1042]:
for update in [update_rn, update_mt]:
    mask = (
        validation_sample_frozen["region"].eq(update["region"])
        &
        validation_sample_frozen["year"].eq(update["year"])
        &
        validation_sample_frozen["direction"].eq(update["direction"])
    )

    for column, value in update.items():
        if column in ["region", "year", "direction"]:
            continue

        validation_sample_frozen.loc[mask, column] = value

In [1043]:
update_to_2024 = {
    "region": "TO",
    "year": 2024,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "InfoGripe em 2024 mostra Tocantins com sinal de crescimento de SRAG "
        "e Palmas com indício de aumento. A atualização referente à SE12 "
        "também registra crescimento de SRAG associado à Influenza A na "
        "Região Norte. Porém, as fontes localizadas não permitem atribuir "
        "com precisão semanal o início do aumento de influenza especificamente "
        "em Tocantins à SE12."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("TO")
    &
    validation_sample_frozen["year"].eq(2024)
    &
    validation_sample_frozen["direction"].eq("UP")
)

for column, value in update_to_2024.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1044]:
update_pb = {
    "region": "PB",
    "year": 2024,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": 14,
    "external_status": "CONFIRMED",
    "external_checked": True,
    "external_notes": (
        "Boletins oficiais da SES-PB mostram crescimento de Influenza A "
        "ao longo das semanas epidemiológicas. Até a SE14 havia 60 "
        "identificações de Influenza A em SRAG por RT-PCR; até a SE16, "
        "esse total chegou a 130. O boletim da SE16 descreve presença "
        "crescente de Influenza A a partir da SE07."
    ),
}

mask = (
    validation_sample_frozen["region"].eq("PB")
    &
    validation_sample_frozen["year"].eq(2024)
    &
    validation_sample_frozen["direction"].eq("UP")
)

for column, value in update_pb.items():
    if column in ["region", "year", "direction"]:
        continue

    validation_sample_frozen.loc[
        mask,
        column,
    ] = value

In [1045]:
validation_sample_frozen[
    "temporal_error_weeks"
] = (
    validation_sample_frozen[
        "change_week_at_alert"
    ]
    -
    validation_sample_frozen[
        "external_reference_week"
    ]
)

validation_sample_frozen[
    "absolute_temporal_error_weeks"
] = (
    validation_sample_frozen[
        "temporal_error_weeks"
    ].abs()
)

In [1046]:
update_mg = {
    "region": "MG",
    "year": 2025,
    "direction": "UP",
    "external_direction": "UP",
    "external_reference_week": pd.NA,
    "external_status": "UNRESOLVED",
    "external_checked": True,
    "external_notes": (
        "SES-MG confirmou aumento da circulação de vírus respiratórios "
        "em 2025, especialmente Influenza A e VSR. A nota informa "
        "crescimento da Influenza A em crianças, adultos e idosos e "
        "crescimento progressivo da demanda respiratória, mas não "
        "delimita com precisão suficiente a semana de início da "
        "ascensão de Influenza A para validar temporalmente a SE14."
    ),
}

In [1047]:
status_summary = (
    validation_sample_frozen[
        "external_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "external_status"
    )
    .reset_index(
        name="events"
    )
)

display(status_summary)

,external_status,events
0,UNRESOLVED,8
1,CONFIRMED,8


In [1048]:
total_sample = len(
    validation_sample_frozen
)

confirmed = (
    validation_sample_frozen[
        "external_status"
    ]
    .eq("CONFIRMED")
    .sum()
)

unresolved = (
    validation_sample_frozen[
        "external_status"
    ]
    .eq("UNRESOLVED")
    .sum()
)

failed = (
    validation_sample_frozen[
        "external_status"
    ]
    .isin(
        [
            "DIRECTION_MISMATCH",
            "TEMPORAL_MISMATCH",
            "NOT_CONFIRMED",
        ]
    )
    .sum()
)

evaluable = confirmed + failed

print(
    "Amostra congelada:",
    total_sample
)

print(
    "Avaliáveis:",
    evaluable
)

print(
    "Confirmados:",
    confirmed
)

print(
    "Falhas externas:",
    failed
)

print(
    "Não resolvidos:",
    unresolved
)

print(
    "Resolubilidade externa:",
    f"{evaluable / total_sample:.1%}"
)

if evaluable:
    print(
        "Confirmação entre avaliáveis:",
        f"{confirmed / evaluable:.1%}"
    )

Amostra congelada: 16
Avaliáveis: 8
Confirmados: 8
Falhas externas: 0
Não resolvidos: 8
Resolubilidade externa: 50.0%
Confirmação entre avaliáveis: 100.0%


In [1049]:
confirmed_validation = (
    validation_sample_frozen.loc[
        validation_sample_frozen[
            "external_status"
        ].eq("CONFIRMED")
    ]
    .copy()
)

confirmed_validation[
    "temporal_error_weeks"
] = (
    confirmed_validation[
        "change_week_at_alert"
    ]
    -
    confirmed_validation[
        "external_reference_week"
    ]
)

confirmed_validation[
    "absolute_temporal_error_weeks"
] = (
    confirmed_validation[
        "temporal_error_weeks"
    ]
    .abs()
)

In [1050]:
print(
    "N confirmados:",
    len(
        confirmed_validation
    )
)

print(
    "Erro absoluto mediano:",
    confirmed_validation[
        "absolute_temporal_error_weeks"
    ].median()
)

print(
    "Erro absoluto médio:",
    confirmed_validation[
        "absolute_temporal_error_weeks"
    ].mean()
)

print(
    "Erro absoluto máximo:",
    confirmed_validation[
        "absolute_temporal_error_weeks"
    ].max()
)

for tolerance in [
    1,
    2,
    3,
    4,
]:

    count = (
        confirmed_validation[
            "absolute_temporal_error_weeks"
        ]
        .le(tolerance)
        .sum()
    )

    print(
        f"±{tolerance} semana(s): "
        f"{count}/"
        f"{len(confirmed_validation)} "
        f"({count / len(confirmed_validation):.1%})"
    )

N confirmados: 8
Erro absoluto mediano: 2.0
Erro absoluto médio: 1.625
Erro absoluto máximo: 2
±1 semana(s): 3/8 (37.5%)
±2 semana(s): 8/8 (100.0%)
±3 semana(s): 8/8 (100.0%)
±4 semana(s): 8/8 (100.0%)


In [1051]:
direction_validation = (
    confirmed_validation
    .groupby(
        "direction",
        as_index=False,
    )
    .agg(
        events=(
            "region",
            "size",
        ),
        median_abs_error=(
            "absolute_temporal_error_weeks",
            "median",
        ),
        mean_abs_error=(
            "absolute_temporal_error_weeks",
            "mean",
        ),
        max_abs_error=(
            "absolute_temporal_error_weeks",
            "max",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
    )
)

display(direction_validation)

,direction,events,median_abs_error,mean_abs_error,max_abs_error,median_operational_delay
0,DOWN,5,2.0,1.8,2,3.0
1,UP,3,1.0,1.333333,2,3.0


In [1052]:
stratum_validation = (
    validation_sample_frozen
    .groupby(
        "stratum",
        as_index=False,
    )
    .agg(
        sampled=(
            "region",
            "size",
        ),

        confirmed=(
            "external_status",
            lambda x:
                x.eq(
                    "CONFIRMED"
                ).sum(),
        ),

        unresolved=(
            "external_status",
            lambda x:
                x.eq(
                    "UNRESOLVED"
                ).sum(),
        ),

        failures=(
            "external_status",
            lambda x:
                x.isin(
                    [
                        "DIRECTION_MISMATCH",
                        "TEMPORAL_MISMATCH",
                        "NOT_CONFIRMED",
                    ]
                ).sum(),
        ),
    )
)

display(stratum_validation)

,stratum,sampled,confirmed,unresolved,failures
0,2024_DOWN,4,1,3,0
1,2024_UP,4,2,2,0
2,2025_DOWN,4,4,0,0
3,2025_UP,4,1,3,0


In [1053]:
from math import sqrt


def wilson_interval(
    successes: int,
    total: int,
    z: float = 1.96,
):
    if total == 0:
        return (
            float("nan"),
            float("nan"),
        )

    p = successes / total

    denominator = (
        1
        +
        z**2 / total
    )

    center = (
        p
        +
        z**2
        /
        (2 * total)
    ) / denominator

    margin = (
        z
        *
        sqrt(
            (
                p * (1 - p)
                / total
            )
            +
            (
                z**2
                /
                (
                    4
                    * total**2
                )
            )
        )
        /
        denominator
    )

    return (
        center - margin,
        center + margin,
    )

In [1054]:
lower, upper = (
    wilson_interval(
        confirmed,
        evaluable,
    )
)

print(
    "Taxa de confirmação:",
    f"{confirmed/evaluable:.1%}"
)

print(
    "IC95%:",
    f"{lower:.1%} – {upper:.1%}"
)

Taxa de confirmação: 100.0%
IC95%: 67.6% – 100.0%


In [1055]:
from math import sqrt
import pandas as pd


FAILURE_STATUSES = [
    "DIRECTION_MISMATCH",
    "TEMPORAL_MISMATCH",
    "NOT_CONFIRMED",
]


def wilson_interval(
    successes: int,
    total: int,
    z: float = 1.96,
):
    if total == 0:
        return float("nan"), float("nan")

    p = successes / total

    denominator = 1 + (z**2 / total)

    center = (
        p + z**2 / (2 * total)
    ) / denominator

    margin = (
        z
        * sqrt(
            p * (1 - p) / total
            + z**2 / (4 * total**2)
        )
        / denominator
    )

    return (
        center - margin,
        center + margin,
    )


# ============================================================
# 1. STATUS GERAL DA AMOSTRA
# ============================================================

status_summary = (
    validation_sample_frozen[
        "external_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("external_status")
    .reset_index(name="events")
)

display(status_summary)


total_sample = len(validation_sample_frozen)

confirmed = (
    validation_sample_frozen[
        "external_status"
    ]
    .eq("CONFIRMED")
    .sum()
)

unresolved = (
    validation_sample_frozen[
        "external_status"
    ]
    .eq("UNRESOLVED")
    .sum()
)

failed = (
    validation_sample_frozen[
        "external_status"
    ]
    .isin(FAILURE_STATUSES)
    .sum()
)

evaluable = confirmed + failed


print("=" * 60)
print("VALIDAÇÃO EXTERNA V4")
print("=" * 60)

print("Amostra congelada:", total_sample)
print("Avaliáveis:", evaluable)
print("Confirmados:", confirmed)
print("Falhas externas:", failed)
print("Não resolvidos:", unresolved)

print(
    "Resolubilidade externa:",
    f"{evaluable / total_sample:.1%}"
    if total_sample
    else "N/A"
)

print(
    "Confirmação entre avaliáveis:",
    f"{confirmed / evaluable:.1%}"
    if evaluable
    else "N/A"
)


# ============================================================
# 2. ERRO TEMPORAL DOS CONFIRMADOS
# ============================================================

confirmed_validation = (
    validation_sample_frozen.loc[
        validation_sample_frozen[
            "external_status"
        ].eq("CONFIRMED")
    ]
    .copy()
)

confirmed_validation[
    "temporal_error_weeks"
] = (
    confirmed_validation[
        "change_week_at_alert"
    ]
    -
    confirmed_validation[
        "external_reference_week"
    ]
)

confirmed_validation[
    "absolute_temporal_error_weeks"
] = (
    confirmed_validation[
        "temporal_error_weeks"
    ].abs()
)


print("\n" + "=" * 60)
print("PRECISÃO TEMPORAL")
print("=" * 60)

print(
    "N confirmados:",
    len(confirmed_validation)
)

if not confirmed_validation.empty:

    print(
        "Erro absoluto mediano:",
        confirmed_validation[
            "absolute_temporal_error_weeks"
        ].median()
    )

    print(
        "Erro absoluto médio:",
        confirmed_validation[
            "absolute_temporal_error_weeks"
        ].mean()
    )

    print(
        "Erro absoluto máximo:",
        confirmed_validation[
            "absolute_temporal_error_weeks"
        ].max()
    )

    for tolerance in [1, 2, 3, 4]:

        count = (
            confirmed_validation[
                "absolute_temporal_error_weeks"
            ]
            .le(tolerance)
            .sum()
        )

        print(
            f"±{tolerance} semana(s): "
            f"{count}/{len(confirmed_validation)} "
            f"({count / len(confirmed_validation):.1%})"
        )


# ============================================================
# 3. RESULTADO POR DIREÇÃO
# ============================================================

direction_validation = (
    confirmed_validation
    .groupby(
        "direction",
        as_index=False,
    )
    .agg(
        events=(
            "region",
            "size",
        ),
        median_abs_error=(
            "absolute_temporal_error_weeks",
            "median",
        ),
        mean_abs_error=(
            "absolute_temporal_error_weeks",
            "mean",
        ),
        max_abs_error=(
            "absolute_temporal_error_weeks",
            "max",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
    )
)

print("\n" + "=" * 60)
print("RESULTADO POR DIREÇÃO")
print("=" * 60)

display(direction_validation)


# ============================================================
# 4. RESULTADO POR ESTRATO
# ============================================================

stratum_validation = (
    validation_sample_frozen
    .groupby(
        "stratum",
        as_index=False,
    )
    .agg(
        sampled=(
            "region",
            "size",
        ),
        confirmed=(
            "external_status",
            lambda x:
                x.eq("CONFIRMED").sum(),
        ),
        unresolved=(
            "external_status",
            lambda x:
                x.eq("UNRESOLVED").sum(),
        ),
        failures=(
            "external_status",
            lambda x:
                x.isin(
                    FAILURE_STATUSES
                ).sum(),
        ),
    )
)

print("\n" + "=" * 60)
print("RESULTADO POR ESTRATO")
print("=" * 60)

display(stratum_validation)


# ============================================================
# IC95% DA TAXA DE CONFIRMAÇÃO
# ============================================================

lower, upper = wilson_interval(
    confirmed,
    evaluable,
)

print("\n" + "=" * 60)
print("INCERTEZA")
print("=" * 60)

if evaluable:
    print(
        "Taxa de confirmação entre avaliáveis:",
        f"{confirmed / evaluable:.1%}"
    )

    print(
        "IC95% de Wilson:",
        f"{lower:.1%} – {upper:.1%}"
    )
else:
    print("Ainda não há casos avaliáveis.")

,external_status,events
0,UNRESOLVED,8
1,CONFIRMED,8


VALIDAÇÃO EXTERNA V4
Amostra congelada: 16
Avaliáveis: 8
Confirmados: 8
Falhas externas: 0
Não resolvidos: 8
Resolubilidade externa: 50.0%
Confirmação entre avaliáveis: 100.0%

PRECISÃO TEMPORAL
N confirmados: 8
Erro absoluto mediano: 2.0
Erro absoluto médio: 1.625
Erro absoluto máximo: 2
±1 semana(s): 3/8 (37.5%)
±2 semana(s): 8/8 (100.0%)
±3 semana(s): 8/8 (100.0%)
±4 semana(s): 8/8 (100.0%)

RESULTADO POR DIREÇÃO


,direction,events,median_abs_error,mean_abs_error,max_abs_error,median_operational_delay
0,DOWN,5,2.0,1.8,2,3.0
1,UP,3,1.0,1.333333,2,3.0



RESULTADO POR ESTRATO


,stratum,sampled,confirmed,unresolved,failures
0,2024_DOWN,4,1,3,0
1,2024_UP,4,2,2,0
2,2025_DOWN,4,4,0,0
3,2025_UP,4,1,3,0



INCERTEZA
Taxa de confirmação entre avaliáveis: 100.0%
IC95% de Wilson: 67.6% – 100.0%


In [1056]:
V4_FINAL_PATH = (
    "../data/processed/"
    "shift_v4_external_validation_FINAL_FROZEN.csv"
)

validation_sample_frozen.to_csv(
    V4_FINAL_PATH,
    index=False,
)

print("V4 final congelada em:")
print(V4_FINAL_PATH)

V4 final congelada em:
../data/processed/shift_v4_external_validation_FINAL_FROZEN.csv


In [1057]:
v4_final_summary = pd.DataFrame(
    [
        {
            "algorithm_version": "V4_FROZEN",
            "operational_version": "PSEUDO_V1_FROZEN",

            "sampled_events": total_sample,
            "evaluable_events": evaluable,
            "confirmed_events": confirmed,
            "failed_events": failed,
            "unresolved_events": unresolved,

            "external_resolvability":
                (
                    evaluable / total_sample
                    if total_sample
                    else pd.NA
                ),

            "confirmation_rate_evaluable":
                (
                    confirmed / evaluable
                    if evaluable
                    else pd.NA
                ),

            "confirmation_ci95_lower":
                lower,

            "confirmation_ci95_upper":
                upper,

            "median_absolute_temporal_error":
                (
                    confirmed_validation[
                        "absolute_temporal_error_weeks"
                    ].median()
                    if not confirmed_validation.empty
                    else pd.NA
                ),

            "max_absolute_temporal_error":
                (
                    confirmed_validation[
                        "absolute_temporal_error_weeks"
                    ].max()
                    if not confirmed_validation.empty
                    else pd.NA
                ),

            "v4_retuned_after_validation":
                False,
        }
    ]
)

v4_final_summary.to_json(
    "../data/processed/"
    "shift_v4_external_validation_summary_FINAL_FROZEN.json",
    orient="records",
    indent=2,
)

SHIFT V4 — Resultado final da validação

A versão V4 foi congelada antes das etapas de holdout e validação externa.

Em uma amostra externa pré-definida de 16 eventos pseudo-prospectivos,
8 apresentaram evidência externa com resolução temporal suficiente para
avaliação e 8 permaneceram não resolvidos.

Todos os 8 eventos avaliáveis foram confirmados quanto à direção da mudança,
sem casos classificados como direction mismatch, temporal mismatch ou
not confirmed.

Entre os eventos confirmados, o erro temporal absoluto mediano foi de
2 semanas, com erro máximo de 2 semanas. Todos os eventos confirmados
ficaram dentro de ±2 semanas da referência externa.

A taxa de confirmação entre eventos avaliáveis foi de 100%
(IC95% de Wilson: 67,6%–100,0%).

Esses resultados sustentam generalização temporal e geográfica do detector,
mas ainda não constituem evidência suficiente para uso operacional em saúde
pública sem validação prospectiva independente.

In [1058]:
V4_FREEZE_MANIFEST = {
    "algorithm_version": "V4_FROZEN",
    "operational_version": "PSEUDO_V1_FROZEN",

    "development_complete": True,
    "holdout_complete": True,
    "pseudo_prospective_complete": True,
    "external_validation_complete": True,

    "external_validation_sample_n": 16,
    "external_evaluable_n": 8,
    "external_confirmed_n": 8,
    "external_unresolved_n": 8,
    "external_failures_n": 0,

    "confirmation_rate_evaluable": 1.0,
    "confirmation_ci95_lower": 0.676,
    "confirmation_ci95_upper": 1.0,

    "median_absolute_temporal_error_weeks": 2.0,
    "max_absolute_temporal_error_weeks": 2.0,

    "v4_parameters_locked": True,
    "v4_future_retuning_allowed": False,

    "next_phase": "V5_PROSPECTIVE_PROTOCOL",
}

In [1059]:
pd.DataFrame(
    [V4_FREEZE_MANIFEST]
).to_json(
    "../data/processed/"
    "shift_v4_freeze_manifest.json",
    orient="records",
    indent=2,
)

In [1060]:
V5_PROTOCOL = {
    "base_algorithm": "V4_FROZEN",

    "prospective_only": True,
    "future_information_allowed": False,

    "weekly_execution": True,

    "outputs": [
        "NO_SIGNAL",
        "SHIFT_UP",
        "SHIFT_DOWN",
    ],

    "observability_layer": True,

    "observability_variables": [
        "weekly_test_count",
        "rolling_test_count",
        "weeks_with_low_testing",
    ],

    "evaluation_primary_metrics": [
        "sensitivity",
        "specificity",
        "false_positive_rate",
        "median_detection_delay",
        "temporal_error",
    ],

    "no_retuning_on_v4_validation_set": True,
}

In [ ]:
V5_PROTOCOL = {
    "version": "V5_PROTOCOL_1",
    "base_algorithm": "V4_FROZEN",
    "operational_layer": "PSEUDO_V1_FROZEN",

    "evaluation_year": 2026,

    "prospective_replay": True,
    "future_information_allowed": False,

    "week_start": 8,
    "week_end": 38,

    "required_stable_snapshots": 2,
    "stability_tolerance_weeks": 1,

    "outputs": [
        "NO_SIGNAL",
        "SHIFT_UP",
        "SHIFT_DOWN",
    ],

    "observability_layer": True,

    "v4_retuning_allowed": False,
    "v4_validation_data_used_for_threshold_tuning": False,
}

In [1061]:
pd.DataFrame(
    [V5_PROTOCOL]
).to_json(
    "../data/processed/"
    "shift_v5_protocol_1_frozen.json",
    orient="records",
    indent=2,
)

In [1074]:
def build_v5_weekly_log(
    pseudo_history: pd.DataFrame,
) -> pd.DataFrame:

    if pseudo_history.empty:
        return pd.DataFrame()

    history = (
        pseudo_history
        .sort_values(
            [
                "region",
                "year",
                "data_available_until",
            ]
        )
        .copy()
    )

    history.loc[
        history["direction"].eq("UP"),
        "signal",
    ] = "SHIFT_UP"

    history.loc[
        history["direction"].eq("DOWN"),
        "signal",
    ] = "SHIFT_DOWN"

    return history[
        [
            "region",
            "year",
            "data_available_until",
            "detected_change_week",
            "direction",
            "signal",
            "absolute_change",
            "relative_change",
            "llr",
        ]
    ]

In [1063]:
V5_STATES = [
    "AC", "AL", "AM", "AP", "BA",
    "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB",
    "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE",
    "SP", "TO",
]

In [1071]:
weekly["flu_positivity"] = (
    weekly["flu_positive"]
    .div(
        weekly["flu_tests"]
        .replace(0, np.nan)
    )
)

In [1081]:
to_history = build_shift_event_history(
    weekly_df=to_2026,
    week_start=8,
    week_end=40,
    min_weeks_each_side=3,
)

display(to_history.tail(15))

,data_available_until,detected_change_week,detected_split_after,left_positivity,right_positivity,absolute_change,relative_change,direction,llr
13,26,16,15,0.084746,0.245614,0.160868,2.898246,UP,11.309043
14,27,16,15,0.084746,0.239316,0.154570,2.823932,UP,10.708876
15,28,16,15,0.084746,0.237288,0.152542,2.800000,UP,10.516417
16,29,16,15,0.084746,0.241667,0.156921,2.851667,UP,11.099758
17,30,11,10,0.000000,0.179724,0.179724,NaN,UP,10.023495
18,31,11,10,0.000000,0.178082,0.178082,NaN,UP,9.928584
19,32,11,10,0.000000,0.177778,0.177778,NaN,UP,9.926154
20,33,11,10,0.000000,0.176991,0.176991,NaN,UP,9.880562
21,34,11,10,0.000000,0.175439,0.175439,NaN,UP,9.790627
22,35,11,10,0.000000,0.173913,0.173913,NaN,UP,9.702322


In [1082]:
to_result = run_pseudo_prospective_case_fast(
    region="TO",
    year=2026,
)

display(to_result)

,data_available_until,detected_change_week,detected_split_after,left_positivity,right_positivity,absolute_change,relative_change,direction,llr,region,year
0,13,11,10,0.000000,0.147541,0.147541,NaN,UP,7.041254,TO,2026
1,14,11,10,0.000000,0.133333,0.133333,NaN,UP,6.532484,TO,2026
2,15,11,10,0.000000,0.109890,0.109890,NaN,UP,5.465559,TO,2026
3,16,11,10,0.000000,0.133333,0.133333,NaN,UP,6.822893,TO,2026
4,17,11,10,0.000000,0.147541,0.147541,NaN,UP,7.726473,TO,2026
5,18,11,10,0.000000,0.141791,0.141791,NaN,UP,7.470644,TO,2026
6,19,16,15,0.084746,0.283582,0.198836,3.346269,UP,12.279066,TO,2026
7,20,16,15,0.084746,0.258824,0.174078,3.054118,UP,11.206199,TO,2026
8,21,16,15,0.084746,0.247191,0.162445,2.916854,UP,10.231272,TO,2026
9,22,16,15,0.084746,0.239583,0.154838,2.827083,UP,9.807745,TO,2026


In [1083]:
if not to_result.empty:
    v5_results.append(to_result)

v5_2026_raw = pd.concat(
    v5_results,
    ignore_index=True,
)

In [1084]:
v5_2026_regimes = (
    build_pseudo_prospective_regimes(
        v5_2026_raw,
        tolerance_weeks=1,
        required_snapshots=2,
    )
)

In [1085]:
v5_2026_raw.to_csv(
    "../data/processed/shift_v5_2026_replay_raw.csv",
    index=False,
)

v5_2026_regimes.to_csv(
    "../data/processed/shift_v5_2026_regimes_frozen.csv",
    index=False,
)

In [1086]:
display(
    v5_2026_regimes[
        [
            "region",
            "direction",
            "first_observed_week",
            "first_alert_week",
            "change_week_at_alert",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
        ]
    ].sort_values(
        [
            "first_alert_week",
            "region",
        ]
    )
)

,region,direction,first_observed_week,first_alert_week,change_week_at_alert,operational_delay_weeks,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime
0,AC,UP,13,14,11,3,13,21.279454,0.132908,28
1,AL,UP,13,14,12,2,12,12.959599,0.290278,5
3,AM,DOWN,13,14,11,3,11,9.648021,-0.037075,28
4,AP,DOWN,13,14,12,2,13,77.911285,-0.333748,28
6,CE,DOWN,13,14,12,2,15,1296.959825,-0.603118,28
7,DF,UP,13,14,11,3,15,50.034169,0.094048,28
8,ES,UP,13,14,11,3,11,3.668694,0.092793,10
10,GO,UP,13,14,11,3,11,67.311311,0.155099,28
11,MA,DOWN,13,14,12,2,15,182.586227,-0.423763,28
12,MG,UP,13,14,11,3,11,59.684927,0.191171,14


In [1087]:
v5_2026_summary = (
    v5_2026_regimes
    .groupby(
        "direction",
        as_index=False,
    )
    .agg(
        states=(
            "region",
            "nunique",
        ),
        regimes=(
            "region",
            "size",
        ),
        median_change_week=(
            "change_week_at_alert",
            "median",
        ),
        median_alert_week=(
            "first_alert_week",
            "median",
        ),
        median_operational_delay=(
            "operational_delay_weeks",
            "median",
        ),
        max_operational_delay=(
            "operational_delay_weeks",
            "max",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
    )
)

display(v5_2026_summary)

,direction,states,regimes,median_change_week,median_alert_week,median_operational_delay,max_operational_delay,median_llr
0,DOWN,22,25,14.0,17.0,3.0,11,172.421787
1,UP,17,17,11.0,14.0,3.0,3,23.897957


In [1088]:
v5_2026_regimes[
    "data_quality_status"
] = "OK"

v5_2026_regimes.loc[
    v5_2026_regimes[
        "region"
    ].eq("TO"),
    "data_quality_status",
] = "RIGHT_CENSORED_LOW_OBSERVABILITY"

In [1089]:
def classify_signal_strength(row):
    llr = row["peak_llr"]
    change = abs(row["peak_absolute_change"])
    snapshots = row["snapshots_in_regime"]

    if llr < 5 or change < 0.05 or snapshots < 3:
        return "WEAK"

    if llr < 20 or change < 0.10 or snapshots < 5:
        return "MODERATE"

    return "STRONG"


v5_2026_regimes["signal_strength"] = (
    v5_2026_regimes.apply(
        classify_signal_strength,
        axis=1,
    )
)

In [1090]:
strength_summary = (
    v5_2026_regimes
    .groupby(
        [
            "direction",
            "signal_strength",
        ],
        as_index=False,
    )
    .agg(
        regimes=("region","size"),
        states=("region", "nunique"),
        median_delay=(
            "operational_delay_weeks",
            "median",
        ),
        median_llr=(
            "peak_llr",
            "median",
        ),
        median_change=(
            "peak_absolute_change",
            lambda x: x.abs().median(),
        ),
    )
)

display(strength_summary)

,direction,signal_strength,regimes,states,median_delay,median_llr,median_change
0,DOWN,MODERATE,1,1,2.0,19.288796,0.113170
1,DOWN,STRONG,19,19,3.0,182.586227,0.327513
2,DOWN,WEAK,5,4,3.0,6.085024,0.060928
3,UP,MODERATE,5,5,3.0,13.094538,0.179405
4,UP,STRONG,9,9,3.0,59.684927,0.191171
5,UP,WEAK,3,3,3.0,3.668694,0.092793


In [1091]:
multi_regime_states = (
    v5_2026_regimes
    .groupby("region")
    .filter(
        lambda g: len(g) > 1
    )
    .sort_values(
        [
            "region",
            "first_alert_week",
        ]
    )
)

display(
    multi_regime_states[
        [
            "region",
            "direction",
            "first_alert_week",
            "change_week_at_alert",
            "operational_delay_weeks",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
        ]
    ]
)

,region,direction,first_alert_week,change_week_at_alert,operational_delay_weeks,peak_llr,peak_absolute_change,snapshots_in_regime
1,AL,UP,14,12,2,12.959599,0.290278,5
2,AL,DOWN,19,16,3,100.476484,-0.358883,23
8,ES,UP,14,11,3,3.668694,0.092793,10
9,ES,DOWN,24,22,2,19.288796,-0.113170,18
12,MG,UP,14,11,3,59.684927,0.191171,14
13,MG,DOWN,28,20,8,30.555778,-0.060928,2
14,MG,DOWN,31,20,11,84.576548,-0.148598,11
15,MS,UP,14,11,3,48.432311,0.200494,16
16,MS,DOWN,31,28,3,189.173368,-0.207002,12
17,MT,UP,14,11,3,23.897957,0.236772,3


In [1092]:
v5_2026_validation_frame = (
    v5_2026_regimes[
        [
            "region",
            "direction",
            "first_alert_week",
            "change_week_at_alert",
            "operational_delay_weeks",
            "representative_change_week",
            "peak_llr",
            "peak_absolute_change",
            "snapshots_in_regime",
            "signal_strength",
        ]
    ]
    .copy()
)

v5_2026_validation_frame[
    "external_status"
] = "NOT_CHECKED_YET"

v5_2026_validation_frame[
    "external_reference_week"
] = pd.NA

v5_2026_validation_frame[
    "external_direction"
] = pd.NA

v5_2026_validation_frame.to_csv(
    "../data/processed/"
    "shift_v5_2026_validation_frame_frozen.csv",
    index=False,
)

In [1094]:
event_source = (
   v5_2026_regimes
   .sort_values(
        [
            "region",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
        ]
   )
   .copy() 
)

v5_2026_events = (
    event_source
    .groupby(
        [
            "region",
            "direction",
            "change_week_at_alert",
        ],
        as_index=False,
    )
    .agg(
        first_observed_week=(
            "first_observed_week",
            "min",
        ),

        first_alert_week=(
            "first_alert_week",
            "min",
        ),

        representative_change_week=(
            "representative_change_week",
            "median",
        ),

        peak_llr=(
            "peak_llr",
            "max",
        ),

        peak_absolute_change=(
            "peak_absolute_change",
            lambda x: x.loc[x.abs().idxmax()],
        ),

        snapshots_in_regime=(
            "snapshots_in_regime",
            "sum",
        ),

        source_regimes=(
            "region",
            "size",
        ),
    )
)

v5_2026_events[
    "operational_delay_weeks"
] = (
    v5_2026_events[
        "first_alert_week"
    ]
    -
    v5_2026_events[
        "change_week_at_alert"
    ]
)

In [1095]:
print(
    "Regimes originais:",
    len(v5_2026_regimes),
)

print(
    "Eventos consolidados:",
    len(v5_2026_events),
)

display(
    v5_2026_events.loc[
        v5_2026_events[
            "source_regimes"
        ].gt(1)
    ]
)

Regimes originais: 42
Eventos consolidados: 40


,region,direction,change_week_at_alert,first_observed_week,first_alert_week,representative_change_week,peak_llr,peak_absolute_change,snapshots_in_regime,source_regimes,operational_delay_weeks
12,MG,DOWN,20,27,28,25.5,84.576548,-0.148598,13,2,8
28,RO,DOWN,30,32,33,30.0,6.085024,-0.177370,8,2,3


In [1096]:
def classify_signal_strength(row):
    llr = row["peak_llr"]
    change = abs(
        row["peak_absolute_change"]
    )
    snapshots = row[
        "snapshots_in_regime"
    ]

    if (
        llr < 5
        or change < 0.05
        or snapshots < 3
    ):
        return "WEAK"

    if (
        llr < 20
        or change < 0.10
        or snapshots < 5
    ):
        return "MODERATE"

    return "STRONG"


v5_2026_events[
    "signal_strength"
] = (
    v5_2026_events.apply(
        classify_signal_strength,
        axis=1,
    )
)

In [1097]:
v5_2026_external = (
    v5_2026_events.copy()
)

v5_2026_external[
    "algorithm_version"
] = "V4_FROZEN"

v5_2026_external[
    "protocol_version"
] = "V5_PROTOCOL_1"

v5_2026_external[
    "external_checked"
] = False

v5_2026_external[
    "external_status"
] = "NOT_CHECKED_YET"

v5_2026_external[
    "external_direction"
] = pd.NA

v5_2026_external[
    "external_reference_week"
] = pd.NA

v5_2026_external[
    "external_source"
] = pd.NA

v5_2026_external[
    "external_notes"
] = pd.NA

In [1098]:
V5_2026_EXTERNAL_PATH = (
    "../data/processed/"
    "shift_v5_2026_external_validation_"
    "FROZEN.csv"
)

v5_2026_external.to_csv(
    V5_2026_EXTERNAL_PATH,
    index=False,
)

print(
    "Validação 2026 congelada:",
    V5_2026_EXTERNAL_PATH,
)

Validação 2026 congelada: ../data/processed/shift_v5_2026_external_validation_FROZEN.csv


In [1099]:
V5_EXTERNAL_RULES = {
    "direction_match_required": True,

    "primary_temporal_tolerance_weeks": 2,

    "secondary_temporal_tolerance_weeks": 3,

    "absence_of_public_evidence_is_not_failure":
        True,

    "ambiguous_external_evidence":
        "UNRESOLVED",

    "explicit_opposite_direction":
        "DIRECTION_MISMATCH",

    "correct_direction_wrong_timing":
        "TEMPORAL_MISMATCH",

    "explicit_absence_of_predicted_event":
        "NOT_CONFIRMED",

    "algorithm_changes_after_ground_truth":
        False,
}

In [1100]:
pd.DataFrame(
    [V5_EXTERNAL_RULES]
).to_json(
    "../data/processed/"
    "shift_v5_2026_external_validation_"
    "rules_FROZEN.json",
    orient="records",
    indent=2,
)

In [1101]:
print(
    "Regimes:",
    len(v5_2026_regimes)
)

print(
    "Eventos:",
    len(v5_2026_events)
)

display(
    v5_2026_events[
        [
            "region",
            "direction",
            "change_week_at_alert",
            "first_alert_week",
            "operational_delay_weeks",
            "peak_llr",
            "peak_absolute_change",
            "signal_strength",
            "source_regimes",
        ]
    ]
    .sort_values(
        [
            "first_alert_week",
            "region",
        ]
    )
)

Regimes: 42
Eventos: 40


,region,direction,change_week_at_alert,first_alert_week,operational_delay_weeks,peak_llr,peak_absolute_change,signal_strength,source_regimes
0,AC,UP,11,14,3,21.279454,0.132908,STRONG,1
2,AL,UP,12,14,2,12.959599,0.290278,MODERATE,1
3,AM,DOWN,11,14,3,9.648021,-0.037075,WEAK,1
4,AP,DOWN,12,14,2,77.911285,-0.333748,STRONG,1
6,CE,DOWN,12,14,2,1296.959825,-0.603118,STRONG,1
7,DF,UP,11,14,3,50.034169,0.094048,MODERATE,1
9,ES,UP,11,14,3,3.668694,0.092793,WEAK,1
10,GO,UP,11,14,3,67.311311,0.155099,STRONG,1
11,MA,DOWN,12,14,2,182.586227,-0.423763,STRONG,1
13,MG,UP,11,14,3,59.684927,0.191171,STRONG,1


In [1102]:
v5_2026_external = (
    v5_2026_events
    .sort_values(
        [
            "region",
            "direction",
            "change_week_at_alert",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

v5_2026_external["event_id"] = [
    f"V5-2026-{i:03d}"
    for i in range(
        1,
        len(v5_2026_external) + 1
    )
]

v5_2026_external["algorithm_version"] = (
    "V4_FROZEN"
)

v5_2026_external["protocol_version"] = (
    "V5_PROTOCOL_1"
)

v5_2026_external["external_checked"] = False
v5_2026_external["external_status"] = (
    "NOT_CHECKED_YET"
)

v5_2026_external["external_direction"] = pd.NA
v5_2026_external["external_reference_week"] = pd.NA
v5_2026_external["external_source"] = pd.NA
v5_2026_external["external_notes"] = pd.NA

v5_2026_external.to_csv(
    "../data/processed/"
    "shift_v5_2026_external_validation_FINAL_FROZEN.csv",
    index=False,
)

In [1103]:
v5_2026_manifest = {
    "protocol": "V5_PROTOCOL_1",
    "algorithm": "V4_FROZEN",

    "year": 2026,

    "raw_regimes": 42,
    "consolidated_events": 40,

    "consolidation_rule":
        (
            "same region + direction + "
            "change_week_at_alert"
        ),

    "signal_strength_used_for_exclusion":
        False,

    "ground_truth_checked_before_freeze":
        False,

    "algorithm_retuning_allowed":
        False,
}

pd.DataFrame(
    [v5_2026_manifest]
).to_json(
    "../data/processed/"
    "shift_v5_2026_validation_manifest_FROZEN.json",
    orient="records",
    indent=2,
)